=== Confirmed paths ===
Current directory : C:\Projects\Infer RozviDrought\RozviDrought\tests
Project root      : C:\Projects\Infer RozviDrought\RozviDrought
Workspace root    : C:\Projects\Infer RozviDrought
Data root         : C:\Projects\Infer RozviDrought\data
Master input path : C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Output directory  : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng
Run ID            : 20260620T092408Z

=== Parquet metadata ===
Rows      : 38,957,625
Columns   : 13
Row groups: 38
Schema    : ['pixel_id', 'row', 'col', 'lon', 'lat', 'scenario', 'yyyymm', 't2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']

=== Required column check ===
Required columns: ['pixel_id', 'row', 'col', 'lon', 'lat', 'scenario', 'yyyymm', 't2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']
Missing columns : []

=== Input preview ===
 pixel_id  row  col       lon        lat   scenario yyyymm        t2m        d2m       pet       sm  ndvi  t

In [2]:
# validate_events_ng.ipynb — Cell 1
# Purpose:
# - Discover the real v2 model artifacts and candidate v2 input datasets.
# - Inspect parquet schemas and manifests without loading large datasets.
# - Confirm whether a reusable v2-ready feature source exists for seed/event tests.
# - Do NOT run any model.
# - Do NOT copy artifacts into this repo.
# - Do NOT treat master_df as sufficient for v2 inference.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json

import pandas as pd


# -----------------------------------------------------------------------------
# 1. Reconfirm current workspace paths from Cell 0
# -----------------------------------------------------------------------------
if "PROJECT_ROOT" not in globals():
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = (
        CURRENT_DIR.parent
        if CURRENT_DIR.name == "tests"
        else CURRENT_DIR
    )

if "WORKSPACE_DIR" not in globals():
    WORKSPACE_DIR = PROJECT_ROOT.parent

if "DATA_ROOT" not in globals():
    DATA_ROOT = WORKSPACE_DIR / "data"

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = DATA_ROOT / "backtests" / "validate_events_ng"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL1_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Existing current-app master input. This is not assumed to be v2-ready.
MASTER_INPUT_PATH = (
    DATA_ROOT
    / "master_inputs"
    / "master_inputs_long_198001_205012.parquet"
)

# Previous drought modelling workspace holding training/release materials.
# This is inspected only. Nothing will be copied or changed.
TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

# Existing RozviDrought package repository.
# This is inspected only. Nothing will be copied or changed.
ROZVIDROUGHT_REPO_ROOT = Path(
    r"C:\Projects\rozvi\models\rozvidrought"
)

ROZVIDROUGHT_ARTIFACT_ROOT = (
    ROZVIDROUGHT_REPO_ROOT
    / "src"
    / "rozvidrought"
    / "artifacts"
)

print("=== Cell 1 purpose ===")
print(
    "Discover and audit v2 model input/artifact sources before "
    "seed or event inference."
)

print("\n=== Confirmed local paths ===")
print("Current inference workspace :", WORKSPACE_DIR)
print("Current data root           :", DATA_ROOT)
print("Current master input        :", MASTER_INPUT_PATH)
print("Training drought root       :", TRAINING_DROUGHT_ROOT)
print("RozviDrought package root   :", ROZVIDROUGHT_REPO_ROOT)
print("Package artifact root       :", ROZVIDROUGHT_ARTIFACT_ROOT)
print("Output directory            :", OUTPUT_DIR)
print("Cell 1 run ID               :", CELL1_RUN_ID)


# -----------------------------------------------------------------------------
# 2. Safe parquet metadata helper
# -----------------------------------------------------------------------------
try:
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "pyarrow is required for v2 parquet schema inspection."
    ) from exc


IDENTITY_COLUMNS = {
    "pixel_id",
    "pixel_key",
    "row",
    "col",
    "lon",
    "lat",
    "yyyymm",
    "scenario",
}

RAINFALL_SPI_TERMS = [
    "rain",
    "precip",
    "chirps",
    "spi",
    "spei",
]

TEMPORAL_TERMS = [
    "lag",
    "roll",
    "rolling",
    "month",
    "season",
    "time_type",
    "run_yyyymm",
]

V2_OUTPUT_TERMS = [
    "fusion_v2",
    "spi3",
    "spi6",
    "spi12",
    "expected_severity",
    "warning",
    "persistent",
]


def find_matching_columns(
    columns: list[str],
    terms: list[str],
) -> list[str]:
    return sorted(
        [
            str(column)
            for column in columns
            if any(
                term in str(column).lower()
                for term in terms
            )
        ]
    )


def parquet_schema_record(
    path: Path,
    source_group: str,
) -> dict[str, Any]:
    record: dict[str, Any] = {
        "source_group": source_group,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": None,
        "read_status": None,
        "row_count": None,
        "column_count": None,
        "columns": None,
        "identity_columns_found": None,
        "rainfall_spi_columns": None,
        "temporal_columns": None,
        "v2_output_columns": None,
        "has_pixel_and_month": False,
        "has_rainfall_or_spi_signal": False,
        "has_temporal_features": False,
        "is_possible_v2_feature_source": False,
        "error": None,
    }

    if not path.exists():
        record["read_status"] = "missing"
        return record

    try:
        record["size_mb"] = round(
            path.stat().st_size / (1024 * 1024),
            3,
        )

        parquet_file = pq.ParquetFile(path)
        columns = list(parquet_file.schema.names)

        identity_found = sorted(
            IDENTITY_COLUMNS.intersection(columns)
        )
        rainfall_spi_columns = find_matching_columns(
            columns,
            RAINFALL_SPI_TERMS,
        )
        temporal_columns = find_matching_columns(
            columns,
            TEMPORAL_TERMS,
        )
        v2_output_columns = find_matching_columns(
            columns,
            V2_OUTPUT_TERMS,
        )

        has_pixel_and_month = (
            ("pixel_id" in columns or "pixel_key" in columns)
            and "yyyymm" in columns
        )

        has_rainfall_or_spi_signal = len(
            rainfall_spi_columns
        ) > 0

        has_temporal_features = len(
            temporal_columns
        ) > 0

        possible_v2_feature_source = (
            has_pixel_and_month
            and has_rainfall_or_spi_signal
            and has_temporal_features
        )

        record.update(
            {
                "read_status": "read_ok",
                "row_count": int(
                    parquet_file.metadata.num_rows
                ),
                "column_count": int(
                    parquet_file.metadata.num_columns
                ),
                "columns": ", ".join(columns),
                "identity_columns_found": ", ".join(
                    identity_found
                ),
                "rainfall_spi_columns": ", ".join(
                    rainfall_spi_columns
                ),
                "temporal_columns": ", ".join(
                    temporal_columns
                ),
                "v2_output_columns": ", ".join(
                    v2_output_columns
                ),
                "has_pixel_and_month": has_pixel_and_month,
                "has_rainfall_or_spi_signal": (
                    has_rainfall_or_spi_signal
                ),
                "has_temporal_features": has_temporal_features,
                "is_possible_v2_feature_source": (
                    possible_v2_feature_source
                ),
            }
        )

    except Exception as exc:
        record["read_status"] = "read_failed"
        record["error"] = repr(exc)

    return record


# -----------------------------------------------------------------------------
# 3. Discover only in narrow, known directories
# -----------------------------------------------------------------------------
CANDIDATE_DATA_DIRECTORIES = {
    "current_infer_master_inputs": (
        DATA_ROOT / "master_inputs"
    ),
    "training_model_inputs": (
        TRAINING_DROUGHT_ROOT
        / "processed"
        / "model_inputs"
    ),
    "training_release_prediction_outputs": (
        TRAINING_DROUGHT_ROOT
        / "processed"
        / "model_outputs"
        / "release_v2_package_handoff"
        / "release_bundle_predictions"
    ),
    "training_fusion_outputs": (
        TRAINING_DROUGHT_ROOT
        / "processed"
        / "model_outputs"
        / "release_v2_package_handoff"
        / "shadow_compare_legacy_vs_fusion_v2"
    ),
}

parquet_records: list[dict[str, Any]] = []

for source_group, directory in CANDIDATE_DATA_DIRECTORIES.items():
    print(f"\nInspecting: {source_group}")
    print("Path:", directory)
    print("Exists:", directory.exists())

    if not directory.exists():
        continue

    parquet_files = sorted(directory.glob("*.parquet"))

    print("Parquet files found:", len(parquet_files))

    for parquet_path in parquet_files:
        parquet_records.append(
            parquet_schema_record(
                path=parquet_path,
                source_group=source_group,
            )
        )

parquet_catalog_df = pd.DataFrame(parquet_records)

if parquet_catalog_df.empty:
    print(
        "\nWARNING: No parquet candidates were found in the "
        "known v2 data directories."
    )
else:
    parquet_catalog_df = parquet_catalog_df.sort_values(
        by=[
            "is_possible_v2_feature_source",
            "has_rainfall_or_spi_signal",
            "has_temporal_features",
            "row_count",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    print("\n=== Candidate parquet catalog ===")

    display_columns = [
        "source_group",
        "path",
        "size_mb",
        "read_status",
        "row_count",
        "column_count",
        "has_pixel_and_month",
        "has_rainfall_or_spi_signal",
        "has_temporal_features",
        "is_possible_v2_feature_source",
        "rainfall_spi_columns",
        "temporal_columns",
        "v2_output_columns",
    ]

    print(
        parquet_catalog_df[
            display_columns
        ].to_string(
            index=False,
            max_colwidth=150,
        )
    )


# -----------------------------------------------------------------------------
# 4. Inspect artifact/manifests without loading models
# -----------------------------------------------------------------------------
MANIFEST_ROOTS = {
    "training_release_package_artifacts": (
        TRAINING_DROUGHT_ROOT
        / "processed"
        / "package_releases"
        / "rozvidrought_v2"
        / "rozvidrought"
        / "artifacts"
    ),
    "package_repo_artifacts": ROZVIDROUGHT_ARTIFACT_ROOT,
}

artifact_records: list[dict[str, Any]] = []
manifest_records: list[dict[str, Any]] = []


def recursive_key_names(
    value: Any,
    prefix: str = "",
) -> list[str]:
    found: list[str] = []

    if isinstance(value, dict):
        for key, child in value.items():
            key_path = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )
            found.append(key_path)
            found.extend(
                recursive_key_names(
                    child,
                    prefix=key_path,
                )
            )

    elif isinstance(value, list):
        for index, child in enumerate(value[:20]):
            found.extend(
                recursive_key_names(
                    child,
                    prefix=f"{prefix}[{index}]",
                )
            )

    return found


for source_group, artifact_root in MANIFEST_ROOTS.items():
    print(f"\nInspecting artifact root: {source_group}")
    print("Path:", artifact_root)
    print("Exists:", artifact_root.exists())

    if not artifact_root.exists():
        continue

    for artifact_path in sorted(
        path
        for path in artifact_root.rglob("*")
        if path.is_file()
    ):
        suffix = artifact_path.suffix.lower()

        artifact_records.append(
            {
                "source_group": source_group,
                "path": str(artifact_path),
                "relative_path": str(
                    artifact_path.relative_to(artifact_root)
                ),
                "suffix": suffix,
                "size_mb": round(
                    artifact_path.stat().st_size
                    / (1024 * 1024),
                    3,
                ),
                "is_model_binary": suffix
                in {".joblib", ".keras", ".pkl"},
                "is_manifest": suffix == ".json",
            }
        )

        if suffix != ".json":
            continue

        try:
            payload = json.loads(
                artifact_path.read_text(encoding="utf-8")
            )

            key_names = recursive_key_names(payload)
            key_text = " | ".join(key_names).lower()
            payload_text = json.dumps(payload).lower()

            feature_key_hits = sorted(
                {
                    key
                    for key in key_names
                    if any(
                        term in key.lower()
                        for term in [
                            "feature",
                            "column",
                            "contract",
                            "schema",
                        ]
                    )
                }
            )

            horizon_hits = [
                horizon
                for horizon in ["spi3", "spi6", "spi12", "fusion_v2"]
                if horizon in payload_text
            ]

            manifest_records.append(
                {
                    "source_group": source_group,
                    "path": str(artifact_path),
                    "relative_path": str(
                        artifact_path.relative_to(
                            artifact_root
                        )
                    ),
                    "manifest_read_status": "read_ok",
                    "top_level_keys": ", ".join(
                        list(payload.keys())
                        if isinstance(payload, dict)
                        else []
                    ),
                    "feature_or_contract_keys": " | ".join(
                        feature_key_hits[:80]
                    ),
                    "horizon_references": ", ".join(
                        horizon_hits
                    ),
                    "mentions_rainfall_or_spi": any(
                        term in payload_text
                        for term in RAINFALL_SPI_TERMS
                    ),
                    "mentions_temporal_features": any(
                        term in payload_text
                        for term in TEMPORAL_TERMS
                    ),
                    "error": None,
                }
            )

        except Exception as exc:
            manifest_records.append(
                {
                    "source_group": source_group,
                    "path": str(artifact_path),
                    "relative_path": str(
                        artifact_path.relative_to(
                            artifact_root
                        )
                    ),
                    "manifest_read_status": "read_failed",
                    "top_level_keys": None,
                    "feature_or_contract_keys": None,
                    "horizon_references": None,
                    "mentions_rainfall_or_spi": None,
                    "mentions_temporal_features": None,
                    "error": repr(exc),
                }
            )

artifact_inventory_df = pd.DataFrame(artifact_records)
manifest_catalog_df = pd.DataFrame(manifest_records)

print("\n=== Artifact inventory summary ===")

if artifact_inventory_df.empty:
    print("No artifact files found in inspected artifact roots.")
else:
    artifact_summary_df = (
        artifact_inventory_df.groupby("source_group")
        .agg(
            file_count=("path", "size"),
            model_binary_count=("is_model_binary", "sum"),
            manifest_count=("is_manifest", "sum"),
            total_size_mb=("size_mb", "sum"),
        )
        .reset_index()
    )

    print(artifact_summary_df.to_string(index=False))

    print("\nArtifact files:")
    print(
        artifact_inventory_df[
            [
                "source_group",
                "relative_path",
                "suffix",
                "size_mb",
                "is_model_binary",
                "is_manifest",
            ]
        ].to_string(
            index=False,
            max_colwidth=160,
        )
    )

print("\n=== Manifest catalog ===")

if manifest_catalog_df.empty:
    print("No JSON manifests found in inspected artifact roots.")
else:
    print(
        manifest_catalog_df.to_string(
            index=False,
            max_colwidth=180,
        )
    )


# -----------------------------------------------------------------------------
# 5. Explicitly compare master inputs against v2 source requirements
# -----------------------------------------------------------------------------
master_record = parquet_schema_record(
    path=MASTER_INPUT_PATH,
    source_group="current_infer_master_inputs_explicit",
)

master_has_rainfall_or_spi = bool(
    master_record["has_rainfall_or_spi_signal"]
)

master_has_temporal_features = bool(
    master_record["has_temporal_features"]
)

master_is_possible_v2_source = bool(
    master_record["is_possible_v2_feature_source"]
)

possible_v2_sources_df = pd.DataFrame()

if not parquet_catalog_df.empty:
    possible_v2_sources_df = parquet_catalog_df[
        parquet_catalog_df[
            "is_possible_v2_feature_source"
        ] == True
    ].copy()

print("\n=== Master input vs v2 conclusion ===")
print(
    "Master has rainfall/SPI-related fields:",
    master_has_rainfall_or_spi,
)
print(
    "Master has temporal feature fields:",
    master_has_temporal_features,
)
print(
    "Master is a possible direct v2 feature source:",
    master_is_possible_v2_source,
)

print("\nPossible v2-ready parquet sources found:")
print(len(possible_v2_sources_df))

if not possible_v2_sources_df.empty:
    print(
        possible_v2_sources_df[
            [
                "source_group",
                "path",
                "row_count",
                "column_count",
                "rainfall_spi_columns",
                "temporal_columns",
                "v2_output_columns",
            ]
        ].to_string(
            index=False,
            max_colwidth=180,
        )
    )
else:
    print(
        "None identified automatically. This means we need either:\n"
        "1. an explicit v2 feature dataset path,\n"
        "2. a targeted feature rebuild, or\n"
        "3. direct data-source/API ingestion for selected events."
    )


# -----------------------------------------------------------------------------
# 6. Lock this cell's scope and save discovery outputs
# -----------------------------------------------------------------------------
if not possible_v2_sources_df.empty:
    CELL1_STATUS = (
        "v2_feature_source_candidates_found_awaiting_contract_selection"
    )
else:
    CELL1_STATUS = (
        "no_v2_feature_source_auto_confirmed_rebuild_or_explicit_path_required"
    )

print("\n=== Cell 1 status ===")
print("Status:", CELL1_STATUS)
print(
    "\nNo model was loaded."
    "\nNo artifact was copied."
    "\nNo source has been selected yet."
)

parquet_catalog_path = (
    OUTPUT_DIR
    / f"cell1_v2_parquet_catalog_{CELL1_RUN_ID}.csv"
)

artifact_inventory_path = (
    OUTPUT_DIR
    / f"cell1_v2_artifact_inventory_{CELL1_RUN_ID}.csv"
)

manifest_catalog_path = (
    OUTPUT_DIR
    / f"cell1_v2_manifest_catalog_{CELL1_RUN_ID}.csv"
)

possible_sources_path = (
    OUTPUT_DIR
    / f"cell1_possible_v2_feature_sources_{CELL1_RUN_ID}.csv"
)

cell1_manifest_path = (
    OUTPUT_DIR
    / f"cell1_v2_source_discovery_manifest_{CELL1_RUN_ID}.json"
)

if parquet_catalog_df.empty:
    pd.DataFrame(
        columns=[
            "source_group",
            "path",
            "read_status",
        ]
    ).to_csv(
        parquet_catalog_path,
        index=False,
    )
else:
    parquet_catalog_df.to_csv(
        parquet_catalog_path,
        index=False,
    )

if artifact_inventory_df.empty:
    pd.DataFrame(
        columns=[
            "source_group",
            "path",
            "relative_path",
        ]
    ).to_csv(
        artifact_inventory_path,
        index=False,
    )
else:
    artifact_inventory_df.to_csv(
        artifact_inventory_path,
        index=False,
    )

if manifest_catalog_df.empty:
    pd.DataFrame(
        columns=[
            "source_group",
            "path",
            "manifest_read_status",
        ]
    ).to_csv(
        manifest_catalog_path,
        index=False,
    )
else:
    manifest_catalog_df.to_csv(
        manifest_catalog_path,
        index=False,
    )

possible_v2_sources_df.to_csv(
    possible_sources_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if hasattr(value, "item"):
        return value.item()

    raise TypeError(
        f"Cannot JSON serialize: {type(value).__name__}"
    )


cell1_manifest = {
    "run_id": CELL1_RUN_ID,
    "status": CELL1_STATUS,
    "purpose": (
        "Discover v2 feature-data candidates and artifact/manifests "
        "before seed or historical event testing."
    ),
    "current_workspace": str(WORKSPACE_DIR),
    "current_data_root": str(DATA_ROOT),
    "master_input_path": str(MASTER_INPUT_PATH),
    "training_drought_root": str(TRAINING_DROUGHT_ROOT),
    "rozvidrought_repo_root": str(ROZVIDROUGHT_REPO_ROOT),
    "master_input_v2_assessment": {
        "has_rainfall_or_spi_signal": master_has_rainfall_or_spi,
        "has_temporal_features": master_has_temporal_features,
        "is_possible_direct_v2_source": (
            master_is_possible_v2_source
        ),
    },
    "possible_v2_feature_source_count": int(
        len(possible_v2_sources_df)
    ),
    "possible_v2_feature_sources": (
        possible_v2_sources_df.to_dict(orient="records")
        if not possible_v2_sources_df.empty
        else []
    ),
    "outputs": {
        "parquet_catalog_csv": str(parquet_catalog_path),
        "artifact_inventory_csv": str(
            artifact_inventory_path
        ),
        "manifest_catalog_csv": str(
            manifest_catalog_path
        ),
        "possible_sources_csv": str(
            possible_sources_path
        ),
        "discovery_manifest_json": str(
            cell1_manifest_path
        ),
    },
}

cell1_manifest_path.write_text(
    json.dumps(
        cell1_manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 1 outputs ===")
print("Parquet catalog      ->", parquet_catalog_path)
print("Artifact inventory   ->", artifact_inventory_path)
print("Manifest catalog     ->", manifest_catalog_path)
print("Possible v2 sources  ->", possible_sources_path)
print("Discovery manifest   ->", cell1_manifest_path)

print("\nCell 1 complete.")
print(
    "Paste the output before we choose a v2 source or write a "
    "targeted rebuild cell."
)

=== Cell 1 purpose ===
Discover and audit v2 model input/artifact sources before seed or event inference.

=== Confirmed local paths ===
Current inference workspace : C:\Projects\Infer RozviDrought
Current data root           : C:\Projects\Infer RozviDrought\data
Current master input        : C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Training drought root       : C:\Projects\ramangwana\risks\data\drought_model
RozviDrought package root   : C:\Projects\rozvi\models\rozvidrought
Package artifact root       : C:\Projects\rozvi\models\rozvidrought\src\rozvidrought\artifacts
Output directory            : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng
Cell 1 run ID               : 20260620T093140Z

Inspecting: current_infer_master_inputs
Path: C:\Projects\Infer RozviDrought\data\master_inputs
Exists: True
Parquet files found: 1

Inspecting: training_model_inputs
Path: C:\Projects\ramangwana\risks\data\drought_model\processed\

In [3]:
# validate_events_ng.ipynb — Cell 2
# Purpose:
# - Build realistic observed seed cases from the v2 release feature frame.
# - Read SPI3/SPI6/SPI12 feature contracts.
# - Confirm source-column compatibility with each target model.
# - Select representative normal, short-term, seasonal, persistent,
#   and multi-horizon drought cases from real observed feature rows.
# - Save all outputs under:
#   C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng
#
# Important:
# - This is seed/pipeline validation, not independent event validation.
# - Do not manually edit rainfall/lag/rolling columns here.
# - Artificial edits without recomputing temporal features would violate
#   the trained model feature contract.
# - No model is loaded or run in this cell.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Recover confirmed paths from prior cells, with safe fallbacks
# -----------------------------------------------------------------------------
if "PROJECT_ROOT" not in globals():
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = (
        CURRENT_DIR.parent
        if CURRENT_DIR.name == "tests"
        else CURRENT_DIR
    )

if "WORKSPACE_DIR" not in globals():
    WORKSPACE_DIR = PROJECT_ROOT.parent

if "DATA_ROOT" not in globals():
    DATA_ROOT = WORKSPACE_DIR / "data"

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = DATA_ROOT / "backtests" / "validate_events_ng"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL2_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

V2_RELEASE_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

RELEASE_FEATURE_FRAME_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
    / "release_bundle_prediction_feature_frame_20260618T152226Z.parquet"
)

TARGET_KEYS = ["spi3", "spi6", "spi12"]

FEATURE_CONTRACT_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "feature_contract.json"
    )
    for target_key in TARGET_KEYS
}

TARGET_CONTRACT_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "target_contract.json"
    )
    for target_key in TARGET_KEYS
}

PREPROCESSING_CONTRACT_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "preprocessing_contract.json"
    )
    for target_key in TARGET_KEYS
}

SEED_OUTPUT_DIR = OUTPUT_DIR / "seed_cases"
SEED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== Cell 2 purpose ===")
print(
    "Create observed, model-compatible seed rows for a controlled "
    "v2 model smoke test."
)

print("\n=== Paths ===")
print("Release feature frame :", RELEASE_FEATURE_FRAME_PATH)
print("Release artifact root :", V2_RELEASE_ARTIFACT_ROOT)
print("Seed output directory :", SEED_OUTPUT_DIR)
print("Cell 2 run ID         :", CELL2_RUN_ID)

if not RELEASE_FEATURE_FRAME_PATH.exists():
    raise FileNotFoundError(
        "Release feature frame not found:\n"
        f"{RELEASE_FEATURE_FRAME_PATH}"
    )

for target_key, contract_path in FEATURE_CONTRACT_PATHS.items():
    if not contract_path.exists():
        raise FileNotFoundError(
            f"{target_key} feature contract not found:\n"
            f"{contract_path}"
        )

for target_key, contract_path in TARGET_CONTRACT_PATHS.items():
    if not contract_path.exists():
        raise FileNotFoundError(
            f"{target_key} target contract not found:\n"
            f"{contract_path}"
        )

for target_key, contract_path in PREPROCESSING_CONTRACT_PATHS.items():
    if not contract_path.exists():
        raise FileNotFoundError(
            f"{target_key} preprocessing contract not found:\n"
            f"{contract_path}"
        )


# -----------------------------------------------------------------------------
# 2. Contract helpers
# -----------------------------------------------------------------------------
def read_json_utf8_sig(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"Expected JSON object in {path.name}, "
            f"got {type(payload).__name__}."
        )

    return payload


def first_string_list(
    payload: dict[str, Any],
    candidate_keys: list[str],
) -> list[str]:
    for key in candidate_keys:
        value = payload.get(key)

        if isinstance(value, list) and all(
            isinstance(item, str)
            for item in value
        ):
            return list(value)

    return []


def get_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    feature_list = first_string_list(
        feature_contract,
        candidate_keys=[
            "model_features",
            "features",
            "feature_columns",
            "required_features",
        ],
    )

    if not feature_list:
        raise ValueError(
            "Could not find a model feature list in the "
            "feature contract."
        )

    return feature_list


def get_label_column(
    target_key: str,
    target_contract: dict[str, Any],
) -> str:
    label_col = target_contract.get("label_col")

    if isinstance(label_col, str) and label_col:
        return label_col

    return f"drought_class_{target_key}"


def choose_existing_columns(
    source_columns: list[str],
    candidates: list[str],
) -> list[str]:
    return [
        column
        for column in candidates
        if column in source_columns
    ]


def stable_seed_order(
    dataframe: pd.DataFrame,
    source_id_column: str,
) -> pd.DataFrame:
    ordered = dataframe.copy()

    ordered["_stable_order"] = (
        (
            ordered[source_id_column]
            .astype("int64")
            .mul(1_103_515_245)
            .add(12_345)
        )
        % 2_147_483_647
    )

    return ordered.sort_values(
        ["_stable_order", source_id_column]
    ).drop(columns="_stable_order")


# -----------------------------------------------------------------------------
# 3. Read release contracts and extract exact target feature lists
# -----------------------------------------------------------------------------
feature_contracts: dict[str, dict[str, Any]] = {}
target_contracts: dict[str, dict[str, Any]] = {}
preprocessing_contracts: dict[str, dict[str, Any]] = {}

target_feature_lists: dict[str, list[str]] = {}
target_label_columns: dict[str, str] = {}

contract_summary_records: list[dict[str, Any]] = []

for target_key in TARGET_KEYS:
    feature_contract = read_json_utf8_sig(
        FEATURE_CONTRACT_PATHS[target_key]
    )
    target_contract = read_json_utf8_sig(
        TARGET_CONTRACT_PATHS[target_key]
    )
    preprocessing_contract = read_json_utf8_sig(
        PREPROCESSING_CONTRACT_PATHS[target_key]
    )

    feature_contracts[target_key] = feature_contract
    target_contracts[target_key] = target_contract
    preprocessing_contracts[target_key] = preprocessing_contract

    model_features = get_feature_list(feature_contract)
    label_column = get_label_column(
        target_key=target_key,
        target_contract=target_contract,
    )

    target_feature_lists[target_key] = model_features
    target_label_columns[target_key] = label_column

    contract_summary_records.append(
        {
            "target_key": target_key,
            "feature_contract_path": str(
                FEATURE_CONTRACT_PATHS[target_key]
            ),
            "target_contract_path": str(
                TARGET_CONTRACT_PATHS[target_key]
            ),
            "preprocessing_contract_path": str(
                PREPROCESSING_CONTRACT_PATHS[target_key]
            ),
            "model_feature_count": len(model_features),
            "label_column": label_column,
            "temporal_reconstruction_required": (
                preprocessing_contract.get(
                    "temporal_reconstruction_required"
                )
            ),
            "expected_time_column": preprocessing_contract.get(
                "expected_time_column"
            ),
            "expected_pixel_columns": json.dumps(
                preprocessing_contract.get(
                    "expected_pixel_columns",
                    [],
                )
            ),
            "missing_value_policy": json.dumps(
                preprocessing_contract.get(
                    "missing_value_policy",
                    {}
                )
            ),
        }
    )

contract_summary_df = pd.DataFrame(
    contract_summary_records
)

print("\n=== Target contract summary ===")
print(
    contract_summary_df.to_string(
        index=False,
        max_colwidth=140,
    )
)


# -----------------------------------------------------------------------------
# 4. Inspect release feature-frame schema and map identity columns
# -----------------------------------------------------------------------------
release_parquet = pq.ParquetFile(
    RELEASE_FEATURE_FRAME_PATH
)

release_row_count = int(
    release_parquet.metadata.num_rows
)

release_columns = list(
    release_parquet.schema.names
)

IDENTITY_CANDIDATES = [
    "pixel_id",
    "pixel_key",
    "row",
    "col",
    "lon",
    "lat",
    "yyyymm",
    "scenario",
    "h3_index",
    "h3",
    "cell_id",
]

identity_columns = choose_existing_columns(
    source_columns=release_columns,
    candidates=IDENTITY_CANDIDATES,
)

if "yyyymm" not in release_columns:
    raise AssertionError(
        "The release feature frame has no yyyymm column. "
        "Cannot create time-aware seed cases."
    )

label_columns = list(
    dict.fromkeys(
        target_label_columns.values()
    )
)

missing_label_columns = [
    column
    for column in label_columns
    if column not in release_columns
]

if missing_label_columns:
    raise AssertionError(
        "Release feature frame is missing required label columns:\n"
        f"{missing_label_columns}"
    )

print("\n=== Release feature-frame schema ===")
print("Rows                  :", f"{release_row_count:,}")
print("Columns               :", len(release_columns))
print("Identity columns found:", identity_columns)
print("Label columns         :", label_columns)


# -----------------------------------------------------------------------------
# 5. Confirm feature availability for each v2 target model
# -----------------------------------------------------------------------------
feature_availability_records: list[dict[str, Any]] = []

for target_key, feature_list in target_feature_lists.items():
    source_present = [
        feature
        for feature in feature_list
        if feature in release_columns
    ]

    source_missing = [
        feature
        for feature in feature_list
        if feature not in release_columns
    ]

    feature_availability_records.append(
        {
            "target_key": target_key,
            "required_feature_count": len(feature_list),
            "source_feature_count": len(source_present),
            "missing_feature_count": len(source_missing),
            "all_required_feature_columns_present": (
                len(source_missing) == 0
            ),
            "missing_features_preview": ", ".join(
                source_missing[:50]
            ),
        }
    )

feature_availability_df = pd.DataFrame(
    feature_availability_records
)

print("\n=== Release-frame feature compatibility ===")
print(
    feature_availability_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

missing_contract_features = feature_availability_df[
    feature_availability_df[
        "all_required_feature_columns_present"
    ] == False
].copy()

if not missing_contract_features.empty:
    raise AssertionError(
        "The release feature frame does not satisfy all target "
        "feature contracts. Stop before seed creation."
    )


# -----------------------------------------------------------------------------
# 6. Read only identity + labels to identify real observed seed candidates
# -----------------------------------------------------------------------------
SOURCE_RECORD_ID = "__source_record_id"

label_scan_columns = list(
    dict.fromkeys(
        identity_columns + label_columns
    )
)

print("\n=== Scanning label/identity columns only ===")
print("Columns scanned:", label_scan_columns)

label_chunks: list[pd.DataFrame] = []
row_cursor = 0
scan_batch_size = 250_000

for batch_number, batch in enumerate(
    release_parquet.iter_batches(
        batch_size=scan_batch_size,
        columns=label_scan_columns,
    ),
    start=1,
):
    chunk = batch.to_pandas()

    chunk[SOURCE_RECORD_ID] = np.arange(
        row_cursor,
        row_cursor + len(chunk),
        dtype=np.int64,
    )

    row_cursor += len(chunk)
    label_chunks.append(chunk)

    print(
        f"Label scan batch {batch_number}: "
        f"{len(chunk):,} rows "
        f"(scanned {row_cursor:,}/{release_row_count:,})"
    )

label_index_df = pd.concat(
    label_chunks,
    ignore_index=True,
)

del label_chunks

if len(label_index_df) != release_row_count:
    raise AssertionError(
        "Label scan row count does not match parquet metadata."
    )

label_index_df["yyyymm"] = (
    label_index_df["yyyymm"]
    .astype(str)
)

if "scenario" in label_index_df.columns:
    label_index_df["scenario"] = (
        label_index_df["scenario"]
        .astype(str)
    )

for label_column in label_columns:
    label_index_df[label_column] = pd.to_numeric(
        label_index_df[label_column],
        errors="coerce",
    )

historical_seed_pool_df = label_index_df.copy()

if "scenario" in historical_seed_pool_df.columns:
    historical_rows_df = historical_seed_pool_df[
        historical_seed_pool_df["scenario"]
        .eq("historical")
    ].copy()

    if historical_rows_df.empty:
        print(
            "\nWARNING: No historical scenario rows found in release "
            "feature frame. Retaining all scenarios for seed selection."
        )
    else:
        historical_seed_pool_df = historical_rows_df

print("\n=== Seed source pool ===")
print("Rows available for seeds:", f"{len(historical_seed_pool_df):,}")

print("\nScenario counts:")
if "scenario" in label_index_df.columns:
    print(
        label_index_df["scenario"]
        .value_counts(dropna=False)
        .to_string()
    )
else:
    print("No scenario column in release feature frame.")


# -----------------------------------------------------------------------------
# 7. Inspect label distributions and define drought-state seed strata
# -----------------------------------------------------------------------------
label_distribution_records: list[dict[str, Any]] = []

for target_key, label_column in target_label_columns.items():
    distribution = (
        historical_seed_pool_df[label_column]
        .value_counts(dropna=False)
        .sort_index()
    )

    for class_value, row_count in distribution.items():
        label_distribution_records.append(
            {
                "target_key": target_key,
                "label_column": label_column,
                "class_value": (
                    None
                    if pd.isna(class_value)
                    else int(class_value)
                ),
                "row_count": int(row_count),
            }
        )

label_distribution_df = pd.DataFrame(
    label_distribution_records
)

print("\n=== Historical label distributions ===")
print(
    label_distribution_df.to_string(
        index=False,
    )
)

label_matrix_columns = [
    target_label_columns[target_key]
    for target_key in TARGET_KEYS
]

valid_label_pool_df = historical_seed_pool_df.dropna(
    subset=label_matrix_columns
).copy()

if valid_label_pool_df.empty:
    raise AssertionError(
        "No rows have all SPI3/SPI6/SPI12 labels available. "
        "Cannot construct observed seed cases."
    )

all_label_values = sorted(
    {
        int(value)
        for label_column in label_matrix_columns
        for value in valid_label_pool_df[
            label_column
        ].dropna().unique()
    }
)

if not all_label_values:
    raise AssertionError(
        "No numeric drought-class values found in release feature frame."
    )

normal_class = int(min(all_label_values))

if 2 in all_label_values:
    stress_class = 2
elif len(all_label_values) >= 3:
    stress_class = int(all_label_values[2])
else:
    stress_class = int(max(all_label_values))

mild_or_lower_class = min(
    normal_class + 1,
    int(max(all_label_values)),
)

spi3_label = target_label_columns["spi3"]
spi6_label = target_label_columns["spi6"]
spi12_label = target_label_columns["spi12"]

valid_label_pool_df["_label_sum"] = (
    valid_label_pool_df[
        label_matrix_columns
    ]
    .sum(axis=1)
)

print("\n=== Seed-state rules ===")
print("Available classes        :", all_label_values)
print("Normal class             :", normal_class)
print("Drought stress threshold :", stress_class)
print("Mild-or-lower threshold  :", mild_or_lower_class)


# -----------------------------------------------------------------------------
# 8. Select deterministic observed seeds
# -----------------------------------------------------------------------------
SEEDS_PER_CASE = 3

seed_case_specs = [
    {
        "seed_case": "observed_normal_all_horizons",
        "preferred_mask": (
            (valid_label_pool_df[spi3_label] == normal_class)
            & (valid_label_pool_df[spi6_label] == normal_class)
            & (valid_label_pool_df[spi12_label] == normal_class)
        ),
        "fallback_mask": (
            valid_label_pool_df["_label_sum"]
            == valid_label_pool_df["_label_sum"].min()
        ),
        "expected_behavior": (
            "All target models and Fusion v2 should favour the "
            "lowest drought class."
        ),
    },
    {
        "seed_case": "observed_short_term_drought",
        "preferred_mask": (
            (valid_label_pool_df[spi3_label] >= stress_class)
            & (valid_label_pool_df[spi6_label] <= mild_or_lower_class)
            & (valid_label_pool_df[spi12_label] <= mild_or_lower_class)
        ),
        "fallback_mask": (
            valid_label_pool_df[spi3_label] >= stress_class
        ),
        "expected_behavior": (
            "SPI3 should show the strongest drought response; "
            "SPI6/SPI12 may remain lower if the signal is recent."
        ),
    },
    {
        "seed_case": "observed_seasonal_drought",
        "preferred_mask": (
            (valid_label_pool_df[spi6_label] >= stress_class)
            & (valid_label_pool_df[spi12_label] <= mild_or_lower_class)
        ),
        "fallback_mask": (
            valid_label_pool_df[spi6_label] >= stress_class
        ),
        "expected_behavior": (
            "SPI6 should show a strong seasonal drought response; "
            "SPI12 may remain lower if drought is not persistent."
        ),
    },
    {
        "seed_case": "observed_persistent_drought",
        "preferred_mask": (
            (valid_label_pool_df[spi12_label] >= stress_class)
            & (valid_label_pool_df[spi6_label] >= mild_or_lower_class)
        ),
        "fallback_mask": (
            valid_label_pool_df[spi12_label] >= stress_class
        ),
        "expected_behavior": (
            "SPI12 should show the strongest persistent drought "
            "response and Fusion v2 should reflect persistence."
        ),
    },
    {
        "seed_case": "observed_multi_horizon_drought",
        "preferred_mask": (
            (valid_label_pool_df[spi3_label] >= stress_class)
            & (valid_label_pool_df[spi6_label] >= stress_class)
            & (valid_label_pool_df[spi12_label] >= stress_class)
        ),
        "fallback_mask": (
            valid_label_pool_df["_label_sum"]
            >= valid_label_pool_df["_label_sum"].quantile(0.99)
        ),
        "expected_behavior": (
            "SPI3, SPI6, SPI12, and Fusion v2 should all show a "
            "strong drought response."
        ),
    },
]

selected_seed_index_frames: list[pd.DataFrame] = []
seed_selection_summary_records: list[dict[str, Any]] = []

for seed_spec in seed_case_specs:
    preferred_candidates = valid_label_pool_df.loc[
        seed_spec["preferred_mask"]
    ].copy()

    if not preferred_candidates.empty:
        selection_mode = "preferred_rule"
        candidate_pool = preferred_candidates
    else:
        fallback_candidates = valid_label_pool_df.loc[
            seed_spec["fallback_mask"]
        ].copy()

        selection_mode = "fallback_rule"

        if fallback_candidates.empty:
            candidate_pool = valid_label_pool_df.copy()
            selection_mode = "global_fallback"

        else:
            candidate_pool = fallback_candidates

    candidate_pool = stable_seed_order(
        dataframe=candidate_pool,
        source_id_column=SOURCE_RECORD_ID,
    )

    selected_case_df = (
        candidate_pool
        .head(SEEDS_PER_CASE)
        .copy()
        .reset_index(drop=True)
    )

    selected_case_df.insert(
        0,
        "seed_case",
        seed_spec["seed_case"],
    )

    selected_case_df.insert(
        1,
        "seed_rank",
        np.arange(
            1,
            len(selected_case_df) + 1,
            dtype=int,
        ),
    )

    selected_case_df.insert(
        2,
        "selection_mode",
        selection_mode,
    )

    selected_case_df.insert(
        3,
        "expected_behavior",
        seed_spec["expected_behavior"],
    )

    selected_seed_index_frames.append(
        selected_case_df
    )

    seed_selection_summary_records.append(
        {
            "seed_case": seed_spec["seed_case"],
            "selection_mode": selection_mode,
            "preferred_candidate_count": int(
                len(preferred_candidates)
            ),
            "selected_row_count": int(
                len(selected_case_df)
            ),
            "expected_behavior": seed_spec[
                "expected_behavior"
            ],
        }
    )

seed_selection_summary_df = pd.DataFrame(
    seed_selection_summary_records
)

seed_index_df = pd.concat(
    selected_seed_index_frames,
    ignore_index=True,
)

if seed_index_df.empty:
    raise AssertionError(
        "No observed seed rows were selected."
    )

if seed_index_df[SOURCE_RECORD_ID].duplicated().any():
    seed_index_df = (
        seed_index_df
        .sort_values(
            ["seed_case", "seed_rank"]
        )
        .drop_duplicates(
            subset=[SOURCE_RECORD_ID],
            keep="first",
        )
        .reset_index(drop=True)
    )

print("\n=== Seed selection summary ===")
print(
    seed_selection_summary_df.to_string(
        index=False,
        max_colwidth=150,
    )
)

seed_display_columns = [
    column
    for column in [
        "seed_case",
        "seed_rank",
        "selection_mode",
        SOURCE_RECORD_ID,
        "pixel_id",
        "pixel_key",
        "row",
        "col",
        "lon",
        "lat",
        "yyyymm",
        "scenario",
        spi3_label,
        spi6_label,
        spi12_label,
        "_label_sum",
    ]
    if column in seed_index_df.columns
]

print("\n=== Selected observed seed rows ===")
print(
    seed_index_df[
        seed_display_columns
    ].to_string(
        index=False,
        max_colwidth=140,
    )
)


# -----------------------------------------------------------------------------
# 9. Retrieve only the exact selected rows with all target-model features
# -----------------------------------------------------------------------------
all_model_feature_columns = sorted(
    set().union(
        *[
            set(feature_list)
            for feature_list in target_feature_lists.values()
        ]
    )
)

missing_union_features = [
    column
    for column in all_model_feature_columns
    if column not in release_columns
]

if missing_union_features:
    raise AssertionError(
        "Release source is missing one or more required union "
        "model features:\n"
        f"{missing_union_features[:100]}"
    )

selected_source_ids = set(
    seed_index_df[SOURCE_RECORD_ID]
    .astype(int)
    .tolist()
)

print("\n=== Retrieving complete feature rows for selected seeds ===")
print("Selected seed rows :", len(selected_source_ids))
print("Union feature count:", len(all_model_feature_columns))

feature_row_frames: list[pd.DataFrame] = []
row_cursor = 0
feature_batch_size = 25_000

for batch_number, batch in enumerate(
    release_parquet.iter_batches(
        batch_size=feature_batch_size,
        columns=all_model_feature_columns,
    ),
    start=1,
):
    batch_length = len(batch)

    batch_source_ids = np.arange(
        row_cursor,
        row_cursor + batch_length,
        dtype=np.int64,
    )

    selected_positions = np.flatnonzero(
        np.isin(
            batch_source_ids,
            list(selected_source_ids),
        )
    )

    if len(selected_positions):
        chunk = batch.to_pandas().iloc[
            selected_positions
        ].copy()

        chunk[SOURCE_RECORD_ID] = batch_source_ids[
            selected_positions
        ]

        feature_row_frames.append(chunk)

    row_cursor += batch_length

    if batch_number % 10 == 0 or row_cursor == release_row_count:
        print(
            f"Feature scan batch {batch_number}: "
            f"scanned {row_cursor:,}/{release_row_count:,}"
        )

seed_feature_rows_df = pd.concat(
    feature_row_frames,
    ignore_index=True,
)

if len(seed_feature_rows_df) != len(selected_source_ids):
    missing_ids = sorted(
        selected_source_ids
        - set(
            seed_feature_rows_df[
                SOURCE_RECORD_ID
            ]
        )
    )

    raise AssertionError(
        "Could not retrieve all selected seed feature rows. "
        f"Missing source IDs: {missing_ids}"
    )

seed_rows_df = (
    seed_index_df
    .merge(
        seed_feature_rows_df,
        on=SOURCE_RECORD_ID,
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["seed_case", "seed_rank"]
    )
    .reset_index(drop=True)
)

print("\nComplete seed rows retrieved:", len(seed_rows_df))
print("Total columns in seed table :", len(seed_rows_df.columns))


# -----------------------------------------------------------------------------
# 10. Per-seed contract and missing-value compatibility report
# -----------------------------------------------------------------------------
seed_compatibility_records: list[dict[str, Any]] = []

for _, seed_row in seed_rows_df.iterrows():
    for target_key, feature_list in target_feature_lists.items():
        row_missing_feature_values = [
            feature
            for feature in feature_list
            if pd.isna(seed_row[feature])
        ]

        seed_compatibility_records.append(
            {
                "seed_case": seed_row["seed_case"],
                "seed_rank": int(seed_row["seed_rank"]),
                "source_record_id": int(
                    seed_row[SOURCE_RECORD_ID]
                ),
                "target_key": target_key,
                "required_feature_count": len(feature_list),
                "missing_feature_value_count": len(
                    row_missing_feature_values
                ),
                "missing_feature_values_preview": ", ".join(
                    row_missing_feature_values[:40]
                ),
                "all_feature_values_present": (
                    len(row_missing_feature_values) == 0
                ),
                "model_input_status": (
                    "ready_no_missing_values"
                    if len(row_missing_feature_values) == 0
                    else "ready_subject_to_model_missing_value_policy"
                ),
            }
        )

seed_compatibility_df = pd.DataFrame(
    seed_compatibility_records
)

print("\n=== Seed-row compatibility by target ===")
print(
    seed_compatibility_df.to_string(
        index=False,
        max_colwidth=150,
    )
)


# -----------------------------------------------------------------------------
# 11. Final seed readiness decision
# -----------------------------------------------------------------------------
required_seed_count = (
    len(seed_case_specs) * SEEDS_PER_CASE
)

all_targets_have_columns = bool(
    feature_availability_df[
        "all_required_feature_columns_present"
    ].all()
)

seed_case_count = int(
    seed_rows_df["seed_case"].nunique()
)

seed_row_count = int(len(seed_rows_df))

if (
    all_targets_have_columns
    and seed_case_count == len(seed_case_specs)
    and seed_row_count >= len(seed_case_specs)
):
    CELL2_STATUS = (
        "observed_v2_seed_cases_ready_for_target_model_smoke_test"
    )
else:
    CELL2_STATUS = (
        "observed_v2_seed_cases_created_with_coverage_warnings"
    )

print("\n=== Cell 2 conclusion ===")
print("Status              :", CELL2_STATUS)
print("Seed cases          :", seed_case_count)
print("Seed rows           :", seed_row_count)
print("Target contracts OK :", all_targets_have_columns)

print(
    "\nInterpretation:\n"
    "- These are observed model-ready rows, not manually fabricated inputs.\n"
    "- The next cell can load SPI3/SPI6/SPI12 artifacts, predict each seed,\n"
    "  then apply Fusion v2 rules and compare output behaviour to seed intent.\n"
    "- Missing feature values are retained exactly as observed and must be\n"
    "  interpreted through each target model's missing-value policy."
)


# -----------------------------------------------------------------------------
# 12. Save all seed setup outputs
# -----------------------------------------------------------------------------
contract_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell2_contract_summary_{CELL2_RUN_ID}.csv"
)

feature_availability_path = (
    SEED_OUTPUT_DIR
    / f"cell2_feature_availability_{CELL2_RUN_ID}.csv"
)

label_distribution_path = (
    SEED_OUTPUT_DIR
    / f"cell2_historical_label_distribution_{CELL2_RUN_ID}.csv"
)

seed_selection_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell2_seed_selection_summary_{CELL2_RUN_ID}.csv"
)

seed_index_path = (
    SEED_OUTPUT_DIR
    / f"cell2_seed_index_{CELL2_RUN_ID}.csv"
)

seed_rows_path = (
    SEED_OUTPUT_DIR
    / f"cell2_observed_seed_rows_{CELL2_RUN_ID}.parquet"
)

seed_compatibility_path = (
    SEED_OUTPUT_DIR
    / f"cell2_seed_compatibility_{CELL2_RUN_ID}.csv"
)

seed_manifest_path = (
    SEED_OUTPUT_DIR
    / f"cell2_seed_setup_manifest_{CELL2_RUN_ID}.json"
)

contract_summary_df.to_csv(
    contract_summary_path,
    index=False,
)

feature_availability_df.to_csv(
    feature_availability_path,
    index=False,
)

label_distribution_df.to_csv(
    label_distribution_path,
    index=False,
)

seed_selection_summary_df.to_csv(
    seed_selection_summary_path,
    index=False,
)

seed_index_df.to_csv(
    seed_index_path,
    index=False,
)

seed_rows_df.to_parquet(
    seed_rows_path,
    index=False,
)

seed_compatibility_df.to_csv(
    seed_compatibility_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


seed_manifest = {
    "run_id": CELL2_RUN_ID,
    "status": CELL2_STATUS,
    "purpose": (
        "Construct observed, contract-compatible seed rows for "
        "SPI3/SPI6/SPI12/Fusion v2 model smoke testing."
    ),
    "release_feature_frame_path": str(
        RELEASE_FEATURE_FRAME_PATH
    ),
    "release_feature_frame_row_count": release_row_count,
    "release_feature_frame_column_count": len(
        release_columns
    ),
    "release_artifact_root": str(
        V2_RELEASE_ARTIFACT_ROOT
    ),
    "target_feature_contracts": {
        target_key: str(
            FEATURE_CONTRACT_PATHS[target_key]
        )
        for target_key in TARGET_KEYS
    },
    "target_label_columns": target_label_columns,
    "identity_columns": identity_columns,
    "normal_class": normal_class,
    "stress_class": stress_class,
    "mild_or_lower_class": mild_or_lower_class,
    "seeds_per_case_requested": SEEDS_PER_CASE,
    "seed_case_count": seed_case_count,
    "seed_row_count": seed_row_count,
    "selection_summary": seed_selection_summary_df.to_dict(
        orient="records"
    ),
    "feature_availability": feature_availability_df.to_dict(
        orient="records"
    ),
    "outputs": {
        "contract_summary_csv": str(
            contract_summary_path
        ),
        "feature_availability_csv": str(
            feature_availability_path
        ),
        "label_distribution_csv": str(
            label_distribution_path
        ),
        "seed_selection_summary_csv": str(
            seed_selection_summary_path
        ),
        "seed_index_csv": str(seed_index_path),
        "observed_seed_rows_parquet": str(
            seed_rows_path
        ),
        "seed_compatibility_csv": str(
            seed_compatibility_path
        ),
        "seed_manifest_json": str(
            seed_manifest_path
        ),
    },
}

seed_manifest_path.write_text(
    json.dumps(
        seed_manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 2 outputs ===")
print("Contract summary       ->", contract_summary_path)
print("Feature availability   ->", feature_availability_path)
print("Label distribution     ->", label_distribution_path)
print("Selection summary      ->", seed_selection_summary_path)
print("Seed index             ->", seed_index_path)
print("Observed seed rows     ->", seed_rows_path)
print("Seed compatibility     ->", seed_compatibility_path)
print("Seed setup manifest    ->", seed_manifest_path)

print("\nCell 2 complete.")
print("Paste the output before we write the model-loading smoke-test cell.")

=== Cell 2 purpose ===
Create observed, model-compatible seed rows for a controlled v2 model smoke test.

=== Paths ===
Release feature frame : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_prediction_feature_frame_20260618T152226Z.parquet
Release artifact root : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts
Seed output directory : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\seed_cases
Cell 2 run ID         : 20260620T094358Z

=== Target contract summary ===
target_key                                                                                                                        feature_contract_path                                                                                                                         target_contract_path                                                             

In [5]:
# validate_events_ng.ipynb — Cell 3
# Purpose:
# - Load released SPI3, SPI6, and SPI12 v2 models.
# - Predict observed, contract-compatible seed rows from Cell 2.
# - Build the released weighted-fusion baseline.
# - Compare direct target predictions with stored release predictions.
# - Check whether seed behaviour follows the expected drought horizon.
#
# Scope:
# - Technical seed smoke test only.
# - Not independent accuracy validation.
# - No model/artifact copying.
# - No API ingestion, Earth Engine, or event-area backtesting yet.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths and prior-cell recovery
# -----------------------------------------------------------------------------
if "PROJECT_ROOT" not in globals():
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = (
        CURRENT_DIR.parent
        if CURRENT_DIR.name == "tests"
        else CURRENT_DIR
    )

if "WORKSPACE_DIR" not in globals():
    WORKSPACE_DIR = PROJECT_ROOT.parent

if "DATA_ROOT" not in globals():
    DATA_ROOT = WORKSPACE_DIR / "data"

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = DATA_ROOT / "backtests" / "validate_events_ng"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED_OUTPUT_DIR = OUTPUT_DIR / "seed_cases"
SEED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL3_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

V2_RELEASE_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

RELEASE_PREDICTION_DIR = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
)

TARGET_KEYS = ["spi3", "spi6", "spi12"]
NUM_CLASSES = 5
CLASS_VALUES = np.arange(NUM_CLASSES, dtype=int)

SEED_KEY_COLUMNS = ["row", "col", "yyyymm"]

MODEL_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "model.joblib"
    )
    for target_key in TARGET_KEYS
}

FEATURE_CONTRACT_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "feature_contract.json"
    )
    for target_key in TARGET_KEYS
}

TARGET_CONTRACT_PATHS = {
    target_key: (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "target_contract.json"
    )
    for target_key in TARGET_KEYS
}

FUSION_RULES_PATH = (
    V2_RELEASE_ARTIFACT_ROOT
    / "fusion_v2"
    / "v2_20260617"
    / "fusion_rules.json"
)

TARGET_PREDICTIONS_PATH = (
    RELEASE_PREDICTION_DIR
    / "release_bundle_target_predictions_20260618T152729Z.parquet"
)

FUSION_PREDICTIONS_PATH = (
    RELEASE_PREDICTION_DIR
    / "release_bundle_fusion_v2_predictions_20260618T185446Z.parquet"
)

print("=== Cell 3 purpose ===")
print(
    "Run staged v2 target models on observed seed rows and "
    "validate direct prediction parity."
)

print("\n=== Paths ===")
print("Artifact root       :", V2_RELEASE_ARTIFACT_ROOT)
print("Target predictions :", TARGET_PREDICTIONS_PATH)
print("Fusion predictions :", FUSION_PREDICTIONS_PATH)
print("Seed output dir    :", SEED_OUTPUT_DIR)
print("Run ID             :", CELL3_RUN_ID)

for target_key, model_path in MODEL_PATHS.items():
    if not model_path.exists():
        raise FileNotFoundError(
            f"{target_key} model missing:\n{model_path}"
        )

for target_key, contract_path in FEATURE_CONTRACT_PATHS.items():
    if not contract_path.exists():
        raise FileNotFoundError(
            f"{target_key} feature contract missing:\n{contract_path}"
        )

for target_key, contract_path in TARGET_CONTRACT_PATHS.items():
    if not contract_path.exists():
        raise FileNotFoundError(
            f"{target_key} target contract missing:\n{contract_path}"
        )

if not FUSION_RULES_PATH.exists():
    raise FileNotFoundError(
        f"Fusion rules missing:\n{FUSION_RULES_PATH}"
    )

if "seed_rows_df" not in globals():
    seed_files = sorted(
        SEED_OUTPUT_DIR.glob("cell2_observed_seed_rows_*.parquet")
    )

    if not seed_files:
        raise FileNotFoundError(
            "seed_rows_df is not in memory and no Cell 2 seed parquet "
            "was found. Rerun Cell 2."
        )

    latest_seed_file = max(
        seed_files,
        key=lambda path: path.stat().st_mtime,
    )

    seed_rows_df = pd.read_parquet(latest_seed_file)

    print("\nReloaded seed rows from:")
    print(latest_seed_file)

else:
    print("\nUsing seed_rows_df already in memory.")

if seed_rows_df.empty:
    raise AssertionError("seed_rows_df is empty.")

print("Seed rows:", len(seed_rows_df))


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def read_json(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"Expected a JSON object in {path.name}."
        )

    return payload


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    for key in [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(isinstance(item, str) for item in value)
        ):
            return list(value)

    raise ValueError(
        "Could not find a valid model feature list in the contract."
    )


def extract_label_column(
    target_key: str,
    target_contract: dict[str, Any],
) -> str:
    label_column = target_contract.get("label_col")

    if isinstance(label_column, str) and label_column:
        return label_column

    return f"drought_class_{target_key}"


def normalize_keys(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    result = dataframe.copy()

    for column in SEED_KEY_COLUMNS:
        if column not in result.columns:
            raise KeyError(
                f"Missing required key column: {column}"
            )

    result["row"] = pd.to_numeric(
        result["row"],
        errors="raise",
    ).astype("int64")

    result["col"] = pd.to_numeric(
        result["col"],
        errors="raise",
    ).astype("int64")

    result["yyyymm"] = result["yyyymm"].astype(str)

    return result


def assert_unique_keys(
    dataframe: pd.DataFrame,
    name: str,
) -> None:
    duplicate_count = int(
        dataframe.duplicated(
            subset=SEED_KEY_COLUMNS
        ).sum()
    )

    if duplicate_count:
        preview = (
            dataframe.loc[
                dataframe.duplicated(
                    subset=SEED_KEY_COLUMNS,
                    keep=False,
                ),
                SEED_KEY_COLUMNS,
            ]
            .sort_values(SEED_KEY_COLUMNS)
            .head(20)
        )

        raise AssertionError(
            f"{name} has {duplicate_count} duplicate "
            f"row/col/month keys.\n"
            f"{preview.to_string(index=False)}"
        )


def key_series(
    dataframe: pd.DataFrame,
) -> pd.Series:
    normalized = normalize_keys(
        dataframe[SEED_KEY_COLUMNS]
    )

    return (
        normalized["row"].astype(str)
        + "|"
        + normalized["col"].astype(str)
        + "|"
        + normalized["yyyymm"]
    )


def scan_release_rows(
    parquet_path: Path,
    required_columns: list[str],
    selected_keys: set[str],
    batch_size: int = 100_000,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    if not parquet_path.exists():
        return (
            pd.DataFrame(),
            {
                "status": "source_missing",
                "path": str(parquet_path),
                "rows_scanned": 0,
                "rows_found": 0,
                "missing_columns": [],
            },
        )

    parquet_file = pq.ParquetFile(parquet_path)
    available_columns = list(parquet_file.schema.names)

    missing_columns = [
        column
        for column in required_columns
        if column not in available_columns
    ]

    if missing_columns:
        return (
            pd.DataFrame(),
            {
                "status": "required_columns_missing",
                "path": str(parquet_path),
                "rows_scanned": 0,
                "rows_found": 0,
                "missing_columns": missing_columns,
            },
        )

    result_frames: list[pd.DataFrame] = []
    rows_scanned = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=required_columns,
    ):
        chunk = normalize_keys(batch.to_pandas())
        rows_scanned += len(chunk)

        chunk["_seed_key"] = key_series(chunk)

        matched = chunk.loc[
            chunk["_seed_key"].isin(selected_keys)
        ].copy()

        if not matched.empty:
            result_frames.append(matched)

    if result_frames:
        result_df = pd.concat(
            result_frames,
            ignore_index=True,
        )
    else:
        result_df = pd.DataFrame(
            columns=required_columns + ["_seed_key"]
        )

    return (
        result_df,
        {
            "status": "read_ok",
            "path": str(parquet_path),
            "rows_scanned": int(rows_scanned),
            "rows_found": int(len(result_df)),
            "missing_columns": [],
        },
    )


def compare_probability_tables(
    direct_df: pd.DataFrame,
    stored_df: pd.DataFrame,
    probability_columns: list[str],
    source_name: str,
    tolerance: float = 1e-8,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if direct_df.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(
                [{
                    "source_name": source_name,
                    "status": "direct_predictions_empty",
                    "expected_seed_rows": 0,
                    "matched_rows": 0,
                    "max_abs_delta": np.nan,
                    "mean_abs_delta": np.nan,
                    "tolerance": tolerance,
                }]
            ),
        )

    if stored_df.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(
                [{
                    "source_name": source_name,
                    "status": "stored_predictions_not_found",
                    "expected_seed_rows": int(len(direct_df)),
                    "matched_rows": 0,
                    "max_abs_delta": np.nan,
                    "mean_abs_delta": np.nan,
                    "tolerance": tolerance,
                }]
            ),
        )

    assert_unique_keys(direct_df, f"{source_name} direct")
    assert_unique_keys(stored_df, f"{source_name} stored")

    merged = direct_df.merge(
        stored_df,
        on=SEED_KEY_COLUMNS,
        how="inner",
        suffixes=("_direct", "_stored"),
        validate="one_to_one",
    )

    if merged.empty:
        return (
            merged,
            pd.DataFrame(
                [{
                    "source_name": source_name,
                    "status": "no_matching_release_rows",
                    "expected_seed_rows": int(len(direct_df)),
                    "matched_rows": 0,
                    "max_abs_delta": np.nan,
                    "mean_abs_delta": np.nan,
                    "tolerance": tolerance,
                }]
            ),
        )

    delta_columns: list[str] = []

    for probability_column in probability_columns:
        direct_column = f"{probability_column}_direct"
        stored_column = f"{probability_column}_stored"
        delta_column = f"{probability_column}_abs_delta"

        if (
            direct_column not in merged.columns
            or stored_column not in merged.columns
        ):
            raise KeyError(
                f"Parity comparison missing expected columns for "
                f"{probability_column}."
            )

        merged[delta_column] = (
            merged[direct_column]
            - merged[stored_column]
        ).abs()

        delta_columns.append(delta_column)

    delta_values = merged[delta_columns].to_numpy(
        dtype=float
    )

    max_delta = float(np.nanmax(delta_values))
    mean_delta = float(np.nanmean(delta_values))

    if len(merged) != len(direct_df):
        status = "partial_release_match"
    elif max_delta <= tolerance:
        status = "parity_passed"
    else:
        status = "parity_difference_detected"

    summary = pd.DataFrame(
        [{
            "source_name": source_name,
            "status": status,
            "expected_seed_rows": int(len(direct_df)),
            "matched_rows": int(len(merged)),
            "max_abs_delta": max_delta,
            "mean_abs_delta": mean_delta,
            "tolerance": tolerance,
        }]
    )

    return merged, summary


# -----------------------------------------------------------------------------
# 3. Read exact feature contracts and fusion weights
# -----------------------------------------------------------------------------
target_feature_lists: dict[str, list[str]] = {}
target_label_columns: dict[str, str] = {}

contract_records: list[dict[str, Any]] = []

for target_key in TARGET_KEYS:
    feature_contract = read_json(
        FEATURE_CONTRACT_PATHS[target_key]
    )

    target_contract = read_json(
        TARGET_CONTRACT_PATHS[target_key]
    )

    feature_list = extract_feature_list(
        feature_contract
    )

    label_column = extract_label_column(
        target_key,
        target_contract,
    )

    target_feature_lists[target_key] = feature_list
    target_label_columns[target_key] = label_column

    contract_records.append(
        {
            "target_key": target_key,
            "feature_count": len(feature_list),
            "label_column": label_column,
        }
    )

fusion_rules = read_json(FUSION_RULES_PATH)

raw_weights = fusion_rules.get("target_weights")

if not isinstance(raw_weights, dict):
    raise KeyError(
        "fusion_rules.json has no target_weights dictionary."
    )

fusion_weights = {
    target_key: float(raw_weights[target_key])
    for target_key in TARGET_KEYS
}

if not np.isclose(
    sum(fusion_weights.values()),
    1.0,
    atol=1e-12,
):
    raise AssertionError(
        "Fusion weights do not sum to 1.0:\n"
        f"{fusion_weights}"
    )

contract_summary_df = pd.DataFrame(contract_records)

print("\n=== Target contracts ===")
print(contract_summary_df.to_string(index=False))

print("\n=== Fusion weights ===")
print(json.dumps(fusion_weights, indent=2))


# -----------------------------------------------------------------------------
# 4. Confirm seed rows and build exact model input matrices
# -----------------------------------------------------------------------------
required_seed_columns = (
    SEED_KEY_COLUMNS
    + ["seed_case", "seed_rank", "__source_record_id"]
    + list(target_label_columns.values())
)

missing_seed_columns = [
    column
    for column in required_seed_columns
    if column not in seed_rows_df.columns
]

if missing_seed_columns:
    raise AssertionError(
        "Cell 2 seed rows are missing required columns:\n"
        f"{missing_seed_columns}"
    )

seed_metadata_df = normalize_keys(
    seed_rows_df[required_seed_columns].copy()
)

assert_unique_keys(
    seed_metadata_df,
    "seed metadata",
)

seed_metadata_df = (
    seed_metadata_df
    .sort_values(["seed_case", "seed_rank"])
    .reset_index(drop=True)
)

target_input_frames: dict[str, pd.DataFrame] = {}
feature_matrix_records: list[dict[str, Any]] = []

for target_key in TARGET_KEYS:
    feature_list = target_feature_lists[target_key]

    missing_features = [
        feature
        for feature in feature_list
        if feature not in seed_rows_df.columns
    ]

    if missing_features:
        raise AssertionError(
            f"{target_key} is missing contract feature columns:\n"
            f"{missing_features}"
        )

    x_seed = seed_rows_df[feature_list].copy()

    for column in x_seed.columns:
        if not pd.api.types.is_numeric_dtype(
            x_seed[column]
        ):
            x_seed[column] = pd.to_numeric(
                x_seed[column],
                errors="raise",
            )

    if list(x_seed.columns) != feature_list:
        raise AssertionError(
            f"{target_key} input feature order differs from "
            "its feature contract."
        )

    target_input_frames[target_key] = x_seed

    feature_matrix_records.append(
        {
            "target_key": target_key,
            "feature_count": len(feature_list),
            "seed_rows": len(x_seed),
            "missing_feature_values": int(
                x_seed.isna().sum().sum()
            ),
            "feature_order_matches_contract": True,
        }
    )

feature_matrix_df = pd.DataFrame(
    feature_matrix_records
)

print("\n=== Seed matrices ===")
print(feature_matrix_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 5. Load models and create direct target predictions
# -----------------------------------------------------------------------------
target_probability_arrays: dict[str, np.ndarray] = {}
target_prediction_frames: list[pd.DataFrame] = []
model_status_records: list[dict[str, Any]] = []

for target_key in TARGET_KEYS:
    model = joblib.load(MODEL_PATHS[target_key])

    probabilities_raw = np.asarray(
        model.predict_proba(
            target_input_frames[target_key]
        )
    )

    model_classes = np.asarray(
        getattr(
            model,
            "classes_",
            np.arange(probabilities_raw.shape[1]),
        )
    )

    if probabilities_raw.ndim != 2:
        raise AssertionError(
            f"{target_key} predict_proba output is not 2D."
        )

    if probabilities_raw.shape[1] != len(model_classes):
        raise AssertionError(
            f"{target_key} probability columns do not match "
            "model classes."
        )

    aligned_probabilities = np.zeros(
        (len(seed_metadata_df), NUM_CLASSES),
        dtype=float,
    )

    for model_column_index, class_value in enumerate(
        model_classes
    ):
        class_value = int(class_value)

        if class_value not in CLASS_VALUES:
            raise AssertionError(
                f"{target_key} produced unsupported class "
                f"{class_value}."
            )

        aligned_probabilities[:, class_value] = (
            probabilities_raw[:, model_column_index]
        )

    probability_sums = aligned_probabilities.sum(axis=1)

    if not np.allclose(
        probability_sums,
        1.0,
        atol=1e-6,
        rtol=0.0,
    ):
        raise AssertionError(
            f"{target_key} probabilities do not sum to 1.0."
        )

    target_probability_arrays[target_key] = (
        aligned_probabilities
    )

    label_column = target_label_columns[target_key]

    target_frame = seed_metadata_df[
        SEED_KEY_COLUMNS
        + [
            "seed_case",
            "seed_rank",
            "__source_record_id",
            label_column,
        ]
    ].copy()

    target_frame = target_frame.rename(
        columns={label_column: "true_class"}
    )

    target_frame["target_key"] = target_key
    target_frame["predicted_class"] = (
        aligned_probabilities.argmax(axis=1)
    )
    target_frame["confidence"] = (
        aligned_probabilities.max(axis=1)
    )
    target_frame["expected_severity"] = (
        aligned_probabilities @ CLASS_VALUES
    )
    target_frame["probability_sum"] = probability_sums
    target_frame["prediction_matches_label"] = (
        target_frame["predicted_class"]
        == target_frame["true_class"]
    )
    target_frame["class_error"] = (
        target_frame["predicted_class"]
        - target_frame["true_class"]
    )

    for class_value in CLASS_VALUES:
        target_frame[f"p{class_value}"] = (
            aligned_probabilities[:, class_value]
        )

    target_prediction_frames.append(target_frame)

    model_status_records.append(
        {
            "target_key": target_key,
            "model_path": str(MODEL_PATHS[target_key]),
            "model_type": type(model).__name__,
            "model_classes": ", ".join(
                str(int(value))
                for value in model_classes
            ),
            "prediction_rows": len(target_frame),
            "probability_sum_min": float(
                probability_sums.min()
            ),
            "probability_sum_max": float(
                probability_sums.max()
            ),
        }
    )

target_seed_predictions_df = pd.concat(
    target_prediction_frames,
    ignore_index=True,
)

model_status_df = pd.DataFrame(
    model_status_records
)

print("\n=== Model prediction status ===")
print(model_status_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 6. Build direct target probability table by keys, never position
# -----------------------------------------------------------------------------
direct_target_probability_df = seed_metadata_df[
    SEED_KEY_COLUMNS
].copy()

for target_key in TARGET_KEYS:
    target_probability_df = target_seed_predictions_df.loc[
        target_seed_predictions_df["target_key"].eq(target_key),
        SEED_KEY_COLUMNS
        + [
            f"p{class_value}"
            for class_value in CLASS_VALUES
        ],
    ].copy()

    target_probability_df = target_probability_df.rename(
        columns={
            f"p{class_value}": (
                f"{target_key}_p{class_value}"
            )
            for class_value in CLASS_VALUES
        }
    )

    assert_unique_keys(
        target_probability_df,
        f"{target_key} direct predictions",
    )

    direct_target_probability_df = direct_target_probability_df.merge(
        target_probability_df,
        on=SEED_KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )

target_probability_columns = [
    f"{target_key}_p{class_value}"
    for target_key in TARGET_KEYS
    for class_value in CLASS_VALUES
]

missing_direct_target_probabilities = [
    column
    for column in target_probability_columns
    if column not in direct_target_probability_df.columns
]

if missing_direct_target_probabilities:
    raise AssertionError(
        "Direct target probability table is incomplete:\n"
        f"{missing_direct_target_probabilities}"
    )

if direct_target_probability_df[
    target_probability_columns
].isna().any().any():
    raise AssertionError(
        "One or more direct target probabilities were lost during "
        "keyed table construction."
    )


# -----------------------------------------------------------------------------
# 7. Build weighted-fusion probabilities with release column names
# -----------------------------------------------------------------------------
weighted_fusion_probabilities = np.zeros(
    (len(seed_metadata_df), NUM_CLASSES),
    dtype=float,
)

for target_key in TARGET_KEYS:
    weighted_fusion_probabilities += (
        fusion_weights[target_key]
        * target_probability_arrays[target_key]
    )

weighted_fusion_probability_sums = (
    weighted_fusion_probabilities.sum(axis=1)
)

if not np.allclose(
    weighted_fusion_probability_sums,
    1.0,
    atol=1e-6,
    rtol=0.0,
):
    raise AssertionError(
        "Weighted fusion probabilities do not sum to 1.0."
    )

weighted_fusion_seed_predictions_df = seed_metadata_df[
    SEED_KEY_COLUMNS
    + [
        "seed_case",
        "seed_rank",
        "__source_record_id",
    ]
].copy()

weighted_fusion_seed_predictions_df[
    "weighted_fusion_predicted_class"
] = weighted_fusion_probabilities.argmax(axis=1)

weighted_fusion_seed_predictions_df[
    "weighted_fusion_confidence"
] = weighted_fusion_probabilities.max(axis=1)

weighted_fusion_seed_predictions_df[
    "weighted_fusion_expected_severity"
] = weighted_fusion_probabilities @ CLASS_VALUES

weighted_fusion_seed_predictions_df[
    "weighted_fusion_probability_sum"
] = weighted_fusion_probability_sums

for class_value in CLASS_VALUES:
    weighted_fusion_seed_predictions_df[
        f"fusion_v2_p{class_value}"
    ] = weighted_fusion_probabilities[:, class_value]

fusion_probability_columns = [
    f"fusion_v2_p{class_value}"
    for class_value in CLASS_VALUES
]

print("\n=== Weighted fusion seed predictions ===")
print(
    weighted_fusion_seed_predictions_df.to_string(
        index=False,
        max_colwidth=120,
    )
)


# -----------------------------------------------------------------------------
# 8. Seed-direction diagnostics
# -----------------------------------------------------------------------------
expected_severity_wide_df = (
    target_seed_predictions_df.pivot(
        index=SEED_KEY_COLUMNS
        + [
            "seed_case",
            "seed_rank",
            "__source_record_id",
        ],
        columns="target_key",
        values="expected_severity",
    )
    .reset_index()
    .rename(
        columns={
            "spi3": "spi3_expected_severity",
            "spi6": "spi6_expected_severity",
            "spi12": "spi12_expected_severity",
        }
    )
)

seed_direction_df = expected_severity_wide_df.merge(
    weighted_fusion_seed_predictions_df[
        SEED_KEY_COLUMNS
        + [
            "weighted_fusion_predicted_class",
            "weighted_fusion_confidence",
            "weighted_fusion_expected_severity",
        ]
    ],
    on=SEED_KEY_COLUMNS,
    how="left",
    validate="one_to_one",
)

expected_horizon_map = {
    "observed_short_term_drought": "spi3",
    "observed_seasonal_drought": "spi6",
    "observed_persistent_drought": "spi12",
}

horizon_columns = {
    "spi3": "spi3_expected_severity",
    "spi6": "spi6_expected_severity",
    "spi12": "spi12_expected_severity",
}

direction_records: list[dict[str, Any]] = []

for _, seed_row in seed_direction_df.iterrows():
    horizon_scores = {
        horizon: float(seed_row[column])
        for horizon, column in horizon_columns.items()
    }

    dominant_horizon = max(
        horizon_scores,
        key=horizon_scores.get,
    )

    seed_case = seed_row["seed_case"]

    if seed_case == "observed_normal_all_horizons":
        result = (
            "normal_signal_low"
            if (
                max(horizon_scores.values()) <= 1.0
                and float(
                    seed_row[
                        "weighted_fusion_expected_severity"
                    ]
                ) <= 1.0
            )
            else "normal_signal_not_low"
        )

    elif seed_case == "observed_multi_horizon_drought":
        result = (
            "multi_horizon_drought_signal"
            if min(horizon_scores.values()) >= 1.0
            else "multi_horizon_signal_mixed"
        )

    else:
        expected_horizon = expected_horizon_map[seed_case]

        result = (
            "expected_horizon_dominant"
            if dominant_horizon == expected_horizon
            else "different_horizon_dominant"
        )

    direction_records.append(
        {
            "seed_case": seed_case,
            "seed_rank": int(seed_row["seed_rank"]),
            "source_record_id": int(
                seed_row["__source_record_id"]
            ),
            "row": int(seed_row["row"]),
            "col": int(seed_row["col"]),
            "yyyymm": str(seed_row["yyyymm"]),
            "spi3_expected_severity": horizon_scores["spi3"],
            "spi6_expected_severity": horizon_scores["spi6"],
            "spi12_expected_severity": horizon_scores["spi12"],
            "weighted_fusion_expected_severity": float(
                seed_row[
                    "weighted_fusion_expected_severity"
                ]
            ),
            "dominant_target_horizon": dominant_horizon,
            "directional_result": result,
        }
    )

seed_direction_results_df = pd.DataFrame(
    direction_records
)

seed_direction_summary_df = (
    seed_direction_results_df.groupby(
        ["seed_case", "directional_result"],
        dropna=False,
    )
    .size()
    .reset_index(name="row_count")
    .sort_values(["seed_case", "directional_result"])
    .reset_index(drop=True)
)

print("\n=== Seed-direction summary ===")
print(seed_direction_summary_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 9. Read stored release predictions for exact keyed parity checks
# -----------------------------------------------------------------------------
selected_seed_keys = set(
    key_series(seed_metadata_df).tolist()
)

stored_target_required_columns = (
    SEED_KEY_COLUMNS
    + target_probability_columns
)

stored_fusion_required_columns = (
    SEED_KEY_COLUMNS
    + fusion_probability_columns
)

stored_target_df, target_scan_status = scan_release_rows(
    parquet_path=TARGET_PREDICTIONS_PATH,
    required_columns=stored_target_required_columns,
    selected_keys=selected_seed_keys,
)

stored_fusion_df, fusion_scan_status = scan_release_rows(
    parquet_path=FUSION_PREDICTIONS_PATH,
    required_columns=stored_fusion_required_columns,
    selected_keys=selected_seed_keys,
)

release_scan_status_df = pd.DataFrame(
    [
        {
            "source": "target_predictions",
            **target_scan_status,
        },
        {
            "source": "fusion_predictions",
            **fusion_scan_status,
        },
    ]
)

print("\n=== Stored-release scan ===")
print(
    release_scan_status_df.to_string(
        index=False,
        max_colwidth=160,
    )
)


# -----------------------------------------------------------------------------
# 10. Direct target parity and weighted-fusion diagnostic parity
# -----------------------------------------------------------------------------
target_parity_detail_df, target_parity_summary_df = (
    compare_probability_tables(
        direct_df=direct_target_probability_df,
        stored_df=stored_target_df,
        probability_columns=target_probability_columns,
        source_name="target_models_direct_vs_release",
    )
)

weighted_fusion_parity_detail_df, weighted_fusion_parity_summary_df = (
    compare_probability_tables(
        direct_df=weighted_fusion_seed_predictions_df[
            SEED_KEY_COLUMNS
            + fusion_probability_columns
        ],
        stored_df=stored_fusion_df,
        probability_columns=fusion_probability_columns,
        source_name="weighted_fusion_vs_release_fusion",
    )
)

parity_summary_df = pd.concat(
    [
        target_parity_summary_df,
        weighted_fusion_parity_summary_df,
    ],
    ignore_index=True,
)

print("\n=== Prediction parity summary ===")
print(parity_summary_df.to_string(index=False))

if not target_parity_detail_df.empty:
    target_delta_columns = [
        column
        for column in target_parity_detail_df.columns
        if column.endswith("_abs_delta")
    ]

    print("\n=== Target-model parity detail ===")
    print(
        target_parity_detail_df[
            SEED_KEY_COLUMNS
            + target_delta_columns
        ].to_string(
            index=False,
            max_colwidth=120,
        )
    )

if not weighted_fusion_parity_detail_df.empty:
    fusion_delta_columns = [
        column
        for column in weighted_fusion_parity_detail_df.columns
        if column.endswith("_abs_delta")
    ]

    print("\n=== Weighted-fusion parity detail ===")
    print(
        weighted_fusion_parity_detail_df[
            SEED_KEY_COLUMNS
            + fusion_delta_columns
        ].to_string(
            index=False,
            max_colwidth=120,
        )
    )


# -----------------------------------------------------------------------------
# 11. Target label diagnostics
# -----------------------------------------------------------------------------
target_label_summary_df = (
    target_seed_predictions_df.groupby(
        "target_key",
        dropna=False,
    )
    .agg(
        seed_rows=("target_key", "size"),
        exact_label_match_rate=(
            "prediction_matches_label",
            "mean",
        ),
        mean_absolute_class_error=(
            "class_error",
            lambda values: float(
                np.mean(np.abs(values))
            ),
        ),
        mean_confidence=("confidence", "mean"),
        mean_true_class_probability=(
            "true_class",
            lambda values: np.nan,
        ),
        minimum_probability_sum=(
            "probability_sum",
            "min",
        ),
        maximum_probability_sum=(
            "probability_sum",
            "max",
        ),
    )
    .reset_index()
)

true_probability_records: list[dict[str, Any]] = []

for _, prediction_row in target_seed_predictions_df.iterrows():
    true_class = int(prediction_row["true_class"])

    true_probability_records.append(
        {
            "target_key": prediction_row["target_key"],
            "true_class_probability": float(
                prediction_row[f"p{true_class}"]
            ),
        }
    )

true_probability_df = pd.DataFrame(
    true_probability_records
)

mean_true_probability_df = (
    true_probability_df.groupby("target_key")
    .agg(
        mean_true_class_probability=(
            "true_class_probability",
            "mean",
        )
    )
    .reset_index()
)

target_label_summary_df = (
    target_label_summary_df.drop(
        columns=["mean_true_class_probability"]
    )
    .merge(
        mean_true_probability_df,
        on="target_key",
        how="left",
        validate="one_to_one",
    )
)

print("\n=== Target label diagnostics ===")
print(target_label_summary_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 12. Final technical status
# -----------------------------------------------------------------------------
target_parity_status = parity_summary_df.loc[
    parity_summary_df["source_name"].eq(
        "target_models_direct_vs_release"
    ),
    "status",
]

weighted_fusion_parity_status = parity_summary_df.loc[
    parity_summary_df["source_name"].eq(
        "weighted_fusion_vs_release_fusion"
    ),
    "status",
]

target_parity_passed = bool(
    len(target_parity_status) == 1
    and target_parity_status.iloc[0] == "parity_passed"
)

weighted_fusion_parity_passed = bool(
    len(weighted_fusion_parity_status) == 1
    and weighted_fusion_parity_status.iloc[0] == "parity_passed"
)

technical_checks = {
    "all_target_probability_sums_valid": bool(
        np.allclose(
            target_seed_predictions_df[
                "probability_sum"
            ].to_numpy(dtype=float),
            1.0,
            atol=1e-6,
            rtol=0.0,
        )
    ),
    "weighted_fusion_probability_sums_valid": bool(
        np.allclose(
            weighted_fusion_seed_predictions_df[
                "weighted_fusion_probability_sum"
            ].to_numpy(dtype=float),
            1.0,
            atol=1e-6,
            rtol=0.0,
        )
    ),
    "all_contract_feature_orders_valid": bool(
        feature_matrix_df[
            "feature_order_matches_contract"
        ].all()
    ),
    "target_model_release_parity_passed": (
        target_parity_passed
    ),
    "weighted_fusion_matches_release_exactly": (
        weighted_fusion_parity_passed
    ),
}

if (
    technical_checks[
        "all_target_probability_sums_valid"
    ]
    and technical_checks[
        "weighted_fusion_probability_sums_valid"
    ]
    and technical_checks[
        "all_contract_feature_orders_valid"
    ]
    and technical_checks[
        "target_model_release_parity_passed"
    ]
):
    CELL3_STATUS = (
        "target_model_seed_smoke_test_passed"
    )
else:
    CELL3_STATUS = (
        "target_model_seed_smoke_test_completed_with_diagnostics"
    )

print("\n=== Cell 3 conclusion ===")
print("Status:", CELL3_STATUS)

print("\nTechnical checks:")
for check_name, check_value in technical_checks.items():
    print(f"- {check_name}: {check_value}")

print(
    "\nInterpretation:\n"
    "- Target parity is the hard technical check.\n"
    "- Weighted-fusion parity is diagnostic only until the complete "
    "release fusion-rule logic is reproduced.\n"
    "- Seed label agreement is not an independent accuracy result."
)


# -----------------------------------------------------------------------------
# 13. Save complete Cell 3 outputs
# -----------------------------------------------------------------------------
contract_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell3_contract_summary_{CELL3_RUN_ID}.csv"
)

feature_matrix_path = (
    SEED_OUTPUT_DIR
    / f"cell3_feature_matrix_validation_{CELL3_RUN_ID}.csv"
)

model_status_path = (
    SEED_OUTPUT_DIR
    / f"cell3_model_status_{CELL3_RUN_ID}.csv"
)

target_predictions_path = (
    SEED_OUTPUT_DIR
    / f"cell3_target_seed_predictions_{CELL3_RUN_ID}.csv"
)

weighted_fusion_path = (
    SEED_OUTPUT_DIR
    / f"cell3_weighted_fusion_seed_predictions_{CELL3_RUN_ID}.csv"
)

direction_results_path = (
    SEED_OUTPUT_DIR
    / f"cell3_seed_direction_results_{CELL3_RUN_ID}.csv"
)

direction_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell3_seed_direction_summary_{CELL3_RUN_ID}.csv"
)

release_scan_path = (
    SEED_OUTPUT_DIR
    / f"cell3_release_scan_status_{CELL3_RUN_ID}.csv"
)

parity_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell3_prediction_parity_summary_{CELL3_RUN_ID}.csv"
)

target_parity_detail_path = (
    SEED_OUTPUT_DIR
    / f"cell3_target_parity_detail_{CELL3_RUN_ID}.csv"
)

weighted_fusion_parity_detail_path = (
    SEED_OUTPUT_DIR
    / f"cell3_weighted_fusion_parity_detail_{CELL3_RUN_ID}.csv"
)

label_summary_path = (
    SEED_OUTPUT_DIR
    / f"cell3_target_label_diagnostics_{CELL3_RUN_ID}.csv"
)

manifest_path = (
    SEED_OUTPUT_DIR
    / f"cell3_seed_smoke_test_manifest_{CELL3_RUN_ID}.json"
)

contract_summary_df.to_csv(
    contract_summary_path,
    index=False,
)

feature_matrix_df.to_csv(
    feature_matrix_path,
    index=False,
)

model_status_df.to_csv(
    model_status_path,
    index=False,
)

target_seed_predictions_df.to_csv(
    target_predictions_path,
    index=False,
)

weighted_fusion_seed_predictions_df.to_csv(
    weighted_fusion_path,
    index=False,
)

seed_direction_results_df.to_csv(
    direction_results_path,
    index=False,
)

seed_direction_summary_df.to_csv(
    direction_summary_path,
    index=False,
)

release_scan_status_df.to_csv(
    release_scan_path,
    index=False,
)

parity_summary_df.to_csv(
    parity_summary_path,
    index=False,
)

target_parity_detail_df.to_csv(
    target_parity_detail_path,
    index=False,
)

weighted_fusion_parity_detail_df.to_csv(
    weighted_fusion_parity_detail_path,
    index=False,
)

target_label_summary_df.to_csv(
    label_summary_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot serialize {type(value).__name__} to JSON."
    )


manifest = {
    "run_id": CELL3_RUN_ID,
    "status": CELL3_STATUS,
    "purpose": (
        "Observed-seed technical smoke test for SPI3, SPI6, SPI12, "
        "and weighted Fusion v2 baseline."
    ),
    "seed_row_count": int(len(seed_metadata_df)),
    "seed_case_count": int(
        seed_metadata_df["seed_case"].nunique()
    ),
    "target_models": {
        target_key: str(MODEL_PATHS[target_key])
        for target_key in TARGET_KEYS
    },
    "target_feature_contracts": {
        target_key: str(FEATURE_CONTRACT_PATHS[target_key])
        for target_key in TARGET_KEYS
    },
    "fusion_rules_path": str(FUSION_RULES_PATH),
    "fusion_weights": fusion_weights,
    "technical_checks": technical_checks,
    "target_parity_summary": (
        target_parity_summary_df.to_dict(
            orient="records"
        )
    ),
    "weighted_fusion_parity_summary": (
        weighted_fusion_parity_summary_df.to_dict(
            orient="records"
        )
    ),
    "outputs": {
        "contract_summary_csv": str(contract_summary_path),
        "feature_matrix_csv": str(feature_matrix_path),
        "model_status_csv": str(model_status_path),
        "target_predictions_csv": str(target_predictions_path),
        "weighted_fusion_predictions_csv": str(
            weighted_fusion_path
        ),
        "seed_direction_results_csv": str(
            direction_results_path
        ),
        "seed_direction_summary_csv": str(
            direction_summary_path
        ),
        "release_scan_csv": str(release_scan_path),
        "parity_summary_csv": str(parity_summary_path),
        "target_parity_detail_csv": str(
            target_parity_detail_path
        ),
        "weighted_fusion_parity_detail_csv": str(
            weighted_fusion_parity_detail_path
        ),
        "label_diagnostics_csv": str(label_summary_path),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 3 outputs ===")
print("Contract summary           ->", contract_summary_path)
print("Feature-matrix validation  ->", feature_matrix_path)
print("Model status               ->", model_status_path)
print("Target predictions         ->", target_predictions_path)
print("Weighted fusion predictions->", weighted_fusion_path)
print("Seed directions            ->", direction_results_path)
print("Direction summary          ->", direction_summary_path)
print("Release scan               ->", release_scan_path)
print("Parity summary             ->", parity_summary_path)
print("Target parity detail       ->", target_parity_detail_path)
print("Weighted fusion parity     ->", weighted_fusion_parity_detail_path)
print("Label diagnostics          ->", label_summary_path)
print("Manifest                   ->", manifest_path)

print("\nCell 3 complete.")

=== Cell 3 purpose ===
Run staged v2 target models on observed seed rows and validate direct prediction parity.

=== Paths ===
Artifact root       : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts
Target predictions : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_target_predictions_20260618T152729Z.parquet
Fusion predictions : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_fusion_v2_predictions_20260618T185446Z.parquet
Seed output dir    : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\seed_cases
Run ID             : 20260620T105208Z

Using seed_rows_df already in memory.
Seed rows: 13

=== Target contracts ===
target_key  feature_count        label_column
      spi3             23  drought_class_spi3
      spi6      

In [6]:
# validate_events_ng.ipynb — Cell 4
# Purpose:
# - Prepare real-event testing from the v2 release feature frame.
# - Confirm release-month availability for documented drought windows.
# - Verify that release-frame row/col cells map cleanly to lon/lat.
# - Create a reusable release-grid coordinate lookup for later ADM/polygon tests.
#
# Scope:
# - Input/readiness and spatial-key validation only.
# - No model inference in this cell.
# - No API sourcing or feature rebuilding.
# - Initial event windows are broad timeline-year windows and will be
#   refined before final event-performance interpretation.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Required prior-cell objects and confirmed paths
# -----------------------------------------------------------------------------
REQUIRED_OBJECTS = [
    "master_df",
    "DATA_ROOT",
    "OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in REQUIRED_OBJECTS
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Cell 4 requires Cell 0 to have completed in this kernel.\n"
        f"Missing objects: {missing_objects}"
    )

CELL4_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

RELEASE_FEATURE_FRAME_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
    / "release_bundle_prediction_feature_frame_20260618T152226Z.parquet"
)

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RELEASE_KEY_COLUMNS = ["row", "col", "yyyymm"]
MASTER_COORDINATE_COLUMNS = ["row", "col", "lon", "lat"]

print("=== Cell 4 purpose ===")
print(
    "Prepare release-feature inputs and spatial lookup for "
    "real drought-event testing."
)

print("\n=== Paths ===")
print("Release feature frame:", RELEASE_FEATURE_FRAME_PATH)
print("Event output directory:", EVENT_OUTPUT_DIR)
print("Run ID:", CELL4_RUN_ID)

if not RELEASE_FEATURE_FRAME_PATH.exists():
    raise FileNotFoundError(
        "Release feature frame not found:\n"
        f"{RELEASE_FEATURE_FRAME_PATH}"
    )


# -----------------------------------------------------------------------------
# 2. Initial documented event windows
# -----------------------------------------------------------------------------
# These are broad year-range windows from the drought timeline.
# They are used here only to assess release-data availability.
# Final event interpretation will narrow timing where evidence supports it.
INITIAL_EVENT_WINDOWS = [
    {
        "event_id": "zim_2015_2016",
        "event_name": "Zimbabwe drought 2015–2016",
        "reported_severity": "extreme",
        "start_yyyymm": "201501",
        "end_yyyymm": "201612",
        "initial_geography": "national",
        "window_definition": "timeline_year_range_unrefined",
    },
    {
        "event_id": "zim_2018_2019",
        "event_name": "Zimbabwe drought 2018–2019",
        "reported_severity": "moderate",
        "start_yyyymm": "201801",
        "end_yyyymm": "201912",
        "initial_geography": "national",
        "window_definition": "timeline_year_range_unrefined",
    },
    {
        "event_id": "zim_2023_2024",
        "event_name": "Zimbabwe drought 2023–2024",
        "reported_severity": "catastrophic",
        "start_yyyymm": "202301",
        "end_yyyymm": "202412",
        "initial_geography": "national",
        "window_definition": "timeline_year_range_unrefined",
    },
]

event_windows_df = pd.DataFrame(INITIAL_EVENT_WINDOWS)

print("\n=== Initial event windows ===")
print(event_windows_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 3. Load only release-frame spatial/time keys
# -----------------------------------------------------------------------------
print("\nLoading release row/col/month index only...")

release_index_df = pd.read_parquet(
    RELEASE_FEATURE_FRAME_PATH,
    columns=RELEASE_KEY_COLUMNS,
)

release_index_df["row"] = pd.to_numeric(
    release_index_df["row"],
    errors="raise",
).astype("int64")

release_index_df["col"] = pd.to_numeric(
    release_index_df["col"],
    errors="raise",
).astype("int64")

release_index_df["yyyymm"] = (
    release_index_df["yyyymm"]
    .astype(str)
)

invalid_month_mask = (
    ~release_index_df["yyyymm"].str.fullmatch(r"\d{6}")
    | ~pd.to_numeric(
        release_index_df["yyyymm"].str[-2:],
        errors="coerce",
    ).between(1, 12)
)

invalid_month_count = int(invalid_month_mask.sum())

if invalid_month_count:
    raise AssertionError(
        "Release feature frame has invalid yyyymm values.\n"
        f"Invalid rows: {invalid_month_count}"
    )

release_duplicate_key_count = int(
    release_index_df.duplicated(
        subset=RELEASE_KEY_COLUMNS
    ).sum()
)

if release_duplicate_key_count:
    raise AssertionError(
        "Release feature frame has duplicate row/col/month keys.\n"
        f"Duplicate rows: {release_duplicate_key_count}"
    )

release_grid_df = (
    release_index_df[["row", "col"]]
    .drop_duplicates()
    .sort_values(["row", "col"])
    .reset_index(drop=True)
)

release_month_summary_df = (
    release_index_df.groupby("yyyymm")
    .agg(
        release_rows=("yyyymm", "size"),
        release_cells=("row", "size"),
        unique_rows=("row", "nunique"),
        unique_cols=("col", "nunique"),
    )
    .reset_index()
    .sort_values("yyyymm")
    .reset_index(drop=True)
)

release_months = set(
    release_month_summary_df["yyyymm"].tolist()
)

print("\n=== Release feature-frame temporal coverage ===")
print("Release rows :", f"{len(release_index_df):,}")
print("Release cells:", f"{len(release_grid_df):,}")
print(
    "First month  :",
    release_month_summary_df["yyyymm"].min(),
)
print(
    "Last month   :",
    release_month_summary_df["yyyymm"].max(),
)
print(
    "Month count  :",
    len(release_month_summary_df),
)
print(
    "Duplicate row/col/month keys:",
    release_duplicate_key_count,
)

print("\nMonthly release-grid summary:")
print(
    release_month_summary_df.to_string(
        index=False,
        max_rows=100,
    )
)


# -----------------------------------------------------------------------------
# 4. Build coordinate lookup from the already-audited master_df
# -----------------------------------------------------------------------------
# master_df is used only as a coordinate lookup.
# It is NOT used as v2 model input.

MASTER_LOOKUP_MONTH = "198001"

master_lookup_df = (
    master_df.loc[
        master_df["scenario"].eq("historical")
        & master_df["yyyymm"].eq(MASTER_LOOKUP_MONTH),
        MASTER_COORDINATE_COLUMNS,
    ]
    .drop_duplicates()
    .copy()
)

if master_lookup_df.empty:
    raise AssertionError(
        "Could not extract coordinate lookup rows from master_df "
        f"for historical month {MASTER_LOOKUP_MONTH}."
    )

master_lookup_df["row"] = pd.to_numeric(
    master_lookup_df["row"],
    errors="raise",
).astype("int64")

master_lookup_df["col"] = pd.to_numeric(
    master_lookup_df["col"],
    errors="raise",
).astype("int64")

master_coordinate_duplicate_count = int(
    master_lookup_df.duplicated(
        subset=["row", "col"]
    ).sum()
)

if master_coordinate_duplicate_count:
    raise AssertionError(
        "master_df has duplicate row/col coordinate mappings.\n"
        f"Duplicate mappings: {master_coordinate_duplicate_count}"
    )

master_grid_count = int(len(master_lookup_df))

release_coordinate_lookup_df = release_grid_df.merge(
    master_lookup_df,
    on=["row", "col"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

unmatched_release_cells_df = release_coordinate_lookup_df.loc[
    release_coordinate_lookup_df["_merge"].ne("both")
].copy()

release_coordinate_lookup_df = (
    release_coordinate_lookup_df
    .drop(columns="_merge")
    .sort_values(["row", "col"])
    .reset_index(drop=True)
)

matched_release_cells = int(
    len(release_coordinate_lookup_df)
    - len(unmatched_release_cells_df)
)

coordinate_lookup_status = (
    "release_grid_fully_matched_to_master_coordinates"
    if unmatched_release_cells_df.empty
    else "release_grid_has_unmatched_master_coordinates"
)

print("\n=== Release grid vs master coordinate lookup ===")
print("Master grid cells         :", f"{master_grid_count:,}")
print("Release grid cells        :", f"{len(release_grid_df):,}")
print("Matched release cells     :", f"{matched_release_cells:,}")
print(
    "Unmatched release cells   :",
    f"{len(unmatched_release_cells_df):,}",
)
print("Coordinate lookup status  :", coordinate_lookup_status)

if not unmatched_release_cells_df.empty:
    print("\nUnmatched release-grid cells:")
    print(
        unmatched_release_cells_df[
            ["row", "col"]
        ]
        .head(30)
        .to_string(index=False)
    )

if not unmatched_release_cells_df.empty:
    raise AssertionError(
        "Stop before event testing: release grid cannot be fully "
        "mapped to master lon/lat coordinates."
    )


# -----------------------------------------------------------------------------
# 5. Build exact month lists and assess event coverage
# -----------------------------------------------------------------------------
def month_range(
    start_yyyymm: str,
    end_yyyymm: str,
) -> list[str]:
    start_period = pd.Period(start_yyyymm, freq="M")
    end_period = pd.Period(end_yyyymm, freq="M")

    return [
        period.strftime("%Y%m")
        for period in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


event_month_records: list[dict[str, Any]] = []
event_coverage_records: list[dict[str, Any]] = []

release_cells_per_month = (
    release_month_summary_df
    .set_index("yyyymm")["release_cells"]
    .to_dict()
)

for event_record in INITIAL_EVENT_WINDOWS:
    expected_months = month_range(
        event_record["start_yyyymm"],
        event_record["end_yyyymm"],
    )

    available_months = [
        month
        for month in expected_months
        if month in release_months
    ]

    missing_months = [
        month
        for month in expected_months
        if month not in release_months
    ]

    full_grid_available_months = [
        month
        for month in available_months
        if release_cells_per_month[month]
        == len(release_grid_df)
    ]

    if len(available_months) == 0:
        readiness_status = (
            "not_available_in_release_feature_frame"
        )
    elif len(available_months) == len(expected_months):
        readiness_status = (
            "full_timeline_window_available"
        )
    else:
        readiness_status = (
            "partial_timeline_window_available"
        )

    event_coverage_records.append(
        {
            **event_record,
            "expected_month_count": len(expected_months),
            "available_month_count": len(available_months),
            "missing_month_count": len(missing_months),
            "full_grid_available_month_count": len(
                full_grid_available_months
            ),
            "first_available_month": (
                min(available_months)
                if available_months
                else None
            ),
            "last_available_month": (
                max(available_months)
                if available_months
                else None
            ),
            "available_months": ", ".join(available_months),
            "missing_months": ", ".join(missing_months),
            "readiness_status": readiness_status,
        }
    )

    for month in expected_months:
        event_month_records.append(
            {
                **event_record,
                "yyyymm": month,
                "release_month_available": month in release_months,
                "release_cell_count": int(
                    release_cells_per_month.get(month, 0)
                ),
                "full_release_grid_available": (
                    release_cells_per_month.get(month, 0)
                    == len(release_grid_df)
                ),
            }
        )

event_month_availability_df = pd.DataFrame(
    event_month_records
)

event_coverage_summary_df = pd.DataFrame(
    event_coverage_records
)

print("\n=== Event-window release availability ===")
print(
    event_coverage_summary_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

print("\n=== Event-month availability ===")
print(
    event_month_availability_df.to_string(
        index=False,
        max_rows=100,
        max_colwidth=140,
    )
)


# -----------------------------------------------------------------------------
# 6. Build available event month plan for the model-run cell
# -----------------------------------------------------------------------------
available_event_month_plan_df = (
    event_month_availability_df.loc[
        event_month_availability_df[
            "release_month_available"
        ]
    ]
    .copy()
    .sort_values(["event_id", "yyyymm"])
    .reset_index(drop=True)
)

ready_event_ids = (
    event_coverage_summary_df.loc[
        event_coverage_summary_df[
            "available_month_count"
        ].gt(0),
        "event_id",
    ]
    .tolist()
)

if not ready_event_ids:
    raise AssertionError(
        "None of the initial drought windows overlap the "
        "release feature frame. Stop before event testing."
    )

print("\n=== Model-run candidate event plan ===")
print(
    available_event_month_plan_df[
        [
            "event_id",
            "event_name",
            "reported_severity",
            "yyyymm",
            "release_cell_count",
            "full_release_grid_available",
        ]
    ].to_string(
        index=False,
        max_rows=100,
    )
)


# -----------------------------------------------------------------------------
# 7. Cell conclusion
# -----------------------------------------------------------------------------
full_window_event_count = int(
    event_coverage_summary_df[
        "readiness_status"
    ]
    .eq("full_timeline_window_available")
    .sum()
)

partial_window_event_count = int(
    event_coverage_summary_df[
        "readiness_status"
    ]
    .eq("partial_timeline_window_available")
    .sum()
)

CELL4_STATUS = (
    "release_event_input_index_ready_for_national_model_runs"
)

print("\n=== Cell 4 conclusion ===")
print("Status:", CELL4_STATUS)
print(
    "Events with full broad-window coverage:",
    full_window_event_count,
)
print(
    "Events with partial broad-window coverage:",
    partial_window_event_count,
)
print(
    "Events with any release-data coverage:",
    len(ready_event_ids),
)

print(
    "\nInterpretation:\n"
    "- The v2 release feature frame is now the direct input source.\n"
    "- master_df contributes only the verified row/col-to-coordinate lookup.\n"
    "- The next cell will run the released models across every available\n"
    "  national-grid row in the first supported event window.\n"
    "- These broad event windows must later be refined before making final\n"
    "  claims about monthly drought onset, peak, or recovery."
)


# -----------------------------------------------------------------------------
# 8. Save Cell 4 outputs
# -----------------------------------------------------------------------------
release_month_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_release_month_summary_{CELL4_RUN_ID}.csv"
)

coordinate_lookup_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_release_grid_coordinate_lookup_{CELL4_RUN_ID}.parquet"
)

event_windows_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_initial_event_windows_{CELL4_RUN_ID}.csv"
)

event_month_availability_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_event_month_availability_{CELL4_RUN_ID}.csv"
)

event_coverage_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_event_coverage_summary_{CELL4_RUN_ID}.csv"
)

available_event_plan_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_available_event_month_plan_{CELL4_RUN_ID}.csv"
)

unmatched_grid_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_unmatched_release_grid_cells_{CELL4_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell4_event_input_readiness_manifest_{CELL4_RUN_ID}.json"
)

release_month_summary_df.to_csv(
    release_month_summary_path,
    index=False,
)

release_coordinate_lookup_df.to_parquet(
    coordinate_lookup_path,
    index=False,
)

event_windows_df.to_csv(
    event_windows_path,
    index=False,
)

event_month_availability_df.to_csv(
    event_month_availability_path,
    index=False,
)

event_coverage_summary_df.to_csv(
    event_coverage_summary_path,
    index=False,
)

available_event_month_plan_df.to_csv(
    available_event_plan_path,
    index=False,
)

unmatched_release_cells_df.to_csv(
    unmatched_grid_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL4_RUN_ID,
    "status": CELL4_STATUS,
    "purpose": (
        "Prepare and validate v2 release-feature inputs for "
        "national drought-event model testing."
    ),
    "release_feature_frame_path": str(
        RELEASE_FEATURE_FRAME_PATH
    ),
    "release_feature_rows": int(len(release_index_df)),
    "release_grid_cell_count": int(
        len(release_grid_df)
    ),
    "master_coordinate_lookup_month": (
        MASTER_LOOKUP_MONTH
    ),
    "master_grid_cell_count": master_grid_count,
    "coordinate_lookup_status": coordinate_lookup_status,
    "unmatched_release_cell_count": int(
        len(unmatched_release_cells_df)
    ),
    "event_window_definition": (
        "Broad timeline-year ranges, not final seasonal-month windows."
    ),
    "event_coverage_summary": (
        event_coverage_summary_df.to_dict(
            orient="records"
        )
    ),
    "outputs": {
        "release_month_summary_csv": str(
            release_month_summary_path
        ),
        "release_grid_coordinate_lookup_parquet": str(
            coordinate_lookup_path
        ),
        "initial_event_windows_csv": str(
            event_windows_path
        ),
        "event_month_availability_csv": str(
            event_month_availability_path
        ),
        "event_coverage_summary_csv": str(
            event_coverage_summary_path
        ),
        "available_event_month_plan_csv": str(
            available_event_plan_path
        ),
        "unmatched_release_grid_cells_csv": str(
            unmatched_grid_path
        ),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 4 outputs ===")
print("Release month summary ->", release_month_summary_path)
print("Coordinate lookup     ->", coordinate_lookup_path)
print("Event windows         ->", event_windows_path)
print("Month availability    ->", event_month_availability_path)
print("Coverage summary      ->", event_coverage_summary_path)
print("Available event plan  ->", available_event_plan_path)
print("Unmatched grid cells  ->", unmatched_grid_path)
print("Readiness manifest    ->", manifest_path)

print("\nCell 4 complete.")

=== Cell 4 purpose ===
Prepare release-feature inputs and spatial lookup for real drought-event testing.

=== Paths ===
Release feature frame: C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_prediction_feature_frame_20260618T152226Z.parquet
Event output directory: C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID: 20260620T111045Z

=== Initial event windows ===
     event_id                 event_name reported_severity start_yyyymm end_yyyymm initial_geography             window_definition
zim_2015_2016 Zimbabwe drought 2015–2016           extreme       201501     201612          national timeline_year_range_unrefined
zim_2018_2019 Zimbabwe drought 2018–2019          moderate       201801     201912          national timeline_year_range_unrefined
zim_2023_2024 Zimbabwe drought 2023–2024      catastrophic       202301     202412          national timel

In [7]:
# validate_events_ng.ipynb — Cell 5
# Purpose:
# - Read the Excel drought-event reference directly from the new workspace.
# - Audit workbook sheets, headers, and candidate event/severity fields.
# - Build a transparent candidate event-month validation ledger.
# - Check every required event month against:
#     1) v2 release feature frame,
#     2) full temporal predictor source,
#     3) full training source.
# - Identify months ready now versus genuinely requiring reconstruction.
#
# Scope:
# - No model inference.
# - No raw-data/API rebuild.
# - No automatic interpretation of Excel severity categories.
# - Any parsed year range remains an auditable candidate until reviewed.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths and workspace setup
# -----------------------------------------------------------------------------
if "DATA_ROOT" not in globals():
    raise RuntimeError(
        "DATA_ROOT is not available. Rerun Cell 0 first."
    )

if "OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "OUTPUT_DIR is not available. Rerun Cell 0 first."
    )

CELL5_RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

EVENT_WORKBOOK_PATH = (
    DATA_ROOT
    / "events"
    / "Zimbabwe_Drought_Timeline_1902_2024.xlsx"
)

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

RELEASE_FEATURE_FRAME_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
    / "release_bundle_prediction_feature_frame_20260618T152226Z.parquet"
)

FULL_PREDICTOR_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "temporal_predictors_full_20260614T141118Z.parquet"
)

FULL_TRAINING_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "training_dataset_temporal_multi_spi_20260614T144111Z.parquet"
)

V2_RELEASE_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

TARGET_KEYS = ["spi3", "spi6", "spi12"]
YEAR_PATTERN = re.compile(r"\b(?:18|19|20)\d{2}\b")

print("=== Cell 5 purpose ===")
print(
    "Audit Excel event records and identify the source status of "
    "every candidate event month."
)

print("\n=== Paths ===")
print("Event workbook         :", EVENT_WORKBOOK_PATH)
print("Release feature frame  :", RELEASE_FEATURE_FRAME_PATH)
print("Full predictor source  :", FULL_PREDICTOR_SOURCE_PATH)
print("Full training source   :", FULL_TRAINING_SOURCE_PATH)
print("Event output directory :", EVENT_OUTPUT_DIR)
print("Run ID                 :", CELL5_RUN_ID)

required_paths = [
    EVENT_WORKBOOK_PATH,
    RELEASE_FEATURE_FRAME_PATH,
    FULL_PREDICTOR_SOURCE_PATH,
    FULL_TRAINING_SOURCE_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Cell 5 input paths are missing:\n"
        + "\n".join(missing_paths)
    )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def cell_text(value: Any) -> str:
    if pd.isna(value):
        return ""

    return str(value).strip()


def normalize_text(value: Any) -> str:
    text = cell_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_years(value: Any) -> list[int]:
    return sorted(
        {
            int(match)
            for match in YEAR_PATTERN.findall(
                cell_text(value)
            )
        }
    )


def safe_column_name(
    value: Any,
    fallback_index: int,
    existing_names: set[str],
) -> str:
    candidate = cell_text(value)

    if not candidate:
        candidate = f"unnamed_column_{fallback_index}"

    base_name = candidate
    suffix = 2

    while candidate in existing_names:
        candidate = f"{base_name}_{suffix}"
        suffix += 1

    existing_names.add(candidate)

    return candidate


def month_range_from_years(
    start_year: int,
    end_year: int,
) -> list[str]:
    start_period = pd.Period(
        f"{start_year}01",
        freq="M",
    )

    end_period = pd.Period(
        f"{end_year}12",
        freq="M",
    )

    return [
        period.strftime("%Y%m")
        for period in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    for key in [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            return list(value)

    raise ValueError(
        "Could not find a model feature list in a feature contract."
    )


def normalize_yyyymm_series(
    values: pd.Series,
) -> pd.Series:
    normalized = (
        values.astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    valid = normalized.str.fullmatch(r"\d{6}")

    if not valid.all():
        invalid_values = (
            normalized.loc[~valid]
            .drop_duplicates()
            .head(20)
            .tolist()
        )

        raise AssertionError(
            "Encountered invalid yyyymm values while scanning "
            "a model-input source:\n"
            f"{invalid_values}"
        )

    return normalized


def collect_parquet_months(
    parquet_path: Path,
    source_name: str,
    cached_months: set[str] | None = None,
    batch_size: int = 250_000,
) -> tuple[set[str], list[str], int]:
    parquet_file = pq.ParquetFile(parquet_path)
    schema_columns = list(parquet_file.schema.names)

    if "yyyymm" not in schema_columns:
        raise KeyError(
            f"{source_name} has no yyyymm column."
        )

    if cached_months is not None:
        return (
            set(cached_months),
            schema_columns,
            int(parquet_file.metadata.num_rows),
        )

    month_values: set[str] = set()

    for batch in parquet_file.iter_batches(
        columns=["yyyymm"],
        batch_size=batch_size,
    ):
        month_series = normalize_yyyymm_series(
            batch.to_pandas()["yyyymm"]
        )

        month_values.update(month_series.unique().tolist())

    return (
        month_values,
        schema_columns,
        int(parquet_file.metadata.num_rows),
    )


def choose_best_header_row(
    raw_sheet_df: pd.DataFrame,
    sheet_name: str,
    search_rows: int = 40,
) -> tuple[int, float, list[str]]:
    keyword_terms = [
        "event",
        "drought",
        "year",
        "start",
        "end",
        "period",
        "severity",
        "intensity",
        "location",
        "province",
        "district",
        "timeline",
        "description",
    ]

    sheet_name_text = normalize_text(sheet_name)
    sheet_bonus = sum(
        term in sheet_name_text
        for term in [
            "drought",
            "event",
            "timeline",
        ]
    )

    best_row_index = 0
    best_score = float("-inf")
    best_values: list[str] = []

    max_row_index = min(
        search_rows,
        len(raw_sheet_df),
    )

    for row_index in range(max_row_index):
        values = [
            cell_text(value)
            for value in raw_sheet_df.iloc[
                row_index
            ].tolist()
        ]

        nonempty_values = [
            value
            for value in values
            if value
        ]

        if len(nonempty_values) < 2:
            continue

        normalized_values = [
            normalize_text(value)
            for value in nonempty_values
        ]

        keyword_hits = sum(
            any(
                keyword in value
                for value in normalized_values
            )
            for keyword in keyword_terms
        )

        unique_value_count = len(
            set(normalized_values)
        )

        score = (
            keyword_hits * 20
            + min(len(nonempty_values), 20)
            + min(unique_value_count, 20)
            + sheet_bonus * 5
        )

        if score > best_score:
            best_row_index = row_index
            best_score = float(score)
            best_values = nonempty_values

    return (
        best_row_index,
        best_score,
        best_values,
    )


def promote_sheet(
    raw_sheet_df: pd.DataFrame,
    header_row_index: int,
) -> pd.DataFrame:
    raw_headers = raw_sheet_df.iloc[
        header_row_index
    ].tolist()

    existing_names: set[str] = set()

    promoted_columns = [
        safe_column_name(
            value=raw_header,
            fallback_index=column_index,
            existing_names=existing_names,
        )
        for column_index, raw_header in enumerate(
            raw_headers
        )
    ]

    promoted_df = raw_sheet_df.iloc[
        header_row_index + 1 :
    ].copy()

    promoted_df.columns = promoted_columns

    promoted_df = promoted_df.dropna(
        axis=0,
        how="all",
    ).dropna(
        axis=1,
        how="all",
    )

    return promoted_df.reset_index(drop=True)


def select_column(
    column_names: list[str],
    include_terms: list[str],
    exclude_terms: list[str] | None = None,
) -> str | None:
    exclude_terms = exclude_terms or []

    scored_columns: list[tuple[int, str]] = []

    for column_name in column_names:
        normalized_column = normalize_text(column_name)

        if any(
            excluded in normalized_column
            for excluded in exclude_terms
        ):
            continue

        include_score = sum(
            term in normalized_column
            for term in include_terms
        )

        if include_score == 0:
            continue

        exact_bonus = int(
            normalized_column in include_terms
        ) * 10

        scored_columns.append(
            (
                include_score * 10 + exact_bonus,
                column_name,
            )
        )

    if not scored_columns:
        return None

    return max(
        scored_columns,
        key=lambda item: item[0],
    )[1]


# -----------------------------------------------------------------------------
# 3. Read workbook with raw headers preserved
# -----------------------------------------------------------------------------
excel_file = pd.ExcelFile(
    EVENT_WORKBOOK_PATH,
    engine="openpyxl",
)

raw_sheets: dict[str, pd.DataFrame] = {
    sheet_name: pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
    )
    for sheet_name in excel_file.sheet_names
}

if not raw_sheets:
    raise AssertionError(
        "The event workbook has no readable sheets."
    )

sheet_audit_records: list[dict[str, Any]] = []
sheet_candidates: list[dict[str, Any]] = []

for sheet_name, raw_sheet_df in raw_sheets.items():
    (
        header_row_index,
        header_score,
        header_values,
    ) = choose_best_header_row(
        raw_sheet_df=raw_sheet_df,
        sheet_name=sheet_name,
    )

    sheet_audit_records.append(
        {
            "sheet_name": sheet_name,
            "raw_rows": int(len(raw_sheet_df)),
            "raw_columns": int(len(raw_sheet_df.columns)),
            "candidate_header_excel_row": int(
                header_row_index + 1
            ),
            "candidate_header_score": header_score,
            "candidate_header_values": " | ".join(
                header_values
            ),
        }
    )

    sheet_candidates.append(
        {
            "sheet_name": sheet_name,
            "header_row_index": header_row_index,
            "header_score": header_score,
        }
    )

sheet_audit_df = pd.DataFrame(
    sheet_audit_records
).sort_values(
    "candidate_header_score",
    ascending=False,
).reset_index(drop=True)

best_sheet_candidate = max(
    sheet_candidates,
    key=lambda item: item["header_score"],
)

SELECTED_SHEET_NAME = best_sheet_candidate[
    "sheet_name"
]

SELECTED_HEADER_ROW_INDEX = int(
    best_sheet_candidate["header_row_index"]
)

selected_raw_sheet_df = raw_sheets[
    SELECTED_SHEET_NAME
]

selected_sheet_df = promote_sheet(
    raw_sheet_df=selected_raw_sheet_df,
    header_row_index=SELECTED_HEADER_ROW_INDEX,
)

if selected_sheet_df.empty:
    raise AssertionError(
        "The selected event sheet has no data rows after its "
        "candidate header."
    )

print("\n=== Workbook audit ===")
print(
    sheet_audit_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

print("\nSelected event sheet:", SELECTED_SHEET_NAME)
print(
    "Selected header Excel row:",
    SELECTED_HEADER_ROW_INDEX + 1,
)

print("\n=== Selected sheet columns ===")
for column_index, column_name in enumerate(
    selected_sheet_df.columns,
    start=1,
):
    print(f"{column_index:>2}. {column_name}")

print("\n=== Selected sheet preview ===")
print(
    selected_sheet_df.head(12).to_string(
        index=False,
        max_colwidth=80,
    )
)


# -----------------------------------------------------------------------------
# 4. Identify candidate Excel fields without assuming a fixed workbook schema
# -----------------------------------------------------------------------------
selected_columns = list(
    selected_sheet_df.columns
)

event_name_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "event",
        "drought",
        "title",
        "name",
        "description",
    ],
    exclude_terms=[
        "year",
        "severity",
        "source",
    ],
)

severity_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "severity",
        "intensity",
        "classification",
        "category",
        "level",
    ],
)

start_year_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "start",
        "year",
    ],
)

end_year_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "end",
        "year",
    ],
)

year_or_period_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "year",
        "period",
        "date",
        "timeline",
        "duration",
    ],
)

geography_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "location",
        "geography",
        "province",
        "district",
        "region",
        "area",
        "place",
    ],
)

source_column = select_column(
    column_names=selected_columns,
    include_terms=[
        "source",
        "reference",
        "citation",
    ],
)

field_mapping_df = pd.DataFrame(
    [
        {
            "field_role": "event_name",
            "selected_excel_column": event_name_column,
        },
        {
            "field_role": "reported_severity",
            "selected_excel_column": severity_column,
        },
        {
            "field_role": "start_year",
            "selected_excel_column": start_year_column,
        },
        {
            "field_role": "end_year",
            "selected_excel_column": end_year_column,
        },
        {
            "field_role": "year_or_period",
            "selected_excel_column": year_or_period_column,
        },
        {
            "field_role": "geography",
            "selected_excel_column": geography_column,
        },
        {
            "field_role": "source_reference",
            "selected_excel_column": source_column,
        },
    ]
)

print("\n=== Candidate Excel field mapping ===")
print(field_mapping_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 5. Build an auditable candidate-event table
# -----------------------------------------------------------------------------
selected_sheet_context = normalize_text(
    SELECTED_SHEET_NAME
)

sheet_is_drought_context = any(
    token in selected_sheet_context
    for token in [
        "drought",
        "event",
        "timeline",
    ]
)

candidate_event_records: list[dict[str, Any]] = []

for row_index, row in selected_sheet_df.iterrows():
    row_values = {
        column_name: cell_text(row[column_name])
        for column_name in selected_columns
    }

    source_text = " | ".join(
        value
        for value in row_values.values()
        if value
    )

    text_years = extract_years(source_text)

    explicit_start_years = (
        extract_years(
            row_values.get(start_year_column, "")
        )
        if start_year_column
        else []
    )

    explicit_end_years = (
        extract_years(
            row_values.get(end_year_column, "")
        )
        if end_year_column
        else []
    )

    period_years = (
        extract_years(
            row_values.get(
                year_or_period_column,
                "",
            )
        )
        if year_or_period_column
        else []
    )

    if explicit_start_years:
        start_year = min(explicit_start_years)
        year_parse_method = "explicit_start_year_column"

    elif period_years:
        start_year = min(period_years)
        year_parse_method = "year_or_period_column"

    elif text_years:
        start_year = min(text_years)
        year_parse_method = "all_row_text"

    else:
        start_year = None
        year_parse_method = "no_year_found"

    if explicit_end_years:
        end_year = max(explicit_end_years)

    elif period_years:
        end_year = max(period_years)

    elif text_years:
        end_year = max(text_years)

    else:
        end_year = None

    if (
        start_year is not None
        and end_year is not None
        and end_year < start_year
    ):
        start_year, end_year = (
            end_year,
            start_year,
        )

    event_name_raw = (
        row_values.get(event_name_column, "")
        if event_name_column
        else ""
    )

    if not event_name_raw:
        event_name_raw = source_text[:250]

    severity_raw = (
        row_values.get(severity_column, "")
        if severity_column
        else ""
    )

    geography_raw = (
        row_values.get(geography_column, "")
        if geography_column
        else ""
    )

    source_reference_raw = (
        row_values.get(source_column, "")
        if source_column
        else ""
    )

    has_drought_text = (
        "drought" in normalize_text(source_text)
    )

    has_valid_year_range = (
        start_year is not None
        and end_year is not None
        and 1800 <= start_year <= 2100
        and 1800 <= end_year <= 2100
    )

    candidate_event = bool(
        has_valid_year_range
        and (
            has_drought_text
            or sheet_is_drought_context
        )
    )

    excel_row_number = (
        SELECTED_HEADER_ROW_INDEX + 2 + row_index
    )

    candidate_event_records.append(
        {
            "source_sheet": SELECTED_SHEET_NAME,
            "source_excel_row": int(
                excel_row_number
            ),
            "candidate_event_id": (
                f"xlsx_{normalize_text(SELECTED_SHEET_NAME).replace(' ', '_')}"
                f"_row_{excel_row_number}"
            ),
            "candidate_event": candidate_event,
            "event_name_raw": event_name_raw,
            "reported_severity_raw": severity_raw,
            "geography_raw": geography_raw,
            "source_reference_raw": source_reference_raw,
            "start_year": start_year,
            "end_year": end_year,
            "year_parse_method": year_parse_method,
            "years_found_in_row_text": (
                ", ".join(
                    str(year)
                    for year in text_years
                )
            ),
            "has_drought_text": has_drought_text,
            "source_text_preview": source_text[:500],
        }
    )

candidate_event_df = pd.DataFrame(
    candidate_event_records
)

candidate_event_df = candidate_event_df.sort_values(
    [
        "candidate_event",
        "start_year",
        "end_year",
        "source_excel_row",
    ],
    ascending=[
        False,
        True,
        True,
        True,
    ],
    na_position="last",
).reset_index(drop=True)

event_ledger_df = candidate_event_df.loc[
    candidate_event_df["candidate_event"]
].copy()

if event_ledger_df.empty:
    raise AssertionError(
        "No candidate drought events could be extracted from the "
        "selected sheet. Review the printed field mapping and sheet "
        "preview before proceeding."
    )

event_ledger_df = event_ledger_df.reset_index(
    drop=True
)

print("\n=== Candidate event rows ===")
print(
    candidate_event_df[
        [
            "source_excel_row",
            "candidate_event",
            "event_name_raw",
            "reported_severity_raw",
            "geography_raw",
            "start_year",
            "end_year",
            "year_parse_method",
            "has_drought_text",
        ]
    ].head(60).to_string(
        index=False,
        max_colwidth=120,
    )
)

print("\n=== Extracted event ledger ===")
print(
    event_ledger_df[
        [
            "candidate_event_id",
            "source_excel_row",
            "event_name_raw",
            "reported_severity_raw",
            "geography_raw",
            "start_year",
            "end_year",
            "year_parse_method",
        ]
    ].to_string(
        index=False,
        max_colwidth=120,
    )
)


# -----------------------------------------------------------------------------
# 6. Read v2 feature contracts and inspect source schemas
# -----------------------------------------------------------------------------
target_feature_lists: dict[str, list[str]] = {}

for target_key in TARGET_KEYS:
    feature_contract_path = (
        V2_RELEASE_ARTIFACT_ROOT
        / target_key
        / "v2_20260617"
        / "feature_contract.json"
    )

    if not feature_contract_path.exists():
        raise FileNotFoundError(
            f"Missing {target_key} feature contract:\n"
            f"{feature_contract_path}"
        )

    feature_contract = json.loads(
        feature_contract_path.read_text(
            encoding="utf-8-sig"
        )
    )

    target_feature_lists[target_key] = (
        extract_feature_list(feature_contract)
    )

if "release_month_summary_df" in globals():
    release_cached_months = set(
        release_month_summary_df["yyyymm"]
        .astype(str)
        .tolist()
    )
else:
    release_cached_months = None

source_specs = [
    {
        "source_key": "release_feature_frame",
        "source_path": RELEASE_FEATURE_FRAME_PATH,
        "cached_months": release_cached_months,
    },
    {
        "source_key": "full_predictor_source",
        "source_path": FULL_PREDICTOR_SOURCE_PATH,
        "cached_months": None,
    },
    {
        "source_key": "full_training_source",
        "source_path": FULL_TRAINING_SOURCE_PATH,
        "cached_months": None,
    },
]

source_month_sets: dict[str, set[str]] = {}
source_schema_columns: dict[str, list[str]] = {}
source_feature_records: list[dict[str, Any]] = []

print("\n=== Source scans ===")

for source_spec in source_specs:
    source_key = source_spec["source_key"]
    source_path = source_spec["source_path"]

    (
        available_months,
        schema_columns,
        source_row_count,
    ) = collect_parquet_months(
        parquet_path=source_path,
        source_name=source_key,
        cached_months=source_spec["cached_months"],
    )

    source_month_sets[source_key] = available_months
    source_schema_columns[source_key] = schema_columns

    print(
        f"{source_key}: "
        f"{source_row_count:,} rows, "
        f"{len(available_months)} distinct months, "
        f"{min(available_months)} to {max(available_months)}"
    )

    schema_column_set = set(schema_columns)

    for target_key in TARGET_KEYS:
        missing_contract_features = [
            feature
            for feature in target_feature_lists[target_key]
            if feature not in schema_column_set
        ]

        source_feature_records.append(
            {
                "source_key": source_key,
                "target_key": target_key,
                "source_row_count": source_row_count,
                "source_month_count": len(
                    available_months
                ),
                "first_month": min(available_months),
                "last_month": max(available_months),
                "required_feature_count": len(
                    target_feature_lists[target_key]
                ),
                "missing_contract_feature_count": len(
                    missing_contract_features
                ),
                "missing_contract_features": ", ".join(
                    missing_contract_features
                ),
                "target_contract_compatible": (
                    len(missing_contract_features) == 0
                ),
            }
        )

source_feature_compatibility_df = pd.DataFrame(
    source_feature_records
).sort_values(
    ["source_key", "target_key"]
).reset_index(drop=True)

print("\n=== Source-to-contract compatibility ===")
print(
    source_feature_compatibility_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

source_ready_for_all_targets = {
    source_key: bool(
        source_feature_compatibility_df.loc[
            source_feature_compatibility_df[
                "source_key"
            ].eq(source_key),
            "target_contract_compatible",
        ].all()
    )
    for source_key in source_month_sets
}


# -----------------------------------------------------------------------------
# 7. Expand Excel candidate events to an auditable event-month ledger
# -----------------------------------------------------------------------------
event_month_records: list[dict[str, Any]] = []

for _, event_row in event_ledger_df.iterrows():
    start_year = int(event_row["start_year"])
    end_year = int(event_row["end_year"])

    required_months = month_range_from_years(
        start_year=start_year,
        end_year=end_year,
    )

    for yyyymm in required_months:
        release_available = (
            yyyymm
            in source_month_sets[
                "release_feature_frame"
            ]
        )

        predictor_available = (
            yyyymm
            in source_month_sets[
                "full_predictor_source"
            ]
        )

        training_available = (
            yyyymm
            in source_month_sets[
                "full_training_source"
            ]
        )

        if (
            release_available
            and source_ready_for_all_targets[
                "release_feature_frame"
            ]
        ):
            inference_source_status = (
                "ready_release_feature_frame"
            )

        elif (
            predictor_available
            and source_ready_for_all_targets[
                "full_predictor_source"
            ]
        ):
            inference_source_status = (
                "ready_full_predictor_source"
            )

        elif (
            training_available
            and source_ready_for_all_targets[
                "full_training_source"
            ]
        ):
            inference_source_status = (
                "ready_full_training_source"
            )

        else:
            inference_source_status = (
                "requires_raw_feature_reconstruction"
            )

        event_month_records.append(
            {
                "candidate_event_id": event_row[
                    "candidate_event_id"
                ],
                "event_name_raw": event_row[
                    "event_name_raw"
                ],
                "reported_severity_raw": event_row[
                    "reported_severity_raw"
                ],
                "geography_raw": event_row[
                    "geography_raw"
                ],
                "start_year": start_year,
                "end_year": end_year,
                "yyyymm": yyyymm,
                "year_parse_method": event_row[
                    "year_parse_method"
                ],
                "release_feature_frame_available": (
                    release_available
                ),
                "full_predictor_source_available": (
                    predictor_available
                ),
                "full_training_source_available": (
                    training_available
                ),
                "inference_source_status": (
                    inference_source_status
                ),
            }
        )

event_month_source_ledger_df = pd.DataFrame(
    event_month_records
)

event_coverage_summary_df = (
    event_month_source_ledger_df.groupby(
        [
            "candidate_event_id",
            "event_name_raw",
            "reported_severity_raw",
            "geography_raw",
            "start_year",
            "end_year",
            "year_parse_method",
        ],
        dropna=False,
    )
    .agg(
        required_month_count=("yyyymm", "size"),
        release_feature_frame_month_count=(
            "release_feature_frame_available",
            "sum",
        ),
        full_predictor_source_month_count=(
            "full_predictor_source_available",
            "sum",
        ),
        full_training_source_month_count=(
            "full_training_source_available",
            "sum",
        ),
        months_requiring_reconstruction=(
            "inference_source_status",
            lambda values: int(
                (
                    values
                    == "requires_raw_feature_reconstruction"
                ).sum()
            ),
        ),
    )
    .reset_index()
)

for source_key, count_column in [
    (
        "release_feature_frame",
        "release_feature_frame_month_count",
    ),
    (
        "full_predictor_source",
        "full_predictor_source_month_count",
    ),
    (
        "full_training_source",
        "full_training_source_month_count",
    ),
]:
    event_coverage_summary_df[
        f"{source_key}_coverage_rate"
    ] = (
        event_coverage_summary_df[count_column]
        / event_coverage_summary_df[
            "required_month_count"
        ]
    )

event_coverage_summary_df[
    "recommended_next_action"
] = np.select(
    [
        event_coverage_summary_df[
            "months_requiring_reconstruction"
        ].eq(0),
        event_coverage_summary_df[
            "full_predictor_source_month_count"
        ].gt(0),
    ],
    [
        "validate_with_existing_model_ready_source",
        "validate_available_months_then_rebuild_missing_months",
    ],
    default="rebuild_required_before_event_validation",
)

event_month_source_ledger_df = (
    event_month_source_ledger_df.sort_values(
        [
            "candidate_event_id",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
)

event_coverage_summary_df = (
    event_coverage_summary_df.sort_values(
        [
            "start_year",
            "end_year",
            "candidate_event_id",
        ]
    )
    .reset_index(drop=True)
)

print("\n=== Excel event-to-source coverage summary ===")
print(
    event_coverage_summary_df.to_string(
        index=False,
        max_colwidth=120,
    )
)

print("\n=== Months requiring reconstruction ===")
rebuild_months_df = event_month_source_ledger_df.loc[
    event_month_source_ledger_df[
        "inference_source_status"
    ].eq("requires_raw_feature_reconstruction")
].copy()

if rebuild_months_df.empty:
    print(
        "None. Every extracted Excel event month exists in at least "
        "one checked v2-compatible source."
    )
else:
    print(
        rebuild_months_df[
            [
                "candidate_event_id",
                "event_name_raw",
                "yyyymm",
                "reported_severity_raw",
                "geography_raw",
            ]
        ].to_string(
            index=False,
            max_rows=200,
            max_colwidth=100,
        )
    )


# -----------------------------------------------------------------------------
# 8. Cell conclusion
# -----------------------------------------------------------------------------
event_count = int(
    event_coverage_summary_df[
        "candidate_event_id"
    ].nunique()
)

fully_ready_event_count = int(
    event_coverage_summary_df[
        "months_requiring_reconstruction"
    ].eq(0)
    .sum()
)

rebuild_month_count = int(
    len(rebuild_months_df)
)

CELL5_STATUS = (
    "excel_event_reference_and_source_coverage_audited"
)

print("\n=== Cell 5 conclusion ===")
print("Status:", CELL5_STATUS)
print("Extracted candidate events:", event_count)
print(
    "Events fully covered by an existing checked source:",
    fully_ready_event_count,
)
print(
    "Candidate event months requiring reconstruction:",
    rebuild_month_count,
)

print(
    "\nImportant interpretation:\n"
    "- This cell identifies data availability only.\n"
    "- It does not yet claim model agreement or disagreement with "
    "the Excel severity.\n"
    "- Parsed year spans are transparent candidates derived from the "
    "Excel and must be reviewed before final event comparison."
)


# -----------------------------------------------------------------------------
# 9. Save Cell 5 outputs
# -----------------------------------------------------------------------------
sheet_audit_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_excel_sheet_audit_{CELL5_RUN_ID}.csv"
)

field_mapping_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_excel_field_mapping_{CELL5_RUN_ID}.csv"
)

candidate_rows_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_excel_candidate_rows_{CELL5_RUN_ID}.csv"
)

event_ledger_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_excel_event_ledger_{CELL5_RUN_ID}.csv"
)

source_contract_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_source_contract_compatibility_{CELL5_RUN_ID}.csv"
)

event_month_source_ledger_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_event_month_source_ledger_{CELL5_RUN_ID}.csv"
)

coverage_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_event_source_coverage_summary_{CELL5_RUN_ID}.csv"
)

rebuild_months_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_months_requiring_reconstruction_{CELL5_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell5_excel_source_audit_manifest_{CELL5_RUN_ID}.json"
)

sheet_audit_df.to_csv(
    sheet_audit_path,
    index=False,
)

field_mapping_df.to_csv(
    field_mapping_path,
    index=False,
)

candidate_event_df.to_csv(
    candidate_rows_path,
    index=False,
)

event_ledger_df.to_csv(
    event_ledger_path,
    index=False,
)

source_feature_compatibility_df.to_csv(
    source_contract_path,
    index=False,
)

event_month_source_ledger_df.to_csv(
    event_month_source_ledger_path,
    index=False,
)

event_coverage_summary_df.to_csv(
    coverage_summary_path,
    index=False,
)

rebuild_months_df.to_csv(
    rebuild_months_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL5_RUN_ID,
    "status": CELL5_STATUS,
    "event_workbook_path": str(EVENT_WORKBOOK_PATH),
    "selected_sheet_name": SELECTED_SHEET_NAME,
    "selected_header_excel_row": int(
        SELECTED_HEADER_ROW_INDEX + 1
    ),
    "candidate_event_count": event_count,
    "fully_ready_event_count": fully_ready_event_count,
    "months_requiring_reconstruction_count": (
        rebuild_month_count
    ),
    "source_ready_for_all_targets": (
        source_ready_for_all_targets
    ),
    "candidate_field_mapping": (
        field_mapping_df.to_dict(
            orient="records"
        )
    ),
    "notes": [
        (
            "Event dates are expanded to every calendar month between "
            "the parsed inclusive start and end years."
        ),
        (
            "This is an availability ledger, not a final scientific "
            "event-duration determination."
        ),
        (
            "Raw reconstruction is needed only where no checked "
            "source contains the required month with all v2 contract "
            "features."
        ),
    ],
    "outputs": {
        "sheet_audit_csv": str(sheet_audit_path),
        "field_mapping_csv": str(field_mapping_path),
        "candidate_rows_csv": str(candidate_rows_path),
        "event_ledger_csv": str(event_ledger_path),
        "source_contract_compatibility_csv": str(
            source_contract_path
        ),
        "event_month_source_ledger_csv": str(
            event_month_source_ledger_path
        ),
        "event_source_coverage_summary_csv": str(
            coverage_summary_path
        ),
        "months_requiring_reconstruction_csv": str(
            rebuild_months_path
        ),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 5 outputs ===")
print("Workbook sheet audit       ->", sheet_audit_path)
print("Field mapping              ->", field_mapping_path)
print("Candidate Excel rows       ->", candidate_rows_path)
print("Event ledger               ->", event_ledger_path)
print("Source contract audit      ->", source_contract_path)
print("Event-month source ledger  ->", event_month_source_ledger_path)
print("Event coverage summary     ->", coverage_summary_path)
print("Months for reconstruction  ->", rebuild_months_path)
print("Audit manifest             ->", manifest_path)

print("\nCell 5 complete.")

=== Cell 5 purpose ===
Audit Excel event records and identify the source status of every candidate event month.

=== Paths ===
Event workbook         : C:\Projects\Infer RozviDrought\data\events\Zimbabwe_Drought_Timeline_1902_2024.xlsx
Release feature frame  : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_prediction_feature_frame_20260618T152226Z.parquet
Full predictor source  : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Full training source   : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\training_dataset_temporal_multi_spi_20260614T144111Z.parquet
Event output directory : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID                 : 20260620T112540Z

=== Workbook audit ===
            sheet_name  raw_rows  raw_columns  candidate_header_exc

In [9]:
# validate_events_ng.ipynb — Cell 6
# Purpose:
# - Audit every SPI12 contract feature absent from temporal_predictors_full.
# - Distinguish:
#     1) deterministic missingness indicators,
#     2) temporal engineered features,
#     3) features with no identifiable derivation family.
# - Check whether each missing feature's direct base field exists in:
#     - the full predictor source,
#     - the release feature frame.
# - Produce an evidence-based decision on whether a derivation audit is possible.
#
# Scope:
# - No inference.
# - No feature creation.
# - No raw/API reconstruction.
# - No assumption that sequence features can be recreated until their formula
#   and temporal alignment are validated against the release frame.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "DATA_ROOT" not in globals():
    raise RuntimeError("DATA_ROOT is missing. Rerun Cell 0 first.")

if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL6_RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

RELEASE_FEATURE_FRAME_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
    / "release_bundle_prediction_feature_frame_20260618T152226Z.parquet"
)

FULL_PREDICTOR_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "temporal_predictors_full_20260614T141118Z.parquet"
)

FULL_TRAINING_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "training_dataset_temporal_multi_spi_20260614T144111Z.parquet"
)

V2_RELEASE_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

SPI12_FEATURE_CONTRACT_PATH = (
    V2_RELEASE_ARTIFACT_ROOT
    / "spi12"
    / "v2_20260617"
    / "feature_contract.json"
)

MISSINGNESS_SUFFIX = "__is_missing"

SEQUENCE_PATTERN = re.compile(
    r"^(?P<base>.+)_seq_"
    r"(?P<operation>lag|delta|rollmean|rollstd)"
    r"(?P<window>\d+)$"
)

print("=== Cell 6 purpose ===")
print(
    "Audit the actual provenance gap between the SPI12 contract "
    "and temporal_predictors_full."
)

print("\n=== Paths ===")
print("Release feature frame :", RELEASE_FEATURE_FRAME_PATH)
print("Full predictor source :", FULL_PREDICTOR_SOURCE_PATH)
print("Full training source  :", FULL_TRAINING_SOURCE_PATH)
print("SPI12 contract        :", SPI12_FEATURE_CONTRACT_PATH)
print("Output directory      :", EVENT_OUTPUT_DIR)
print("Run ID                :", CELL6_RUN_ID)

for required_path in [
    RELEASE_FEATURE_FRAME_PATH,
    FULL_PREDICTOR_SOURCE_PATH,
    FULL_TRAINING_SOURCE_PATH,
    SPI12_FEATURE_CONTRACT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 6 input is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"{path.name} did not contain a JSON object."
        )

    return payload


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    candidate_keys = [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]

    for key in candidate_keys:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            return list(value)

    raise ValueError(
        "No usable model-feature list was found in the SPI12 "
        "feature contract."
    )


def related_schema_columns(
    base_feature: str | None,
    schema_columns: set[str],
    max_columns: int = 30,
) -> list[str]:
    if not base_feature:
        return []

    related = sorted(
        column
        for column in schema_columns
        if (
            column == base_feature
            or column.startswith(
                f"{base_feature}_"
            )
        )
    )

    return related[:max_columns]


def classify_missing_feature(
    missing_feature: str,
) -> dict[str, Any]:
    if missing_feature.endswith(
        MISSINGNESS_SUFFIX
    ):
        base_feature = missing_feature[
            : -len(MISSINGNESS_SUFFIX)
        ]

        return {
            "missing_spi12_feature": missing_feature,
            "feature_family": "missingness_indicator",
            "base_feature": base_feature,
            "temporal_operation": None,
            "temporal_window": None,
            "candidate_derivation": (
                f"int8({base_feature}.isna())"
            ),
        }

    sequence_match = SEQUENCE_PATTERN.match(
        missing_feature
    )

    if sequence_match:
        base_feature = sequence_match.group("base")
        operation = sequence_match.group("operation")
        window = int(sequence_match.group("window"))

        return {
            "missing_spi12_feature": missing_feature,
            "feature_family": "temporal_engineered",
            "base_feature": base_feature,
            "temporal_operation": operation,
            "temporal_window": window,
            "candidate_derivation": (
                "requires_formula_and_calendar_alignment_validation"
            ),
        }

    return {
        "missing_spi12_feature": missing_feature,
        "feature_family": "unclassified",
        "base_feature": None,
        "temporal_operation": None,
        "temporal_window": None,
        "candidate_derivation": (
            "unknown_requires_provenance_investigation"
        ),
    }


def serialize_list(values: list[str]) -> str:
    return " | ".join(values)


# -----------------------------------------------------------------------------
# 3. Read contract and source schemas only
# -----------------------------------------------------------------------------
spi12_contract = read_json_object(
    SPI12_FEATURE_CONTRACT_PATH
)

spi12_required_features = extract_feature_list(
    spi12_contract
)

release_schema_columns = set(
    pq.ParquetFile(
        RELEASE_FEATURE_FRAME_PATH
    ).schema.names
)

predictor_schema_columns = set(
    pq.ParquetFile(
        FULL_PREDICTOR_SOURCE_PATH
    ).schema.names
)

training_schema_columns = set(
    pq.ParquetFile(
        FULL_TRAINING_SOURCE_PATH
    ).schema.names
)

missing_from_release = [
    feature
    for feature in spi12_required_features
    if feature not in release_schema_columns
]

missing_from_predictor = [
    feature
    for feature in spi12_required_features
    if feature not in predictor_schema_columns
]

missing_from_training = [
    feature
    for feature in spi12_required_features
    if feature not in training_schema_columns
]

if missing_from_release:
    raise AssertionError(
        "The release feature frame is unexpectedly missing "
        "SPI12 contract features:\n"
        f"{missing_from_release}"
    )

print("\n=== SPI12 schema summary ===")
print(
    "SPI12 contract features          :",
    len(spi12_required_features),
)
print(
    "Missing from release frame       :",
    len(missing_from_release),
)
print(
    "Missing from full predictor      :",
    len(missing_from_predictor),
)
print(
    "Missing from full training source:",
    len(missing_from_training),
)


# -----------------------------------------------------------------------------
# 4. Classify every missing full-predictor feature
# -----------------------------------------------------------------------------
provenance_records: list[dict[str, Any]] = []

for missing_feature in missing_from_predictor:
    record = classify_missing_feature(
        missing_feature
    )

    base_feature = record["base_feature"]

    record.update(
        {
            "base_exists_in_full_predictor": (
                bool(
                    base_feature
                    and base_feature
                    in predictor_schema_columns
                )
            ),
            "base_exists_in_full_training": (
                bool(
                    base_feature
                    and base_feature
                    in training_schema_columns
                )
            ),
            "base_exists_in_release_frame": (
                bool(
                    base_feature
                    and base_feature
                    in release_schema_columns
                )
            ),
            "related_predictor_columns": serialize_list(
                related_schema_columns(
                    base_feature=base_feature,
                    schema_columns=predictor_schema_columns,
                )
            ),
            "related_training_columns": serialize_list(
                related_schema_columns(
                    base_feature=base_feature,
                    schema_columns=training_schema_columns,
                )
            ),
            "related_release_columns": serialize_list(
                related_schema_columns(
                    base_feature=base_feature,
                    schema_columns=release_schema_columns,
                )
            ),
        }
    )

    provenance_records.append(record)

spi12_provenance_df = pd.DataFrame(
    provenance_records
).sort_values(
    [
        "feature_family",
        "base_feature",
        "temporal_operation",
        "temporal_window",
        "missing_spi12_feature",
    ],
    na_position="last",
).reset_index(drop=True)

family_summary_df = (
    spi12_provenance_df.groupby(
        "feature_family",
        dropna=False,
    )
    .agg(
        feature_count=(
            "missing_spi12_feature",
            "size",
        ),
        bases_present_in_predictor=(
            "base_exists_in_full_predictor",
            "sum",
        ),
        bases_present_in_training=(
            "base_exists_in_full_training",
            "sum",
        ),
        bases_present_in_release=(
            "base_exists_in_release_frame",
            "sum",
        ),
    )
    .reset_index()
    .sort_values("feature_family")
    .reset_index(drop=True)
)

print("\n=== Missing SPI12 feature families ===")
print(
    family_summary_df.to_string(
        index=False
    )
)

print("\n=== Complete SPI12 provenance audit ===")
print(
    spi12_provenance_df.to_string(
        index=False,
        max_colwidth=130,
    )
)


# -----------------------------------------------------------------------------
# 5. Separate and validate the three provenance categories
# -----------------------------------------------------------------------------
indicator_gap_df = spi12_provenance_df.loc[
    spi12_provenance_df[
        "feature_family"
    ].eq("missingness_indicator")
].copy()

temporal_gap_df = spi12_provenance_df.loc[
    spi12_provenance_df[
        "feature_family"
    ].eq("temporal_engineered")
].copy()

unclassified_gap_df = spi12_provenance_df.loc[
    spi12_provenance_df[
        "feature_family"
    ].eq("unclassified")
].copy()

indicator_without_predictor_base_df = indicator_gap_df.loc[
    ~indicator_gap_df[
        "base_exists_in_full_predictor"
    ]
].copy()

temporal_without_predictor_base_df = temporal_gap_df.loc[
    ~temporal_gap_df[
        "base_exists_in_full_predictor"
    ]
].copy()

print("\n=== Gap classification ===")
print(
    "Missingness indicators:",
    len(indicator_gap_df),
)
print(
    "Temporal engineered features:",
    len(temporal_gap_df),
)
print(
    "Unclassified features:",
    len(unclassified_gap_df),
)
print(
    "Indicators lacking predictor base:",
    len(indicator_without_predictor_base_df),
)
print(
    "Temporal features lacking predictor base:",
    len(temporal_without_predictor_base_df),
)

if not indicator_without_predictor_base_df.empty:
    print("\nIndicators with no direct predictor base:")
    print(
        indicator_without_predictor_base_df.to_string(
            index=False,
            max_colwidth=130,
        )
    )

if not temporal_without_predictor_base_df.empty:
    print("\nTemporal features with no direct predictor base:")
    print(
        temporal_without_predictor_base_df.to_string(
            index=False,
            max_colwidth=130,
        )
    )

if not unclassified_gap_df.empty:
    print("\nUnclassified missing features:")
    print(
        unclassified_gap_df.to_string(
            index=False,
            max_colwidth=130,
        )
    )


# -----------------------------------------------------------------------------
# 6. Build a strict decision table; no derivation happens here
# -----------------------------------------------------------------------------
indicator_ready_for_formula_test = bool(
    not indicator_gap_df.empty
    and indicator_without_predictor_base_df.empty
)

temporal_ready_for_formula_test = bool(
    not temporal_gap_df.empty
    and temporal_without_predictor_base_df.empty
)

all_missing_features_classified = bool(
    unclassified_gap_df.empty
)

if (
    all_missing_features_classified
    and indicator_ready_for_formula_test
    and temporal_ready_for_formula_test
):
    CELL6_STATUS = (
        "spi12_gap_fully_classified_"
        "formula_validation_required_before_inference"
    )

elif (
    all_missing_features_classified
    and indicator_ready_for_formula_test
):
    CELL6_STATUS = (
        "spi12_indicator_formula_test_possible_"
        "temporal_feature_provenance_unresolved"
    )

else:
    CELL6_STATUS = (
        "spi12_full_predictor_not_ready_"
        "upstream_feature_provenance_required"
    )

decision_df = pd.DataFrame(
    [
        {
            "check": (
                "all_missing_features_classified"
            ),
            "result": all_missing_features_classified,
        },
        {
            "check": (
                "all_indicator_bases_exist_in_predictor"
            ),
            "result": indicator_without_predictor_base_df.empty,
        },
        {
            "check": (
                "all_temporal_bases_exist_in_predictor"
            ),
            "result": temporal_without_predictor_base_df.empty,
        },
        {
            "check": (
                "SPI12_ready_for_direct_full_predictor_inference"
            ),
            "result": False,
        },
        {
            "check": (
                "next_step_is_formula_validation_not_reconstruction"
            ),
            "result": (
                all_missing_features_classified
                and indicator_ready_for_formula_test
                and temporal_ready_for_formula_test
            ),
        },
    ]
)

print("\n=== Cell 6 decision ===")
print("Status:", CELL6_STATUS)
print(decision_df.to_string(index=False))

print(
    "\nInterpretation:\n"
    "- The full predictor source cannot yet be passed into SPI12.\n"
    "- Missingness flags and temporal sequence fields must be "
    "validated separately.\n"
    "- The next step depends on whether their direct base fields "
    "actually exist in the full predictor source.\n"
    "- No raw-data reconstruction decision is made by this cell."
)


# -----------------------------------------------------------------------------
# 7. Save outputs
# -----------------------------------------------------------------------------
schema_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_schema_summary_{CELL6_RUN_ID}.csv"
)

provenance_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_missing_feature_provenance_{CELL6_RUN_ID}.csv"
)

family_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_missing_feature_families_{CELL6_RUN_ID}.csv"
)

indicator_gap_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_indicator_gap_{CELL6_RUN_ID}.csv"
)

temporal_gap_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_temporal_gap_{CELL6_RUN_ID}.csv"
)

unclassified_gap_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_unclassified_gap_{CELL6_RUN_ID}.csv"
)

decision_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_readiness_decision_{CELL6_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell6_spi12_provenance_audit_manifest_{CELL6_RUN_ID}.json"
)

schema_summary_df = pd.DataFrame(
    [
        {
            "source": "release_feature_frame",
            "schema_column_count": len(
                release_schema_columns
            ),
            "missing_spi12_contract_features": len(
                missing_from_release
            ),
        },
        {
            "source": "full_predictor_source",
            "schema_column_count": len(
                predictor_schema_columns
            ),
            "missing_spi12_contract_features": len(
                missing_from_predictor
            ),
        },
        {
            "source": "full_training_source",
            "schema_column_count": len(
                training_schema_columns
            ),
            "missing_spi12_contract_features": len(
                missing_from_training
            ),
        },
    ]
)

schema_summary_df.to_csv(
    schema_summary_path,
    index=False,
)

spi12_provenance_df.to_csv(
    provenance_path,
    index=False,
)

family_summary_df.to_csv(
    family_summary_path,
    index=False,
)

indicator_gap_df.to_csv(
    indicator_gap_path,
    index=False,
)

temporal_gap_df.to_csv(
    temporal_gap_path,
    index=False,
)

unclassified_gap_df.to_csv(
    unclassified_gap_path,
    index=False,
)

decision_df.to_csv(
    decision_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL6_RUN_ID,
    "status": CELL6_STATUS,
    "purpose": (
        "Classify all SPI12 contract features absent from the full "
        "predictor source before any reconstruction or inference."
    ),
    "spi12_contract_feature_count": len(
        spi12_required_features
    ),
    "missing_from_full_predictor_count": len(
        missing_from_predictor
    ),
    "missingness_indicator_count": len(
        indicator_gap_df
    ),
    "temporal_engineered_feature_count": len(
        temporal_gap_df
    ),
    "unclassified_feature_count": len(
        unclassified_gap_df
    ),
    "indicator_features_without_base_count": len(
        indicator_without_predictor_base_df
    ),
    "temporal_features_without_base_count": len(
        temporal_without_predictor_base_df
    ),
    "outputs": {
        "schema_summary_csv": str(
            schema_summary_path
        ),
        "feature_provenance_csv": str(
            provenance_path
        ),
        "family_summary_csv": str(
            family_summary_path
        ),
        "indicator_gap_csv": str(
            indicator_gap_path
        ),
        "temporal_gap_csv": str(
            temporal_gap_path
        ),
        "unclassified_gap_csv": str(
            unclassified_gap_path
        ),
        "readiness_decision_csv": str(
            decision_path
        ),
        "manifest_json": str(
            manifest_path
        ),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 6 outputs ===")
print("Schema summary     ->", schema_summary_path)
print("Feature provenance ->", provenance_path)
print("Family summary     ->", family_summary_path)
print("Indicator gap      ->", indicator_gap_path)
print("Temporal gap       ->", temporal_gap_path)
print("Unclassified gap   ->", unclassified_gap_path)
print("Readiness decision ->", decision_path)
print("Manifest           ->", manifest_path)

print("\nCell 6 complete.")

=== Cell 6 purpose ===
Audit the actual provenance gap between the SPI12 contract and temporal_predictors_full.

=== Paths ===
Release feature frame : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_prediction_feature_frame_20260618T152226Z.parquet
Full predictor source : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Full training source  : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\training_dataset_temporal_multi_spi_20260614T144111Z.parquet
SPI12 contract        : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts\spi12\v2_20260617\feature_contract.json
Output directory      : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID                : 20260620T113143Z

=== SPI12 schema summary =

In [10]:
# validate_events_ng.ipynb — Cell 7
# Purpose:
# - Locate the original SPI12 sequence-feature generation logic.
# - Search release manifests, model manifests, notebooks, and Python scripts
#   for the exact missing temporal feature names and related formula terms.
# - Extract relevant code/text contexts without modifying any artifact.
# - Decide whether the next cell can reproduce documented logic directly
#   or must perform a formula-parity audit against the release frame.
#
# Scope:
# - No feature derivation.
# - No inference.
# - No raw-data reconstruction.
# - Read-only source and metadata provenance discovery.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import os
import re

import pandas as pd


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL7_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RISKS_ROOT = Path(
    r"C:\Projects\ramangwana\risks"
)

DROUGHT_ROOT = (
    RISKS_ROOT
    / "data"
    / "drought_model"
)

RELEASE_HANDOFF_ROOT = (
    DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
)

V2_ARTIFACT_ROOT = (
    DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

SPI12_FEATURE_CONTRACT_PATH = (
    V2_ARTIFACT_ROOT
    / "spi12"
    / "v2_20260617"
    / "feature_contract.json"
)

TEXT_EXTENSIONS = {
    ".py",
    ".ipynb",
    ".json",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
}

EXCLUDED_DIRECTORY_NAMES = {
    ".git",
    ".venv",
    "venv",
    "env",
    "__pycache__",
    "node_modules",
    ".pytest_cache",
    ".mypy_cache",
}

# Search terms are deliberately specific to the unresolved feature family.
SEARCH_TERMS = [
    "rain_chirps_seq_lag12",
    "rain_chirps_seq_rollmean24",
    "tws_seq_lag24",
    "tws_grace_seq_rollstd24",
    "seq_lag12",
    "seq_lag24",
    "seq_delta12",
    "seq_delta24",
    "seq_rollmean24",
    "seq_rollstd24",
    "__is_missing",
]

MAX_FILE_SIZE_BYTES = 15 * 1024 * 1024
MAX_CONTEXT_LINES_PER_HIT = 2

print("=== Cell 7 purpose ===")
print(
    "Locate original SPI12 temporal-feature engineering logic "
    "before reproducing any missing features."
)

print("\n=== Search roots ===")
print("Risks root            :", RISKS_ROOT)
print("Release handoff root  :", RELEASE_HANDOFF_ROOT)
print("V2 artifact root      :", V2_ARTIFACT_ROOT)
print("SPI12 contract        :", SPI12_FEATURE_CONTRACT_PATH)
print("Output directory      :", EVENT_OUTPUT_DIR)
print("Run ID                :", CELL7_RUN_ID)

for required_path in [
    RISKS_ROOT,
    RELEASE_HANDOFF_ROOT,
    V2_ARTIFACT_ROOT,
    SPI12_FEATURE_CONTRACT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 7 path is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"{path.name} is not a JSON object."
        )

    return payload


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    for key in [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            return list(value)

    raise ValueError(
        "Could not find a model-feature list in the SPI12 contract."
    )


def extract_text_lines(
    path: Path,
) -> list[str]:
    """
    Extract text from readable source-like files.
    For notebooks, return numbered code-cell lines so formula context
    remains meaningful.
    """
    suffix = path.suffix.lower()

    if suffix == ".ipynb":
        notebook = json.loads(
            path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        )

        lines: list[str] = []

        for cell_index, cell in enumerate(
            notebook.get("cells", []),
            start=1,
        ):
            if cell.get("cell_type") != "code":
                continue

            source = cell.get("source", [])

            if isinstance(source, str):
                source_lines = source.splitlines()

            else:
                source_lines = "".join(source).splitlines()

            for source_line_index, source_line in enumerate(
                source_lines,
                start=1,
            ):
                lines.append(
                    f"[cell {cell_index}, line {source_line_index}] "
                    f"{source_line}"
                )

        return lines

    return path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()


def iter_source_files(
    root: Path,
    allowed_extensions: set[str],
) -> list[Path]:
    source_files: list[Path] = []

    for current_root, dir_names, file_names in os.walk(root):
        dir_names[:] = [
            directory_name
            for directory_name in dir_names
            if directory_name.lower()
            not in EXCLUDED_DIRECTORY_NAMES
        ]

        current_root_path = Path(current_root)

        for file_name in file_names:
            path = current_root_path / file_name

            if path.suffix.lower() not in allowed_extensions:
                continue

            try:
                if path.stat().st_size > MAX_FILE_SIZE_BYTES:
                    continue
            except OSError:
                continue

            source_files.append(path)

    return source_files


def search_file_for_terms(
    path: Path,
    search_terms: list[str],
) -> list[dict[str, Any]]:
    try:
        lines = extract_text_lines(path)
    except Exception as error:
        return [
            {
                "path": str(path),
                "status": "read_failed",
                "term": None,
                "line_number": None,
                "context": repr(error),
            }
        ]

    records: list[dict[str, Any]] = []

    for line_index, line in enumerate(lines):
        line_lower = line.lower()

        matched_terms = [
            term
            for term in search_terms
            if term.lower() in line_lower
        ]

        if not matched_terms:
            continue

        context_start = max(
            0,
            line_index - MAX_CONTEXT_LINES_PER_HIT,
        )

        context_end = min(
            len(lines),
            line_index + MAX_CONTEXT_LINES_PER_HIT + 1,
        )

        context = "\n".join(
            lines[context_start:context_end]
        )

        for term in matched_terms:
            records.append(
                {
                    "path": str(path),
                    "status": "match",
                    "term": term,
                    "line_number": line_index + 1,
                    "context": context,
                }
            )

    return records


# -----------------------------------------------------------------------------
# 3. Derive the exact unresolved SPI12 feature set
# -----------------------------------------------------------------------------
spi12_contract = read_json_object(
    SPI12_FEATURE_CONTRACT_PATH
)

spi12_features = extract_feature_list(
    spi12_contract
)

temporal_sequence_features = [
    feature
    for feature in spi12_features
    if "_seq_" in feature
]

missingness_features = [
    feature
    for feature in spi12_features
    if feature.endswith("__is_missing")
]

print("\n=== SPI12 feature-family scope ===")
print("Total SPI12 contract features :", len(spi12_features))
print("Sequence feature count        :", len(temporal_sequence_features))
print("Missingness indicator count   :", len(missingness_features))

print("\nSequence features:")
for feature in temporal_sequence_features:
    print("-", feature)


# -----------------------------------------------------------------------------
# 4. Search high-value release and artifact metadata first
# -----------------------------------------------------------------------------
priority_metadata_roots = [
    RELEASE_HANDOFF_ROOT,
    V2_ARTIFACT_ROOT,
]

priority_files: list[Path] = []

for root in priority_metadata_roots:
    priority_files.extend(
        iter_source_files(
            root=root,
            allowed_extensions=TEXT_EXTENSIONS,
        )
    )

priority_files = sorted(
    set(priority_files),
    key=lambda path: str(path).lower(),
)

print("\n=== Priority metadata search ===")
print("Files scanned:", len(priority_files))

priority_records: list[dict[str, Any]] = []

for path in priority_files:
    priority_records.extend(
        search_file_for_terms(
            path=path,
            search_terms=SEARCH_TERMS,
        )
    )

priority_hits_df = pd.DataFrame(
    priority_records
)

if priority_hits_df.empty:
    priority_hits_df = pd.DataFrame(
        columns=[
            "path",
            "status",
            "term",
            "line_number",
            "context",
        ]
    )

priority_match_df = priority_hits_df.loc[
    priority_hits_df["status"].eq("match")
].copy()

print("Matches found:", len(priority_match_df))

if not priority_match_df.empty:
    print(
        priority_match_df[
            [
                "term",
                "path",
                "line_number",
                "context",
            ]
        ].head(100).to_string(
            index=False,
            max_colwidth=220,
        )
    )


# -----------------------------------------------------------------------------
# 5. Search notebooks and Python source outside the large data tree
# -----------------------------------------------------------------------------
# The root scan excludes the entire data tree to avoid traversing large data
# artifacts. Release/data metadata were already searched explicitly above.

source_code_roots = [
    path
    for path in RISKS_ROOT.iterdir()
    if (
        path.is_dir()
        and path.name.lower()
        not in {
            "data",
            ".git",
            ".venv",
            "venv",
            "env",
            "__pycache__",
        }
    )
]

source_code_files: list[Path] = []

for root in source_code_roots:
    source_code_files.extend(
        iter_source_files(
            root=root,
            allowed_extensions={
                ".py",
                ".ipynb",
                ".md",
                ".txt",
                ".json",
            },
        )
    )

source_code_files = sorted(
    set(source_code_files),
    key=lambda path: str(path).lower(),
)

print("\n=== Source-code search outside data tree ===")
print("Roots scanned:")
for root in source_code_roots:
    print("-", root)

print("Files scanned:", len(source_code_files))

source_records: list[dict[str, Any]] = []

for path in source_code_files:
    source_records.extend(
        search_file_for_terms(
            path=path,
            search_terms=SEARCH_TERMS,
        )
    )

source_hits_df = pd.DataFrame(source_records)

if source_hits_df.empty:
    source_hits_df = pd.DataFrame(
        columns=[
            "path",
            "status",
            "term",
            "line_number",
            "context",
        ]
    )

source_match_df = source_hits_df.loc[
    source_hits_df["status"].eq("match")
].copy()

print("Matches found:", len(source_match_df))

if not source_match_df.empty:
    print(
        source_match_df[
            [
                "term",
                "path",
                "line_number",
                "context",
            ]
        ].head(150).to_string(
            index=False,
            max_colwidth=220,
        )
    )


# -----------------------------------------------------------------------------
# 6. Search likely drought-specific notebook/script paths inside data root,
#    but only source-code-like files—not Parquet, raster, or export folders.
# -----------------------------------------------------------------------------
drought_source_candidate_names = [
    "notebooks",
    "notebook",
    "scripts",
    "src",
    "code",
    "pipeline",
    "pipelines",
    "workflow",
    "workflows",
]

drought_source_roots: list[Path] = []

for candidate_name in drought_source_candidate_names:
    candidate_path = DROUGHT_ROOT / candidate_name

    if candidate_path.exists() and candidate_path.is_dir():
        drought_source_roots.append(candidate_path)

drought_source_files: list[Path] = []

for root in drought_source_roots:
    drought_source_files.extend(
        iter_source_files(
            root=root,
            allowed_extensions={
                ".py",
                ".ipynb",
                ".md",
                ".txt",
                ".json",
            },
        )
    )

drought_source_files = sorted(
    set(drought_source_files),
    key=lambda path: str(path).lower(),
)

print("\n=== Drought-specific source search ===")
print("Candidate drought source roots:")
for root in drought_source_roots:
    print("-", root)

print("Files scanned:", len(drought_source_files))

drought_records: list[dict[str, Any]] = []

for path in drought_source_files:
    drought_records.extend(
        search_file_for_terms(
            path=path,
            search_terms=SEARCH_TERMS,
        )
    )

drought_hits_df = pd.DataFrame(drought_records)

if drought_hits_df.empty:
    drought_hits_df = pd.DataFrame(
        columns=[
            "path",
            "status",
            "term",
            "line_number",
            "context",
        ]
    )

drought_match_df = drought_hits_df.loc[
    drought_hits_df["status"].eq("match")
].copy()

print("Matches found:", len(drought_match_df))

if not drought_match_df.empty:
    print(
        drought_match_df[
            [
                "term",
                "path",
                "line_number",
                "context",
            ]
        ].head(150).to_string(
            index=False,
            max_colwidth=220,
        )
    )


# -----------------------------------------------------------------------------
# 7. Consolidate provenance evidence and decide next action
# -----------------------------------------------------------------------------
all_hit_frames = [
    priority_match_df.assign(
        search_scope="release_and_artifact_metadata"
    ),
    source_match_df.assign(
        search_scope="source_outside_data_tree"
    ),
    drought_match_df.assign(
        search_scope="drought_source_candidates"
    ),
]

provenance_hits_df = pd.concat(
    all_hit_frames,
    ignore_index=True,
)

if provenance_hits_df.empty:
    provenance_hits_df = pd.DataFrame(
        columns=[
            "search_scope",
            "term",
            "path",
            "line_number",
            "context",
        ]
    )

provenance_hits_df = (
    provenance_hits_df.drop_duplicates(
        subset=[
            "search_scope",
            "term",
            "path",
            "line_number",
            "context",
        ]
    )
    .sort_values(
        [
            "search_scope",
            "path",
            "line_number",
            "term",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

exact_feature_terms = set(
    temporal_sequence_features
)

exact_feature_hits_df = provenance_hits_df.loc[
    provenance_hits_df["term"].isin(
        exact_feature_terms
    )
].copy()

formula_keyword_hits_df = provenance_hits_df.loc[
    provenance_hits_df["term"].isin(
        {
            "seq_lag12",
            "seq_lag24",
            "seq_delta12",
            "seq_delta24",
            "seq_rollmean24",
            "seq_rollstd24",
            "__is_missing",
        }
    )
].copy()

provenance_summary_df = pd.DataFrame(
    [
        {
            "metric": "priority_metadata_files_scanned",
            "value": len(priority_files),
        },
        {
            "metric": "source_code_files_scanned",
            "value": len(source_code_files),
        },
        {
            "metric": "drought_source_files_scanned",
            "value": len(drought_source_files),
        },
        {
            "metric": "all_text_matches",
            "value": len(provenance_hits_df),
        },
        {
            "metric": "exact_missing_sequence_feature_hits",
            "value": len(exact_feature_hits_df),
        },
        {
            "metric": "formula_keyword_hits",
            "value": len(formula_keyword_hits_df),
        },
    ]
)

has_exact_sequence_provenance = bool(
    not exact_feature_hits_df.empty
)

has_formula_context = bool(
    not formula_keyword_hits_df.empty
)

if has_exact_sequence_provenance:
    CELL7_STATUS = (
        "source_provenance_found_review_formula_context_before_derivation"
    )

elif has_formula_context:
    CELL7_STATUS = (
        "partial_formula_provenance_found_manual_context_review_required"
    )

else:
    CELL7_STATUS = (
        "no_explicit_formula_provenance_found_formula_parity_audit_required"
    )

print("\n=== Provenance search summary ===")
print(provenance_summary_df.to_string(index=False))

print("\n=== Cell 7 conclusion ===")
print("Status:", CELL7_STATUS)

if has_exact_sequence_provenance:
    print(
        "\nExact sequence-feature references were found. "
        "We must inspect their surrounding code before producing "
        "a derivation cell."
    )

elif has_formula_context:
    print(
        "\nFormula-related references were found, but no exact feature "
        "reference was confirmed. We must inspect the matched context "
        "before deriving features."
    )

else:
    print(
        "\nNo usable original formula was located in the searched "
        "metadata/source paths. The next step will be a controlled "
        "formula-parity audit against release-frame feature values."
    )


# -----------------------------------------------------------------------------
# 8. Save outputs
# -----------------------------------------------------------------------------
priority_hits_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_priority_metadata_hits_{CELL7_RUN_ID}.csv"
)

source_hits_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_source_code_hits_{CELL7_RUN_ID}.csv"
)

drought_hits_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_drought_source_hits_{CELL7_RUN_ID}.csv"
)

consolidated_hits_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_consolidated_provenance_hits_{CELL7_RUN_ID}.csv"
)

summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_provenance_summary_{CELL7_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell7_feature_provenance_search_manifest_{CELL7_RUN_ID}.json"
)

priority_match_df.to_csv(
    priority_hits_path,
    index=False,
)

source_match_df.to_csv(
    source_hits_path,
    index=False,
)

drought_match_df.to_csv(
    drought_hits_path,
    index=False,
)

provenance_hits_df.to_csv(
    consolidated_hits_path,
    index=False,
)

provenance_summary_df.to_csv(
    summary_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL7_RUN_ID,
    "status": CELL7_STATUS,
    "purpose": (
        "Locate source provenance for SPI12 long-horizon sequence "
        "features before attempting feature recreation."
    ),
    "search_terms": SEARCH_TERMS,
    "exact_sequence_feature_count": len(
        temporal_sequence_features
    ),
    "search_roots": {
        "release_handoff_root": str(
            RELEASE_HANDOFF_ROOT
        ),
        "v2_artifact_root": str(
            V2_ARTIFACT_ROOT
        ),
        "source_roots_outside_data": [
            str(path)
            for path in source_code_roots
        ],
        "drought_source_roots": [
            str(path)
            for path in drought_source_roots
        ],
    },
    "summary": provenance_summary_df.to_dict(
        orient="records"
    ),
    "outputs": {
        "priority_metadata_hits_csv": str(
            priority_hits_path
        ),
        "source_code_hits_csv": str(
            source_hits_path
        ),
        "drought_source_hits_csv": str(
            drought_hits_path
        ),
        "consolidated_hits_csv": str(
            consolidated_hits_path
        ),
        "summary_csv": str(summary_path),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 7 outputs ===")
print("Priority metadata hits ->", priority_hits_path)
print("Source-code hits       ->", source_hits_path)
print("Drought-source hits    ->", drought_hits_path)
print("Consolidated hits      ->", consolidated_hits_path)
print("Summary                ->", summary_path)
print("Manifest               ->", manifest_path)

print("\nCell 7 complete.")

=== Cell 7 purpose ===
Locate original SPI12 temporal-feature engineering logic before reproducing any missing features.

=== Search roots ===
Risks root            : C:\Projects\ramangwana\risks
Release handoff root  : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff
V2 artifact root      : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts
SPI12 contract        : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts\spi12\v2_20260617\feature_contract.json
Output directory      : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID                : 20260620T113613Z

=== SPI12 feature-family scope ===
Total SPI12 contract features : 111
Sequence feature count        : 48
Missingness indicator count   : 20

Sequence features:
- rain_chirps_seq_lag12
- rain_chirps_seq_lag24
- r

In [12]:
# validate_events_ng.ipynb — Cell 8
# Purpose:
# - Extract and preserve the exact SPI12 feature-generation source context.
# - Recreate the 68 SPI12 fields absent from temporal_predictors_full.
# - Use the formula located in the release-bundle generation notebook.
# - Compare recreated values against release-frame values for six
#   deterministic overlapping months.
#
# Scope:
# - No model inference.
# - No raw-data/API rebuilding.
# - No permanent alteration of temporal_predictors_full.
# - A failed parity check stops downstream full-predictor inference.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import re

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths and constants
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "OUTPUT_DIR is missing. Rerun Cell 0 first."
    )

CELL8_RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%SZ"
)

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

DUCKDB_TEMP_DIR = (
    EVENT_OUTPUT_DIR
    / "_duckdb_formula_parity_temp"
)
DUCKDB_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

FULL_PREDICTOR_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "temporal_predictors_full_20260614T141118Z.parquet"
)

RELEASE_FEATURE_FRAME_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_outputs"
    / "release_v2_package_handoff"
    / "release_bundle_predictions"
    / "release_bundle_prediction_feature_frame_20260618T152226Z.parquet"
)

V2_RELEASE_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

SPI12_FEATURE_CONTRACT_PATH = (
    V2_RELEASE_ARTIFACT_ROOT
    / "spi12"
    / "v2_20260617"
    / "feature_contract.json"
)

RELEASE_FORMULA_NOTEBOOK_PATH = Path(
    r"C:\Projects\ramangwana\risks"
    r"\processing_scripts\drought\subsystems"
    r"\release_v2_package_handoff"
    r"\2_generate_release_bundle_predictions.ipynb"
)

KEY_COLUMNS = ["row", "col", "yyyymm"]
MISSINGNESS_SUFFIX = "__is_missing"

SEQUENCE_PATTERN = re.compile(
    r"^(?P<base>.+)_seq_"
    r"(?P<operation>lag|delta|rollmean|rollstd)"
    r"(?P<window>\d+)$"
)

# Spread across the available release period.
# Every selected month was confirmed available in Cell 4.
PARITY_MONTHS = [
    "201501",
    "201601",
    "201805",
    "201905",
    "202301",
    "202410",
]

NUMERIC_TOLERANCE = 1e-5
MAX_MISMATCH_ROWS_PER_FEATURE = 20

print("=== Cell 8 purpose ===")
print(
    "Recreate unresolved SPI12 features from the full predictor "
    "source and test exact formula parity against the release frame."
)

print("\n=== Paths ===")
print("Full predictor source :", FULL_PREDICTOR_SOURCE_PATH)
print("Release feature frame :", RELEASE_FEATURE_FRAME_PATH)
print("SPI12 contract        :", SPI12_FEATURE_CONTRACT_PATH)
print("Release formula source:", RELEASE_FORMULA_NOTEBOOK_PATH)
print("DuckDB temp directory :", DUCKDB_TEMP_DIR)
print("Output directory      :", EVENT_OUTPUT_DIR)
print("Run ID                :", CELL8_RUN_ID)

for required_path in [
    FULL_PREDICTOR_SOURCE_PATH,
    RELEASE_FEATURE_FRAME_PATH,
    SPI12_FEATURE_CONTRACT_PATH,
    RELEASE_FORMULA_NOTEBOOK_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 8 input is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"{path.name} is not a JSON object."
        )

    return payload


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    candidate_keys = [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]

    for key in candidate_keys:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            return list(value)

    raise ValueError(
        "No usable feature list was found in the SPI12 contract."
    )


def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def sql_path(path: Path) -> str:
    return (
        str(path)
        .replace("\\", "/")
        .replace("'", "''")
    )


def normalize_keys(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    result = dataframe.copy()

    missing_keys = [
        column
        for column in KEY_COLUMNS
        if column not in result.columns
    ]

    if missing_keys:
        raise KeyError(
            "Missing key columns:\n"
            f"{missing_keys}"
        )

    result["row"] = pd.to_numeric(
        result["row"],
        errors="raise",
    ).astype("int64")

    result["col"] = pd.to_numeric(
        result["col"],
        errors="raise",
    ).astype("int64")

    result["yyyymm"] = (
        result["yyyymm"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    invalid_month_mask = ~result["yyyymm"].str.fullmatch(
        r"\d{6}"
    )

    if invalid_month_mask.any():
        invalid_values = (
            result.loc[
                invalid_month_mask,
                "yyyymm",
            ]
            .drop_duplicates()
            .head(20)
            .tolist()
        )

        raise AssertionError(
            "Invalid yyyymm values found:\n"
            f"{invalid_values}"
        )

    return result


def assert_unique_keys(
    dataframe: pd.DataFrame,
    dataframe_name: str,
) -> None:
    duplicate_count = int(
        dataframe.duplicated(
            subset=KEY_COLUMNS
        ).sum()
    )

    if duplicate_count:
        preview = (
            dataframe.loc[
                dataframe.duplicated(
                    subset=KEY_COLUMNS,
                    keep=False,
                ),
                KEY_COLUMNS,
            ]
            .sort_values(KEY_COLUMNS)
            .head(20)
        )

        raise AssertionError(
            f"{dataframe_name} has duplicate row/col/month keys: "
            f"{duplicate_count:,}\n"
            f"{preview.to_string(index=False)}"
        )


def parse_sequence_feature(
    feature_name: str,
) -> dict[str, Any]:
    match = SEQUENCE_PATTERN.match(feature_name)

    if not match:
        raise ValueError(
            f"Could not parse sequence feature: {feature_name}"
        )

    return {
        "feature_name": feature_name,
        "base_feature": match.group("base"),
        "operation": match.group("operation"),
        "window": int(match.group("window")),
    }


def read_release_month_subset(
    parquet_path: Path,
    columns: list[str],
    selected_months: list[str],
) -> pd.DataFrame:
    dataset = ds.dataset(
        parquet_path,
        format="parquet",
    )

    schema_columns = set(dataset.schema.names)

    missing_columns = [
        column
        for column in columns
        if column not in schema_columns
    ]

    if missing_columns:
        raise KeyError(
            "Release frame missing required columns:\n"
            f"{missing_columns}"
        )

    yyyymm_field_type = dataset.schema.field(
        "yyyymm"
    ).type

    if pa.types.is_integer(yyyymm_field_type):
        month_filter_values = [
            int(month)
            for month in selected_months
        ]
    else:
        month_filter_values = selected_months

    table = dataset.to_table(
        columns=columns,
        filter=ds.field("yyyymm").isin(
            month_filter_values
        ),
    )

    return normalize_keys(table.to_pandas())


def extract_formula_context(
    notebook_path: Path,
) -> tuple[str, int, list[str]]:
    notebook = json.loads(
        notebook_path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    )

    code_cells = [
        cell
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    ]

    for code_cell_index, cell in enumerate(
        code_cells,
        start=1,
    ):
        source = cell.get("source", [])

        if isinstance(source, str):
            lines = source.splitlines()
        else:
            lines = "".join(source).splitlines()

        marker_candidates = [
            line_index
            for line_index, line in enumerate(lines)
            if "for c in sequence_base_cols" in line
        ]

        if not marker_candidates:
            continue

        marker_line_index = marker_candidates[0]

        context_start = max(
            0,
            marker_line_index - 25,
        )

        context_end = min(
            len(lines),
            marker_line_index + 95,
        )

        context_lines = [
            f"{line_index + 1:>4}: {line}"
            for line_index, line in enumerate(
                lines[context_start:context_end],
                start=context_start,
            )
        ]

        return (
            f"code_cell_{code_cell_index}",
            marker_line_index + 1,
            context_lines,
        )

    raise AssertionError(
        "Could not find the sequence-feature generation loop in "
        "the release formula notebook."
    )


# -----------------------------------------------------------------------------
# 3. Read contract and identify exactly what must be recreated
# -----------------------------------------------------------------------------
spi12_contract = read_json_object(
    SPI12_FEATURE_CONTRACT_PATH
)

spi12_features = extract_feature_list(
    spi12_contract
)

predictor_schema_columns = set(
    pq.ParquetFile(
        FULL_PREDICTOR_SOURCE_PATH
    ).schema.names
)

release_schema_columns = set(
    pq.ParquetFile(
        RELEASE_FEATURE_FRAME_PATH
    ).schema.names
)

missing_from_predictor = [
    feature
    for feature in spi12_features
    if feature not in predictor_schema_columns
]

missing_from_release = [
    feature
    for feature in spi12_features
    if feature not in release_schema_columns
]

if missing_from_release:
    raise AssertionError(
        "Release feature frame is missing SPI12 contract features:\n"
        f"{missing_from_release}"
    )

sequence_features = [
    feature
    for feature in missing_from_predictor
    if "_seq_" in feature
]

indicator_features = [
    feature
    for feature in missing_from_predictor
    if feature.endswith(MISSINGNESS_SUFFIX)
]

unclassified_features = [
    feature
    for feature in missing_from_predictor
    if (
        feature not in sequence_features
        and feature not in indicator_features
    )
]

if unclassified_features:
    raise AssertionError(
        "Cannot proceed: unresolved SPI12 features are neither "
        "sequence features nor missingness indicators:\n"
        f"{unclassified_features}"
    )

if len(missing_from_predictor) != 68:
    raise AssertionError(
        "Unexpected count of missing SPI12 features. "
        f"Expected 68, found {len(missing_from_predictor)}."
    )

print("\n=== SPI12 recreation scope ===")
print("SPI12 contract features       :", len(spi12_features))
print("Missing from full predictor   :", len(missing_from_predictor))
print("Sequence features to recreate :", len(sequence_features))
print("Indicators to recreate        :", len(indicator_features))


# -----------------------------------------------------------------------------
# 4. Preserve the exact release-formula context
# -----------------------------------------------------------------------------
(
    formula_code_cell,
    formula_marker_line,
    formula_context_lines,
) = extract_formula_context(
    RELEASE_FORMULA_NOTEBOOK_PATH
)

print("\n=== Located release formula context ===")
print("Notebook code cell :", formula_code_cell)
print("Marker source line :", formula_marker_line)

for context_line in formula_context_lines:
    print(context_line)


# -----------------------------------------------------------------------------
# 5. Build formal feature-generation specification
# -----------------------------------------------------------------------------
sequence_specs = [
    parse_sequence_feature(feature)
    for feature in sequence_features
]

sequence_spec_df = pd.DataFrame(
    sequence_specs
).sort_values(
    [
        "base_feature",
        "operation",
        "window",
        "feature_name",
    ]
).reset_index(drop=True)

sequence_base_features = sorted(
    sequence_spec_df["base_feature"].unique().tolist()
)

indicator_base_features = sorted(
    {
        feature[: -len(MISSINGNESS_SUFFIX)]
        for feature in indicator_features
    }
)

missing_base_features = [
    base_feature
    for base_feature in (
        sequence_base_features
        + indicator_base_features
    )
    if base_feature not in predictor_schema_columns
]

if missing_base_features:
    raise AssertionError(
        "Full predictor source lacks required base features:\n"
        f"{sorted(set(missing_base_features))}"
    )

formula_records: list[dict[str, Any]] = []

for _, spec in sequence_spec_df.iterrows():
    base_feature = spec["base_feature"]
    operation = spec["operation"]
    window = int(spec["window"])
    feature_name = spec["feature_name"]

    existing_lag12_column = (
        f"{base_feature}_lag12"
    )

    if operation == "lag" and window == 12:
        if existing_lag12_column in predictor_schema_columns:
            derivation = (
                f"use_existing_source_column:"
                f"{existing_lag12_column}"
            )
        else:
            derivation = (
                "grouped_shift_12_over_row_col_sorted_yyyymm"
            )

    elif operation == "lag" and window == 24:
        derivation = (
            "grouped_shift_24_over_row_col_sorted_yyyymm"
        )

    elif operation == "rollmean" and window == 24:
        derivation = (
            "grouped_trailing_24_row_rolling_mean_"
            "min_periods_3_current_month_included"
        )

    elif operation == "rollstd" and window == 24:
        derivation = (
            "grouped_trailing_24_row_rolling_sample_std_"
            "min_periods_3_current_month_included"
        )

    elif operation == "delta" and window == 12:
        derivation = (
            f"{base_feature}_minus_{base_feature}_seq_lag12"
        )

    elif operation == "delta" and window == 24:
        derivation = (
            f"{base_feature}_minus_{base_feature}_seq_lag24"
        )

    else:
        raise AssertionError(
            "Unexpected sequence operation/window combination: "
            f"{feature_name}"
        )

    formula_records.append(
        {
            "feature_name": feature_name,
            "feature_family": "temporal_sequence",
            "base_feature": base_feature,
            "operation": operation,
            "window": window,
            "derivation": derivation,
        }
    )

for indicator_feature in sorted(indicator_features):
    base_feature = indicator_feature[
        : -len(MISSINGNESS_SUFFIX)
    ]

    formula_records.append(
        {
            "feature_name": indicator_feature,
            "feature_family": "missingness_indicator",
            "base_feature": base_feature,
            "operation": "is_missing",
            "window": None,
            "derivation": (
                f"int8({base_feature}.isna())"
            ),
        }
    )

formula_spec_df = pd.DataFrame(
    formula_records
).sort_values(
    [
        "feature_family",
        "base_feature",
        "operation",
        "window",
        "feature_name",
    ],
    na_position="last",
).reset_index(drop=True)

print("\n=== Formula specification ===")
print(
    formula_spec_df.to_string(
        index=False,
        max_colwidth=160,
    )
)


# -----------------------------------------------------------------------------
# 6. Build DuckDB query matching the release-generation feature logic
# -----------------------------------------------------------------------------
source_select_columns = (
    KEY_COLUMNS
    + sequence_base_features
    + indicator_base_features
)

for base_feature in sequence_base_features:
    existing_lag12_column = (
        f"{base_feature}_lag12"
    )

    if existing_lag12_column in predictor_schema_columns:
        source_select_columns.append(
            existing_lag12_column
        )

source_select_columns = list(
    dict.fromkeys(source_select_columns)
)

base_select_sql_parts: list[str] = []

for column_name in source_select_columns:
    quoted_column = quote_identifier(column_name)

    if column_name == "row":
        base_select_sql_parts.append(
            f"CAST({quoted_column} AS BIGINT) AS {quoted_column}"
        )

    elif column_name == "col":
        base_select_sql_parts.append(
            f"CAST({quoted_column} AS BIGINT) AS {quoted_column}"
        )

    elif column_name == "yyyymm":
        base_select_sql_parts.append(
            f"CAST({quoted_column} AS VARCHAR) AS {quoted_column}"
        )

    else:
        base_select_sql_parts.append(
            f"CAST({quoted_column} AS FLOAT) AS {quoted_column}"
        )

sequence_stage_sql_parts = [
    quote_identifier(column_name)
    for column_name in source_select_columns
]

internal_sequence_columns: dict[
    tuple[str, str, int],
    str,
] = {}

for base_feature in sequence_base_features:
    quoted_base = quote_identifier(base_feature)

    existing_lag12_column = (
        f"{base_feature}_lag12"
    )

    if existing_lag12_column in predictor_schema_columns:
        lag12_expression = (
            f"CAST({quote_identifier(existing_lag12_column)} "
            f"AS FLOAT)"
        )
        lag12_strategy = "existing_lag12_source_column"
    else:
        lag12_expression = (
            f"CAST(LAG({quoted_base}, 12) OVER w AS FLOAT)"
        )
        lag12_strategy = "group_shift_12"

    lag12_internal = (
        f"__{base_feature}_seq_lag12_internal"
    )

    lag24_internal = (
        f"__{base_feature}_seq_lag24_internal"
    )

    rollmean24_internal = (
        f"__{base_feature}_seq_rollmean24_internal"
    )

    rollstd24_internal = (
        f"__{base_feature}_seq_rollstd24_internal"
    )

    internal_sequence_columns[
        (base_feature, "lag", 12)
    ] = lag12_internal

    internal_sequence_columns[
        (base_feature, "lag", 24)
    ] = lag24_internal

    internal_sequence_columns[
        (base_feature, "rollmean", 24)
    ] = rollmean24_internal

    internal_sequence_columns[
        (base_feature, "rollstd", 24)
    ] = rollstd24_internal

    sequence_stage_sql_parts.extend(
        [
            (
                f"{lag12_expression} AS "
                f"{quote_identifier(lag12_internal)}"
            ),
            (
                f"CAST(LAG({quoted_base}, 24) OVER w AS FLOAT) AS "
                f"{quote_identifier(lag24_internal)}"
            ),
            (
                "CASE "
                f"WHEN COUNT({quoted_base}) OVER w24 >= 3 "
                f"THEN CAST(AVG({quoted_base}) OVER w24 AS FLOAT) "
                "ELSE CAST(NULL AS FLOAT) "
                f"END AS {quote_identifier(rollmean24_internal)}"
            ),
            (
                "CASE "
                f"WHEN COUNT({quoted_base}) OVER w24 >= 3 "
                f"THEN CAST(STDDEV_SAMP({quoted_base}) OVER w24 AS FLOAT) "
                "ELSE CAST(NULL AS FLOAT) "
                f"END AS {quote_identifier(rollstd24_internal)}"
            ),
        ]
    )

derived_select_sql_parts = [
    quote_identifier(column_name)
    for column_name in KEY_COLUMNS
]

for _, spec in sequence_spec_df.iterrows():
    feature_name = spec["feature_name"]
    base_feature = spec["base_feature"]
    operation = spec["operation"]
    window = int(spec["window"])

    quoted_feature = quote_identifier(feature_name)
    quoted_base = quote_identifier(base_feature)

    if operation == "lag":
        internal_column = internal_sequence_columns[
            (base_feature, "lag", window)
        ]

        expression = quote_identifier(internal_column)

    elif operation == "rollmean":
        internal_column = internal_sequence_columns[
            (base_feature, "rollmean", window)
        ]

        expression = quote_identifier(internal_column)

    elif operation == "rollstd":
        internal_column = internal_sequence_columns[
            (base_feature, "rollstd", window)
        ]

        expression = quote_identifier(internal_column)

    elif operation == "delta":
        lag_internal = internal_sequence_columns[
            (base_feature, "lag", window)
        ]

        expression = (
            f"CAST({quoted_base} - "
            f"{quote_identifier(lag_internal)} AS FLOAT)"
        )

    else:
        raise AssertionError(
            f"Unsupported operation: {operation}"
        )

    derived_select_sql_parts.append(
        f"{expression} AS {quoted_feature}"
    )

for indicator_feature in sorted(indicator_features):
    base_feature = indicator_feature[
        : -len(MISSINGNESS_SUFFIX)
    ]

    derived_select_sql_parts.append(
        (
            "CAST(CASE "
            f"WHEN {quote_identifier(base_feature)} IS NULL "
            "THEN 1 ELSE 0 END AS TINYINT) "
            f"AS {quote_identifier(indicator_feature)}"
        )
    )

month_list_sql = ", ".join(
    f"'{month}'"
    for month in PARITY_MONTHS
)

query_sql = f"""
WITH base_context AS (
    SELECT
        {", ".join(base_select_sql_parts)}
    FROM read_parquet('{sql_path(FULL_PREDICTOR_SOURCE_PATH)}')
),
sequence_stage AS (
    SELECT
        {", ".join(sequence_stage_sql_parts)}
    FROM base_context
    WINDOW
        w AS (
            PARTITION BY "row", "col"
            ORDER BY "yyyymm"
        ),
        w24 AS (
            PARTITION BY "row", "col"
            ORDER BY "yyyymm"
            ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
        )
),
derived_features AS (
    SELECT
        {", ".join(derived_select_sql_parts)}
    FROM sequence_stage
)
SELECT *
FROM derived_features
WHERE "yyyymm" IN ({month_list_sql})
ORDER BY "yyyymm", "row", "col"
"""

print("\n=== Formula execution plan ===")
print("Parity months:", ", ".join(PARITY_MONTHS))
print(
    "Sequence base features:",
    ", ".join(sequence_base_features),
)
print(
    "Indicator source features:",
    len(indicator_base_features),
)
print(
    "Expected recreated feature count:",
    len(missing_from_predictor),
)


# -----------------------------------------------------------------------------
# 7. Recreate feature values from full predictor source
# -----------------------------------------------------------------------------
duckdb_connection = duckdb.connect(
    database=":memory:"
)

duckdb_connection.execute(
    "SET memory_limit = '6GB'"
)

duckdb_connection.execute(
    f"SET temp_directory = '{sql_path(DUCKDB_TEMP_DIR)}'"
)

duckdb_connection.execute(
    "PRAGMA threads = 4"
)

try:
    recreated_feature_df = duckdb_connection.execute(
        query_sql
    ).fetch_df()

finally:
    duckdb_connection.close()

recreated_feature_df = normalize_keys(
    recreated_feature_df
)

assert_unique_keys(
    recreated_feature_df,
    "recreated SPI12 feature frame",
)

expected_recreated_rows = (
    len(PARITY_MONTHS) * 26775
)

print("\n=== Recreated full-predictor feature frame ===")
print("Rows recreated:", f"{len(recreated_feature_df):,}")
print(
    "Expected rows from six complete months:",
    f"{expected_recreated_rows:,}",
)
print(
    "Columns returned:",
    len(recreated_feature_df.columns),
)

recreated_month_counts_df = (
    recreated_feature_df.groupby("yyyymm")
    .size()
    .reset_index(name="row_count")
    .sort_values("yyyymm")
    .reset_index(drop=True)
)

print("\nRows by parity month:")
print(
    recreated_month_counts_df.to_string(
        index=False
    )
)

if len(recreated_feature_df) != expected_recreated_rows:
    raise AssertionError(
        "Unexpected recreated row count. "
        "Stop before formula comparison."
    )


# -----------------------------------------------------------------------------
# 8. Read matching release-frame values
# -----------------------------------------------------------------------------
release_comparison_columns = (
    KEY_COLUMNS
    + missing_from_predictor
)

release_feature_df = read_release_month_subset(
    parquet_path=RELEASE_FEATURE_FRAME_PATH,
    columns=release_comparison_columns,
    selected_months=PARITY_MONTHS,
)

assert_unique_keys(
    release_feature_df,
    "release SPI12 comparison frame",
)

print("\n=== Matching release feature frame ===")
print("Release rows loaded:", f"{len(release_feature_df):,}")

release_month_counts_df = (
    release_feature_df.groupby("yyyymm")
    .size()
    .reset_index(name="row_count")
    .sort_values("yyyymm")
    .reset_index(drop=True)
)

print("\nRelease rows by parity month:")
print(
    release_month_counts_df.to_string(
        index=False
    )
)

if len(release_feature_df) != expected_recreated_rows:
    raise AssertionError(
        "Unexpected release row count for parity months. "
        "Stop before formula comparison."
    )


# -----------------------------------------------------------------------------
# 9. Feature-by-feature parity comparison
# -----------------------------------------------------------------------------
comparison_df = recreated_feature_df.merge(
    release_feature_df,
    on=KEY_COLUMNS,
    how="inner",
    suffixes=("_recreated", "_release"),
    validate="one_to_one",
)

if len(comparison_df) != expected_recreated_rows:
    raise AssertionError(
        "Feature comparison lost row/col/month matches.\n"
        f"Expected: {expected_recreated_rows:,}\n"
        f"Matched : {len(comparison_df):,}"
    )

formula_lookup = (
    formula_spec_df.set_index("feature_name")
    .to_dict(orient="index")
)

feature_parity_records: list[dict[str, Any]] = []
mismatch_frames: list[pd.DataFrame] = []

for feature_name in missing_from_predictor:
    recreated_column = f"{feature_name}_recreated"
    release_column = f"{feature_name}_release"

    recreated_values = pd.to_numeric(
        comparison_df[recreated_column],
        errors="coerce",
    )

    release_values = pd.to_numeric(
        comparison_df[release_column],
        errors="coerce",
    )

    both_null_mask = (
        recreated_values.isna()
        & release_values.isna()
    )

    one_null_mask = (
        recreated_values.isna()
        ^ release_values.isna()
    )

    both_numeric_mask = (
        recreated_values.notna()
        & release_values.notna()
    )

    absolute_delta = pd.Series(
        np.nan,
        index=comparison_df.index,
        dtype="float64",
    )

    absolute_delta.loc[both_numeric_mask] = (
        recreated_values.loc[both_numeric_mask]
        - release_values.loc[both_numeric_mask]
    ).abs()

    numeric_mismatch_mask = (
        both_numeric_mask
        & absolute_delta.gt(NUMERIC_TOLERANCE)
    )

    mismatch_mask = (
        one_null_mask
        | numeric_mismatch_mask
    )

    valid_numeric_deltas = absolute_delta.loc[
        both_numeric_mask
    ].dropna()

    feature_metadata = formula_lookup[feature_name]

    feature_parity_records.append(
        {
            "feature_name": feature_name,
            "feature_family": feature_metadata[
                "feature_family"
            ],
            "base_feature": feature_metadata[
                "base_feature"
            ],
            "operation": feature_metadata[
                "operation"
            ],
            "window": feature_metadata[
                "window"
            ],
            "derivation": feature_metadata[
                "derivation"
            ],
            "comparison_rows": int(len(comparison_df)),
            "both_null_count": int(both_null_mask.sum()),
            "one_null_mismatch_count": int(
                one_null_mask.sum()
            ),
            "numeric_comparison_count": int(
                both_numeric_mask.sum()
            ),
            "numeric_mismatch_count": int(
                numeric_mismatch_mask.sum()
            ),
            "total_mismatch_count": int(
                mismatch_mask.sum()
            ),
            "max_abs_delta": (
                float(valid_numeric_deltas.max())
                if not valid_numeric_deltas.empty
                else np.nan
            ),
            "mean_abs_delta": (
                float(valid_numeric_deltas.mean())
                if not valid_numeric_deltas.empty
                else np.nan
            ),
            "parity_passed": bool(
                mismatch_mask.sum() == 0
            ),
        }
    )

    if mismatch_mask.any():
        mismatch_sample_df = comparison_df.loc[
            mismatch_mask,
            KEY_COLUMNS
            + [
                recreated_column,
                release_column,
            ],
        ].copy()

        mismatch_sample_df.insert(
            3,
            "feature_name",
            feature_name,
        )

        mismatch_sample_df["abs_delta"] = (
            absolute_delta.loc[
                mismatch_sample_df.index
            ].to_numpy()
        )

        mismatch_sample_df["mismatch_type"] = np.where(
            one_null_mask.loc[
                mismatch_sample_df.index
            ].to_numpy(),
            "null_pattern_mismatch",
            "numeric_tolerance_exceeded",
        )

        mismatch_frames.append(
            mismatch_sample_df.head(
                MAX_MISMATCH_ROWS_PER_FEATURE
            )
        )

feature_parity_summary_df = pd.DataFrame(
    feature_parity_records
).sort_values(
    [
        "feature_family",
        "base_feature",
        "operation",
        "window",
        "feature_name",
    ],
    na_position="last",
).reset_index(drop=True)

if mismatch_frames:
    mismatch_sample_df = pd.concat(
        mismatch_frames,
        ignore_index=True,
    )
else:
    mismatch_sample_df = pd.DataFrame(
        columns=
        KEY_COLUMNS
        + [
            "feature_name",
            "abs_delta",
            "mismatch_type",
        ]
    )

print("\n=== SPI12 formula parity summary ===")
print(
    feature_parity_summary_df.to_string(
        index=False,
        max_colwidth=150,
    )
)

total_feature_count = int(
    len(feature_parity_summary_df)
)

passed_feature_count = int(
    feature_parity_summary_df[
        "parity_passed"
    ].sum()
)

failed_feature_count = int(
    total_feature_count
    - passed_feature_count
)

total_value_mismatch_count = int(
    feature_parity_summary_df[
        "total_mismatch_count"
    ].sum()
)

print("\n=== Formula parity totals ===")
print("Features compared       :", total_feature_count)
print("Features passed         :", passed_feature_count)
print("Features failed         :", failed_feature_count)
print(
    "Total value mismatches :",
    f"{total_value_mismatch_count:,}",
)
print("Tolerance used          :", NUMERIC_TOLERANCE)

if not mismatch_sample_df.empty:
    print("\n=== Mismatch sample ===")
    print(
        mismatch_sample_df.to_string(
            index=False,
            max_rows=200,
            max_colwidth=120,
        )
    )


# -----------------------------------------------------------------------------
# 10. Final Cell 8 decision
# -----------------------------------------------------------------------------
formula_parity_passed = bool(
    failed_feature_count == 0
)

if formula_parity_passed:
    CELL8_STATUS = (
        "spi12_full_predictor_feature_recreation_"
        "parity_passed_for_release_overlap_months"
    )
else:
    CELL8_STATUS = (
        "spi12_full_predictor_feature_recreation_"
        "parity_failed_stop_before_inference"
    )

formula_parity_decision_df = pd.DataFrame(
    [
        {
            "check": "release_formula_source_found",
            "result": True,
        },
        {
            "check": "all_68_missing_features_recreated",
            "result": (
                len(missing_from_predictor) == 68
            ),
        },
        {
            "check": "all_parity_months_matched",
            "result": (
                len(comparison_df)
                == expected_recreated_rows
            ),
        },
        {
            "check": "all_feature_values_pass_parity",
            "result": formula_parity_passed,
        },
        {
            "check": (
                "full_predictor_ready_for_spi12_event_inference"
            ),
            "result": formula_parity_passed,
        },
    ]
)

print("\n=== Cell 8 conclusion ===")
print("Status:", CELL8_STATUS)
print(
    formula_parity_decision_df.to_string(
        index=False
    )
)

print(
    "\nInterpretation:\n"
    "- A pass means the full predictor source can supply all "
    "SPI12 inputs for months it contains, using the verified "
    "release formula.\n"
    "- A fail means no full-predictor SPI12 inference should occur "
    "until the mismatch is explained.\n"
    "- This test does not yet assess drought-event agreement with "
    "the Excel severity scores."
)


# -----------------------------------------------------------------------------
# 11. Save outputs
# -----------------------------------------------------------------------------
formula_context_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_release_formula_context_{CELL8_RUN_ID}.txt"
)

formula_spec_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_spi12_formula_specification_{CELL8_RUN_ID}.csv"
)

recreated_month_counts_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_recreated_month_counts_{CELL8_RUN_ID}.csv"
)

release_month_counts_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_release_month_counts_{CELL8_RUN_ID}.csv"
)

feature_parity_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_spi12_feature_parity_summary_{CELL8_RUN_ID}.csv"
)

mismatch_sample_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_spi12_feature_mismatch_sample_{CELL8_RUN_ID}.csv"
)

decision_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_spi12_formula_parity_decision_{CELL8_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell8_spi12_formula_parity_manifest_{CELL8_RUN_ID}.json"
)

formula_context_path.write_text(
    "\n".join(
        [
            "Release formula notebook:",
            str(RELEASE_FORMULA_NOTEBOOK_PATH),
            "",
            f"Located code cell: {formula_code_cell}",
            f"Marker source line: {formula_marker_line}",
            "",
            *formula_context_lines,
        ]
    ),
    encoding="utf-8",
)

formula_spec_df.to_csv(
    formula_spec_path,
    index=False,
)

recreated_month_counts_df.to_csv(
    recreated_month_counts_path,
    index=False,
)

release_month_counts_df.to_csv(
    release_month_counts_path,
    index=False,
)

feature_parity_summary_df.to_csv(
    feature_parity_summary_path,
    index=False,
)

mismatch_sample_df.to_csv(
    mismatch_sample_path,
    index=False,
)

formula_parity_decision_df.to_csv(
    decision_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL8_RUN_ID,
    "status": CELL8_STATUS,
    "purpose": (
        "Formula-parity validation for SPI12 fields absent from "
        "temporal_predictors_full."
    ),
    "release_formula_notebook": str(
        RELEASE_FORMULA_NOTEBOOK_PATH
    ),
    "formula_code_cell": formula_code_cell,
    "formula_marker_line": formula_marker_line,
    "full_predictor_source_path": str(
        FULL_PREDICTOR_SOURCE_PATH
    ),
    "release_feature_frame_path": str(
        RELEASE_FEATURE_FRAME_PATH
    ),
    "parity_months": PARITY_MONTHS,
    "expected_comparison_rows": int(
        expected_recreated_rows
    ),
    "matched_comparison_rows": int(
        len(comparison_df)
    ),
    "numeric_tolerance": NUMERIC_TOLERANCE,
    "recreated_feature_count": int(
        len(missing_from_predictor)
    ),
    "passed_feature_count": passed_feature_count,
    "failed_feature_count": failed_feature_count,
    "total_value_mismatch_count": (
        total_value_mismatch_count
    ),
    "formula_parity_passed": formula_parity_passed,
    "outputs": {
        "release_formula_context_txt": str(
            formula_context_path
        ),
        "formula_specification_csv": str(
            formula_spec_path
        ),
        "recreated_month_counts_csv": str(
            recreated_month_counts_path
        ),
        "release_month_counts_csv": str(
            release_month_counts_path
        ),
        "feature_parity_summary_csv": str(
            feature_parity_summary_path
        ),
        "mismatch_sample_csv": str(
            mismatch_sample_path
        ),
        "decision_csv": str(decision_path),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 8 outputs ===")
print("Release formula context ->", formula_context_path)
print("Formula specification   ->", formula_spec_path)
print("Recreated month counts  ->", recreated_month_counts_path)
print("Release month counts    ->", release_month_counts_path)
print("Parity summary          ->", feature_parity_summary_path)
print("Mismatch sample         ->", mismatch_sample_path)
print("Decision                ->", decision_path)
print("Manifest                ->", manifest_path)

print("\nCell 8 complete.")

=== Cell 8 purpose ===
Recreate unresolved SPI12 features from the full predictor source and test exact formula parity against the release frame.

=== Paths ===
Full predictor source : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Release feature frame : C:\Projects\ramangwana\risks\data\drought_model\processed\model_outputs\release_v2_package_handoff\release_bundle_predictions\release_bundle_prediction_feature_frame_20260618T152226Z.parquet
SPI12 contract        : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts\spi12\v2_20260617\feature_contract.json
Release formula source: C:\Projects\ramangwana\risks\processing_scripts\drought\subsystems\release_v2_package_handoff\2_generate_release_bundle_predictions.ipynb
DuckDB temp directory : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\_duckdb_formula_parity_temp


In [14]:
# validate_events_ng.ipynb — Cell 9
# Purpose:
# - Read the Excel Chronological Timeline directly.
# - Define each drought window from:
#       Growing Season start year
#       + stated Duration (Months)
# - Use October of the Growing Season's first year as the explicit
#   season-window anchor, then retain exactly the stated duration.
# - Keep Excel Severity Score (1–5) and Severity Label untouched as the
#   event-level external comparison reference.
# - Determine which exact event months are currently eligible for v2
#   inference from temporal_predictors_full with complete 24-month context.
#
# Scope:
# - No old-notebook search.
# - No model inference yet.
# - No raw-data reconstruction.
# - No mapping of Excel severity scores to WMO model classes yet.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import re

import duckdb
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Paths and fixed protocol
# -----------------------------------------------------------------------------
if "DATA_ROOT" not in globals():
    raise RuntimeError("DATA_ROOT is missing. Rerun Cell 0 first.")

if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL9_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

EVENT_OUTPUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
)
EVENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVENT_WORKBOOK_PATH = (
    DATA_ROOT
    / "events"
    / "Zimbabwe_Drought_Timeline_1902_2024.xlsx"
)

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

FULL_PREDICTOR_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "temporal_predictors_full_20260614T141118Z.parquet"
)

# Protocol:
# Zimbabwe agricultural season is represented in the workbook as YYYY/YY.
# The event window therefore starts in October of the first season year.
# The event then continues for exactly the Excel-stated duration.
SEASON_ANCHOR_MONTH = 10
REQUIRED_HISTORY_MONTHS = 24

YEAR_PATTERN = re.compile(
    r"(?<!\d)(?:18|19|20)\d{2}(?!\d)"
)

print("=== Cell 9 purpose ===")
print(
    "Build the direct Excel-derived, season-aligned drought-event "
    "ledger for final severity validation."
)

print("\n=== Inputs ===")
print("Event workbook       :", EVENT_WORKBOOK_PATH)
print("Full predictor source:", FULL_PREDICTOR_SOURCE_PATH)
print("Output directory     :", EVENT_OUTPUT_DIR)
print("Run ID               :", CELL9_RUN_ID)

print("\n=== Event-window protocol ===")
print(
    "Anchor month         :",
    f"{SEASON_ANCHOR_MONTH:02d} "
    "(October of Growing Season first year)",
)
print(
    "Window length        :",
    "exact Excel Duration (Months)",
)
print(
    "Required SPI12 context:",
    f"{REQUIRED_HISTORY_MONTHS} prior months plus event month",
)

for required_path in [
    EVENT_WORKBOOK_PATH,
    FULL_PREDICTOR_SOURCE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 9 input is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def normalize_column_name(value: Any) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def resolve_required_column(
    dataframe: pd.DataFrame,
    required_terms: list[str],
    role_name: str,
) -> str:
    normalized_columns = {
        column_name: normalize_column_name(column_name)
        for column_name in dataframe.columns
    }

    matches = [
        column_name
        for column_name, normalized_name in normalized_columns.items()
        if all(
            term in normalized_name
            for term in required_terms
        )
    ]

    if len(matches) != 1:
        raise KeyError(
            f"Could not uniquely identify the {role_name} column.\n"
            f"Required terms: {required_terms}\n"
            f"Matches: {matches}\n"
            f"Available columns: {list(dataframe.columns)}"
        )

    return matches[0]


def extract_years(value: Any) -> list[int]:
    return [
        int(year)
        for year in YEAR_PATTERN.findall(
            str(value)
        )
    ]


def parse_duration_months(value: Any) -> int:
    numbers = re.findall(
        r"\d+",
        str(value),
    )

    if not numbers:
        raise ValueError(
            f"Could not parse duration months from: {value!r}"
        )

    duration = int(numbers[0])

    if duration <= 0 or duration > 60:
        raise ValueError(
            f"Unreasonable duration extracted: {duration}"
        )

    return duration


def yyyymm_from_period(period: pd.Period) -> str:
    return period.strftime("%Y%m")


def period_from_yyyymm(yyyymm: str) -> pd.Period:
    return pd.Period(
        yyyymm,
        freq="M",
    )


def list_months(
    start_period: pd.Period,
    duration_months: int,
) -> list[str]:
    return [
        yyyymm_from_period(
            start_period + offset
        )
        for offset in range(duration_months)
    ]


def has_complete_spi12_context(
    event_month: str,
    complete_source_months: set[str],
    required_history_months: int,
) -> bool:
    event_period = period_from_yyyymm(event_month)

    required_periods = [
        event_period - offset
        for offset in range(
            required_history_months + 1
        )
    ]

    required_months = {
        yyyymm_from_period(period)
        for period in required_periods
    }

    return required_months.issubset(
        complete_source_months
    )


def source_status_for_month(
    event_month: str,
    source_months: set[str],
    complete_source_months: set[str],
) -> str:
    if event_month not in source_months:
        return "requires_raw_month_reconstruction"

    if event_month not in complete_source_months:
        return "source_month_incomplete_grid"

    if not has_complete_spi12_context(
        event_month=event_month,
        complete_source_months=complete_source_months,
        required_history_months=REQUIRED_HISTORY_MONTHS,
    ):
        return "requires_pre_source_history"

    return "ready_for_v2_inference"


# -----------------------------------------------------------------------------
# 3. Read exactly the authoritative Excel event table
# -----------------------------------------------------------------------------
timeline_df = pd.read_excel(
    EVENT_WORKBOOK_PATH,
    sheet_name="Chronological Timeline",
    header=2,
)

timeline_df = timeline_df.dropna(
    axis=0,
    how="all",
).copy()

if timeline_df.empty:
    raise AssertionError(
        "The Chronological Timeline sheet has no event rows."
    )

id_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["id"],
    role_name="ID",
)

event_period_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["event", "period"],
    role_name="Event Period",
)

growing_season_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["growing", "season"],
    role_name="Growing Season",
)

duration_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["duration", "months"],
    role_name="Duration (Months)",
)

severity_score_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["severity", "score"],
    role_name="Severity Score",
)

severity_label_column = resolve_required_column(
    dataframe=timeline_df,
    required_terms=["severity", "label"],
    role_name="Severity Label",
)

print("\n=== Excel fields used directly ===")
print("ID               :", id_column)
print("Event Period     :", event_period_column)
print("Growing Season   :", growing_season_column)
print("Duration Months  :", duration_column)
print("Severity Score   :", severity_score_column)
print("Severity Label   :", severity_label_column)

print("\n=== Excel timeline preview ===")
print(
    timeline_df[
        [
            id_column,
            event_period_column,
            growing_season_column,
            duration_column,
            severity_score_column,
            severity_label_column,
        ]
    ].to_string(
        index=False,
        max_colwidth=80,
    )
)


# -----------------------------------------------------------------------------
# 4. Inspect exact monthly coverage in the validated predictor source
# -----------------------------------------------------------------------------
source_path_sql = str(
    FULL_PREDICTOR_SOURCE_PATH
).replace("\\", "/").replace("'", "''")

duckdb_connection = duckdb.connect(
    database=":memory:"
)

try:
    source_month_coverage_df = duckdb_connection.execute(
        f"""
        SELECT
            CAST("yyyymm" AS VARCHAR) AS yyyymm,
            COUNT(*) AS source_row_count
        FROM read_parquet('{source_path_sql}')
        GROUP BY 1
        ORDER BY 1
        """
    ).fetch_df()

finally:
    duckdb_connection.close()

source_month_coverage_df["yyyymm"] = (
    source_month_coverage_df["yyyymm"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

source_month_coverage_df["source_row_count"] = (
    pd.to_numeric(
        source_month_coverage_df[
            "source_row_count"
        ],
        errors="raise",
    ).astype("int64")
)

source_month_coverage_df["month_period"] = pd.PeriodIndex(
    source_month_coverage_df["yyyymm"],
    freq="M",
)

source_grid_cell_count = int(
    source_month_coverage_df[
        "source_row_count"
    ].max()
)

source_month_coverage_df[
    "complete_grid_month"
] = source_month_coverage_df[
    "source_row_count"
].eq(source_grid_cell_count)

source_months = set(
    source_month_coverage_df["yyyymm"]
)

complete_source_months = set(
    source_month_coverage_df.loc[
        source_month_coverage_df[
            "complete_grid_month"
        ],
        "yyyymm",
    ]
)

source_first_period = source_month_coverage_df[
    "month_period"
].min()

source_last_period = source_month_coverage_df[
    "month_period"
].max()

expected_source_periods = pd.period_range(
    start=source_first_period,
    end=source_last_period,
    freq="M",
)

observed_source_periods = set(
    source_month_coverage_df["month_period"]
)

source_gap_periods = [
    period
    for period in expected_source_periods
    if period not in observed_source_periods
]

first_spi12_context_complete_period = (
    source_first_period
    + REQUIRED_HISTORY_MONTHS
)

print("\n=== Full predictor monthly coverage ===")
print(
    "Source range                 :",
    f"{source_first_period} -> {source_last_period}",
)
print(
    "Distinct source months       :",
    len(source_month_coverage_df),
)
print(
    "Expected continuous months   :",
    len(expected_source_periods),
)
print(
    "Internal source month gaps   :",
    len(source_gap_periods),
)
print(
    "Full-grid cells per month    :",
    f"{source_grid_cell_count:,}",
)
print(
    "First month with 24m context:",
    first_spi12_context_complete_period,
)

if source_gap_periods:
    raise AssertionError(
        "The full predictor source has internal monthly gaps. "
        "Stop before defining event eligibility.\n"
        f"Missing months: {[str(period) for period in source_gap_periods]}"
    )


# -----------------------------------------------------------------------------
# 5. Build the direct season-aligned Excel event ledger
# -----------------------------------------------------------------------------
event_records: list[dict[str, Any]] = []
event_month_records: list[dict[str, Any]] = []

for _, row in timeline_df.iterrows():
    event_id = pd.to_numeric(
        row[id_column],
        errors="raise",
    )

    event_period_raw = str(
        row[event_period_column]
    ).strip()

    growing_season_raw = str(
        row[growing_season_column]
    ).strip()

    duration_months = parse_duration_months(
        row[duration_column]
    )

    severity_score = pd.to_numeric(
        row[severity_score_column],
        errors="raise",
    )

    severity_label = str(
        row[severity_label_column]
    ).strip()

    event_period_years = extract_years(
        event_period_raw
    )

    season_years = extract_years(
        growing_season_raw
    )

    if not event_period_years:
        raise AssertionError(
            "Could not extract a year from Event Period for "
            f"event ID {event_id}: {event_period_raw!r}"
        )

    if not season_years:
        raise AssertionError(
            "Could not extract a year from Growing Season for "
            f"event ID {event_id}: {growing_season_raw!r}"
        )

    event_period_start_year = min(
        event_period_years
    )

    event_period_end_year = max(
        event_period_years
    )

    growing_season_start_year = min(
        season_years
    )

    event_start_period = pd.Period(
        year=growing_season_start_year,
        month=SEASON_ANCHOR_MONTH,
        freq="M",
    )

    event_end_period = (
        event_start_period
        + duration_months
        - 1
    )

    event_months = list_months(
        start_period=event_start_period,
        duration_months=duration_months,
    )

    event_period_start_period = pd.Period(
        year=event_period_start_year,
        month=1,
        freq="M",
    )

    event_period_end_period = pd.Period(
        year=event_period_end_year,
        month=12,
        freq="M",
    )

    window_within_event_period_years = bool(
        event_start_period >= event_period_start_period
        and event_end_period <= event_period_end_period
    )

    event_key = (
        f"excel_event_{int(event_id):02d}_"
        f"{event_period_raw.replace(' ', '_')}"
    )

    event_record = {
        "event_key": event_key,
        "excel_event_id": int(event_id),
        "event_period_raw": event_period_raw,
        "growing_season_raw": growing_season_raw,
        "duration_months": int(duration_months),
        "excel_severity_score": int(severity_score),
        "excel_severity_label": severity_label,
        "season_anchor_month": SEASON_ANCHOR_MONTH,
        "season_anchor_rule": (
            "October of Growing Season first year"
        ),
        "event_window_rule": (
            "season anchor plus stated Duration (Months), inclusive"
        ),
        "event_period_start_year": (
            event_period_start_year
        ),
        "event_period_end_year": (
            event_period_end_year
        ),
        "growing_season_start_year": (
            growing_season_start_year
        ),
        "event_start_yyyymm": yyyymm_from_period(
            event_start_period
        ),
        "event_end_yyyymm": yyyymm_from_period(
            event_end_period
        ),
        "event_month_count": len(event_months),
        "window_within_event_period_years": (
            window_within_event_period_years
        ),
        "comparison_reference": (
            "Excel Severity Score (1-5), event-level"
        ),
    }

    event_records.append(event_record)

    for month_position, yyyymm in enumerate(
        event_months,
        start=1,
    ):
        month_status = source_status_for_month(
            event_month=yyyymm,
            source_months=source_months,
            complete_source_months=complete_source_months,
        )

        event_month_records.append(
            {
                **event_record,
                "event_month_position": month_position,
                "yyyymm": yyyymm,
                "source_month_present": (
                    yyyymm in source_months
                ),
                "source_month_complete_grid": (
                    yyyymm in complete_source_months
                ),
                "has_complete_24m_spi12_context": (
                    month_status
                    == "ready_for_v2_inference"
                ),
                "inference_month_status": month_status,
            }
        )

event_ledger_df = pd.DataFrame(
    event_records
).sort_values(
    "excel_event_id"
).reset_index(drop=True)

event_month_ledger_df = pd.DataFrame(
    event_month_records
).sort_values(
    [
        "excel_event_id",
        "event_month_position",
    ]
).reset_index(drop=True)

if len(event_ledger_df) != 17:
    raise AssertionError(
        "Expected 17 chronological Excel events, found "
        f"{len(event_ledger_df)}."
    )

if (
    event_month_ledger_df.groupby(
        "event_key"
    )["event_month_position"].max()
    != event_month_ledger_df.groupby(
        "event_key"
    )["duration_months"].first()
).any():
    raise AssertionError(
        "At least one event-month ledger length differs from its "
        "Excel-stated duration."
    )


# -----------------------------------------------------------------------------
# 6. Summarize event eligibility for direct model-versus-Excel comparison
# -----------------------------------------------------------------------------
event_eligibility_summary_df = (
    event_month_ledger_df.groupby(
        [
            "event_key",
            "excel_event_id",
            "event_period_raw",
            "growing_season_raw",
            "duration_months",
            "excel_severity_score",
            "excel_severity_label",
            "event_start_yyyymm",
            "event_end_yyyymm",
            "window_within_event_period_years",
        ],
        dropna=False,
    )
    .agg(
        source_months_present=(
            "source_month_present",
            "sum",
        ),
        complete_grid_months=(
            "source_month_complete_grid",
            "sum",
        ),
        spi12_context_ready_months=(
            "has_complete_24m_spi12_context",
            "sum",
        ),
        raw_reconstruction_months=(
            "inference_month_status",
            lambda values: int(
                (
                    values
                    == "requires_raw_month_reconstruction"
                ).sum()
            ),
        ),
        source_history_months_required=(
            "inference_month_status",
            lambda values: int(
                (
                    values
                    == "requires_pre_source_history"
                ).sum()
            ),
        ),
    )
    .reset_index()
)

event_eligibility_summary_df[
    "event_coverage_rate"
] = (
    event_eligibility_summary_df[
        "spi12_context_ready_months"
    ]
    / event_eligibility_summary_df[
        "duration_months"
    ]
)

event_eligibility_summary_df[
    "event_inference_status"
] = np.select(
    [
        event_eligibility_summary_df[
            "spi12_context_ready_months"
        ].eq(
            event_eligibility_summary_df[
                "duration_months"
            ]
        ),
        event_eligibility_summary_df[
            "spi12_context_ready_months"
        ].gt(0),
    ],
    [
        "ready_for_full_event_model_vs_excel_comparison",
        "partial_event_only_not_valid_for_full_comparison",
    ],
    default="requires_historical_reconstruction_before_comparison",
)

event_eligibility_summary_df = (
    event_eligibility_summary_df.sort_values(
        "excel_event_id"
    )
    .reset_index(drop=True)
)

ready_event_df = event_eligibility_summary_df.loc[
    event_eligibility_summary_df[
        "event_inference_status"
    ].eq(
        "ready_for_full_event_model_vs_excel_comparison"
    )
].copy()

print("\n=== Direct Excel event ledger ===")
print(
    event_ledger_df[
        [
            "excel_event_id",
            "event_period_raw",
            "growing_season_raw",
            "duration_months",
            "event_start_yyyymm",
            "event_end_yyyymm",
            "excel_severity_score",
            "excel_severity_label",
            "window_within_event_period_years",
        ]
    ].to_string(
        index=False,
        max_colwidth=100,
    )
)

print("\n=== Event inference eligibility ===")
print(
    event_eligibility_summary_df.to_string(
        index=False,
        max_colwidth=100,
    )
)

print("\n=== Events ready now for full comparison ===")
if ready_event_df.empty:
    print("None.")
else:
    print(
        ready_event_df[
            [
                "excel_event_id",
                "event_period_raw",
                "event_start_yyyymm",
                "event_end_yyyymm",
                "duration_months",
                "excel_severity_score",
                "excel_severity_label",
            ]
        ].to_string(
            index=False,
            max_colwidth=100,
        )
    )


# -----------------------------------------------------------------------------
# 7. Final status
# -----------------------------------------------------------------------------
ready_event_count = int(len(ready_event_df))

partial_event_count = int(
    event_eligibility_summary_df[
        "event_inference_status"
    ].eq(
        "partial_event_only_not_valid_for_full_comparison"
    ).sum()
)

reconstruction_event_count = int(
    event_eligibility_summary_df[
        "event_inference_status"
    ].eq(
        "requires_historical_reconstruction_before_comparison"
    ).sum()
)

CELL9_STATUS = (
    "excel_season_aligned_event_ledger_and_"
    "v2_inference_eligibility_completed"
)

print("\n=== Cell 9 conclusion ===")
print("Status:", CELL9_STATUS)
print("Excel events processed:", len(event_ledger_df))
print(
    "Events ready for full model-vs-Excel comparison:",
    ready_event_count,
)
print(
    "Partial events, not valid for full comparison:",
    partial_event_count,
)
print(
    "Events needing historical reconstruction:",
    reconstruction_event_count,
)

print(
    "\nThis cell does not run predictions. It creates the exact "
    "event-month plan that the next inference cell must use."
)


# -----------------------------------------------------------------------------
# 8. Save outputs
# -----------------------------------------------------------------------------
source_month_coverage_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_full_predictor_month_coverage_{CELL9_RUN_ID}.csv"
)

event_ledger_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_excel_season_aligned_event_ledger_{CELL9_RUN_ID}.csv"
)

event_month_ledger_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_excel_season_aligned_event_month_ledger_{CELL9_RUN_ID}.csv"
)

eligibility_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_event_inference_eligibility_{CELL9_RUN_ID}.csv"
)

ready_events_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_events_ready_for_comparison_{CELL9_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell9_excel_event_ledger_manifest_{CELL9_RUN_ID}.json"
)

source_month_coverage_df.to_csv(
    source_month_coverage_path,
    index=False,
)

event_ledger_df.to_csv(
    event_ledger_path,
    index=False,
)

event_month_ledger_df.to_csv(
    event_month_ledger_path,
    index=False,
)

event_eligibility_summary_df.to_csv(
    eligibility_summary_path,
    index=False,
)

ready_event_df.to_csv(
    ready_events_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


manifest = {
    "run_id": CELL9_RUN_ID,
    "status": CELL9_STATUS,
    "event_workbook_path": str(EVENT_WORKBOOK_PATH),
    "sheet_name": "Chronological Timeline",
    "header_row_excel_number": 3,
    "event_window_protocol": {
        "anchor": (
            "October of Growing Season first year"
        ),
        "anchor_month_number": SEASON_ANCHOR_MONTH,
        "duration": (
            "Excel Duration (Months), inclusive"
        ),
        "spi12_context_requirement": (
            f"event month plus prior {REQUIRED_HISTORY_MONTHS} "
            "complete source months"
        ),
    },
    "source_coverage": {
        "first_month": str(source_first_period),
        "last_month": str(source_last_period),
        "full_grid_cell_count": source_grid_cell_count,
        "internal_month_gap_count": len(
            source_gap_periods
        ),
        "first_full_spi12_context_month": str(
            first_spi12_context_complete_period
        ),
    },
    "event_count": int(len(event_ledger_df)),
    "ready_event_count": ready_event_count,
    "partial_event_count": partial_event_count,
    "historical_reconstruction_event_count": (
        reconstruction_event_count
    ),
    "outputs": {
        "full_predictor_month_coverage_csv": str(
            source_month_coverage_path
        ),
        "event_ledger_csv": str(
            event_ledger_path
        ),
        "event_month_ledger_csv": str(
            event_month_ledger_path
        ),
        "event_inference_eligibility_csv": str(
            eligibility_summary_path
        ),
        "events_ready_for_comparison_csv": str(
            ready_events_path
        ),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 9 outputs ===")
print("Predictor monthly coverage ->", source_month_coverage_path)
print("Excel event ledger         ->", event_ledger_path)
print("Excel event-month ledger   ->", event_month_ledger_path)
print("Inference eligibility      ->", eligibility_summary_path)
print("Ready event list           ->", ready_events_path)
print("Manifest                   ->", manifest_path)

print("\nCell 9 complete.")

=== Cell 9 purpose ===
Build the direct Excel-derived, season-aligned drought-event ledger for final severity validation.

=== Inputs ===
Event workbook       : C:\Projects\Infer RozviDrought\data\events\Zimbabwe_Drought_Timeline_1902_2024.xlsx
Full predictor source: C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Output directory     : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID               : 20260620T115347Z

=== Event-window protocol ===
Anchor month         : 10 (October of Growing Season first year)
Window length        : exact Excel Duration (Months)
Required SPI12 context: 24 prior months plus event month

=== Excel fields used directly ===
ID               : ID
Event Period     : Event Period
Growing Season   : Growing Season
Duration Months  : Duration
(Months)
Severity Score   : Severity
Score (1–5)
Severity Label   : Severity Label

=== Excel timeline 

In [16]:
# validate_events_ng.ipynb — Cell 10 (REWRITE)
# Purpose:
# - Run national v2 model inference for the three citable drought windows:
#       2015/16: 201510–201602
#       2018/19: 201810–201905
#       2023/24: 202310–202403
# - Rebuild the verified 68 absent SPI12 fields from temporal_predictors_full.
# - Run SPI3, SPI6, SPI12, and the stored weighted fusion.
# - Aggregate monthly and event-window predictions nationally.
# - Compare event ordering descriptively against the Excel severity scores.
#
# Evidence rules:
# - Uses Cell 9 ONLY as a cached direct extract of Excel event ID / period /
#   severity fields. It does not use Cell 9's generated event dates.
# - Uses the source-backed month windows below; no calendar-anchor rule is used.
# - Requires Cell 8's exact SPI12 feature-reconstruction parity pass.
#
# Scope:
# - No raw-data reconstruction.
# - No cell-level prediction output.
# - No forced conversion of Excel severity scores (1–5) to model classes (0–4).

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import gc
import json
import re

import duckdb
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths and fixed, source-backed validation windows
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL10_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

EVENT_OUTPUT_DIR = OUTPUT_DIR / "real_event_validation"
EVENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_TEMP_DIR = (
    EVENT_OUTPUT_DIR / "_duckdb_event_inference_temp"
)
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_DROUGHT_ROOT = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
)

FULL_PREDICTOR_SOURCE_PATH = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "model_inputs"
    / "temporal_predictors_full_20260614T141118Z.parquet"
)

V2_ARTIFACT_ROOT = (
    TRAINING_DROUGHT_ROOT
    / "processed"
    / "package_releases"
    / "rozvidrought_v2"
    / "rozvidrought"
    / "artifacts"
)

MODEL_RUNS = {
    "spi3": V2_ARTIFACT_ROOT / "spi3" / "v2_20260617",
    "spi6": V2_ARTIFACT_ROOT / "spi6" / "v2_20260617",
    "spi12": V2_ARTIFACT_ROOT / "spi12" / "v2_20260617",
}

FUSION_RUN_DIR = (
    V2_ARTIFACT_ROOT
    / "fusion_v2"
    / "v2_20260617"
)

KEY_COLUMNS = ["row", "col", "yyyymm"]

NUM_CLASSES = 5
CLASS_VALUES = np.arange(NUM_CLASSES, dtype=np.float64)

REQUIRED_HISTORY_MONTHS = 24
NUMERIC_TOLERANCE = 1e-6
MISSINGNESS_SUFFIX = "__is_missing"

SEQUENCE_PATTERN = re.compile(
    r"^(?P<base>.+)_seq_"
    r"(?P<operation>lag|delta|rollmean|rollstd)"
    r"(?P<window>\d+)$"
)

# These are the only event windows used in this cell.
# They are not derived from Cell 9, Excel duration, or an assumed season start.
SOURCE_BACKED_EVENT_WINDOWS = [
    {
        "event_key": "zimbabwe_2015_2016",
        "excel_event_id": 15,
        "expected_excel_period_years": (2015, 2016),
        "window_start_yyyymm": "201510",
        "window_end_yyyymm": "201602",
        "source_organisation": "FEWS NET",
        "source_window_type": "national_rainfall_and_spi_period",
        "source_title": (
            "Drought conditions to significantly reduce "
            "2015/16 harvests"
        ),
        "source_url": (
            "https://fews.net/southern-africa/zimbabwe/"
            "food-security-outlook/february-2016"
        ),
        "source_month_statement": (
            "October 2015 to February 2016"
        ),
    },
    {
        "event_key": "zimbabwe_2018_2019",
        "excel_event_id": 16,
        "expected_excel_period_years": (2018, 2019),
        "window_start_yyyymm": "201810",
        "window_end_yyyymm": "201905",
        "source_organisation": "ReliefWeb",
        "source_window_type": "poor_rain_and_harvest_impact_period",
        "source_title": (
            "Zimbabwe: Drought Emergency, 19 September 2019"
        ),
        "source_url": (
            "https://reliefweb.int/report/zimbabwe/"
            "zimbabwe-drought-emergency-19-september-2019"
        ),
        "source_month_statement": (
            "October 2018 to May 2019"
        ),
    },
    {
        "event_key": "zimbabwe_2023_2024",
        "excel_event_id": 17,
        "expected_excel_period_years": (2023, 2024),
        "window_start_yyyymm": "202310",
        "window_end_yyyymm": "202403",
        "source_organisation": "ACT Alliance",
        "source_window_type": "national_rainy_season_rainfall_period",
        "source_title": (
            "RRF 07 2024 – Zimbabwe Responding to Drought"
        ),
        "source_url": (
            "https://actalliance.org/"
            "appeals-rapid-response-funds/"
            "rrf-07-2024-zimbabwe-drought/"
        ),
        "source_month_statement": (
            "October 2023 to March 2024"
        ),
    },
]

print("=== Cell 10 purpose ===")
print(
    "Run source-backed national v2 drought inference for three "
    "citable Zimbabwe drought windows."
)

print("\n=== Inputs ===")
print("Full predictor source:", FULL_PREDICTOR_SOURCE_PATH)
print("Artifact root        :", V2_ARTIFACT_ROOT)
print("Fusion run directory :", FUSION_RUN_DIR)
print("Output directory     :", EVENT_OUTPUT_DIR)
print("Run ID               :", CELL10_RUN_ID)

print("\n=== Source-backed event windows ===")
for event in SOURCE_BACKED_EVENT_WINDOWS:
    print(
        f"- {event['event_key']}: "
        f"{event['window_start_yyyymm']}–"
        f"{event['window_end_yyyymm']} | "
        f"{event['source_organisation']}"
    )

for required_path in [
    FULL_PREDICTOR_SOURCE_PATH,
    FUSION_RUN_DIR,
    *MODEL_RUNS.values(),
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required Cell 10 path is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------
def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(
        path.read_text(encoding="utf-8-sig")
    )

    if not isinstance(payload, dict):
        raise TypeError(
            f"{path.name} must contain one JSON object."
        )

    return payload


def extract_feature_list(
    feature_contract: dict[str, Any],
) -> list[str]:
    for key in [
        "model_features",
        "features",
        "feature_columns",
        "required_features",
    ]:
        value = feature_contract.get(key)

        if (
            isinstance(value, list)
            and value
            and all(
                isinstance(item, str)
                for item in value
            )
        ):
            return list(value)

    raise ValueError(
        "No usable model feature list was found in a feature contract."
    )


def quote_identifier(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


def sql_path(path: Path) -> str:
    return (
        str(path)
        .replace("\\", "/")
        .replace("'", "''")
    )


def parse_yyyymm(yyyymm: str) -> pd.Period:
    return pd.Period(str(yyyymm), freq="M")


def enumerate_months(
    start_yyyymm: str,
    end_yyyymm: str,
) -> list[str]:
    start_period = parse_yyyymm(start_yyyymm)
    end_period = parse_yyyymm(end_yyyymm)

    if end_period < start_period:
        raise ValueError(
            f"Invalid event window: {start_yyyymm} -> {end_yyyymm}"
        )

    return [
        month.strftime("%Y%m")
        for month in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


def extract_years(value: Any) -> tuple[int, ...]:
    values = re.findall(
        r"(?<!\d)(?:18|19|20)\d{2}(?!\d)",
        str(value),
    )

    if not values:
        raise ValueError(
            f"Could not parse event years from {value!r}"
        )

    return tuple(sorted({int(value) for value in values}))


def parse_sequence_feature(
    feature_name: str,
) -> dict[str, Any]:
    match = SEQUENCE_PATTERN.match(feature_name)

    if not match:
        raise ValueError(
            f"Could not parse SPI12 sequence feature: {feature_name}"
        )

    return {
        "feature_name": feature_name,
        "base_feature": match.group("base"),
        "operation": match.group("operation"),
        "window": int(match.group("window")),
    }


def normalize_horizon_key(key: Any) -> str | None:
    normalized = re.sub(
        r"[^a-z0-9]+",
        "",
        str(key).lower(),
    )

    mapping = {
        "spi3": "spi3",
        "spi6": "spi6",
        "spi12": "spi12",
        "spi3weight": "spi3",
        "spi6weight": "spi6",
        "spi12weight": "spi12",
        "weightspi3": "spi3",
        "weightspi6": "spi6",
        "weightspi12": "spi12",
    }

    return mapping.get(normalized)


def extract_fusion_weights(
    payload: Any,
    path_hint: str = "root",
) -> tuple[dict[str, float], str]:
    candidates: list[tuple[dict[str, float], str]] = []

    def walk(value: Any, current_path: str) -> None:
        if isinstance(value, dict):
            mapping: dict[str, float] = {}

            for key, item in value.items():
                horizon = normalize_horizon_key(key)

                if (
                    horizon
                    and isinstance(item, (int, float))
                    and not isinstance(item, bool)
                ):
                    mapping[horizon] = float(item)

            if set(mapping) == {"spi3", "spi6", "spi12"}:
                candidates.append((mapping, current_path))

            for key, item in value.items():
                walk(item, f"{current_path}.{key}")

        elif isinstance(value, list):
            for index, item in enumerate(value):
                walk(item, f"{current_path}[{index}]")

    walk(payload, path_hint)

    if not candidates:
        raise KeyError(
            "No numeric SPI3/SPI6/SPI12 fusion-weight mapping was found."
        )

    candidates.sort(
        key=lambda item: (
            0 if "weight" in item[1].lower() else 1,
            item[1],
        )
    )

    weights, source_path = candidates[0]

    if not np.isclose(sum(weights.values()), 1.0):
        raise AssertionError(
            f"Fusion weights do not sum to 1.0: {weights}"
        )

    if any(value < 0 for value in weights.values()):
        raise AssertionError(
            f"Fusion weights include a negative value: {weights}"
        )

    return weights, source_path


def load_model_bundle(
    horizon: str,
    run_dir: Path,
) -> dict[str, Any]:
    model_path = run_dir / "model.joblib"
    contract_path = run_dir / "feature_contract.json"

    for required_path in [model_path, contract_path]:
        if not required_path.exists():
            raise FileNotFoundError(
                f"Missing {horizon} model artifact:\n{required_path}"
            )

    model = joblib.load(model_path)
    feature_contract = read_json_object(contract_path)
    features = extract_feature_list(feature_contract)

    if not hasattr(model, "predict_proba"):
        raise TypeError(
            f"{horizon} model lacks predict_proba."
        )

    classes = [
        int(value)
        for value in np.asarray(model.classes_).tolist()
    ]

    if classes != [0, 1, 2, 3, 4]:
        raise AssertionError(
            f"{horizon} classes are not [0, 1, 2, 3, 4]: {classes}"
        )

    return {
        "horizon": horizon,
        "model": model,
        "features": features,
        "model_path": model_path,
        "contract_path": contract_path,
        "classes": classes,
    }


def predict_probabilities(
    bundle: dict[str, Any],
    feature_frame: pd.DataFrame,
) -> np.ndarray:
    required_features = bundle["features"]

    missing_features = [
        feature
        for feature in required_features
        if feature not in feature_frame.columns
    ]

    if missing_features:
        raise KeyError(
            f"{bundle['horizon']} input is missing contract fields:\n"
            f"{missing_features}"
        )

    X = feature_frame.loc[:, required_features].copy()

    for column_name in required_features:
        if not pd.api.types.is_numeric_dtype(X[column_name]):
            X[column_name] = pd.to_numeric(
                X[column_name],
                errors="raise",
            )

    probabilities = np.asarray(
        bundle["model"].predict_proba(X),
        dtype=np.float64,
    )

    expected_shape = (len(feature_frame), NUM_CLASSES)

    if probabilities.shape != expected_shape:
        raise AssertionError(
            f"{bundle['horizon']} probability shape is "
            f"{probabilities.shape}; expected {expected_shape}."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=NUMERIC_TOLERANCE,
    ):
        raise AssertionError(
            f"{bundle['horizon']} probabilities do not sum to 1.0."
        )

    return probabilities


# -----------------------------------------------------------------------------
# 3. Require Cell 8's verified SPI12 feature reconstruction
# -----------------------------------------------------------------------------
cell8_manifest_paths = sorted(
    EVENT_OUTPUT_DIR.glob(
        "cell8_spi12_formula_parity_manifest_*.json"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not cell8_manifest_paths:
    raise FileNotFoundError(
        "Cell 8 parity manifest not found. Run Cell 8 first."
    )

CELL8_MANIFEST_PATH = cell8_manifest_paths[0]
cell8_manifest = read_json_object(CELL8_MANIFEST_PATH)

if not cell8_manifest.get("formula_parity_passed", False):
    raise AssertionError(
        "Cell 8 does not confirm an exact SPI12 reconstruction pass. "
        "Stop before inference."
    )

if int(cell8_manifest.get("passed_feature_count", -1)) != 68:
    raise AssertionError(
        "Cell 8 did not pass all 68 reconstructed SPI12 fields."
    )

print("\n=== Required Cell 8 precondition ===")
print("Cell 8 manifest :", CELL8_MANIFEST_PATH)
print(
    "Formula parity  :",
    cell8_manifest["formula_parity_passed"],
)
print(
    "Features passed :",
    cell8_manifest["passed_feature_count"],
    "/",
    cell8_manifest.get("recreated_feature_count"),
)


# -----------------------------------------------------------------------------
# 4. Load Excel facts from the explicitly named Cell 9 extract
# -----------------------------------------------------------------------------
# Cell 9's time construction is not used.
# This reads only the direct Excel fields that Cell 9 extracted correctly.
cell9_excel_ledger_paths = sorted(
    EVENT_OUTPUT_DIR.glob(
        "cell9_excel_season_aligned_event_ledger_*.csv"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not cell9_excel_ledger_paths:
    raise FileNotFoundError(
        "The Cell 9 Excel event ledger was not found. "
        "Cell 10 requires its direct Excel fields only."
    )

CELL9_EXCEL_REFERENCE_PATH = cell9_excel_ledger_paths[0]

excel_reference_df = pd.read_csv(CELL9_EXCEL_REFERENCE_PATH)

required_excel_columns = [
    "excel_event_id",
    "event_period_raw",
    "excel_severity_score",
    "excel_severity_label",
]

missing_excel_columns = [
    column_name
    for column_name in required_excel_columns
    if column_name not in excel_reference_df.columns
]

if missing_excel_columns:
    raise KeyError(
        "The selected Cell 9 ledger does not contain the required "
        "direct Excel fields:\n"
        f"{missing_excel_columns}\n"
        f"Available: {list(excel_reference_df.columns)}"
    )

excel_reference_df = excel_reference_df[
    required_excel_columns
].copy()

excel_reference_df["excel_event_id"] = pd.to_numeric(
    excel_reference_df["excel_event_id"],
    errors="raise",
).astype("int64")

excel_reference_df["excel_severity_score"] = pd.to_numeric(
    excel_reference_df["excel_severity_score"],
    errors="raise",
).astype("int64")

if excel_reference_df["excel_event_id"].duplicated().any():
    raise AssertionError(
        "The Cell 9 Excel reference extract has duplicate event IDs."
    )

excel_reference_df["event_period_years"] = (
    excel_reference_df["event_period_raw"].map(extract_years)
)

source_window_records: list[dict[str, Any]] = []

for event in SOURCE_BACKED_EVENT_WINDOWS:
    matched_rows = excel_reference_df.loc[
        excel_reference_df["excel_event_id"].eq(
            event["excel_event_id"]
        )
    ].copy()

    if len(matched_rows) != 1:
        raise AssertionError(
            "Could not uniquely match this source-backed event to "
            "the Excel reference extract:\n"
            f"{event}"
        )

    excel_row = matched_rows.iloc[0]

    if tuple(excel_row["event_period_years"]) != tuple(
        event["expected_excel_period_years"]
    ):
        raise AssertionError(
            "The configured source-backed event does not match "
            "the Excel event-period years.\n"
            f"Event key: {event['event_key']}\n"
            f"Excel years: {excel_row['event_period_years']}\n"
            f"Expected: {event['expected_excel_period_years']}"
        )

    event_months = enumerate_months(
        event["window_start_yyyymm"],
        event["window_end_yyyymm"],
    )

    source_window_records.append(
        {
            **event,
            "excel_event_period": excel_row["event_period_raw"],
            "excel_severity_score": int(
                excel_row["excel_severity_score"]
            ),
            "excel_severity_label": excel_row[
                "excel_severity_label"
            ],
            "event_month_count": len(event_months),
            "event_months_pipe_delimited": " | ".join(event_months),
            "excel_reference_rule": (
                "Direct Excel event ID, period, score, and label only; "
                "Cell 9 generated dates ignored."
            ),
        }
    )

source_evidence_df = pd.DataFrame(
    source_window_records
).sort_values("excel_event_id").reset_index(drop=True)

print("\n=== Source-backed event evidence ===")
print(
    source_evidence_df[
        [
            "excel_event_id",
            "excel_event_period",
            "window_start_yyyymm",
            "window_end_yyyymm",
            "event_month_count",
            "excel_severity_score",
            "excel_severity_label",
            "source_organisation",
        ]
    ].to_string(
        index=False,
        max_colwidth=100,
    )
)


# -----------------------------------------------------------------------------
# 5. Load models, contracts, fusion weights, and source schema
# -----------------------------------------------------------------------------
model_bundles = {
    horizon: load_model_bundle(horizon, run_dir)
    for horizon, run_dir in MODEL_RUNS.items()
}

fusion_rules_path = FUSION_RUN_DIR / "fusion_rules.json"

if not fusion_rules_path.exists():
    raise FileNotFoundError(
        f"Stored fusion rules are missing:\n{fusion_rules_path}"
    )

fusion_rules = read_json_object(fusion_rules_path)

fusion_weights, fusion_weight_object_path = extract_fusion_weights(
    fusion_rules
)

predictor_schema_columns = set(
    pq.ParquetFile(FULL_PREDICTOR_SOURCE_PATH).schema.names
)

model_features_by_horizon = {
    horizon: model_bundles[horizon]["features"]
    for horizon in ["spi3", "spi6", "spi12"]
}

missing_source_features = {
    horizon: sorted(
        feature
        for feature in model_features_by_horizon[horizon]
        if feature not in predictor_schema_columns
    )
    for horizon in ["spi3", "spi6", "spi12"]
}

if missing_source_features["spi3"]:
    raise AssertionError(
        "SPI3 source features unexpectedly missing:\n"
        f"{missing_source_features['spi3']}"
    )

if missing_source_features["spi6"]:
    raise AssertionError(
        "SPI6 source features unexpectedly missing:\n"
        f"{missing_source_features['spi6']}"
    )

spi12_recreated_features = missing_source_features["spi12"]

sequence_features = sorted(
    feature
    for feature in spi12_recreated_features
    if "_seq_" in feature
)

indicator_features = sorted(
    feature
    for feature in spi12_recreated_features
    if feature.endswith(MISSINGNESS_SUFFIX)
)

unclassified_features = sorted(
    set(spi12_recreated_features)
    - set(sequence_features)
    - set(indicator_features)
)

if unclassified_features:
    raise AssertionError(
        "SPI12 contains unsupported source feature gaps:\n"
        f"{unclassified_features}"
    )

if len(sequence_features) != 48 or len(indicator_features) != 20:
    raise AssertionError(
        "Unexpected SPI12 reconstruction scope.\n"
        f"Sequence fields: {len(sequence_features)}\n"
        f"Indicators: {len(indicator_features)}"
    )

sequence_spec_df = pd.DataFrame(
    [
        parse_sequence_feature(feature)
        for feature in sequence_features
    ]
).sort_values(
    ["base_feature", "operation", "window", "feature_name"]
).reset_index(drop=True)

sequence_base_features = sorted(
    sequence_spec_df["base_feature"].unique().tolist()
)

indicator_base_features = sorted(
    {
        feature[: -len(MISSINGNESS_SUFFIX)]
        for feature in indicator_features
    }
)

expected_sequence_base_features = {
    "rain_chirps",
    "rain_chirps_anom",
    "sm_esa_cci",
    "sm_esa_cci_anom",
    "tws",
    "tws_anom",
    "tws_grace",
    "tws_grace_anom",
}

if set(sequence_base_features) != expected_sequence_base_features:
    raise AssertionError(
        "Unexpected SPI12 sequence base features:\n"
        f"{sequence_base_features}"
    )

all_model_features = sorted(
    {
        feature
        for feature_list in model_features_by_horizon.values()
        for feature in feature_list
    }
)

direct_source_features = sorted(
    feature
    for feature in all_model_features
    if feature in predictor_schema_columns
)

source_columns_needed = list(
    dict.fromkeys(
        KEY_COLUMNS
        + direct_source_features
        + sequence_base_features
        + indicator_base_features
        + [
            f"{base_feature}_lag12"
            for base_feature in sequence_base_features
            if (
                f"{base_feature}_lag12"
                in predictor_schema_columns
            )
        ]
    )
)

base_features_missing_from_source = sorted(
    set(sequence_base_features + indicator_base_features)
    - predictor_schema_columns
)

if base_features_missing_from_source:
    raise AssertionError(
        "Required SPI12 base inputs are absent from the predictor source:\n"
        f"{base_features_missing_from_source}"
    )

print("\n=== Models and stored fusion ===")
for horizon in ["spi3", "spi6", "spi12"]:
    print(
        f"{horizon.upper()}: "
        f"{len(model_bundles[horizon]['features'])} contract features | "
        f"{model_bundles[horizon]['model'].__class__.__name__}"
    )

print("Fusion rules :", fusion_rules_path)
print("Weight path  :", fusion_weight_object_path)
print("Weights      :", fusion_weights)

print("\n=== Predictor compatibility ===")
print("SPI3 direct source gaps :", len(missing_source_features["spi3"]))
print("SPI6 direct source gaps :", len(missing_source_features["spi6"]))
print("SPI12 sequence rebuild  :", len(sequence_features))
print("SPI12 missingness flags :", len(indicator_features))
print("Columns read per event  :", len(source_columns_needed))


# -----------------------------------------------------------------------------
# 6. Verify required source-month continuity and national grid completeness
# -----------------------------------------------------------------------------
context_start = min(
    parse_yyyymm(event["window_start_yyyymm"])
    - REQUIRED_HISTORY_MONTHS
    for event in source_window_records
)

target_end = max(
    parse_yyyymm(event["window_end_yyyymm"])
    for event in source_window_records
)

expected_months = [
    month.strftime("%Y%m")
    for month in pd.period_range(
        start=context_start,
        end=target_end,
        freq="M",
    )
]

source_path_sql = sql_path(FULL_PREDICTOR_SOURCE_PATH)

connection = duckdb.connect(database=":memory:")
connection.execute("SET memory_limit = '6GB'")
connection.execute(
    f"SET temp_directory = '{sql_path(DUCKDB_TEMP_DIR)}'"
)
connection.execute("PRAGMA threads = 4")

coverage_query = f"""
SELECT
    CAST("yyyymm" AS VARCHAR) AS yyyymm,
    COUNT(*) AS row_count
FROM read_parquet('{source_path_sql}')
WHERE CAST("yyyymm" AS VARCHAR)
      BETWEEN '{context_start.strftime("%Y%m")}'
          AND '{target_end.strftime("%Y%m")}'
GROUP BY 1
ORDER BY 1
"""

source_coverage_df = connection.execute(
    coverage_query
).fetch_df()

source_coverage_df["yyyymm"] = (
    source_coverage_df["yyyymm"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

source_coverage_df["row_count"] = pd.to_numeric(
    source_coverage_df["row_count"],
    errors="raise",
).astype("int64")

available_months = set(source_coverage_df["yyyymm"])

missing_context_months = [
    month
    for month in expected_months
    if month not in available_months
]

source_grid_cell_count = int(source_coverage_df["row_count"].max())

incomplete_grid_months = source_coverage_df.loc[
    source_coverage_df["row_count"].ne(source_grid_cell_count),
    "yyyymm",
].tolist()

if missing_context_months:
    raise AssertionError(
        "Required event/context months are absent from the full source:\n"
        f"{missing_context_months}"
    )

if incomplete_grid_months:
    raise AssertionError(
        "Required event/context months have incomplete national grids:\n"
        f"{incomplete_grid_months}"
    )

print("\n=== Required full-predictor coverage ===")
print(
    "Context range      :",
    f"{context_start.strftime('%Y%m')} -> {target_end.strftime('%Y%m')}",
)
print("Months available   :", len(expected_months))
print("National cell count:", f"{source_grid_cell_count:,}")


# -----------------------------------------------------------------------------
# 7. Rebuild verified SPI12 fields and return one event input frame
# -----------------------------------------------------------------------------
def build_event_input_frame(
    event: dict[str, Any],
) -> pd.DataFrame:
    event_start = parse_yyyymm(event["window_start_yyyymm"])
    event_end = parse_yyyymm(event["window_end_yyyymm"])
    event_context_start = event_start - REQUIRED_HISTORY_MONTHS

    base_select_parts: list[str] = []

    for column_name in source_columns_needed:
        q = quote_identifier(column_name)

        if column_name in {"row", "col"}:
            expression = f"CAST({q} AS BIGINT) AS {q}"

        elif column_name == "yyyymm":
            expression = f"CAST({q} AS VARCHAR) AS {q}"

        else:
            expression = f"CAST({q} AS FLOAT) AS {q}"

        base_select_parts.append(expression)

    sequence_stage_parts = [
        quote_identifier(column_name)
        for column_name in source_columns_needed
    ]

    internal_feature_names: dict[tuple[str, str, int], str] = {}

    for base_feature in sequence_base_features:
        q_base = quote_identifier(base_feature)

        lag12_source_column = f"{base_feature}_lag12"

        if lag12_source_column in predictor_schema_columns:
            lag12_expression = (
                f"CAST({quote_identifier(lag12_source_column)} AS FLOAT)"
            )
        else:
            lag12_expression = (
                f"CAST(LAG({q_base}, 12) OVER w AS FLOAT)"
            )

        lag12_internal = f"__{base_feature}_lag12"
        lag24_internal = f"__{base_feature}_lag24"
        rollmean24_internal = f"__{base_feature}_rollmean24"
        rollstd24_internal = f"__{base_feature}_rollstd24"

        internal_feature_names[(base_feature, "lag", 12)] = lag12_internal
        internal_feature_names[(base_feature, "lag", 24)] = lag24_internal
        internal_feature_names[
            (base_feature, "rollmean", 24)
        ] = rollmean24_internal
        internal_feature_names[
            (base_feature, "rollstd", 24)
        ] = rollstd24_internal

        sequence_stage_parts.extend(
            [
                (
                    f"{lag12_expression} AS "
                    f"{quote_identifier(lag12_internal)}"
                ),
                (
                    f"CAST(LAG({q_base}, 24) OVER w AS FLOAT) AS "
                    f"{quote_identifier(lag24_internal)}"
                ),
                (
                    "CASE "
                    f"WHEN COUNT({q_base}) OVER w24 >= 3 "
                    f"THEN CAST(AVG({q_base}) OVER w24 AS FLOAT) "
                    "ELSE CAST(NULL AS FLOAT) "
                    f"END AS {quote_identifier(rollmean24_internal)}"
                ),
                (
                    "CASE "
                    f"WHEN COUNT({q_base}) OVER w24 >= 3 "
                    f"THEN CAST(STDDEV_SAMP({q_base}) OVER w24 AS FLOAT) "
                    "ELSE CAST(NULL AS FLOAT) "
                    f"END AS {quote_identifier(rollstd24_internal)}"
                ),
            ]
        )

    output_parts = [
        quote_identifier(column_name)
        for column_name in KEY_COLUMNS
    ]

    output_parts.extend(
        quote_identifier(feature)
        for feature in direct_source_features
    )

    for _, spec in sequence_spec_df.iterrows():
        feature_name = spec["feature_name"]
        base_feature = spec["base_feature"]
        operation = spec["operation"]
        window = int(spec["window"])

        q_base = quote_identifier(base_feature)
        q_output = quote_identifier(feature_name)

        if operation == "lag":
            expression = quote_identifier(
                internal_feature_names[
                    (base_feature, "lag", window)
                ]
            )

        elif operation == "rollmean":
            expression = quote_identifier(
                internal_feature_names[
                    (base_feature, "rollmean", window)
                ]
            )

        elif operation == "rollstd":
            expression = quote_identifier(
                internal_feature_names[
                    (base_feature, "rollstd", window)
                ]
            )

        elif operation == "delta":
            q_lag = quote_identifier(
                internal_feature_names[
                    (base_feature, "lag", window)
                ]
            )

            expression = f"CAST({q_base} - {q_lag} AS FLOAT)"

        else:
            raise AssertionError(
                f"Unsupported sequence operation: {operation}"
            )

        output_parts.append(f"{expression} AS {q_output}")

    for indicator_feature in indicator_features:
        base_feature = indicator_feature[: -len(MISSINGNESS_SUFFIX)]

        output_parts.append(
            (
                "CAST(CASE "
                f"WHEN {quote_identifier(base_feature)} IS NULL "
                "THEN 1 ELSE 0 END AS TINYINT) "
                f"AS {quote_identifier(indicator_feature)}"
            )
        )

    query = f"""
    WITH base_context AS (
        SELECT
            {", ".join(base_select_parts)}
        FROM read_parquet('{source_path_sql}')
        WHERE CAST("yyyymm" AS VARCHAR)
              BETWEEN '{event_context_start.strftime("%Y%m")}'
                  AND '{event_end.strftime("%Y%m")}'
    ),
    sequence_stage AS (
        SELECT
            {", ".join(sequence_stage_parts)}
        FROM base_context
        WINDOW
            w AS (
                PARTITION BY "row", "col"
                ORDER BY "yyyymm"
            ),
            w24 AS (
                PARTITION BY "row", "col"
                ORDER BY "yyyymm"
                ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
            )
    )
    SELECT
        {", ".join(output_parts)}
    FROM sequence_stage
    WHERE "yyyymm"
          BETWEEN '{event_start.strftime("%Y%m")}'
              AND '{event_end.strftime("%Y%m")}'
    ORDER BY "yyyymm", "row", "col"
    """

    frame = connection.execute(query).fetch_df()

    frame["row"] = pd.to_numeric(
        frame["row"],
        errors="raise",
    ).astype("int64")

    frame["col"] = pd.to_numeric(
        frame["col"],
        errors="raise",
    ).astype("int64")

    frame["yyyymm"] = (
        frame["yyyymm"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(6)
    )

    expected_event_months = enumerate_months(
        event["window_start_yyyymm"],
        event["window_end_yyyymm"],
    )

    expected_rows = (
        len(expected_event_months)
        * source_grid_cell_count
    )

    if len(frame) != expected_rows:
        raise AssertionError(
            f"{event['event_key']} has {len(frame):,} input rows; "
            f"expected {expected_rows:,}."
        )

    if frame.duplicated(subset=KEY_COLUMNS).any():
        raise AssertionError(
            f"{event['event_key']} has duplicate row/col/month keys."
        )

    actual_months = sorted(frame["yyyymm"].unique().tolist())

    if actual_months != expected_event_months:
        raise AssertionError(
            f"{event['event_key']} has unexpected returned months.\n"
            f"Expected: {expected_event_months}\n"
            f"Actual:   {actual_months}"
        )

    missing_input_columns = sorted(
        set(all_model_features)
        - set(frame.columns)
    )

    if missing_input_columns:
        raise AssertionError(
            f"{event['event_key']} is missing model inputs:\n"
            f"{missing_input_columns}"
        )

    return frame


# -----------------------------------------------------------------------------
# 8. Run national inference and month-level aggregation
# -----------------------------------------------------------------------------
monthly_records: list[dict[str, Any]] = []
event_run_records: list[dict[str, Any]] = []

try:
    for event in source_window_records:
        print("\n" + "=" * 88)
        print("Running:", event["event_key"])
        print(
            "Window :",
            f"{event['window_start_yyyymm']} -> "
            f"{event['window_end_yyyymm']}",
        )

        feature_frame = build_event_input_frame(event)

        horizon_probabilities = {
            horizon: predict_probabilities(
                model_bundles[horizon],
                feature_frame,
            )
            for horizon in ["spi3", "spi6", "spi12"]
        }

        fusion_probabilities = (
            fusion_weights["spi3"] * horizon_probabilities["spi3"]
            + fusion_weights["spi6"] * horizon_probabilities["spi6"]
            + fusion_weights["spi12"] * horizon_probabilities["spi12"]
        )

        if not np.allclose(
            fusion_probabilities.sum(axis=1),
            1.0,
            atol=NUMERIC_TOLERANCE,
        ):
            raise AssertionError(
                f"{event['event_key']} fusion probabilities do not sum to 1.0."
            )

        fusion_expected_class = fusion_probabilities @ CLASS_VALUES
        fusion_hard_class = fusion_probabilities.argmax(axis=1)
        fusion_confidence = fusion_probabilities.max(axis=1)
        fusion_drought_probability = 1.0 - fusion_probabilities[:, 0]

        event_months = enumerate_months(
            event["window_start_yyyymm"],
            event["window_end_yyyymm"],
        )

        for position, yyyymm in enumerate(event_months, start=1):
            month_mask = feature_frame["yyyymm"].eq(yyyymm).to_numpy()

            cell_count = int(month_mask.sum())

            if cell_count != source_grid_cell_count:
                raise AssertionError(
                    f"{event['event_key']} {yyyymm} has "
                    f"{cell_count:,} cells; expected "
                    f"{source_grid_cell_count:,}."
                )

            month_fusion_probabilities = fusion_probabilities[month_mask]
            month_hard_classes = fusion_hard_class[month_mask]
            hard_counts = np.bincount(
                month_hard_classes,
                minlength=NUM_CLASSES,
            )

            record = {
                "event_key": event["event_key"],
                "excel_event_id": event["excel_event_id"],
                "excel_event_period": event["excel_event_period"],
                "excel_severity_score": event["excel_severity_score"],
                "excel_severity_label": event["excel_severity_label"],
                "source_organisation": event["source_organisation"],
                "source_title": event["source_title"],
                "source_url": event["source_url"],
                "source_month_statement": event["source_month_statement"],
                "source_window_type": event["source_window_type"],
                "window_start_yyyymm": event["window_start_yyyymm"],
                "window_end_yyyymm": event["window_end_yyyymm"],
                "event_month_position": position,
                "event_month_count": event["event_month_count"],
                "yyyymm": yyyymm,
                "national_cell_count": cell_count,
                "spi3_mean_expected_class": float(
                    (horizon_probabilities["spi3"][month_mask] @ CLASS_VALUES).mean()
                ),
                "spi6_mean_expected_class": float(
                    (horizon_probabilities["spi6"][month_mask] @ CLASS_VALUES).mean()
                ),
                "spi12_mean_expected_class": float(
                    (horizon_probabilities["spi12"][month_mask] @ CLASS_VALUES).mean()
                ),
                "fusion_mean_expected_class_0_to_4": float(
                    fusion_expected_class[month_mask].mean()
                ),
                "fusion_mean_drought_probability": float(
                    fusion_drought_probability[month_mask].mean()
                ),
                "fusion_mean_confidence": float(
                    fusion_confidence[month_mask].mean()
                ),
                "fusion_hard_class_mode_0_to_4": int(
                    hard_counts.argmax()
                ),
                "fusion_severe_or_extreme_cell_share": float(
                    (month_hard_classes >= 3).mean()
                ),
            }

            for class_value in range(NUM_CLASSES):
                record[
                    f"fusion_mean_probability_p{class_value}"
                ] = float(
                    month_fusion_probabilities[:, class_value].mean()
                )

                record[
                    f"fusion_hard_class_{class_value}_share"
                ] = float(hard_counts[class_value] / cell_count)

            monthly_records.append(record)

        event_run_records.append(
            {
                "event_key": event["event_key"],
                "excel_event_id": event["excel_event_id"],
                "window_start_yyyymm": event["window_start_yyyymm"],
                "window_end_yyyymm": event["window_end_yyyymm"],
                "event_month_count": event["event_month_count"],
                "national_cell_count": source_grid_cell_count,
                "input_rows_scored": int(len(feature_frame)),
                "spi12_reconstruction_status": (
                    "cell8_exact_formula_parity_pass"
                ),
                "status": "inference_completed",
            }
        )

        print("Rows scored:", f"{len(feature_frame):,}")

        del feature_frame
        del horizon_probabilities
        del fusion_probabilities
        del fusion_expected_class
        del fusion_hard_class
        del fusion_confidence
        del fusion_drought_probability
        gc.collect()

finally:
    connection.close()

monthly_summary_df = pd.DataFrame(monthly_records).sort_values(
    ["excel_event_id", "yyyymm"]
).reset_index(drop=True)

event_run_audit_df = pd.DataFrame(event_run_records).sort_values(
    "excel_event_id"
).reset_index(drop=True)

expected_month_summary_rows = sum(
    event["event_month_count"]
    for event in source_window_records
)

if len(monthly_summary_df) != expected_month_summary_rows:
    raise AssertionError(
        "Unexpected national event-month summary count.\n"
        f"Expected: {expected_month_summary_rows}\n"
        f"Actual:   {len(monthly_summary_df)}"
    )


# -----------------------------------------------------------------------------
# 9. Aggregate model outputs by source-backed event window
# -----------------------------------------------------------------------------
event_summary_records: list[dict[str, Any]] = []

for event_key, group in monthly_summary_df.groupby(
    "event_key",
    sort=False,
):
    group = group.sort_values("yyyymm").reset_index(drop=True)

    peak_expected_class_row = group.loc[
        group["fusion_mean_expected_class_0_to_4"].idxmax()
    ]

    peak_drought_probability_row = group.loc[
        group["fusion_mean_drought_probability"].idxmax()
    ]

    event_summary_records.append(
        {
            "event_key": event_key,
            "excel_event_id": int(group["excel_event_id"].iloc[0]),
            "excel_event_period": group["excel_event_period"].iloc[0],
            "excel_severity_score": int(
                group["excel_severity_score"].iloc[0]
            ),
            "excel_severity_label": group["excel_severity_label"].iloc[0],
            "source_organisation": group["source_organisation"].iloc[0],
            "source_title": group["source_title"].iloc[0],
            "source_url": group["source_url"].iloc[0],
            "source_month_statement": group["source_month_statement"].iloc[0],
            "source_window_type": group["source_window_type"].iloc[0],
            "window_start_yyyymm": group["window_start_yyyymm"].iloc[0],
            "window_end_yyyymm": group["window_end_yyyymm"].iloc[0],
            "event_month_count": int(len(group)),
            "national_cell_count": int(group["national_cell_count"].iloc[0]),
            "fusion_event_mean_expected_class_0_to_4": float(
                group["fusion_mean_expected_class_0_to_4"].mean()
            ),
            "fusion_event_peak_expected_class_0_to_4": float(
                peak_expected_class_row[
                    "fusion_mean_expected_class_0_to_4"
                ]
            ),
            "fusion_peak_expected_class_month": peak_expected_class_row["yyyymm"],
            "fusion_event_mean_drought_probability": float(
                group["fusion_mean_drought_probability"].mean()
            ),
            "fusion_peak_drought_probability": float(
                peak_drought_probability_row[
                    "fusion_mean_drought_probability"
                ]
            ),
            "fusion_peak_drought_probability_month": (
                peak_drought_probability_row["yyyymm"]
            ),
            "fusion_event_mean_confidence": float(
                group["fusion_mean_confidence"].mean()
            ),
            "fusion_event_mean_severe_or_extreme_share": float(
                group["fusion_severe_or_extreme_cell_share"].mean()
            ),
            "comparison_scope": (
                "Descriptive comparison. Excel severity remains 1–5; "
                "model expected class remains 0–4; no direct score mapping applied."
            ),
        }
    )

event_summary_df = pd.DataFrame(event_summary_records).sort_values(
    "excel_event_id"
).reset_index(drop=True)

event_summary_df["excel_ordinal_rank"] = (
    event_summary_df["excel_severity_score"].rank(
        method="average",
        ascending=True,
    )
)

event_summary_df["model_ordinal_rank"] = (
    event_summary_df[
        "fusion_event_mean_expected_class_0_to_4"
    ].rank(
        method="average",
        ascending=True,
    )
)

event_summary_df["ordinal_rank_difference"] = (
    event_summary_df["model_ordinal_rank"]
    - event_summary_df["excel_ordinal_rank"]
)

if (
    event_summary_df["excel_ordinal_rank"].nunique() > 1
    and event_summary_df["model_ordinal_rank"].nunique() > 1
):
    ordinal_rank_correlation = float(
        np.corrcoef(
            event_summary_df["excel_ordinal_rank"],
            event_summary_df["model_ordinal_rank"],
        )[0, 1]
    )
else:
    ordinal_rank_correlation = np.nan

ordinal_diagnostic_df = pd.DataFrame(
    [
        {
            "event_count": int(len(event_summary_df)),
            "ordinal_rank_correlation": ordinal_rank_correlation,
            "interpretation": (
                "Descriptive only; three source-backed events; "
                "not a calibrated score-accuracy metric."
            ),
            "scale_handling": (
                "Excel severity retained as 1–5 and model expected "
                "class retained as 0–4. No forced numerical mapping."
            ),
        }
    ]
)

print("\n=== National monthly results ===")
print(
    monthly_summary_df[
        [
            "excel_event_id",
            "yyyymm",
            "excel_severity_score",
            "fusion_mean_expected_class_0_to_4",
            "fusion_mean_drought_probability",
            "fusion_severe_or_extreme_cell_share",
            "fusion_hard_class_mode_0_to_4",
            "fusion_mean_confidence",
        ]
    ].to_string(
        index=False,
        max_rows=100,
    )
)

print("\n=== Source-backed event summaries ===")
print(
    event_summary_df[
        [
            "excel_event_id",
            "excel_event_period",
            "window_start_yyyymm",
            "window_end_yyyymm",
            "excel_severity_score",
            "fusion_event_mean_expected_class_0_to_4",
            "fusion_event_peak_expected_class_0_to_4",
            "fusion_peak_expected_class_month",
            "fusion_event_mean_drought_probability",
            "fusion_event_mean_severe_or_extreme_share",
        ]
    ].to_string(
        index=False,
        max_colwidth=110,
    )
)

print("\n=== Cross-event ordinal diagnostic ===")
print(ordinal_diagnostic_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 10. Save only auditable aggregate outputs
# -----------------------------------------------------------------------------
source_evidence_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_source_backed_event_evidence_{CELL10_RUN_ID}.csv"
)

coverage_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_required_source_coverage_{CELL10_RUN_ID}.csv"
)

model_bundle_audit_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_model_bundle_audit_{CELL10_RUN_ID}.csv"
)

fusion_weight_audit_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_fusion_weight_audit_{CELL10_RUN_ID}.csv"
)

run_audit_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_event_inference_run_audit_{CELL10_RUN_ID}.csv"
)

monthly_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_national_event_month_summary_{CELL10_RUN_ID}.csv"
)

event_summary_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_national_event_window_summary_{CELL10_RUN_ID}.csv"
)

ordinal_diagnostic_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_ordinal_cross_event_diagnostic_{CELL10_RUN_ID}.csv"
)

manifest_path = (
    EVENT_OUTPUT_DIR
    / f"cell10_source_backed_event_inference_manifest_{CELL10_RUN_ID}.json"
)

source_evidence_df.to_csv(source_evidence_path, index=False)
source_coverage_df.to_csv(coverage_path, index=False)
event_run_audit_df.to_csv(run_audit_path, index=False)
monthly_summary_df.to_csv(monthly_summary_path, index=False)
event_summary_df.to_csv(event_summary_path, index=False)
ordinal_diagnostic_df.to_csv(ordinal_diagnostic_path, index=False)

model_bundle_audit_df = pd.DataFrame(
    [
        {
            "horizon": horizon,
            "model_path": str(model_bundles[horizon]["model_path"]),
            "contract_path": str(model_bundles[horizon]["contract_path"]),
            "feature_count": len(model_bundles[horizon]["features"]),
            "model_class": model_bundles[horizon]["model"].__class__.__name__,
            "classes": " | ".join(
                map(str, model_bundles[horizon]["classes"])
            ),
        }
        for horizon in ["spi3", "spi6", "spi12"]
    ]
)

model_bundle_audit_df.to_csv(model_bundle_audit_path, index=False)

fusion_weight_audit_df = pd.DataFrame(
    [
        {
            "horizon": horizon,
            "weight": fusion_weights[horizon],
            "fusion_rules_path": str(fusion_rules_path),
            "weight_object_path": fusion_weight_object_path,
        }
        for horizon in ["spi3", "spi6", "spi12"]
    ]
)

fusion_weight_audit_df.to_csv(
    fusion_weight_audit_path,
    index=False,
)


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    raise TypeError(
        f"Cannot JSON serialize {type(value).__name__}."
    )


CELL10_STATUS = (
    "source_backed_event_inference_completed_"
    "for_2015_2016_2018_2019_2023_2024"
)

manifest = {
    "run_id": CELL10_RUN_ID,
    "status": CELL10_STATUS,
    "purpose": (
        "National v2 inference across three source-backed Zimbabwe "
        "drought windows with descriptive comparison to Excel severity."
    ),
    "cell8_formula_parity_manifest": str(CELL8_MANIFEST_PATH),
    "cell8_formula_parity_passed": bool(
        cell8_manifest["formula_parity_passed"]
    ),
    "excel_reference_cache_path": str(CELL9_EXCEL_REFERENCE_PATH),
    "excel_fields_used": required_excel_columns,
    "cell9_generated_dates_used": False,
    "full_predictor_source": str(FULL_PREDICTOR_SOURCE_PATH),
    "required_history_months": REQUIRED_HISTORY_MONTHS,
    "national_grid_cell_count": source_grid_cell_count,
    "source_backed_events": source_evidence_df.to_dict(
        orient="records"
    ),
    "fusion_weights": fusion_weights,
    "fusion_rules_path": str(fusion_rules_path),
    "ordinal_rank_correlation": ordinal_rank_correlation,
    "score_mapping_applied": False,
    "outputs": {
        "source_evidence_csv": str(source_evidence_path),
        "required_source_coverage_csv": str(coverage_path),
        "model_bundle_audit_csv": str(model_bundle_audit_path),
        "fusion_weight_audit_csv": str(fusion_weight_audit_path),
        "event_run_audit_csv": str(run_audit_path),
        "national_event_month_summary_csv": str(monthly_summary_path),
        "national_event_window_summary_csv": str(event_summary_path),
        "ordinal_diagnostic_csv": str(ordinal_diagnostic_path),
        "manifest_json": str(manifest_path),
    },
}

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=json_default,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 10 outputs ===")
print("Source evidence       ->", source_evidence_path)
print("Required coverage     ->", coverage_path)
print("Model bundle audit    ->", model_bundle_audit_path)
print("Fusion weight audit   ->", fusion_weight_audit_path)
print("Inference run audit   ->", run_audit_path)
print("National monthly data ->", monthly_summary_path)
print("National event summary->", event_summary_path)
print("Ordinal diagnostic    ->", ordinal_diagnostic_path)
print("Manifest              ->", manifest_path)

print("\nCell 10 complete.")

=== Cell 10 purpose ===
Run source-backed national v2 drought inference for three citable Zimbabwe drought windows.

=== Inputs ===
Full predictor source: C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Artifact root        : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts
Fusion run directory : C:\Projects\ramangwana\risks\data\drought_model\processed\package_releases\rozvidrought_v2\rozvidrought\artifacts\fusion_v2\v2_20260617
Output directory     : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation
Run ID               : 20260620T121715Z

=== Source-backed event windows ===
- zimbabwe_2015_2016: 201510–201602 | FEWS NET
- zimbabwe_2018_2019: 201810–201905 | ReliefWeb
- zimbabwe_2023_2024: 202310–202403 | ACT Alliance

=== Required Cell 8 precondition ===
Cell 8 manifest : C:\Projects\Infer RozviDrought\data\bac

In [18]:
# validate_events_ng.ipynb — Cell 11 (REWRITE)
# Purpose:
# - Resolve the legacy package's required soil-moisture input without reading
#   master_inputs or running inference.
# - Determine whether the legacy `sm` field has a documented mapping to a field
#   in temporal_predictors_full, especially sm_esa_cci.
#
# Scope:
# - Read-only package/source audit.
# - No master_inputs.
# - No model inference.
# - No automatic substitution.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import csv
import inspect
import re

import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL11_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

PREDICTOR_PATH = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
    r"\processed\model_inputs"
    r"\temporal_predictors_full_20260614T141118Z.parquet"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== Cell 11 purpose ===")
print(
    "Resolve the legacy package soil-moisture input mapping "
    "without using master_inputs."
)

print("\nProject root :", PROJECT_ROOT)
print("Predictor    :", PREDICTOR_PATH)
print("master used  :", False)
print("Run ID       :", CELL11_RUN_ID)

for path in [PROJECT_ROOT, PREDICTOR_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required path:\n{path}")


# -----------------------------------------------------------------------------
# 2. Load legacy package source location
# -----------------------------------------------------------------------------
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.polygon_inference_service import (
    PolygonInferenceService,
)

SERVICE_SOURCE_PATH = Path(
    inspect.getsourcefile(PolygonInferenceService)
)

if not SERVICE_SOURCE_PATH.exists():
    raise FileNotFoundError(
        "Could not locate PolygonInferenceService source."
    )

print("\n=== Legacy service source ===")
print(SERVICE_SOURCE_PATH)


# -----------------------------------------------------------------------------
# 3. Check permitted predictor schema
# -----------------------------------------------------------------------------
predictor_columns = sorted(
    pq.ParquetFile(PREDICTOR_PATH).schema.names
)

legacy_raw_inputs = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

available_direct = [
    column
    for column in legacy_raw_inputs
    if column in predictor_columns
]

missing_direct = [
    column
    for column in legacy_raw_inputs
    if column not in predictor_columns
]

soil_moisture_candidates = [
    column
    for column in predictor_columns
    if "sm" in column.lower()
    or "soil" in column.lower()
]

print("\n=== Permitted predictor compatibility ===")
print("Legacy inputs expected :", legacy_raw_inputs)
print("Available directly     :", available_direct)
print("Missing directly       :", missing_direct)
print("Soil-moisture candidates:")
for column in soil_moisture_candidates:
    print(" -", column)


# -----------------------------------------------------------------------------
# 4. Search package code and local metadata for explicit mapping evidence
# -----------------------------------------------------------------------------
SEARCH_ROOTS = [
    PROJECT_ROOT / "app",
    PROJECT_ROOT / "data",
    PROJECT_ROOT,
]

ALLOWED_SUFFIXES = {
    ".py",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".md",
    ".txt",
}

MAX_FILE_SIZE_BYTES = 2 * 1024 * 1024

SM_EXACT_PATTERN = re.compile(
    r"(?<![A-Za-z0-9_])sm(?![A-Za-z0-9_])",
    flags=re.IGNORECASE,
)

mapping_hits = []
seen_paths = set()

for root in SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path in seen_paths:
            continue

        seen_paths.add(path)

        if (
            not path.is_file()
            or path.suffix.lower() not in ALLOWED_SUFFIXES
        ):
            continue

        try:
            if path.stat().st_size > MAX_FILE_SIZE_BYTES:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        except OSError:
            continue

        text_lower = text.lower()

        # Keep only files that contain legacy `sm` or a possible direct replacement.
        if (
            not SM_EXACT_PATTERN.search(text)
            and "sm_esa_cci" not in text_lower
        ):
            continue

        for line_number, line in enumerate(
            text.splitlines(),
            start=1,
        ):
            line_lower = line.lower()

            if (
                SM_EXACT_PATTERN.search(line)
                or "sm_esa_cci" in line_lower
            ):
                mapping_hits.append(
                    {
                        "file_path": str(path),
                        "line_number": line_number,
                        "contains_exact_sm": bool(
                            SM_EXACT_PATTERN.search(line)
                        ),
                        "contains_sm_esa_cci": (
                            "sm_esa_cci" in line_lower
                        ),
                        "line_text": line.strip(),
                    }
                )

mapping_hits = sorted(
    mapping_hits,
    key=lambda record: (
        record["file_path"].lower(),
        record["line_number"],
    ),
)

print("\n=== Mapping evidence found in package/local metadata ===")

if mapping_hits:
    for record in mapping_hits[:120]:
        print(
            f"{record['file_path']}:{record['line_number']}"
        )
        print(" ", record["line_text"])
else:
    print("No package or local metadata references found.")


# -----------------------------------------------------------------------------
# 5. Make a strict decision: no undocumented field substitution
# -----------------------------------------------------------------------------
explicit_sm_esa_mapping_hits = [
    record
    for record in mapping_hits
    if (
        record["contains_exact_sm"]
        and record["contains_sm_esa_cci"]
    )
]

if "sm" in predictor_columns:
    CELL11_STATUS = (
        "legacy_sm_input_available_directly_in_permitted_predictor_source"
    )

elif explicit_sm_esa_mapping_hits:
    CELL11_STATUS = (
        "explicit_sm_to_sm_esa_cci_mapping_evidence_found_review_before_run"
    )

else:
    CELL11_STATUS = (
        "legacy_challenger_blocked_no_documented_sm_mapping_without_master"
    )

print("\n=== Cell 11 conclusion ===")
print("Status:", CELL11_STATUS)

if CELL11_STATUS == (
    "legacy_challenger_blocked_no_documented_sm_mapping_without_master"
):
    print(
        "The legacy package cannot be run legitimately from the permitted "
        "source yet. `sm_esa_cci` exists as a candidate, but no explicit "
        "package/local mapping was found in this audit."
    )

elif CELL11_STATUS == (
    "explicit_sm_to_sm_esa_cci_mapping_evidence_found_review_before_run"
):
    print(
        "A local explicit mapping reference was found. Review the listed "
        "line(s), then use that documented mapping in the challenger run."
    )

else:
    print(
        "The legacy `sm` field is directly present in the permitted "
        "predictor source."
    )


# -----------------------------------------------------------------------------
# 6. Save evidence
# -----------------------------------------------------------------------------
mapping_hits_path = (
    OUT_DIR
    / f"cell11_legacy_sm_mapping_evidence_{CELL11_RUN_ID}.csv"
)

summary_path = (
    OUT_DIR
    / f"cell11_legacy_sm_mapping_summary_{CELL11_RUN_ID}.csv"
)

with mapping_hits_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "file_path",
            "line_number",
            "contains_exact_sm",
            "contains_sm_esa_cci",
            "line_text",
        ],
    )
    writer.writeheader()
    writer.writerows(mapping_hits)

summary_rows = [
    {
        "metric": "status",
        "value": CELL11_STATUS,
    },
    {
        "metric": "master_inputs_used",
        "value": False,
    },
    {
        "metric": "legacy_sm_directly_available",
        "value": "sm" in predictor_columns,
    },
    {
        "metric": "soil_moisture_candidate_columns",
        "value": " | ".join(soil_moisture_candidates),
    },
    {
        "metric": "explicit_sm_and_sm_esa_cci_same_line_hits",
        "value": len(explicit_sm_esa_mapping_hits),
    },
    {
        "metric": "all_mapping_evidence_hits",
        "value": len(mapping_hits),
    },
    {
        "metric": "legacy_service_source",
        "value": str(SERVICE_SOURCE_PATH),
    },
]

with summary_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["metric", "value"],
    )
    writer.writeheader()
    writer.writerows(summary_rows)

print("\n=== Saved Cell 11 outputs ===")
print("Mapping evidence ->", mapping_hits_path)
print("Summary          ->", summary_path)

print("\nCell 11 complete.")

=== Cell 11 purpose ===
Resolve the legacy package soil-moisture input mapping without using master_inputs.

Project root : C:\Projects\Infer RozviDrought\RozviDrought
Predictor    : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
master used  : False
Run ID       : 20260620T124807Z

=== Legacy service source ===
C:\Projects\Infer RozviDrought\RozviDrought\app\services\polygon_inference_service.py

=== Permitted predictor compatibility ===
Legacy inputs expected : ['t2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']
Available directly     : ['t2m', 'd2m', 'pet', 'ndvi', 'tws']
Missing directly       : ['sm']
Soil-moisture candidates:
 - sm_esa_cci
 - sm_esa_cci_anom
 - sm_esa_cci_lag1
 - sm_esa_cci_lag12
 - sm_esa_cci_lag2
 - sm_esa_cci_lag3
 - sm_esa_cci_lag6
 - sm_esa_cci_rollmean12
 - sm_esa_cci_rollmean3
 - sm_esa_cci_rollmean6
 - soil_texture_class_static

=== Mapping evidence found in package/local metadata ===
C:\Pro

In [21]:
# validate_events_ng.ipynb — Cell 12 (REWRITE)
# Purpose:
# - Diagnose the legacy hybrid-package failure before any national run.
# - Use one deterministic pixel and one target month: 201510.
# - Read only temporal_predictors_full, never master_inputs.
# - Apply confirmed mapping: legacy sm <- sm_esa_cci.
# - Run the real package preparation + subsystem path.
# - Stop immediately before fusion and report raw and selected subsystem outputs.
#
# Scope:
# - No full-country loop.
# - No legacy prediction export.
# - No master_inputs.
# - No substitute model logic.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import sys
import traceback

import pandas as pd
import pyarrow.dataset as ds


# -----------------------------------------------------------------------------
# 1. Paths and single preflight target
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL12_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

PREDICTOR_PATH = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model"
    r"\processed\model_inputs"
    r"\temporal_predictors_full_20260614T141118Z.parquet"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROBE_TARGET_YYYYMM = 201510
REQUIRED_HISTORY_MONTHS = 24
EXPECTED_NATIONAL_CELL_COUNT = 26_775

SOURCE_COLUMNS = [
    "row",
    "col",
    "lon",
    "lat",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm_esa_cci",
    "ndvi",
    "tws",
]

LEGACY_COLUMNS = [
    "pixel_id",
    "row",
    "col",
    "lon",
    "lat",
    "scenario",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

print("=== Cell 12 purpose ===")
print(
    "Run one legacy-package preflight through the real subsystem "
    "pipeline and capture the fusion-boundary inputs."
)

print("\n=== Inputs ===")
print("Project root       :", PROJECT_ROOT)
print("Predictor source   :", PREDICTOR_PATH)
print("master_inputs read :", False)
print("Target month       :", PROBE_TARGET_YYYYMM)
print("Required history   :", REQUIRED_HISTORY_MONTHS, "months")
print("Run ID             :", CELL12_RUN_ID)

for required_path in [
    PROJECT_ROOT,
    PREDICTOR_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path is missing:\n{required_path}"
        )


# -----------------------------------------------------------------------------
# 2. Real calendar helpers
# -----------------------------------------------------------------------------
def to_period(yyyymm: int) -> pd.Period:
    return pd.Period(
        str(int(yyyymm)),
        freq="M",
    )


def to_yyyymm(period: pd.Period) -> int:
    return int(period.strftime("%Y%m"))


def month_sequence(
    start_yyyymm: int,
    end_yyyymm: int,
) -> list[int]:
    start_period = to_period(start_yyyymm)
    end_period = to_period(end_yyyymm)

    if end_period < start_period:
        raise ValueError(
            f"Invalid month range: {start_yyyymm} -> {end_yyyymm}"
        )

    return [
        to_yyyymm(period)
        for period in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


PROBE_CONTEXT_START_YYYYMM = to_yyyymm(
    to_period(PROBE_TARGET_YYYYMM)
    - REQUIRED_HISTORY_MONTHS
)

EXPECTED_CONTEXT_MONTHS = month_sequence(
    PROBE_CONTEXT_START_YYYYMM,
    PROBE_TARGET_YYYYMM,
)

print("\n=== Required calendar context ===")
print(
    "Context range:",
    PROBE_CONTEXT_START_YYYYMM,
    "->",
    PROBE_TARGET_YYYYMM,
)
print(
    "Expected months:",
    len(EXPECTED_CONTEXT_MONTHS),
)
print(
    "Months:",
    EXPECTED_CONTEXT_MONTHS,
)


# -----------------------------------------------------------------------------
# 3. Import the unchanged legacy package
# -----------------------------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.polygon_inference_service import (
    PolygonInferenceService,
)

print("\n=== Package under test ===")
print(
    "Service:",
    PolygonInferenceService.__module__,
)
print("Model argument: hybrid")
print("Confirmed mapping: legacy sm <- sm_esa_cci")
print("Mapping transform: direct rename only")


# -----------------------------------------------------------------------------
# 4. Load compact permitted input context
# -----------------------------------------------------------------------------
predictor_dataset = ds.dataset(
    PREDICTOR_PATH,
    format="parquet",
)

source_table = predictor_dataset.to_table(
    columns=SOURCE_COLUMNS,
    filter=(
        (ds.field("yyyymm") >= PROBE_CONTEXT_START_YYYYMM)
        & (ds.field("yyyymm") <= PROBE_TARGET_YYYYMM)
    ),
)

legacy_input_df = source_table.to_pandas()

if legacy_input_df.empty:
    raise AssertionError(
        "No rows were returned for the probe context."
    )

for column_name in [
    "row",
    "col",
    "yyyymm",
]:
    legacy_input_df[column_name] = pd.to_numeric(
        legacy_input_df[column_name],
        errors="raise",
    ).astype("int64")

for column_name in [
    "lon",
    "lat",
    "t2m",
    "d2m",
    "pet",
    "sm_esa_cci",
    "ndvi",
    "tws",
]:
    legacy_input_df[column_name] = pd.to_numeric(
        legacy_input_df[column_name],
        errors="coerce",
    )

observed_context_months = sorted(
    legacy_input_df["yyyymm"]
    .unique()
    .tolist()
)

if observed_context_months != EXPECTED_CONTEXT_MONTHS:
    missing_months = sorted(
        set(EXPECTED_CONTEXT_MONTHS)
        - set(observed_context_months)
    )

    unexpected_months = sorted(
        set(observed_context_months)
        - set(EXPECTED_CONTEXT_MONTHS)
    )

    raise AssertionError(
        "Unexpected source-month coverage.\n"
        f"Expected: {EXPECTED_CONTEXT_MONTHS}\n"
        f"Observed: {observed_context_months}\n"
        f"Missing: {missing_months}\n"
        f"Unexpected: {unexpected_months}"
    )

month_counts_df = (
    legacy_input_df.groupby(
        "yyyymm",
        as_index=False,
    )
    .size()
    .rename(
        columns={"size": "row_count"}
    )
)

incomplete_months_df = month_counts_df.loc[
    month_counts_df["row_count"].ne(
        EXPECTED_NATIONAL_CELL_COUNT
    )
].copy()

if not incomplete_months_df.empty:
    raise AssertionError(
        "The legacy probe context has incomplete national grids:\n"
        f"{incomplete_months_df.to_dict(orient='records')}"
    )

legacy_input_df = legacy_input_df.rename(
    columns={
        "sm_esa_cci": "sm",
    }
)

legacy_input_df["pixel_id"] = (
    legacy_input_df["row"].astype(str)
    + "_"
    + legacy_input_df["col"].astype(str)
)

legacy_input_df["scenario"] = "historical"

legacy_input_df = (
    legacy_input_df[
        LEGACY_COLUMNS
    ]
    .sort_values(
        [
            "yyyymm",
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

target_month_pixels_df = (
    legacy_input_df.loc[
        legacy_input_df["yyyymm"].eq(
            PROBE_TARGET_YYYYMM
        )
    ]
    .sort_values(
        [
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

if target_month_pixels_df.empty:
    raise AssertionError(
        "No target-month pixels were found."
    )

PROBE_PIXEL_ID = target_month_pixels_df.loc[
    0,
    "pixel_id",
]

probe_history_df = (
    legacy_input_df.loc[
        legacy_input_df["pixel_id"].eq(
            PROBE_PIXEL_ID
        ),
        [
            "yyyymm",
            "t2m",
            "d2m",
            "pet",
            "sm",
            "ndvi",
            "tws",
        ],
    ]
    .sort_values("yyyymm")
    .reset_index(drop=True)
)

if len(probe_history_df) != len(EXPECTED_CONTEXT_MONTHS):
    raise AssertionError(
        "Probe pixel does not have a complete required context."
    )

print("\n=== Compact legacy context ===")
print(
    "Rows:",
    f"{len(legacy_input_df):,}",
)
print(
    "Months:",
    len(observed_context_months),
)
print(
    "National cells per month:",
    EXPECTED_NATIONAL_CELL_COUNT,
)
print(
    "Probe pixel:",
    PROBE_PIXEL_ID,
)
print(
    "Probe TWS nulls:",
    int(probe_history_df["tws"].isna().sum()),
    "/",
    len(probe_history_df),
)
print(
    "Probe SM nulls:",
    int(probe_history_df["sm"].isna().sum()),
    "/",
    len(probe_history_df),
)


# -----------------------------------------------------------------------------
# 5. Capture actual package subsystem outputs immediately before fusion
# -----------------------------------------------------------------------------
class StopAfterFusionBoundary(Exception):
    """Intentional stop after subsystem selection, before prediction fusion."""


captured: dict[str, Any] = {
    "prepared": None,
    "raw_subsystem_outputs": None,
    "selected_subsystem_outputs": None,
}

service = PolygonInferenceService(
    master_df=legacy_input_df
)

original_run_subsystems = (
    service.subsystem_service.run_subsystems
)

original_run_fusion = (
    service.fusion_service.run
)


def capture_run_subsystems(
    prepared: Any,
) -> Any:
    captured["prepared"] = prepared

    raw_outputs = original_run_subsystems(
        prepared
    )

    captured["raw_subsystem_outputs"] = raw_outputs

    return raw_outputs


def stop_before_fusion(
    subsystem_outputs: dict[str, Any],
    model: str = "hybrid",
) -> Any:
    captured["selected_subsystem_outputs"] = (
        subsystem_outputs
    )

    raise StopAfterFusionBoundary(
        "Captured selected subsystem outputs before fusion."
    )


service.subsystem_service.run_subsystems = (
    capture_run_subsystems
)

service.fusion_service.run = stop_before_fusion

unexpected_exception = None
unexpected_traceback = None

try:
    service._infer_one_pixel(
        pixel_id=PROBE_PIXEL_ID,
        scenario="historical",
        run_yyyymm=PROBE_TARGET_YYYYMM,
        model="hybrid",
    )

except StopAfterFusionBoundary:
    pass

except Exception as error:
    unexpected_exception = repr(error)
    unexpected_traceback = traceback.format_exc()

finally:
    service.subsystem_service.run_subsystems = (
        original_run_subsystems
    )

    service.fusion_service.run = original_run_fusion


# -----------------------------------------------------------------------------
# 6. Produce compact evidence of raw and selected subsystem outputs
# -----------------------------------------------------------------------------
def inspect_output(
    stage: str,
    subsystem: str,
    value: Any,
) -> dict[str, Any]:
    record = {
        "stage": stage,
        "subsystem": subsystem,
        "object_type": type(value).__name__,
        "row_count": None,
        "column_count": None,
        "columns": None,
        "yyyymm_min": None,
        "yyyymm_max": None,
        "tws_null_count": None,
    }

    if not isinstance(value, pd.DataFrame):
        return record

    record["row_count"] = int(len(value))
    record["column_count"] = int(len(value.columns))
    record["columns"] = " | ".join(
        str(column)
        for column in value.columns
    )

    if "yyyymm" in value.columns:
        valid_months = pd.to_numeric(
            value["yyyymm"],
            errors="coerce",
        ).dropna()

        if not valid_months.empty:
            record["yyyymm_min"] = int(
                valid_months.min()
            )
            record["yyyymm_max"] = int(
                valid_months.max()
            )

    if "tws" in value.columns:
        record["tws_null_count"] = int(
            value["tws"].isna().sum()
        )

    return record


def inspect_output_dict(
    stage: str,
    payload: Any,
) -> list[dict[str, Any]]:
    if not isinstance(payload, dict):
        return [
            inspect_output(
                stage=stage,
                subsystem="__container__",
                value=payload,
            )
        ]

    return [
        inspect_output(
            stage=stage,
            subsystem=str(name),
            value=value,
        )
        for name, value in payload.items()
    ]


subsystem_audit_df = pd.DataFrame(
    inspect_output_dict(
        stage="raw_subsystem_output",
        payload=captured["raw_subsystem_outputs"],
    )
    + inspect_output_dict(
        stage="selected_for_fusion",
        payload=captured["selected_subsystem_outputs"],
    )
)

selected_hydrology_df = subsystem_audit_df.loc[
    subsystem_audit_df["stage"].eq(
        "selected_for_fusion"
    )
    & subsystem_audit_df["subsystem"].eq(
        "hydrology"
    )
].copy()

raw_hydrology_df = subsystem_audit_df.loc[
    subsystem_audit_df["stage"].eq(
        "raw_subsystem_output"
    )
    & subsystem_audit_df["subsystem"].eq(
        "hydrology"
    )
].copy()

if unexpected_exception is not None:
    CELL12_STATUS = (
        "legacy_preflight_failed_before_fusion_boundary"
    )

elif selected_hydrology_df.empty:
    CELL12_STATUS = (
        "legacy_preflight_failed_hydrology_missing_at_fusion_boundary"
    )

elif int(
    selected_hydrology_df["row_count"].iloc[0] or 0
) == 0:
    CELL12_STATUS = (
        "legacy_hybrid_blocked_selected_hydrology_output_empty"
    )

else:
    CELL12_STATUS = (
        "legacy_hybrid_preflight_passed_ready_for_national_run"
    )

prepared = captured["prepared"]

prepared_summary = {
    "prepared_object_type": (
        type(prepared).__name__
        if prepared is not None
        else None
    ),
    "prepared_attribute_names": (
        " | ".join(
            sorted(vars(prepared).keys())
        )
        if prepared is not None
        and hasattr(prepared, "__dict__")
        else None
    ),
}

print("\n=== Raw and selected subsystem outputs ===")
print(
    subsystem_audit_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

print("\n=== Cell 12 conclusion ===")
print("Status:", CELL12_STATUS)

if not raw_hydrology_df.empty:
    print(
        "Raw hydrology rows:",
        raw_hydrology_df["row_count"].iloc[0],
    )

if not selected_hydrology_df.empty:
    print(
        "Selected hydrology rows:",
        selected_hydrology_df["row_count"].iloc[0],
    )

if unexpected_exception is not None:
    print(
        "Unexpected package exception:",
        unexpected_exception,
    )


# -----------------------------------------------------------------------------
# 7. Save compact diagnostic evidence
# -----------------------------------------------------------------------------
subsystem_audit_path = (
    OUT_DIR
    / f"cell12_legacy_subsystem_boundary_{CELL12_RUN_ID}.csv"
)

probe_history_path = (
    OUT_DIR
    / f"cell12_legacy_probe_feature_history_{CELL12_RUN_ID}.csv"
)

month_counts_path = (
    OUT_DIR
    / f"cell12_legacy_context_month_counts_{CELL12_RUN_ID}.csv"
)

summary_path = (
    OUT_DIR
    / f"cell12_legacy_preflight_summary_{CELL12_RUN_ID}.json"
)

subsystem_audit_df.to_csv(
    subsystem_audit_path,
    index=False,
)

probe_history_df.to_csv(
    probe_history_path,
    index=False,
)

month_counts_df.to_csv(
    month_counts_path,
    index=False,
)

summary_payload = {
    "run_id": CELL12_RUN_ID,
    "status": CELL12_STATUS,
    "master_inputs_used": False,
    "input_source": str(PREDICTOR_PATH),
    "legacy_sm_mapping": "sm <- sm_esa_cci",
    "mapping_transformation": "direct_rename_only",
    "probe_pixel_id": PROBE_PIXEL_ID,
    "target_yyyymm": PROBE_TARGET_YYYYMM,
    "context_start_yyyymm": PROBE_CONTEXT_START_YYYYMM,
    "context_month_count": len(EXPECTED_CONTEXT_MONTHS),
    "probe_tws_null_count": int(
        probe_history_df["tws"].isna().sum()
    ),
    "probe_sm_null_count": int(
        probe_history_df["sm"].isna().sum()
    ),
    "unexpected_exception": unexpected_exception,
    "unexpected_traceback": unexpected_traceback,
    "prepared_summary": prepared_summary,
    "outputs": {
        "subsystem_boundary_csv": str(
            subsystem_audit_path
        ),
        "probe_feature_history_csv": str(
            probe_history_path
        ),
        "context_month_counts_csv": str(
            month_counts_path
        ),
        "summary_json": str(summary_path),
    },
}

summary_path.write_text(
    json.dumps(
        summary_payload,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 12 outputs ===")
print("Subsystem boundary ->", subsystem_audit_path)
print("Probe history      ->", probe_history_path)
print("Month counts       ->", month_counts_path)
print("Preflight summary  ->", summary_path)

print("\nCell 12 complete.")

=== Cell 12 purpose ===
Run one legacy-package preflight through the real subsystem pipeline and capture the fusion-boundary inputs.

=== Inputs ===
Project root       : C:\Projects\Infer RozviDrought\RozviDrought
Predictor source   : C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
master_inputs read : False
Target month       : 201510
Required history   : 24 months
Run ID             : 20260620T130907Z

=== Required calendar context ===
Context range: 201310 -> 201510
Expected months: 25
Months: [201310, 201311, 201312, 201401, 201402, 201403, 201404, 201405, 201406, 201407, 201408, 201409, 201410, 201411, 201412, 201501, 201502, 201503, 201504, 201505, 201506, 201507, 201508, 201509, 201510]

=== Package under test ===
Service: app.services.polygon_inference_service
Model argument: hybrid
Confirmed mapping: legacy sm <- sm_esa_cci
Mapping transform: direct rename only

=== Compact legacy context ===
Rows: 669,37

In [35]:
# validate_events_ng.ipynb — Cell 13 (REWRITE 3)
# Purpose:
# - Build expanded legacy-package event context with enough TWS history for
#   hydrology feature warm-up and the packaged GRU sequence.
# - Read predictor data for t2m, d2m, pet, sm_esa_cci, ndvi, and predictor TWS.
# - Read master_inputs only to backfill missing predictor TWS.
# - Preserve non-TWS missing values exactly as found.
# - Audit non-TWS availability only at scored event months.
# - Prove that each event's first scored month has a valid hydrology window.
#
# This cell does not run the legacy model nationally.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import os
import re
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


# -----------------------------------------------------------------------------
# 1. Paths and outputs
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL13_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

PREDICTOR_PATH = Path(
    r"C:\Projects\ramangwana\risks\data\drought_model\processed"
    r"\model_inputs\temporal_predictors_full_20260614T141118Z.parquet"
)

MASTER_TWS_PATH = Path(
    r"C:\Projects\Infer RozviDrought\data\master_inputs"
    r"\master_inputs_long_198001_205012.parquet"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CONTEXT_PATH = (
    OUT_DIR
    / f"cell13_legacy_context_tws_backfilled_{CELL13_RUN_ID}.parquet"
)

TEMP_CONTEXT_PATH = (
    OUT_DIR
    / f".cell13_legacy_context_tws_backfilled_{CELL13_RUN_ID}.tmp.parquet"
)

COVERAGE_PATH = (
    OUT_DIR
    / f"cell13_legacy_context_coverage_{CELL13_RUN_ID}.csv"
)

MONTHLY_PROVENANCE_PATH = (
    OUT_DIR
    / f"cell13_legacy_monthly_tws_provenance_{CELL13_RUN_ID}.csv"
)

EVENT_EXPANSION_PATH = (
    OUT_DIR
    / f"cell13_legacy_event_context_expansion_{CELL13_RUN_ID}.csv"
)

HYDROLOGY_PREFLIGHT_PATH = (
    OUT_DIR
    / f"cell13_legacy_hydrology_sequence_preflight_{CELL13_RUN_ID}.csv"
)

TARGET_RAW_COVERAGE_PATH = (
    OUT_DIR
    / f"cell13_legacy_target_raw_coverage_{CELL13_RUN_ID}.csv"
)

MANIFEST_PATH = (
    OUT_DIR
    / f"cell13_legacy_context_manifest_{CELL13_RUN_ID}.json"
)

PREDICTOR_COLUMNS = [
    "row",
    "col",
    "lon",
    "lat",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm_esa_cci",
    "ndvi",
    "tws",
]

MASTER_TWS_COLUMNS = [
    "row",
    "col",
    "yyyymm",
    "tws",
]

FINAL_CONTEXT_COLUMNS = [
    "pixel_id",
    "row",
    "col",
    "lon",
    "lat",
    "scenario",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
    "tws_source",
]

NON_TWS_RAW_FEATURES = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
]

LEGACY_RAW_FEATURES = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

print("=== Cell 13 purpose ===")
print(
    "Build expanded legacy context using predictor data and "
    "master TWS-only backfill."
)

print("\n=== Paths ===")
print("Project root    :", PROJECT_ROOT)
print("Predictor source:", PREDICTOR_PATH)
print("Master TWS only :", MASTER_TWS_PATH)
print("Output parquet  :", FINAL_CONTEXT_PATH)
print("Run ID          :", CELL13_RUN_ID)

for required_path, label in [
    (PROJECT_ROOT, "Project root"),
    (PREDICTOR_PATH, "Predictor source"),
    (MASTER_TWS_PATH, "Master TWS source"),
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"{label} is missing:\n{required_path}"
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# 2. Read the installed hydrology contract
# -----------------------------------------------------------------------------
from app.services.feature_service import FeatureService
from rozvidrought_subsystems.hydrology_windows import (
    build_hydrology_sequences,
)
from rozvidrought_subsystems.loaders import (
    load_subsystem_spec,
)

hydrology_spec = load_subsystem_spec("hydrology")

HYDROLOGY_FEATURES = list(
    hydrology_spec.feature_list
)

HYDROLOGY_SEQUENCE_CONFIG = dict(
    hydrology_spec.sequence_config
)

HYDROLOGY_SEQ_LEN = int(
    HYDROLOGY_SEQUENCE_CONFIG["seq_len"]
)

HYDROLOGY_MAX_GAP_MONTHS = int(
    HYDROLOGY_SEQUENCE_CONFIG["max_gap_months"]
)


def feature_warmup_months(
    feature_list: list[str],
) -> int:
    warmup = 0

    for feature_name in feature_list:
        lag_match = re.search(
            r"_lag(\d+)$",
            feature_name,
        )

        if lag_match:
            warmup = max(
                warmup,
                int(lag_match.group(1)),
            )

        rolling_match = re.search(
            r"_roll(?:mean|std)(\d+)$",
            feature_name,
        )

        if rolling_match:
            warmup = max(
                warmup,
                int(rolling_match.group(1)) - 1,
            )

    return warmup


HYDROLOGY_WARMUP_MONTHS = feature_warmup_months(
    HYDROLOGY_FEATURES
)

TOTAL_REQUIRED_RAW_MONTHS = (
    HYDROLOGY_WARMUP_MONTHS
    + HYDROLOGY_SEQ_LEN
)

print("\n=== Installed hydrology contract ===")
print("Feature list        :", HYDROLOGY_FEATURES)
print("Warm-up months      :", HYDROLOGY_WARMUP_MONTHS)
print("GRU sequence length :", HYDROLOGY_SEQ_LEN)
print("Maximum month gap   :", HYDROLOGY_MAX_GAP_MONTHS)
print("Required raw months :", TOTAL_REQUIRED_RAW_MONTHS)

if TOTAL_REQUIRED_RAW_MONTHS < 1:
    raise AssertionError(
        "Hydrology history requirement is invalid."
    )


# -----------------------------------------------------------------------------
# 3. Calendar helpers and event windows
# -----------------------------------------------------------------------------
def to_period(
    yyyymm: int,
) -> pd.Period:
    return pd.Period(
        str(int(yyyymm)),
        freq="M",
    )


def to_yyyymm(
    period: pd.Period,
) -> int:
    return int(period.strftime("%Y%m"))


def month_range(
    start_yyyymm: int,
    end_yyyymm: int,
) -> list[int]:
    start = to_period(start_yyyymm)
    end = to_period(end_yyyymm)

    if end < start:
        raise ValueError(
            f"Invalid date range: "
            f"{start_yyyymm} -> {end_yyyymm}"
        )

    return [
        to_yyyymm(period)
        for period in pd.period_range(
            start=start,
            end=end,
            freq="M",
        )
    ]


def required_raw_start(
    first_scored_yyyymm: int,
) -> int:
    return to_yyyymm(
        to_period(first_scored_yyyymm)
        - (TOTAL_REQUIRED_RAW_MONTHS - 1)
    )


EVENT_SPECS = [
    {
        "event_id": "zimbabwe_2015_2016",
        "first_scored_yyyymm": 201510,
        "last_scored_yyyymm": 201602,
    },
    {
        "event_id": "zimbabwe_2018_2019",
        "first_scored_yyyymm": 201810,
        "last_scored_yyyymm": 201905,
    },
    {
        "event_id": "zimbabwe_2023_2024",
        "first_scored_yyyymm": 202310,
        "last_scored_yyyymm": 202403,
    },
]

for event in EVENT_SPECS:
    event["required_raw_context_start"] = (
        required_raw_start(
            event["first_scored_yyyymm"]
        )
    )

    event["scored_months"] = month_range(
        event["first_scored_yyyymm"],
        event["last_scored_yyyymm"],
    )

    event["raw_context_months"] = month_range(
        event["required_raw_context_start"],
        event["last_scored_yyyymm"],
    )

event_expansion_df = pd.DataFrame(
    [
        {
            "event_id": event["event_id"],
            "required_raw_context_start": event[
                "required_raw_context_start"
            ],
            "first_scored_yyyymm": event[
                "first_scored_yyyymm"
            ],
            "last_scored_yyyymm": event[
                "last_scored_yyyymm"
            ],
            "scored_month_count": len(
                event["scored_months"]
            ),
            "raw_context_month_count": len(
                event["raw_context_months"]
            ),
        }
        for event in EVENT_SPECS
    ]
)

print("\n=== Expanded event contexts ===")
print(event_expansion_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 4. Merge overlapping context intervals
# -----------------------------------------------------------------------------
intervals = sorted(
    [
        {
            "event_id": event["event_id"],
            "start_yyyymm": event[
                "required_raw_context_start"
            ],
            "end_yyyymm": event[
                "last_scored_yyyymm"
            ],
        }
        for event in EVENT_SPECS
    ],
    key=lambda record: (
        record["start_yyyymm"],
        record["end_yyyymm"],
    ),
)

segments: list[dict] = []

for interval in intervals:
    if not segments:
        segments.append(
            {
                "start_yyyymm": interval[
                    "start_yyyymm"
                ],
                "end_yyyymm": interval[
                    "end_yyyymm"
                ],
                "event_ids": [interval["event_id"]],
            }
        )
        continue

    prior = segments[-1]

    month_after_prior = to_yyyymm(
        to_period(prior["end_yyyymm"]) + 1
    )

    if interval["start_yyyymm"] <= month_after_prior:
        prior["end_yyyymm"] = max(
            prior["end_yyyymm"],
            interval["end_yyyymm"],
        )

        prior["event_ids"].append(
            interval["event_id"]
        )

    else:
        segments.append(
            {
                "start_yyyymm": interval[
                    "start_yyyymm"
                ],
                "end_yyyymm": interval[
                    "end_yyyymm"
                ],
                "event_ids": [interval["event_id"]],
            }
        )

for index, segment in enumerate(
    segments,
    start=1,
):
    segment["segment_id"] = (
        f"segment_{index:02d}_"
        f"{segment['start_yyyymm']}_"
        f"{segment['end_yyyymm']}"
    )

    segment["months"] = month_range(
        segment["start_yyyymm"],
        segment["end_yyyymm"],
    )

print("\n=== Unique context segments ===")
for segment in segments:
    print(
        f"{segment['segment_id']}: "
        f"{segment['start_yyyymm']} -> "
        f"{segment['end_yyyymm']} | "
        f"{len(segment['months'])} months | "
        f"events={segment['event_ids']}"
    )


# -----------------------------------------------------------------------------
# 5. Open datasets and validate source fields
# -----------------------------------------------------------------------------
predictor_dataset = ds.dataset(
    PREDICTOR_PATH,
    format="parquet",
)

master_dataset = ds.dataset(
    MASTER_TWS_PATH,
    format="parquet",
)

missing_predictor_columns = [
    column_name
    for column_name in PREDICTOR_COLUMNS
    if column_name not in predictor_dataset.schema.names
]

missing_master_columns = [
    column_name
    for column_name in MASTER_TWS_COLUMNS
    if column_name not in master_dataset.schema.names
]

if missing_predictor_columns:
    raise AssertionError(
        "Predictor source lacks required fields:\n"
        f"{missing_predictor_columns}"
    )

if missing_master_columns:
    raise AssertionError(
        "Master source lacks required TWS fields:\n"
        f"{missing_master_columns}"
    )

print("\n=== Source fields ===")
print("Predictor fields:", PREDICTOR_COLUMNS)
print("Master fields   :", MASTER_TWS_COLUMNS)
print("pixel_id rule   : '<row>_<col>'")
print(
    "scenario rule   : null; not provided by either "
    "source and unused in this direct component path."
)


# -----------------------------------------------------------------------------
# 6. Source-normalisation and validation helpers
# -----------------------------------------------------------------------------
def typed_month_value(
    arrow_type: pa.DataType,
    yyyymm: int,
):
    if (
        pa.types.is_string(arrow_type)
        or pa.types.is_large_string(arrow_type)
    ):
        return str(int(yyyymm))

    if pa.types.is_integer(arrow_type):
        return int(yyyymm)

    raise TypeError(
        f"Unsupported yyyymm type: {arrow_type}"
    )


def month_filter(
    dataset: ds.Dataset,
    start_yyyymm: int,
    end_yyyymm: int,
):
    month_type = dataset.schema.field(
        "yyyymm"
    ).type

    return (
        ds.field("yyyymm")
        >= typed_month_value(
            month_type,
            start_yyyymm,
        )
    ) & (
        ds.field("yyyymm")
        <= typed_month_value(
            month_type,
            end_yyyymm,
        )
    )


def build_pixel_id(
    frame: pd.DataFrame,
) -> pd.Series:
    return (
        frame["row"].astype("int64").astype(str)
        + "_"
        + frame["col"].astype("int64").astype(str)
    )


def standardise_predictor(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    work = frame.rename(
        columns={
            "sm_esa_cci": "sm",
            "tws": "tws_predictor",
        }
    ).copy()

    for column_name in ["row", "col", "yyyymm"]:
        work[column_name] = pd.to_numeric(
            work[column_name],
            errors="raise",
        ).astype("int64")

    for column_name in [
        "t2m",
        "d2m",
        "pet",
        "sm",
        "ndvi",
        "tws_predictor",
    ]:
        work[column_name] = pd.to_numeric(
            work[column_name],
            errors="coerce",
        )

    work["pixel_id"] = build_pixel_id(work)

    return work


def standardise_master_tws(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    work = frame.rename(
        columns={
            "tws": "tws_master",
        }
    ).copy()

    for column_name in ["row", "col", "yyyymm"]:
        work[column_name] = pd.to_numeric(
            work[column_name],
            errors="raise",
        ).astype("int64")

    work["tws_master"] = pd.to_numeric(
        work["tws_master"],
        errors="coerce",
    )

    return work


def validate_grid(
    frame: pd.DataFrame,
    expected_months: list[int],
    source_label: str,
) -> pd.Series:
    observed_months = sorted(
        frame["yyyymm"].unique().tolist()
    )

    if observed_months != expected_months:
        raise AssertionError(
            f"{source_label}: unexpected month coverage.\n"
            f"Expected: {expected_months}\n"
            f"Observed: {observed_months}"
        )

    if frame.duplicated(
        subset=["row", "col", "yyyymm"]
    ).any():
        raise AssertionError(
            f"{source_label}: duplicate row/col/month values."
        )

    monthly_counts = (
        frame.groupby("yyyymm")
        .size()
        .sort_index()
    )

    if monthly_counts.nunique() != 1:
        raise AssertionError(
            f"{source_label}: grid count changes by month.\n"
            f"{monthly_counts.to_string()}"
        )

    return monthly_counts


# -----------------------------------------------------------------------------
# 7. Build context segments
# -----------------------------------------------------------------------------
feature_service = FeatureService()

coverage_records: list[dict] = []
monthly_provenance_frames: list[pd.DataFrame] = []
hydrology_preflight_records: list[dict] = []
target_raw_coverage_records: list[dict] = []

parquet_writer = None
rows_written = 0
grid_rows_per_month = None

try:
    for segment in segments:
        segment_id = segment["segment_id"]
        segment_start = segment["start_yyyymm"]
        segment_end = segment["end_yyyymm"]
        segment_months = segment["months"]

        print("\n" + "=" * 90)
        print(
            f"Building {segment_id}: "
            f"{segment_start} -> {segment_end}"
        )

        predictor_table = predictor_dataset.to_table(
            columns=PREDICTOR_COLUMNS,
            filter=month_filter(
                predictor_dataset,
                segment_start,
                segment_end,
            ),
        )

        master_table = master_dataset.to_table(
            columns=MASTER_TWS_COLUMNS,
            filter=month_filter(
                master_dataset,
                segment_start,
                segment_end,
            ),
        )

        predictor_df = standardise_predictor(
            predictor_table.to_pandas()
        )

        master_tws_df = standardise_master_tws(
            master_table.to_pandas()
        )

        del predictor_table
        del master_table

        predictor_counts = validate_grid(
            frame=predictor_df,
            expected_months=segment_months,
            source_label=f"{segment_id} predictor",
        )

        master_counts = validate_grid(
            frame=master_tws_df,
            expected_months=segment_months,
            source_label=f"{segment_id} master TWS",
        )

        if not predictor_counts.equals(master_counts):
            raise AssertionError(
                f"{segment_id}: predictor and master TWS grids "
                "do not have identical monthly row counts.\n\n"
                f"Predictor:\n{predictor_counts.to_string()}\n\n"
                f"Master:\n{master_counts.to_string()}"
            )

        current_grid_rows = int(
            predictor_counts.iloc[0]
        )

        if grid_rows_per_month is None:
            grid_rows_per_month = current_grid_rows

        elif current_grid_rows != grid_rows_per_month:
            raise AssertionError(
                "Grid row count changed between segments.\n"
                f"Prior: {grid_rows_per_month}\n"
                f"Current: {current_grid_rows}"
            )

        merged_df = predictor_df.merge(
            master_tws_df,
            on=["row", "col", "yyyymm"],
            how="left",
            validate="one_to_one",
            indicator=True,
            sort=False,
        )

        unmatched_master_rows = int(
            (~merged_df["_merge"].eq("both")).sum()
        )

        if unmatched_master_rows:
            raise AssertionError(
                f"{segment_id}: predictor rows without matching "
                f"master TWS rows: {unmatched_master_rows}"
            )

        merged_df["tws"] = np.where(
            merged_df["tws_predictor"].notna(),
            merged_df["tws_predictor"],
            merged_df["tws_master"],
        )

        merged_df["tws_source"] = np.where(
            merged_df["tws_predictor"].notna(),
            "predictor",
            "master_backfill",
        )

        remaining_tws_missing = int(
            merged_df["tws"].isna().sum()
        )

        if remaining_tws_missing:
            raise AssertionError(
                f"{segment_id}: TWS remains missing after "
                f"allowed master-only fallback: "
                f"{remaining_tws_missing}"
            )

        merged_df["scenario"] = pd.Series(
            pd.NA,
            index=merged_df.index,
            dtype="string",
        )

        merged_df["tws_source"] = (
            merged_df["tws_source"]
            .astype("string")
        )

        final_df = merged_df[
            FINAL_CONTEXT_COLUMNS
        ].copy()

        expected_rows = (
            len(segment_months)
            * current_grid_rows
        )

        if len(final_df) != expected_rows:
            raise AssertionError(
                f"{segment_id}: final row mismatch.\n"
                f"Expected: {expected_rows}\n"
                f"Observed: {len(final_df)}"
            )

        if final_df.duplicated(
            subset=["pixel_id", "yyyymm"]
        ).any():
            raise AssertionError(
                f"{segment_id}: duplicate pixel/month rows "
                "after context construction."
            )

        # ---------------------------------------------------------------------
        # 7A. Audit raw non-TWS availability at the scored event months only.
        # No imputation. No failure here: this is descriptive evidence for
        # later legacy component coverage checks.
        # ---------------------------------------------------------------------
        segment_events = [
            event
            for event in EVENT_SPECS
            if (
                segment_start
                <= event["first_scored_yyyymm"]
                <= segment_end
            )
        ]

        for event in segment_events:
            event_target_df = (
                final_df.loc[
                    final_df["yyyymm"].isin(
                        event["scored_months"]
                    )
                ]
                .copy()
            )

            for target_yyyymm, target_month_df in (
                event_target_df.groupby(
                    "yyyymm",
                    sort=True,
                )
            ):
                complete_raw_rows = int(
                    target_month_df[
                        LEGACY_RAW_FEATURES
                    ]
                    .notna()
                    .all(axis=1)
                    .sum()
                )

                record = {
                    "event_id": event["event_id"],
                    "yyyymm": int(target_yyyymm),
                    "grid_rows": int(
                        len(target_month_df)
                    ),
                    "complete_all_legacy_raw_rows": (
                        complete_raw_rows
                    ),
                }

                for feature_name in LEGACY_RAW_FEATURES:
                    record[
                        f"{feature_name}_null_rows"
                    ] = int(
                        target_month_df[
                            feature_name
                        ]
                        .isna()
                        .sum()
                    )

                target_raw_coverage_records.append(
                    record
                )

        # ---------------------------------------------------------------------
        # 7B. Prove the hydrology sequence is valid for each first scored month.
        # The preflight constructs windows but does not call model.predict().
        # ---------------------------------------------------------------------
        for event in segment_events:
            event_id = event["event_id"]
            raw_start = event[
                "required_raw_context_start"
            ]
            first_scored = event[
                "first_scored_yyyymm"
            ]

            hydrology_context_df = (
                final_df.loc[
                    final_df["yyyymm"].between(
                        raw_start,
                        first_scored,
                    )
                ]
                .copy()
            )

            expected_hydrology_rows = (
                TOTAL_REQUIRED_RAW_MONTHS
                * current_grid_rows
            )

            if len(hydrology_context_df) != expected_hydrology_rows:
                raise AssertionError(
                    f"{event_id}: incomplete raw TWS history.\n"
                    f"Expected rows: "
                    f"{expected_hydrology_rows}\n"
                    f"Observed rows: "
                    f"{len(hydrology_context_df)}"
                )

            probe_target_df = (
                hydrology_context_df.loc[
                    hydrology_context_df["yyyymm"].eq(
                        first_scored
                    )
                ]
                .sort_values(["row", "col"])
                .reset_index(drop=True)
            )

            probe_pixel_id = probe_target_df.loc[
                0,
                "pixel_id",
            ]

            probe_df = (
                hydrology_context_df.loc[
                    hydrology_context_df["pixel_id"].eq(
                        probe_pixel_id
                    )
                ]
                .sort_values("yyyymm")
                .reset_index(drop=True)
                .copy()
            )

            if len(probe_df) != TOTAL_REQUIRED_RAW_MONTHS:
                raise AssertionError(
                    f"{event_id}: hydrology probe has "
                    "incomplete monthly history."
                )

            if probe_df["tws"].isna().any():
                raise AssertionError(
                    f"{event_id}: hydrology probe contains "
                    "missing final TWS."
                )

            probe_df["yyyymm"] = (
                probe_df["yyyymm"]
                .astype("int64")
                .astype(str)
            )

            prepared = feature_service.prepare_subsystem_inputs(
                timeseries=probe_df,
                run_yyyymm=first_scored,
            )

            hydrology_df = (
                prepared["hydrology"]
                .sort_values("yyyymm")
                .reset_index(drop=True)
                .copy()
            )

            hydrology_df["yyyymm"] = (
                hydrology_df["yyyymm"]
                .astype(str)
            )

            sequence_array, valid_index = (
                build_hydrology_sequences(
                    df=hydrology_df,
                    pixel_col="pixel_key",
                    time_col="yyyymm",
                )
            )

            target_index = hydrology_df.index[
                hydrology_df["yyyymm"].eq(
                    str(first_scored)
                )
            ].tolist()

            if len(target_index) != 1:
                raise AssertionError(
                    f"{event_id}: expected one hydrology "
                    f"target row, got {len(target_index)}."
                )

            target_has_valid_window = bool(
                target_index[0]
                in valid_index.tolist()
            )

            hydrology_preflight_records.append(
                {
                    "event_id": event_id,
                    "probe_pixel_id": probe_pixel_id,
                    "required_raw_context_start": raw_start,
                    "first_scored_yyyymm": first_scored,
                    "raw_context_months": int(
                        len(probe_df)
                    ),
                    "seq_len": HYDROLOGY_SEQ_LEN,
                    "valid_window_count": int(
                        len(valid_index)
                    ),
                    "target_has_valid_window": (
                        target_has_valid_window
                    ),
                    "sequence_shape": "x".join(
                        map(
                            str,
                            sequence_array.shape,
                        )
                    ),
                }
            )

            if not target_has_valid_window:
                raise AssertionError(
                    f"{event_id}: no valid hydrology sequence "
                    f"for {first_scored}."
                )

            print(
                f"{event_id}: hydrology target window confirmed "
                f"for pixel {probe_pixel_id}."
            )

        # ---------------------------------------------------------------------
        # 7C. Record provenance and write the segment.
        # ---------------------------------------------------------------------
        monthly_provenance_df = (
            final_df.groupby(
                ["yyyymm", "tws_source"],
                as_index=False,
            )
            .size()
            .pivot(
                index="yyyymm",
                columns="tws_source",
                values="size",
            )
            .fillna(0)
            .reset_index()
        )

        for expected_column in [
            "predictor",
            "master_backfill",
        ]:
            if expected_column not in monthly_provenance_df.columns:
                monthly_provenance_df[
                    expected_column
                ] = 0

        monthly_provenance_df["segment_id"] = segment_id
        monthly_provenance_df["event_ids"] = "|".join(
            segment["event_ids"]
        )

        monthly_provenance_frames.append(
            monthly_provenance_df[
                [
                    "segment_id",
                    "event_ids",
                    "yyyymm",
                    "predictor",
                    "master_backfill",
                ]
            ]
        )

        non_tws_null_counts = (
            final_df[NON_TWS_RAW_FEATURES]
            .isna()
            .sum()
        )

        coverage_record = {
            "segment_id": segment_id,
            "event_ids": "|".join(
                segment["event_ids"]
            ),
            "start_yyyymm": segment_start,
            "end_yyyymm": segment_end,
            "month_count": len(segment_months),
            "grid_rows_per_month": current_grid_rows,
            "final_rows": int(len(final_df)),
            "predictor_tws_rows": int(
                final_df["tws_source"]
                .eq("predictor")
                .sum()
            ),
            "master_tws_backfill_rows": int(
                final_df["tws_source"]
                .eq("master_backfill")
                .sum()
            ),
            "final_tws_missing_rows": int(
                final_df["tws"].isna().sum()
            ),
        }

        for feature_name in NON_TWS_RAW_FEATURES:
            coverage_record[
                f"{feature_name}_null_rows"
            ] = int(
                non_tws_null_counts[
                    feature_name
                ]
            )

        coverage_records.append(coverage_record)

        output_table = pa.Table.from_pandas(
            final_df,
            preserve_index=False,
        )

        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(
                TEMP_CONTEXT_PATH,
                output_table.schema,
                compression="zstd",
            )

        parquet_writer.write_table(output_table)
        rows_written += len(final_df)

        print(
            f"{segment_id} written: "
            f"{len(final_df):,} rows | "
            f"predictor TWS="
            f"{int(final_df['tws_source'].eq('predictor').sum()):,} | "
            f"master backfill="
            f"{int(final_df['tws_source'].eq('master_backfill').sum()):,}"
        )

        del predictor_df
        del master_tws_df
        del merged_df
        del final_df
        del output_table
        gc.collect()

    if parquet_writer is None:
        raise AssertionError(
            "No output context was written."
        )

    parquet_writer.close()
    parquet_writer = None

    os.replace(
        TEMP_CONTEXT_PATH,
        FINAL_CONTEXT_PATH,
    )

except Exception:
    if parquet_writer is not None:
        parquet_writer.close()

    if TEMP_CONTEXT_PATH.exists():
        TEMP_CONTEXT_PATH.unlink()

    raise


# -----------------------------------------------------------------------------
# 8. Save evidence
# -----------------------------------------------------------------------------
coverage_df = pd.DataFrame(
    coverage_records
)

monthly_provenance_out_df = (
    pd.concat(
        monthly_provenance_frames,
        ignore_index=True,
    )
    .sort_values(["yyyymm", "segment_id"])
    .reset_index(drop=True)
)

hydrology_preflight_df = (
    pd.DataFrame(hydrology_preflight_records)
    .sort_values("event_id")
    .reset_index(drop=True)
)

target_raw_coverage_df = (
    pd.DataFrame(target_raw_coverage_records)
    .sort_values(["event_id", "yyyymm"])
    .reset_index(drop=True)
)

if hydrology_preflight_df.empty:
    raise AssertionError(
        "No hydrology preflight records were produced."
    )

if not hydrology_preflight_df[
    "target_has_valid_window"
].all():
    raise AssertionError(
        "At least one event lacks a valid hydrology "
        "window at its first scored month."
    )

coverage_df.to_csv(
    COVERAGE_PATH,
    index=False,
)

monthly_provenance_out_df.to_csv(
    MONTHLY_PROVENANCE_PATH,
    index=False,
)

event_expansion_df.to_csv(
    EVENT_EXPANSION_PATH,
    index=False,
)

hydrology_preflight_df.to_csv(
    HYDROLOGY_PREFLIGHT_PATH,
    index=False,
)

target_raw_coverage_df.to_csv(
    TARGET_RAW_COVERAGE_PATH,
    index=False,
)

manifest = {
    "run_id": CELL13_RUN_ID,
    "purpose": (
        "Expanded legacy hydrology context with predictor inputs "
        "and master TWS-only fallback."
    ),
    "predictor_source": str(PREDICTOR_PATH),
    "master_tws_only_source": str(MASTER_TWS_PATH),
    "predictor_columns_read": PREDICTOR_COLUMNS,
    "master_columns_read": MASTER_TWS_COLUMNS,
    "pixel_id_rule": "pixel_id = '<row>_<col>'",
    "scenario_rule": (
        "scenario is null because neither source supplies it."
    ),
    "raw_feature_mapping": {
        "t2m": "predictor:t2m",
        "d2m": "predictor:d2m",
        "pet": "predictor:pet",
        "sm": "predictor:sm_esa_cci",
        "ndvi": "predictor:ndvi",
        "tws": (
            "predictor:tws where non-null; "
            "otherwise master_inputs:tws"
        ),
    },
    "hydrology_contract": {
        "feature_list": HYDROLOGY_FEATURES,
        "sequence_config": HYDROLOGY_SEQUENCE_CONFIG,
        "warmup_months": HYDROLOGY_WARMUP_MONTHS,
        "required_raw_months": TOTAL_REQUIRED_RAW_MONTHS,
    },
    "events": [
        {
            key: value
            for key, value in event.items()
            if key not in {
                "scored_months",
                "raw_context_months",
            }
        }
        for event in EVENT_SPECS
    ],
    "grid_rows_per_month": grid_rows_per_month,
    "rows_written": rows_written,
    "hydrology_preflight_passed": bool(
        hydrology_preflight_df[
            "target_has_valid_window"
        ].all()
    ),
    "outputs": {
        "context_parquet": str(FINAL_CONTEXT_PATH),
        "coverage_csv": str(COVERAGE_PATH),
        "monthly_tws_provenance_csv": str(
            MONTHLY_PROVENANCE_PATH
        ),
        "event_expansion_csv": str(
            EVENT_EXPANSION_PATH
        ),
        "hydrology_preflight_csv": str(
            HYDROLOGY_PREFLIGHT_PATH
        ),
        "target_raw_coverage_csv": str(
            TARGET_RAW_COVERAGE_PATH
        ),
        "manifest_json": str(MANIFEST_PATH),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# 9. Final report
# -----------------------------------------------------------------------------
print("\n=== Cell 13 final result ===")
print("Grid rows/month      :", grid_rows_per_month)
print("Context rows written :", rows_written)
print(
    "Predictor TWS rows   :",
    int(
        coverage_df[
            "predictor_tws_rows"
        ].sum()
    ),
)
print(
    "Master TWS backfills :",
    int(
        coverage_df[
            "master_tws_backfill_rows"
        ].sum()
    ),
)
print(
    "Final missing TWS    :",
    int(
        coverage_df[
            "final_tws_missing_rows"
        ].sum()
    ),
)
print(
    "Hydrology preflight  :",
    bool(
        hydrology_preflight_df[
            "target_has_valid_window"
        ].all()
    ),
)

print("\n=== Hydrology sequence preflight ===")
print(hydrology_preflight_df.to_string(index=False))

print("\n=== Saved Cell 13 outputs ===")
print("Context parquet      ->", FINAL_CONTEXT_PATH)
print("Coverage             ->", COVERAGE_PATH)
print("Monthly provenance   ->", MONTHLY_PROVENANCE_PATH)
print("Event expansion      ->", EVENT_EXPANSION_PATH)
print("Hydrology preflight  ->", HYDROLOGY_PREFLIGHT_PATH)
print("Target raw coverage  ->", TARGET_RAW_COVERAGE_PATH)
print("Manifest             ->", MANIFEST_PATH)

print(
    "\nCell 13 complete: expanded context is ready for "
    "the legacy hydrology preflight."
)

=== Cell 13 purpose ===
Build expanded legacy context using predictor data and master TWS-only backfill.

=== Paths ===
Project root    : C:\Projects\Infer RozviDrought\RozviDrought
Predictor source: C:\Projects\ramangwana\risks\data\drought_model\processed\model_inputs\temporal_predictors_full_20260614T141118Z.parquet
Master TWS only : C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Output parquet  : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell13_legacy_context_tws_backfilled_20260620T145557Z.parquet
Run ID          : 20260620T145557Z

=== Installed hydrology contract ===
Feature list        : ['tws', 'tws_lag1', 'tws_lag2', 'tws_lag3', 'tws_lag6', 'tws_rollmean3', 'tws_rollmean6', 'tws_rollmean12']
Warm-up months      : 11
GRU sequence length : 24
Maximum month gap   : 6
Required raw months : 35

=== Expanded event contexts ===
          event_id  required_raw_context_s

In [36]:
# validate_events_ng.ipynb — Cell 14 (REWRITE 2)
# Purpose:
# - Run one full legacy hybrid preflight on the rebuilt Cell 13 context.
# - Derive the exact hydrology raw-history requirement from the installed package.
# - Select one deterministic pixel with:
#       * complete final TWS over the full required hydrology history;
#       * complete raw inputs at the target month.
# - Reproduce:
#       FeatureService -> SubsystemService -> FusionService
# - Do not read master_inputs or temporal_predictors_full directly.
# - Do not run a national inference loop.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import gc
import json
import re
import sys
import traceback

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds


# -----------------------------------------------------------------------------
# 1. Paths and fixed preflight target
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL14_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

PROBE_TARGET_YYYYMM = 201510

CONTEXT_COLUMNS = [
    "pixel_id",
    "row",
    "col",
    "lon",
    "lat",
    "scenario",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
    "tws_source",
]

RAW_FEATURE_COLUMNS = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

print("=== Cell 14 purpose ===")
print(
    "Run one full legacy hybrid preflight using the rebuilt "
    "Cell 13 context and the installed hydrology contract."
)

print("\n=== Inputs ===")
print("Project root       :", PROJECT_ROOT)
print("Context source     : newest successful Cell 13 parquet only")
print("master_inputs read :", False)
print("predictor read     :", False)
print("Target month       :", PROBE_TARGET_YYYYMM)
print("Run ID             :", CELL14_RUN_ID)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root is missing:\n{PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# 2. Imports from the installed legacy package
# -----------------------------------------------------------------------------
from app.services.feature_service import FeatureService
from app.services.fusion_service import FusionService
from app.services.subsystem_service import SubsystemService

from rozvidrought_subsystems.hydrology_windows import (
    build_hydrology_sequences,
)
from rozvidrought_subsystems.loaders import (
    load_subsystem_spec,
)


# -----------------------------------------------------------------------------
# 3. Calendar and packaged hydrology requirements
# -----------------------------------------------------------------------------
def to_period(
    yyyymm: int,
) -> pd.Period:
    return pd.Period(
        str(int(yyyymm)),
        freq="M",
    )


def to_yyyymm(
    period: pd.Period,
) -> int:
    return int(
        period.strftime("%Y%m")
    )


def month_sequence(
    start_yyyymm: int,
    end_yyyymm: int,
) -> list[int]:
    start_period = to_period(start_yyyymm)
    end_period = to_period(end_yyyymm)

    if end_period < start_period:
        raise ValueError(
            f"Invalid month range: "
            f"{start_yyyymm} -> {end_yyyymm}"
        )

    return [
        to_yyyymm(period)
        for period in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


def derive_warmup_months(
    feature_list: list[str],
) -> int:
    warmup = 0

    for feature_name in feature_list:
        lag_match = re.search(
            r"_lag(\d+)$",
            feature_name,
        )

        if lag_match:
            warmup = max(
                warmup,
                int(lag_match.group(1)),
            )

        rolling_match = re.search(
            r"_roll(?:mean|std)(\d+)$",
            feature_name,
        )

        if rolling_match:
            warmup = max(
                warmup,
                int(rolling_match.group(1)) - 1,
            )

    return warmup


hydrology_spec = load_subsystem_spec(
    "hydrology"
)

HYDROLOGY_FEATURES = list(
    hydrology_spec.feature_list
)

HYDROLOGY_SEQUENCE_CONFIG = dict(
    hydrology_spec.sequence_config
)

HYDROLOGY_SEQ_LEN = int(
    HYDROLOGY_SEQUENCE_CONFIG["seq_len"]
)

HYDROLOGY_WARMUP_MONTHS = derive_warmup_months(
    HYDROLOGY_FEATURES
)

TOTAL_REQUIRED_RAW_MONTHS = (
    HYDROLOGY_WARMUP_MONTHS
    + HYDROLOGY_SEQ_LEN
)

PROBE_CONTEXT_START_YYYYMM = to_yyyymm(
    to_period(PROBE_TARGET_YYYYMM)
    - (TOTAL_REQUIRED_RAW_MONTHS - 1)
)

EXPECTED_MONTHS = month_sequence(
    PROBE_CONTEXT_START_YYYYMM,
    PROBE_TARGET_YYYYMM,
)

print("\n=== Installed hydrology contract ===")
print("Feature list           :", HYDROLOGY_FEATURES)
print("Feature warm-up months :", HYDROLOGY_WARMUP_MONTHS)
print("GRU sequence length    :", HYDROLOGY_SEQ_LEN)
print("Required raw months    :", TOTAL_REQUIRED_RAW_MONTHS)

print("\n=== Required probe context ===")
print(
    "Range:",
    PROBE_CONTEXT_START_YYYYMM,
    "->",
    PROBE_TARGET_YYYYMM,
)
print("Month count:", len(EXPECTED_MONTHS))

if len(EXPECTED_MONTHS) != TOTAL_REQUIRED_RAW_MONTHS:
    raise AssertionError(
        "Derived monthly context length is inconsistent."
    )


# -----------------------------------------------------------------------------
# 4. Find newest Cell 13 output
# -----------------------------------------------------------------------------
context_paths = sorted(
    OUT_DIR.glob(
        "cell13_legacy_context_tws_backfilled_*.parquet"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not context_paths:
    raise FileNotFoundError(
        "No Cell 13 context parquet was found."
    )

CELL13_CONTEXT_PATH = context_paths[0]

print("\n=== Cell 13 context used ===")
print(CELL13_CONTEXT_PATH)


# -----------------------------------------------------------------------------
# 5. Load only the required 35-month context
# -----------------------------------------------------------------------------
def typed_month_value(
    arrow_type: pa.DataType,
    yyyymm: int,
):
    if (
        pa.types.is_string(arrow_type)
        or pa.types.is_large_string(arrow_type)
    ):
        return str(int(yyyymm))

    if pa.types.is_integer(arrow_type):
        return int(yyyymm)

    raise TypeError(
        f"Unsupported yyyymm type: {arrow_type}"
    )


context_dataset = ds.dataset(
    CELL13_CONTEXT_PATH,
    format="parquet",
)

missing_context_columns = [
    column_name
    for column_name in CONTEXT_COLUMNS
    if column_name not in context_dataset.schema.names
]

if missing_context_columns:
    raise AssertionError(
        "Cell 13 context lacks required fields:\n"
        f"{missing_context_columns}"
    )

context_month_type = context_dataset.schema.field(
    "yyyymm"
).type

context_table = context_dataset.to_table(
    columns=CONTEXT_COLUMNS,
    filter=(
        ds.field("yyyymm")
        >= typed_month_value(
            context_month_type,
            PROBE_CONTEXT_START_YYYYMM,
        )
    ) & (
        ds.field("yyyymm")
        <= typed_month_value(
            context_month_type,
            PROBE_TARGET_YYYYMM,
        )
    ),
)

context_df = context_table.to_pandas()

if context_df.empty:
    raise AssertionError(
        "No Cell 13 context rows were returned for the "
        "required hydrology period."
    )

for column_name in [
    "row",
    "col",
    "yyyymm",
]:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="raise",
    ).astype("int64")

for column_name in RAW_FEATURE_COLUMNS:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="coerce",
    )

observed_months = sorted(
    context_df["yyyymm"]
    .unique()
    .tolist()
)

if observed_months != EXPECTED_MONTHS:
    raise AssertionError(
        "Cell 13 context does not contain the exact required "
        "continuous history.\n"
        f"Expected: {EXPECTED_MONTHS}\n"
        f"Observed: {observed_months}"
    )

if context_df.duplicated(
    subset=[
        "pixel_id",
        "yyyymm",
    ]
).any():
    raise AssertionError(
        "Cell 13 context contains duplicate pixel/month rows."
    )

rows_per_month = (
    context_df.groupby("yyyymm")
    .size()
    .sort_index()
)

if rows_per_month.nunique() != 1:
    raise AssertionError(
        "Grid row count changes across the probe context.\n"
        f"{rows_per_month.to_string()}"
    )

GRID_ROWS_PER_MONTH = int(
    rows_per_month.iloc[0]
)

print("\n=== Loaded context ===")
print("Rows               :", len(context_df))
print("Grid rows/month    :", GRID_ROWS_PER_MONTH)
print("Context month count:", len(observed_months))


# -----------------------------------------------------------------------------
# 6. Select one deterministic candidate pixel
# -----------------------------------------------------------------------------
# Conditions are intentionally limited to what this preflight needs:
# - exactly the full raw history;
# - no final TWS gaps over that history;
# - complete direct raw values in the target month for all four branches.
history_audit_df = (
    context_df.groupby(
        "pixel_id",
        as_index=False,
    )
    .agg(
        history_month_count=(
            "yyyymm",
            "nunique",
        ),
        tws_non_null_month_count=(
            "tws",
            "count",
        ),
    )
)

target_rows_df = (
    context_df.loc[
        context_df["yyyymm"].eq(
            PROBE_TARGET_YYYYMM
        )
    ]
    .copy()
)

target_rows_df["target_raw_complete"] = (
    target_rows_df[
        RAW_FEATURE_COLUMNS
    ]
    .notna()
    .all(axis=1)
)

candidate_df = (
    target_rows_df[
        [
            "pixel_id",
            "row",
            "col",
            "target_raw_complete",
        ]
    ]
    .merge(
        history_audit_df,
        on="pixel_id",
        how="left",
        validate="one_to_one",
    )
)

eligible_candidates_df = (
    candidate_df.loc[
        candidate_df["target_raw_complete"]
        & candidate_df[
            "history_month_count"
        ].eq(TOTAL_REQUIRED_RAW_MONTHS)
        & candidate_df[
            "tws_non_null_month_count"
        ].eq(TOTAL_REQUIRED_RAW_MONTHS)
    ]
    .sort_values(
        [
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

candidate_audit_path = (
    OUT_DIR
    / f"cell14_candidate_audit_{CELL14_RUN_ID}.csv"
)

candidate_df.to_csv(
    candidate_audit_path,
    index=False,
)

print("\n=== Candidate selection ===")
print(
    "Target-month rows:",
    len(target_rows_df),
)
print(
    "Eligible full-hybrid candidates:",
    len(eligible_candidates_df),
)

if eligible_candidates_df.empty:
    raise AssertionError(
        "No pixel has both complete target raw values and "
        "complete final TWS across the required hydrology history."
    )

PROBE_PIXEL_ID = eligible_candidates_df.loc[
    0,
    "pixel_id",
]

timeseries_df = (
    context_df.loc[
        context_df["pixel_id"].eq(
            PROBE_PIXEL_ID
        )
    ]
    .sort_values("yyyymm")
    .reset_index(drop=True)
    .copy()
)

if len(timeseries_df) != TOTAL_REQUIRED_RAW_MONTHS:
    raise AssertionError(
        "Selected probe pixel does not have the required "
        "raw-month count."
    )

if timeseries_df["tws"].isna().any():
    raise AssertionError(
        "Selected probe pixel has missing final TWS."
    )

if not bool(
    timeseries_df.loc[
        timeseries_df["yyyymm"].eq(
            PROBE_TARGET_YYYYMM
        ),
        RAW_FEATURE_COLUMNS,
    ]
    .notna()
    .all(axis=1)
    .iloc[0]
):
    raise AssertionError(
        "Selected probe pixel lacks a complete target raw row."
    )

timeseries_df["yyyymm"] = (
    timeseries_df["yyyymm"]
    .astype("int64")
    .astype(str)
)

TARGET_MONTH_STRING = str(
    PROBE_TARGET_YYYYMM
)

print("\n=== Selected probe pixel ===")
print("Pixel ID:", PROBE_PIXEL_ID)
print(
    "Row / col:",
    int(timeseries_df["row"].iloc[0]),
    "/",
    int(timeseries_df["col"].iloc[0]),
)
print(
    "Predictor TWS months:",
    int(
        timeseries_df["tws_source"]
        .eq("predictor")
        .sum()
    ),
)
print(
    "Master-backfilled TWS months:",
    int(
        timeseries_df["tws_source"]
        .eq("master_backfill")
        .sum()
    ),
)
print(
    "Target raw null count:",
    int(
        timeseries_df.loc[
            timeseries_df["yyyymm"].eq(
                TARGET_MONTH_STRING
            ),
            RAW_FEATURE_COLUMNS,
        ]
        .isna()
        .sum()
        .sum()
    ),
)


# -----------------------------------------------------------------------------
# 7. Prepare the original legacy component inputs
# -----------------------------------------------------------------------------
feature_service = FeatureService()
subsystem_service = SubsystemService()
fusion_service = FusionService()

print("\n=== Legacy components under test ===")
print("FeatureService   :", FeatureService.__module__)
print("SubsystemService :", SubsystemService.__module__)
print("FusionService    :", FusionService.__module__)
print("Model            : hybrid")

prepared = None
target_prepared: dict[str, pd.DataFrame] = {}
subsystem_outputs = None
fusion_output = None

package_exception = None
package_traceback = None

hydrology_valid_window_count = None
hydrology_target_has_valid_window = None
hydrology_sequence_shape = None

try:
    prepared = feature_service.prepare_subsystem_inputs(
        timeseries=timeseries_df.copy(),
        run_yyyymm=PROBE_TARGET_YYYYMM,
    )

    required_subsystems = {
        "atmospheric",
        "soil",
        "vegetation",
        "hydrology",
    }

    missing_subsystems = sorted(
        required_subsystems
        - set(prepared.keys())
    )

    if missing_subsystems:
        raise AssertionError(
            "FeatureService omitted required subsystems:\n"
            f"{missing_subsystems}"
        )

    for subsystem_name, subsystem_df in prepared.items():
        subsystem_df = subsystem_df.copy()

        if "yyyymm" not in subsystem_df.columns:
            raise AssertionError(
                f"{subsystem_name} prepared frame lacks yyyymm."
            )

        subsystem_df["yyyymm"] = (
            subsystem_df["yyyymm"]
            .astype(str)
        )

        if subsystem_name == "hydrology":
            hydrology_sequence_df = (
                subsystem_df.loc[
                    subsystem_df["yyyymm"].le(
                        TARGET_MONTH_STRING
                    )
                ]
                .sort_values("yyyymm")
                .reset_index(drop=True)
            )

            sequence_array, valid_index = (
                build_hydrology_sequences(
                    df=hydrology_sequence_df,
                    pixel_col="pixel_key",
                    time_col="yyyymm",
                )
            )

            hydrology_valid_window_count = int(
                len(valid_index)
            )

            hydrology_sequence_shape = "x".join(
                map(
                    str,
                    sequence_array.shape,
                )
            )

            hydrology_target_index = hydrology_sequence_df.index[
                hydrology_sequence_df["yyyymm"].eq(
                    TARGET_MONTH_STRING
                )
            ].tolist()

            if len(hydrology_target_index) != 1:
                raise AssertionError(
                    "Expected one hydrology target row, got "
                    f"{len(hydrology_target_index)}."
                )

            hydrology_target_has_valid_window = bool(
                hydrology_target_index[0]
                in valid_index.tolist()
            )

            if not hydrology_target_has_valid_window:
                raise AssertionError(
                    "Hydrology target month has no valid sequence "
                    "before subsystem prediction."
                )

            target_prepared[subsystem_name] = (
                hydrology_sequence_df
            )

        else:
            target_row_df = (
                subsystem_df.loc[
                    subsystem_df["yyyymm"].eq(
                        TARGET_MONTH_STRING
                    )
                ]
                .copy()
                .reset_index(drop=True)
            )

            if target_row_df.empty:
                raise AssertionError(
                    f"{subsystem_name} has no target row for "
                    f"{PROBE_TARGET_YYYYMM}."
                )

            target_prepared[subsystem_name] = target_row_df

    subsystem_outputs = subsystem_service.run_subsystems(
        target_prepared
    )

    if not isinstance(
        subsystem_outputs,
        dict,
    ):
        raise TypeError(
            "SubsystemService did not return a dictionary."
        )

    for subsystem_name in required_subsystems:
        if subsystem_name not in subsystem_outputs:
            raise KeyError(
                "SubsystemService omitted output for "
                f"{subsystem_name}."
            )

        if subsystem_outputs[subsystem_name].empty:
            raise ValueError(
                f"{subsystem_name} returned no probability rows."
            )

    fusion_output = fusion_service.run(
        subsystem_outputs=subsystem_outputs,
        model="hybrid",
    )

except Exception as error:
    package_exception = repr(error)
    package_traceback = traceback.format_exc()


# -----------------------------------------------------------------------------
# 8. Build compact audit evidence
# -----------------------------------------------------------------------------
def inspect_frame(
    stage: str,
    subsystem_name: str,
    value: Any,
) -> dict[str, Any]:
    record = {
        "stage": stage,
        "subsystem": subsystem_name,
        "object_type": type(value).__name__,
        "row_count": None,
        "column_count": None,
        "feature_null_count": None,
        "yyyymm_min": None,
        "yyyymm_max": None,
        "columns": None,
    }

    if not isinstance(
        value,
        pd.DataFrame,
    ):
        return record

    record["row_count"] = int(len(value))
    record["column_count"] = int(
        len(value.columns)
    )
    record["feature_null_count"] = int(
        value.isna().sum().sum()
    )
    record["columns"] = " | ".join(
        map(
            str,
            value.columns.tolist(),
        )
    )

    if "yyyymm" in value.columns:
        month_values = (
            value["yyyymm"]
            .astype(str)
            .dropna()
        )

        if not month_values.empty:
            record["yyyymm_min"] = month_values.min()
            record["yyyymm_max"] = month_values.max()

    return record


audit_records = []

if isinstance(prepared, dict):
    for subsystem_name, frame in prepared.items():
        audit_records.append(
            inspect_frame(
                stage="prepared",
                subsystem_name=subsystem_name,
                value=frame,
            )
        )

for subsystem_name, frame in target_prepared.items():
    audit_records.append(
        inspect_frame(
            stage="target_prepared",
            subsystem_name=subsystem_name,
            value=frame,
        )
    )

if isinstance(subsystem_outputs, dict):
    for subsystem_name, frame in subsystem_outputs.items():
        audit_records.append(
            inspect_frame(
                stage="subsystem_output",
                subsystem_name=subsystem_name,
                value=frame,
            )
        )

audit_df = pd.DataFrame(
    audit_records
)

subsystem_probability_rows = []

if isinstance(subsystem_outputs, dict):
    for subsystem_name, probability_df in subsystem_outputs.items():
        if isinstance(probability_df, pd.DataFrame):
            probability_record = {
                "subsystem": subsystem_name,
                "row_count": int(len(probability_df)),
            }

            for probability_column in [
                "p0",
                "p1",
                "p2",
                "p3",
            ]:
                probability_record[probability_column] = (
                    float(
                        probability_df[
                            probability_column
                        ].iloc[0]
                    )
                    if (
                        not probability_df.empty
                        and probability_column
                        in probability_df.columns
                    )
                    else None
                )

            subsystem_probability_rows.append(
                probability_record
            )

subsystem_probability_df = pd.DataFrame(
    subsystem_probability_rows
)


def to_jsonable(
    value: Any,
) -> Any:
    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass

    if isinstance(value, pd.DataFrame):
        return {
            "object_type": "DataFrame",
            "row_count": int(len(value)),
            "columns": list(
                map(
                    str,
                    value.columns.tolist(),
                )
            ),
            "rows": value.head(5).to_dict(
                orient="records"
            ),
        }

    if isinstance(value, dict):
        return {
            str(key): to_jsonable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_jsonable(item)
            for item in value[:10]
        ]

    return str(value)


if package_exception is not None:
    CELL14_STATUS = (
        "legacy_hybrid_preflight_failed"
    )

elif fusion_output is None:
    CELL14_STATUS = (
        "legacy_hybrid_preflight_failed_no_fusion_output"
    )

else:
    CELL14_STATUS = (
        "legacy_hybrid_preflight_passed_ready_for_national_run"
    )


# -----------------------------------------------------------------------------
# 9. Print result
# -----------------------------------------------------------------------------
print("\n=== Legacy preparation and subsystem audit ===")

if audit_df.empty:
    print("No audit records were produced.")

else:
    print(
        audit_df.to_string(
            index=False,
            max_colwidth=170,
        )
    )

print("\n=== Hydrology sequence evidence ===")
print(
    "Valid hydrology windows:",
    hydrology_valid_window_count,
)
print(
    "Target has valid window:",
    hydrology_target_has_valid_window,
)
print(
    "Sequence array shape:",
    hydrology_sequence_shape,
)

print("\n=== Subsystem probabilities ===")

if subsystem_probability_df.empty:
    print("No subsystem probability rows were returned.")

else:
    print(
        subsystem_probability_df.to_string(
            index=False
        )
    )

print("\n=== Cell 14 conclusion ===")
print("Status                :", CELL14_STATUS)
print(
    "Fusion result returned:",
    fusion_output is not None,
)

if package_exception is not None:
    print("\nPackage exception:")
    print(package_exception)


# -----------------------------------------------------------------------------
# 10. Save evidence without forcing a notebook exception
# -----------------------------------------------------------------------------
audit_path = (
    OUT_DIR
    / f"cell14_legacy_direct_path_audit_{CELL14_RUN_ID}.csv"
)

candidate_path = (
    OUT_DIR
    / f"cell14_candidate_audit_{CELL14_RUN_ID}.csv"
)

probe_history_path = (
    OUT_DIR
    / f"cell14_legacy_probe_history_{CELL14_RUN_ID}.csv"
)

subsystem_probability_path = (
    OUT_DIR
    / f"cell14_legacy_subsystem_probabilities_{CELL14_RUN_ID}.csv"
)

fusion_path = (
    OUT_DIR
    / f"cell14_legacy_probe_fusion_{CELL14_RUN_ID}.json"
)

summary_path = (
    OUT_DIR
    / f"cell14_legacy_direct_path_summary_{CELL14_RUN_ID}.json"
)

audit_df.to_csv(
    audit_path,
    index=False,
)

timeseries_df.to_csv(
    probe_history_path,
    index=False,
)

subsystem_probability_df.to_csv(
    subsystem_probability_path,
    index=False,
)

fusion_path.write_text(
    json.dumps(
        {
            "run_id": CELL14_RUN_ID,
            "probe_pixel_id": PROBE_PIXEL_ID,
            "target_yyyymm": PROBE_TARGET_YYYYMM,
            "fusion_output": to_jsonable(
                fusion_output
            ),
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

summary_path.write_text(
    json.dumps(
        {
            "run_id": CELL14_RUN_ID,
            "status": CELL14_STATUS,
            "master_inputs_read_in_cell14": False,
            "predictor_source_read_in_cell14": False,
            "cell13_context_source": str(
                CELL13_CONTEXT_PATH
            ),
            "probe_pixel_id": PROBE_PIXEL_ID,
            "target_yyyymm": PROBE_TARGET_YYYYMM,
            "context_start_yyyymm": (
                PROBE_CONTEXT_START_YYYYMM
            ),
            "context_month_count": (
                TOTAL_REQUIRED_RAW_MONTHS
            ),
            "hydrology_features": HYDROLOGY_FEATURES,
            "hydrology_feature_warmup_months": (
                HYDROLOGY_WARMUP_MONTHS
            ),
            "hydrology_seq_len": HYDROLOGY_SEQ_LEN,
            "hydrology_valid_window_count": (
                hydrology_valid_window_count
            ),
            "hydrology_target_has_valid_window": (
                hydrology_target_has_valid_window
            ),
            "hydrology_sequence_shape": (
                hydrology_sequence_shape
            ),
            "package_exception": package_exception,
            "package_traceback": package_traceback,
            "outputs": {
                "candidate_audit_csv": str(
                    candidate_path
                ),
                "direct_path_audit_csv": str(
                    audit_path
                ),
                "probe_history_csv": str(
                    probe_history_path
                ),
                "subsystem_probabilities_csv": str(
                    subsystem_probability_path
                ),
                "fusion_json": str(fusion_path),
                "summary_json": str(summary_path),
            },
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 14 outputs ===")
print("Candidate audit        ->", candidate_path)
print("Preparation audit      ->", audit_path)
print("Probe history          ->", probe_history_path)
print("Subsystem probabilities->", subsystem_probability_path)
print("Fusion output          ->", fusion_path)
print("Summary                ->", summary_path)

if CELL14_STATUS == (
    "legacy_hybrid_preflight_passed_ready_for_national_run"
):
    print(
        "\nCell 14 complete: the full legacy hybrid path "
        "passed for one valid event-context pixel."
    )
else:
    print(
        "\nCell 14 completed with a failed preflight. "
        "Use the saved audit and traceback before any national run."
    )

del context_df
gc.collect()

=== Cell 14 purpose ===
Run one full legacy hybrid preflight using the rebuilt Cell 13 context and the installed hydrology contract.

=== Inputs ===
Project root       : C:\Projects\Infer RozviDrought\RozviDrought
Context source     : newest successful Cell 13 parquet only
master_inputs read : False
predictor read     : False
Target month       : 201510
Run ID             : 20260620T150752Z

=== Installed hydrology contract ===
Feature list           : ['tws', 'tws_lag1', 'tws_lag2', 'tws_lag3', 'tws_lag6', 'tws_rollmean3', 'tws_rollmean6', 'tws_rollmean12']
Feature warm-up months : 11
GRU sequence length    : 24
Required raw months    : 35

=== Required probe context ===
Range: 201212 -> 201510
Month count: 35

=== Cell 13 context used ===
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell13_legacy_context_tws_backfilled_20260620T145557Z.parquet

=== Loaded context ===
Rows               : 937125
Grid rows/month    : 2

3727

In [27]:
# validate_events_ng.ipynb — Cell 15
# Purpose:
# - Debug why the legacy hydrology subsystem returns zero rows.
# - Use only the saved Cell 13 mixed-source context.
# - Reproduce FeatureService -> SubsystemService for one complete pixel.
# - Inspect engineered hydrology completeness and package source requirements.
# - Do not run fusion or a national inference loop.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import inspect
import json
import re
import sys
import traceback

import pandas as pd
import pyarrow.dataset as ds


# -----------------------------------------------------------------------------
# 1. Paths and fixed probe
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL15_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

PROBE_TARGET_YYYYMM = 201510
REQUIRED_HISTORY_MONTHS = 24

CONTEXT_COLUMNS = [
    "pixel_id",
    "row",
    "col",
    "lon",
    "lat",
    "scenario",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
    "tws_source",
]

RAW_FEATURE_COLUMNS = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

print("=== Cell 15 purpose ===")
print(
    "Diagnose the legacy hydrology branch using one complete "
    "Cell 13 context pixel."
)

print("\nProject root       :", PROJECT_ROOT)
print("Context source     : Cell 13 parquet only")
print("master_inputs read :", False)
print("Target month       :", PROBE_TARGET_YYYYMM)
print("Run ID             :", CELL15_RUN_ID)


# -----------------------------------------------------------------------------
# 2. Calendar helpers
# -----------------------------------------------------------------------------
def to_period(yyyymm: int) -> pd.Period:
    return pd.Period(str(int(yyyymm)), freq="M")


def to_yyyymm(period: pd.Period) -> int:
    return int(period.strftime("%Y%m"))


def month_sequence(
    start_yyyymm: int,
    end_yyyymm: int,
) -> list[int]:
    return [
        to_yyyymm(period)
        for period in pd.period_range(
            start=to_period(start_yyyymm),
            end=to_period(end_yyyymm),
            freq="M",
        )
    ]


CONTEXT_START_YYYYMM = to_yyyymm(
    to_period(PROBE_TARGET_YYYYMM)
    - REQUIRED_HISTORY_MONTHS
)

EXPECTED_MONTHS = month_sequence(
    CONTEXT_START_YYYYMM,
    PROBE_TARGET_YYYYMM,
)


# -----------------------------------------------------------------------------
# 3. Find and load Cell 13 context
# -----------------------------------------------------------------------------
context_paths = sorted(
    OUT_DIR.glob(
        "cell13_legacy_context_tws_backfilled_*.parquet"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not context_paths:
    raise FileNotFoundError(
        "No Cell 13 TWS-backfilled context parquet was found."
    )

CELL13_CONTEXT_PATH = context_paths[0]

print("\n=== Cell 13 context ===")
print(CELL13_CONTEXT_PATH)

context_dataset = ds.dataset(
    CELL13_CONTEXT_PATH,
    format="parquet",
)

missing_columns = [
    column_name
    for column_name in CONTEXT_COLUMNS
    if column_name not in context_dataset.schema.names
]

if missing_columns:
    raise AssertionError(
        "Cell 13 context lacks required fields:\n"
        f"{missing_columns}"
    )

context_table = context_dataset.to_table(
    columns=CONTEXT_COLUMNS,
    filter=(
        (ds.field("yyyymm") >= CONTEXT_START_YYYYMM)
        & (ds.field("yyyymm") <= PROBE_TARGET_YYYYMM)
    ),
)

context_df = context_table.to_pandas()

if context_df.empty:
    raise AssertionError(
        "No Cell 13 rows were returned for the probe context."
    )

for column_name in ["row", "col", "yyyymm"]:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="raise",
    ).astype("int64")

for column_name in RAW_FEATURE_COLUMNS:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="coerce",
    )

observed_months = sorted(
    context_df["yyyymm"].unique().tolist()
)

if observed_months != EXPECTED_MONTHS:
    raise AssertionError(
        "Unexpected monthly coverage.\n"
        f"Expected: {EXPECTED_MONTHS}\n"
        f"Observed: {observed_months}"
    )


# -----------------------------------------------------------------------------
# 4. Select one complete-history pixel
# -----------------------------------------------------------------------------
feature_counts_df = (
    context_df.groupby(
        "pixel_id",
        as_index=True,
    )[RAW_FEATURE_COLUMNS]
    .count()
)

complete_pixel_ids = feature_counts_df.index[
    feature_counts_df.eq(
        len(EXPECTED_MONTHS)
    ).all(axis=1)
].tolist()

if not complete_pixel_ids:
    raise AssertionError(
        "No Cell 13 pixel has complete raw legacy input history."
    )

probe_candidates_df = (
    context_df.loc[
        context_df["yyyymm"].eq(PROBE_TARGET_YYYYMM)
        & context_df["pixel_id"].isin(
            complete_pixel_ids
        )
    ]
    .sort_values(["row", "col"])
    .reset_index(drop=True)
)

PROBE_PIXEL_ID = probe_candidates_df.loc[0, "pixel_id"]

timeseries_df = (
    context_df.loc[
        context_df["pixel_id"].eq(PROBE_PIXEL_ID)
    ]
    .sort_values("yyyymm")
    .reset_index(drop=True)
    .copy()
)

if timeseries_df[RAW_FEATURE_COLUMNS].isna().any().any():
    raise AssertionError(
        "Selected pixel has raw feature nulls despite complete-history selection."
    )

timeseries_df["yyyymm"] = (
    timeseries_df["yyyymm"]
    .astype("int64")
    .astype(str)
)

TARGET_MONTH_STRING = str(PROBE_TARGET_YYYYMM)

print("\n=== Probe pixel ===")
print("Pixel ID :", PROBE_PIXEL_ID)
print(
    "Row / col:",
    int(timeseries_df["row"].iloc[0]),
    "/",
    int(timeseries_df["col"].iloc[0]),
)
print(
    "Predictor TWS months:",
    int(
        timeseries_df["tws_source"]
        .eq("predictor")
        .sum()
    ),
)
print(
    "Master TWS months:",
    int(
        timeseries_df["tws_source"]
        .eq("master_backfill")
        .sum()
    ),
)


# -----------------------------------------------------------------------------
# 5. Import saved legacy components
# -----------------------------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.feature_service import FeatureService
from app.services.subsystem_service import SubsystemService

feature_service = FeatureService()
subsystem_service = SubsystemService()

print("\n=== Component sources ===")
print(
    "FeatureService source:",
    inspect.getsourcefile(FeatureService),
)
print(
    "SubsystemService source:",
    inspect.getsourcefile(SubsystemService),
)

print("\n=== SubsystemService attributes ===")
for attribute_name, attribute_value in vars(
    subsystem_service
).items():
    print(
        f"{attribute_name}: "
        f"{type(attribute_value).__module__}."
        f"{type(attribute_value).__name__}"
    )


# -----------------------------------------------------------------------------
# 6. Build the actual legacy prepared inputs
# -----------------------------------------------------------------------------
prepared = feature_service.prepare_subsystem_inputs(
    timeseries=timeseries_df.copy(),
    run_yyyymm=PROBE_TARGET_YYYYMM,
)

required_subsystems = {
    "atmospheric",
    "soil",
    "vegetation",
    "hydrology",
}

missing_subsystems = sorted(
    required_subsystems - set(prepared)
)

if missing_subsystems:
    raise AssertionError(
        "FeatureService is missing subsystem outputs:\n"
        f"{missing_subsystems}"
    )

target_prepared: dict[str, pd.DataFrame] = {}

for subsystem_name, subsystem_df in prepared.items():
    subsystem_df = subsystem_df.copy()
    subsystem_df["yyyymm"] = subsystem_df["yyyymm"].astype(str)

    if subsystem_name == "hydrology":
        target_prepared[subsystem_name] = (
            subsystem_df.loc[
                subsystem_df["yyyymm"].le(
                    TARGET_MONTH_STRING
                )
            ]
            .sort_values("yyyymm")
            .reset_index(drop=True)
        )
    else:
        target_prepared[subsystem_name] = (
            subsystem_df.loc[
                subsystem_df["yyyymm"].eq(
                    TARGET_MONTH_STRING
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

hydrology_sequence_df = target_prepared["hydrology"].copy()

hydrology_metadata_columns = {
    "pixel_key",
    "yyyymm",
    "time_type",
    "run_yyyymm",
}

hydrology_model_columns = [
    column_name
    for column_name in hydrology_sequence_df.columns
    if column_name not in hydrology_metadata_columns
]

hydrology_complete_mask = (
    hydrology_sequence_df[
        hydrology_model_columns
    ]
    .notna()
    .all(axis=1)
)

hydrology_complete_df = (
    hydrology_sequence_df.loc[
        hydrology_complete_mask
    ]
    .copy()
    .reset_index(drop=True)
)

hydrology_missingness_df = pd.DataFrame(
    {
        "column": hydrology_model_columns,
        "null_count": [
            int(
                hydrology_sequence_df[column_name]
                .isna()
                .sum()
            )
            for column_name in hydrology_model_columns
        ],
    }
).sort_values(
    [
        "null_count",
        "column",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

print("\n=== Hydrology input diagnosis ===")
print(
    "Hydrology sequence rows:",
    len(hydrology_sequence_df),
)
print(
    "Fully complete engineered rows:",
    len(hydrology_complete_df),
)
print(
    "First complete hydrology month:",
    (
        hydrology_complete_df["yyyymm"].iloc[0]
        if not hydrology_complete_df.empty
        else None
    ),
)
print(
    "Last complete hydrology month:",
    (
        hydrology_complete_df["yyyymm"].iloc[-1]
        if not hydrology_complete_df.empty
        else None
    ),
)

print("\n=== Hydrology feature null counts ===")
print(
    hydrology_missingness_df.to_string(
        index=False,
    )
)


# -----------------------------------------------------------------------------
# 7. Run actual subsystem service and preserve all output details
# -----------------------------------------------------------------------------
subsystem_outputs = None
subsystem_exception = None
subsystem_traceback = None

try:
    subsystem_outputs = subsystem_service.run_subsystems(
        target_prepared
    )

except Exception as error:
    subsystem_exception = repr(error)
    subsystem_traceback = traceback.format_exc()

output_records = []

if isinstance(subsystem_outputs, dict):
    for subsystem_name, output_df in subsystem_outputs.items():
        output_records.append(
            {
                "subsystem": subsystem_name,
                "object_type": type(output_df).__name__,
                "row_count": (
                    int(len(output_df))
                    if isinstance(output_df, pd.DataFrame)
                    else None
                ),
                "columns": (
                    " | ".join(
                        map(
                            str,
                            output_df.columns.tolist(),
                        )
                    )
                    if isinstance(output_df, pd.DataFrame)
                    else None
                ),
            }
        )

subsystem_output_audit_df = pd.DataFrame(output_records)

print("\n=== Actual SubsystemService output ===")

if subsystem_exception is not None:
    print("Subsystem exception:", subsystem_exception)

elif subsystem_output_audit_df.empty:
    print("No subsystem outputs were returned.")

else:
    print(
        subsystem_output_audit_df.to_string(
            index=False,
            max_colwidth=140,
        )
    )


# -----------------------------------------------------------------------------
# 8. Search exact package source lines that control hydrology execution
# -----------------------------------------------------------------------------
SEARCH_TERMS = [
    "hydrology",
    "tws",
    "seq_len",
    "sequence",
    "dropna",
    "empty",
    "rolling",
    "gru",
    "predict",
]

MAX_SOURCE_HITS = 160

source_hits: list[dict[str, object]] = []

for source_path in sorted(
    (PROJECT_ROOT / "app").rglob("*.py")
):
    try:
        source_lines = source_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
    except OSError:
        continue

    for line_number, line_text in enumerate(
        source_lines,
        start=1,
    ):
        lower_line = line_text.lower()

        if not any(
            term in lower_line
            for term in SEARCH_TERMS
        ):
            continue

        if (
            "hydro" not in lower_line
            and "tws" not in lower_line
            and "seq_len" not in lower_line
            and "gru" not in lower_line
            and source_path.name
            != "subsystem_service.py"
        ):
            continue

        source_hits.append(
            {
                "file_path": str(source_path),
                "line_number": line_number,
                "line_text": line_text.strip(),
            }
        )

        if len(source_hits) >= MAX_SOURCE_HITS:
            break

    if len(source_hits) >= MAX_SOURCE_HITS:
        break

source_hits_df = pd.DataFrame(source_hits)

print("\n=== Relevant legacy package source lines ===")

if source_hits_df.empty:
    print("No relevant lines found.")

else:
    print(
        source_hits_df.to_string(
            index=False,
            max_colwidth=180,
        )
    )


# -----------------------------------------------------------------------------
# 9. Result status and saved evidence
# -----------------------------------------------------------------------------
hydrology_output_rows = None

if (
    isinstance(subsystem_outputs, dict)
    and isinstance(
        subsystem_outputs.get("hydrology"),
        pd.DataFrame,
    )
):
    hydrology_output_rows = int(
        len(subsystem_outputs["hydrology"])
    )

if subsystem_exception is not None:
    CELL15_STATUS = (
        "hydrology_debug_subsystem_service_exception"
    )

elif hydrology_output_rows is None:
    CELL15_STATUS = (
        "hydrology_debug_no_hydrology_output_object"
    )

elif hydrology_output_rows == 0:
    CELL15_STATUS = (
        "hydrology_debug_confirmed_empty_output"
    )

else:
    CELL15_STATUS = (
        "hydrology_debug_output_present"
    )

print("\n=== Cell 15 conclusion ===")
print("Status:", CELL15_STATUS)
print(
    "Hydrology sequence rows:",
    len(hydrology_sequence_df),
)
print(
    "Complete engineered rows:",
    len(hydrology_complete_df),
)
print(
    "Hydrology output rows:",
    hydrology_output_rows,
)

hydrology_sequence_path = (
    OUT_DIR
    / f"cell15_hydrology_sequence_{CELL15_RUN_ID}.csv"
)

hydrology_missingness_path = (
    OUT_DIR
    / f"cell15_hydrology_missingness_{CELL15_RUN_ID}.csv"
)

subsystem_output_path = (
    OUT_DIR
    / f"cell15_subsystem_outputs_{CELL15_RUN_ID}.csv"
)

source_hits_path = (
    OUT_DIR
    / f"cell15_hydrology_source_hits_{CELL15_RUN_ID}.csv"
)

summary_path = (
    OUT_DIR
    / f"cell15_hydrology_debug_summary_{CELL15_RUN_ID}.json"
)

hydrology_sequence_df.to_csv(
    hydrology_sequence_path,
    index=False,
)

hydrology_missingness_df.to_csv(
    hydrology_missingness_path,
    index=False,
)

subsystem_output_audit_df.to_csv(
    subsystem_output_path,
    index=False,
)

source_hits_df.to_csv(
    source_hits_path,
    index=False,
)

summary_path.write_text(
    json.dumps(
        {
            "run_id": CELL15_RUN_ID,
            "status": CELL15_STATUS,
            "master_inputs_read_in_cell15": False,
            "cell13_context_source": str(
                CELL13_CONTEXT_PATH
            ),
            "probe_pixel_id": PROBE_PIXEL_ID,
            "target_yyyymm": PROBE_TARGET_YYYYMM,
            "hydrology_sequence_rows": int(
                len(hydrology_sequence_df)
            ),
            "hydrology_complete_engineered_rows": int(
                len(hydrology_complete_df)
            ),
            "hydrology_output_rows": hydrology_output_rows,
            "subsystem_exception": subsystem_exception,
            "subsystem_traceback": subsystem_traceback,
            "outputs": {
                "hydrology_sequence_csv": str(
                    hydrology_sequence_path
                ),
                "hydrology_missingness_csv": str(
                    hydrology_missingness_path
                ),
                "subsystem_outputs_csv": str(
                    subsystem_output_path
                ),
                "source_hits_csv": str(
                    source_hits_path
                ),
                "summary_json": str(summary_path),
            },
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 15 outputs ===")
print("Hydrology sequence ->", hydrology_sequence_path)
print("Missingness        ->", hydrology_missingness_path)
print("Subsystem outputs  ->", subsystem_output_path)
print("Source hits        ->", source_hits_path)
print("Summary            ->", summary_path)

print("\nCell 15 complete.")

=== Cell 15 purpose ===
Diagnose the legacy hydrology branch using one complete Cell 13 context pixel.

Project root       : C:\Projects\Infer RozviDrought\RozviDrought
Context source     : Cell 13 parquet only
master_inputs read : False
Target month       : 201510
Run ID             : 20260620T140931Z

=== Cell 13 context ===
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell13_legacy_context_tws_backfilled_20260620T134540Z.parquet

=== Probe pixel ===
Pixel ID : 0_101
Row / col: 0 / 101
Predictor TWS months: 18
Master TWS months: 7

=== Component sources ===
FeatureService source: C:\Projects\Infer RozviDrought\RozviDrought\app\services\feature_service.py
SubsystemService source: C:\Projects\Infer RozviDrought\RozviDrought\app\services\subsystem_service.py

=== SubsystemService attributes ===

=== Hydrology input diagnosis ===
Hydrology sequence rows: 25
Fully complete engineered rows: 14
First complete hydrology mont

In [28]:
# validate_events_ng.ipynb — Cell 16
# Purpose:
# - Inspect the exact legacy hydrology prediction implementation.
# - Confirm sequence length, feature order, dropna/filter rules, scaler/model use,
#   and any hidden conditions causing empty hydrology output.
# - No model inference.
# - No master_inputs read.
# - No predictor-source read.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import importlib
import inspect
import json
import re
import sys

import pandas as pd


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL16_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== Cell 16 purpose ===")
print(
    "Inspect the exact saved legacy hydrology prediction function "
    "and its runtime configuration."
)

print("\nProject root       :", PROJECT_ROOT)
print("master_inputs read :", False)
print("model inference    :", False)
print("Run ID             :", CELL16_RUN_ID)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root is missing:\n{PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# 2. Load subsystem-service module and resolve hydrology callable
# -----------------------------------------------------------------------------
subsystem_service_module = importlib.import_module(
    "app.services.subsystem_service"
)

subsystem_service_source_path = Path(
    inspect.getsourcefile(subsystem_service_module)
)

if not subsystem_service_source_path.exists():
    raise FileNotFoundError(
        "Could not locate subsystem_service.py source."
    )

print("\n=== Subsystem service source ===")
print(subsystem_service_source_path)

candidate_objects = []

for global_name, global_value in vars(
    subsystem_service_module
).items():
    if hasattr(
        global_value,
        "predict_hydrology_proba_df",
    ):
        candidate_objects.append(
            {
                "global_name": global_name,
                "object": global_value,
            }
        )

if not candidate_objects:
    raise AttributeError(
        "Could not find an object with "
        "'predict_hydrology_proba_df' in subsystem_service module globals."
    )

if len(candidate_objects) > 1:
    print("\n=== Multiple hydrology callable candidates found ===")
    for item in candidate_objects:
        print(
            "-",
            item["global_name"],
            "->",
            type(item["object"]).__module__,
            ".",
            type(item["object"]).__name__,
        )

resolved_candidate = candidate_objects[0]
SUBSYSTEMS_OBJECT_NAME = resolved_candidate["global_name"]
subsystems_object = resolved_candidate["object"]

hydrology_predict_function = getattr(
    subsystems_object,
    "predict_hydrology_proba_df",
)

if not callable(hydrology_predict_function):
    raise TypeError(
        "Resolved predict_hydrology_proba_df is not callable."
    )

hydrology_function_source_path = Path(
    inspect.getsourcefile(hydrology_predict_function)
)

if not hydrology_function_source_path.exists():
    raise FileNotFoundError(
        "Could not locate hydrology prediction function source."
    )

print("\n=== Resolved hydrology callable ===")
print("Resolved object :", SUBSYSTEMS_OBJECT_NAME)
print(
    "Object type     :",
    type(subsystems_object).__module__,
    ".",
    type(subsystems_object).__name__,
)
print(
    "Function source :",
    hydrology_function_source_path,
)
print(
    "Function signature:",
    inspect.signature(hydrology_predict_function),
)


# -----------------------------------------------------------------------------
# 3. Read exact function source
# -----------------------------------------------------------------------------
function_source_text = inspect.getsource(
    hydrology_predict_function
)

function_start_line = inspect.getsourcelines(
    hydrology_predict_function
)[1]

print("\n=== Exact predict_hydrology_proba_df source ===")

for relative_line_number, line_text in enumerate(
    function_source_text.splitlines(),
    start=0,
):
    absolute_line_number = (
        function_start_line
        + relative_line_number
    )

    print(
        f"{absolute_line_number:>5}: {line_text}"
    )


# -----------------------------------------------------------------------------
# 4. Inspect runtime globals referenced by the hydrology function
# -----------------------------------------------------------------------------
IMPORTANT_NAME_TOKENS = [
    "hyd",
    "tws",
    "seq",
    "step",
    "lag",
    "roll",
    "feature",
    "scaler",
    "model",
    "class",
]

MAX_COLLECTION_ITEMS = 200


def serialise_runtime_value(value):
    """Produce a compact, non-model-dumping representation."""
    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (list, tuple)):
        if len(value) <= MAX_COLLECTION_ITEMS:
            return [
                serialise_runtime_value(item)
                for item in value
            ]

        return {
            "type": type(value).__name__,
            "length": len(value),
            "preview": [
                serialise_runtime_value(item)
                for item in value[:20]
            ],
        }

    if isinstance(value, dict):
        if len(value) <= MAX_COLLECTION_ITEMS:
            return {
                str(key): serialise_runtime_value(item)
                for key, item in value.items()
            }

        return {
            "type": "dict",
            "length": len(value),
            "keys_preview": [
                str(key)
                for key in list(value.keys())[:30]
            ],
        }

    if hasattr(value, "shape"):
        try:
            return {
                "type": (
                    f"{type(value).__module__}."
                    f"{type(value).__name__}"
                ),
                "shape": list(value.shape),
            }
        except Exception:
            pass

    return {
        "type": (
            f"{type(value).__module__}."
            f"{type(value).__name__}"
        )
    }


function_globals = hydrology_predict_function.__globals__

runtime_config_records = []

for global_name, global_value in function_globals.items():
    global_name_lower = global_name.lower()

    if not any(
        token in global_name_lower
        for token in IMPORTANT_NAME_TOKENS
    ):
        continue

    if global_name.startswith("__"):
        continue

    runtime_config_records.append(
        {
            "global_name": global_name,
            "value_type": (
                f"{type(global_value).__module__}."
                f"{type(global_value).__name__}"
            ),
            "value_compact": json.dumps(
                serialise_runtime_value(
                    global_value
                ),
                default=str,
            ),
        }
    )

runtime_config_df = pd.DataFrame(
    runtime_config_records
).sort_values(
    "global_name"
).reset_index(drop=True)

print("\n=== Hydrology-related function globals ===")

if runtime_config_df.empty:
    print("No matching runtime globals found.")
else:
    print(
        runtime_config_df.to_string(
            index=False,
            max_colwidth=200,
        )
    )


# -----------------------------------------------------------------------------
# 5. Search the exact package source for hydrology sequence/filter logic
# -----------------------------------------------------------------------------
SEARCH_TERMS = [
    "predict_hydrology_proba_df",
    "hydrology",
    "seq_len",
    "sequence",
    "dropna",
    "drop_duplicates",
    "rolling",
    "tws_lag",
    "tws_roll",
    "scaler",
    "gru",
    "reshape",
    "empty",
]

CONTEXT_LINES_BEFORE = 3
CONTEXT_LINES_AFTER = 6
MAX_SOURCE_CONTEXTS = 80

source_context_records = []
seen_context_keys = set()

for source_path in sorted(
    PROJECT_ROOT.rglob("*.py")
):
    try:
        source_lines = source_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
    except OSError:
        continue

    for line_index, line_text in enumerate(source_lines):
        line_lower = line_text.lower()

        if not any(
            term in line_lower
            for term in SEARCH_TERMS
        ):
            continue

        start_index = max(
            0,
            line_index - CONTEXT_LINES_BEFORE,
        )

        end_index = min(
            len(source_lines),
            line_index + CONTEXT_LINES_AFTER + 1,
        )

        context_key = (
            str(source_path),
            start_index,
            end_index,
        )

        if context_key in seen_context_keys:
            continue

        seen_context_keys.add(context_key)

        source_context_records.append(
            {
                "file_path": str(source_path),
                "match_line_number": line_index + 1,
                "matched_line": line_text.strip(),
                "context_start_line": start_index + 1,
                "context_end_line": end_index,
                "context": "\n".join(
                    f"{position + 1:>5}: "
                    f"{source_lines[position]}"
                    for position in range(
                        start_index,
                        end_index,
                    )
                ),
            }
        )

        if len(source_context_records) >= MAX_SOURCE_CONTEXTS:
            break

    if len(source_context_records) >= MAX_SOURCE_CONTEXTS:
        break

source_context_df = pd.DataFrame(
    source_context_records
)

print("\n=== Hydrology sequence and filter source contexts ===")

if source_context_df.empty:
    print("No matching source contexts found.")
else:
    for record in source_context_df.to_dict(
        orient="records"
    ):
        print("\n" + "=" * 100)
        print(
            f"{record['file_path']}"
            f":{record['match_line_number']}"
        )
        print(record["context"])


# -----------------------------------------------------------------------------
# 6. Extract likely requirements from the function source
# -----------------------------------------------------------------------------
likely_requirement_patterns = {
    "sequence_length": (
        r"(?i)(?:seq_len|sequence_length|timesteps|"
        r"window_size|lookback)\s*=\s*(\d+)"
    ),
    "dropna_call": (
        r"(?i)\.dropna\([^)]*\)"
    ),
    "required_column_selection": (
        r"(?i)(?:feature_cols|hyd_features|"
        r"hydrology_features|features)\s*=\s*\["
    ),
    "reshape_call": (
        r"(?i)\.reshape\([^)]*\)"
    ),
    "empty_return": (
        r"(?i)return\s+pd\.DataFrame"
    ),
}

requirement_hits = []

for requirement_name, pattern in (
    likely_requirement_patterns.items()
):
    matches = re.findall(
        pattern,
        function_source_text,
    )

    requirement_hits.append(
        {
            "requirement": requirement_name,
            "matches": json.dumps(
                matches,
                default=str,
            ),
        }
    )

requirements_df = pd.DataFrame(
    requirement_hits
)

print("\n=== Extracted hydrology requirement clues ===")
print(
    requirements_df.to_string(
        index=False,
        max_colwidth=200,
    )
)


# -----------------------------------------------------------------------------
# 7. Save evidence
# -----------------------------------------------------------------------------
function_source_path = (
    OUT_DIR
    / f"cell16_hydrology_predict_function_{CELL16_RUN_ID}.py"
)

runtime_config_path = (
    OUT_DIR
    / f"cell16_hydrology_runtime_config_{CELL16_RUN_ID}.csv"
)

source_context_path = (
    OUT_DIR
    / f"cell16_hydrology_source_contexts_{CELL16_RUN_ID}.csv"
)

requirements_path = (
    OUT_DIR
    / f"cell16_hydrology_requirement_clues_{CELL16_RUN_ID}.csv"
)

summary_path = (
    OUT_DIR
    / f"cell16_hydrology_code_inspection_summary_{CELL16_RUN_ID}.json"
)

function_source_path.write_text(
    function_source_text,
    encoding="utf-8",
)

runtime_config_df.to_csv(
    runtime_config_path,
    index=False,
)

source_context_df.to_csv(
    source_context_path,
    index=False,
)

requirements_df.to_csv(
    requirements_path,
    index=False,
)

summary_payload = {
    "run_id": CELL16_RUN_ID,
    "purpose": (
        "Inspect the exact legacy hydrology prediction implementation "
        "without running inference."
    ),
    "master_inputs_read": False,
    "predictor_source_read": False,
    "resolved_subsystems_object": SUBSYSTEMS_OBJECT_NAME,
    "hydrology_function_source": str(
        hydrology_function_source_path
    ),
    "hydrology_function_signature": str(
        inspect.signature(
            hydrology_predict_function
        )
    ),
    "outputs": {
        "function_source_py": str(
            function_source_path
        ),
        "runtime_config_csv": str(
            runtime_config_path
        ),
        "source_contexts_csv": str(
            source_context_path
        ),
        "requirement_clues_csv": str(
            requirements_path
        ),
        "summary_json": str(
            summary_path
        ),
    },
}

summary_path.write_text(
    json.dumps(
        summary_payload,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 16 outputs ===")
print("Function source   ->", function_source_path)
print("Runtime config    ->", runtime_config_path)
print("Source contexts   ->", source_context_path)
print("Requirement clues ->", requirements_path)
print("Summary           ->", summary_path)

print("\nCell 16 complete.")

=== Cell 16 purpose ===
Inspect the exact saved legacy hydrology prediction function and its runtime configuration.

Project root       : C:\Projects\Infer RozviDrought\RozviDrought
master_inputs read : False
model inference    : False
Run ID             : 20260620T142324Z

=== Subsystem service source ===
C:\Projects\Infer RozviDrought\RozviDrought\app\services\subsystem_service.py

=== Resolved hydrology callable ===
Resolved object : subs
Object type     : builtins . module
Function source : c:\Projects\Infer RozviDrought\.venv\Lib\site-packages\rozvidrought_subsystems\api.py
Function signature: (df: 'pd.DataFrame', pixel_col: 'str' = 'pixel_key', time_col: 'str' = 'yyyymm') -> 'pd.DataFrame'

=== Exact predict_hydrology_proba_df source ===
   23: def predict_hydrology_proba_df(
   24:     df: pd.DataFrame,
   25:     pixel_col: str = "pixel_key",
   26:     time_col: str = "yyyymm",
   27: ) -> pd.DataFrame:
   28:     return predict_hydrology_proba_from_long_df(
   29:         df=

In [29]:
# validate_events_ng.ipynb — Cell 17
# Purpose:
# - Inspect the real hydrology implementation behind the API wrapper:
#       predict_hydrology_proba_from_long_df
# - Identify sequence length, feature list, drop/filter logic, scaler/model use,
#   and direct callable dependencies.
# - No inference.
# - No data reads.
# - No model modification.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import ast
import inspect
import json
import sys

import pandas as pd


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL17_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=== Cell 17 purpose ===")
print(
    "Inspect the actual legacy hydrology implementation and the "
    "direct dependencies it uses."
)

print("\nProject root       :", PROJECT_ROOT)
print("master_inputs read :", False)
print("predictor read     :", False)
print("model inference    :", False)
print("Run ID             :", CELL17_RUN_ID)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root is missing:\n{PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# 2. Resolve the real hydrology function
# -----------------------------------------------------------------------------
import rozvidrought_subsystems.api as subsystem_api

TARGET_FUNCTION_NAME = (
    "predict_hydrology_proba_from_long_df"
)

if not hasattr(
    subsystem_api,
    TARGET_FUNCTION_NAME,
):
    raise AttributeError(
        f"{TARGET_FUNCTION_NAME} was not found in "
        "rozvidrought_subsystems.api."
    )

hydrology_function = getattr(
    subsystem_api,
    TARGET_FUNCTION_NAME,
)

if not callable(hydrology_function):
    raise TypeError(
        f"{TARGET_FUNCTION_NAME} is not callable."
    )

function_source_path = Path(
    inspect.getsourcefile(hydrology_function)
)

if not function_source_path.exists():
    raise FileNotFoundError(
        "Could not locate the real hydrology function source file."
    )

function_source_text = inspect.getsource(
    hydrology_function
)

function_start_line = inspect.getsourcelines(
    hydrology_function
)[1]

print("\n=== Real hydrology function ===")
print("Function :", TARGET_FUNCTION_NAME)
print("Signature:", inspect.signature(hydrology_function))
print("Source   :", function_source_path)

print("\n=== Exact function source ===")

for relative_line, line_text in enumerate(
    function_source_text.splitlines(),
    start=0,
):
    absolute_line = function_start_line + relative_line

    print(
        f"{absolute_line:>5}: {line_text}"
    )


# -----------------------------------------------------------------------------
# 3. Extract direct names and function calls from AST
# -----------------------------------------------------------------------------
function_tree = ast.parse(
    function_source_text
)

called_names: set[str] = set()
referenced_names: set[str] = set()


class FunctionReferenceVisitor(ast.NodeVisitor):
    def visit_Name(
        self,
        node: ast.Name,
    ) -> None:
        referenced_names.add(node.id)
        self.generic_visit(node)

    def visit_Call(
        self,
        node: ast.Call,
    ) -> None:
        if isinstance(node.func, ast.Name):
            called_names.add(node.func.id)

        elif isinstance(
            node.func,
            ast.Attribute,
        ):
            called_names.add(node.func.attr)

        self.generic_visit(node)


FunctionReferenceVisitor().visit(
    function_tree
)

print("\n=== Direct callable names found ===")
print(
    sorted(called_names)
    if called_names
    else "None"
)


# -----------------------------------------------------------------------------
# 4. Inspect only directly referenced package globals
# -----------------------------------------------------------------------------
function_globals = hydrology_function.__globals__

IGNORE_NAMES = {
    "pd",
    "np",
    "json",
    "Path",
    "Any",
    "Dict",
    "List",
    "Optional",
    "Tuple",
    "Sequence",
    "Iterable",
    "DataFrame",
}

MAX_PREVIEW_ITEMS = 50


def compact_value(
    value,
):
    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (list, tuple)):
        if len(value) <= MAX_PREVIEW_ITEMS:
            return [
                compact_value(item)
                for item in value
            ]

        return {
            "type": type(value).__name__,
            "length": len(value),
            "preview": [
                compact_value(item)
                for item in value[:20]
            ],
        }

    if isinstance(value, dict):
        if len(value) <= MAX_PREVIEW_ITEMS:
            return {
                str(key): compact_value(item)
                for key, item in value.items()
            }

        return {
            "type": "dict",
            "length": len(value),
            "keys_preview": [
                str(key)
                for key in list(value.keys())[:20]
            ],
        }

    if hasattr(value, "shape"):
        try:
            return {
                "type": (
                    f"{type(value).__module__}."
                    f"{type(value).__name__}"
                ),
                "shape": list(value.shape),
            }
        except Exception:
            pass

    return {
        "type": (
            f"{type(value).__module__}."
            f"{type(value).__name__}"
        )
    }


referenced_global_records = []

for name in sorted(referenced_names):
    if name in IGNORE_NAMES:
        continue

    if name not in function_globals:
        continue

    value = function_globals[name]

    referenced_global_records.append(
        {
            "name": name,
            "object_type": (
                f"{type(value).__module__}."
                f"{type(value).__name__}"
            ),
            "is_callable": callable(value),
            "compact_value": json.dumps(
                compact_value(value),
                default=str,
            ),
        }
    )

referenced_globals_df = pd.DataFrame(
    referenced_global_records
)

print("\n=== Directly referenced runtime globals ===")

if referenced_globals_df.empty:
    print("No package globals were directly referenced.")

else:
    print(
        referenced_globals_df.to_string(
            index=False,
            max_colwidth=220,
        )
    )


# -----------------------------------------------------------------------------
# 5. Print source of direct callable dependencies only
# -----------------------------------------------------------------------------
dependency_records = []
dependency_source_blocks = []

for name in sorted(called_names):
    if name not in function_globals:
        continue

    dependency = function_globals[name]

    if not callable(dependency):
        continue

    if dependency is hydrology_function:
        continue

    try:
        dependency_source_path = Path(
            inspect.getsourcefile(dependency)
        )

        dependency_source_text = inspect.getsource(
            dependency
        )

        dependency_start_line = inspect.getsourcelines(
            dependency
        )[1]

        dependency_records.append(
            {
                "dependency_name": name,
                "dependency_type": (
                    f"{type(dependency).__module__}."
                    f"{type(dependency).__name__}"
                ),
                "source_path": str(
                    dependency_source_path
                ),
                "source_start_line": dependency_start_line,
                "source_available": True,
            }
        )

        dependency_source_blocks.append(
            {
                "dependency_name": name,
                "source_path": str(
                    dependency_source_path
                ),
                "source_start_line": dependency_start_line,
                "source": dependency_source_text,
            }
        )

    except (
        OSError,
        IOError,
        TypeError,
    ):
        dependency_records.append(
            {
                "dependency_name": name,
                "dependency_type": (
                    f"{type(dependency).__module__}."
                    f"{type(dependency).__name__}"
                ),
                "source_path": None,
                "source_start_line": None,
                "source_available": False,
            }
        )

dependency_df = pd.DataFrame(
    dependency_records
)

print("\n=== Direct callable dependencies ===")

if dependency_df.empty:
    print("No inspectable direct callable dependencies found.")

else:
    print(
        dependency_df.to_string(
            index=False,
            max_colwidth=180,
        )
    )

for block in dependency_source_blocks:
    print("\n" + "=" * 100)
    print(
        f"DEPENDENCY: {block['dependency_name']}"
    )
    print(
        f"SOURCE: {block['source_path']}"
    )
    print(
        f"START LINE: {block['source_start_line']}"
    )
    print("-" * 100)

    for relative_line, line_text in enumerate(
        block["source"].splitlines(),
        start=0,
    ):
        print(
            f"{block['source_start_line'] + relative_line:>5}: "
            f"{line_text}"
        )


# -----------------------------------------------------------------------------
# 6. Search the real function's source file for key hydrology rules
# -----------------------------------------------------------------------------
SEARCH_TERMS = [
    "seq_len",
    "sequence",
    "timesteps",
    "lookback",
    "dropna",
    "feature",
    "scaler",
    "model",
    "gru",
    "reshape",
    "tws_lag",
    "tws_roll",
    "empty",
]

source_lines = function_source_path.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

CONTEXT_BEFORE = 3
CONTEXT_AFTER = 6
MAX_CONTEXT_BLOCKS = 80

source_rule_records = []
seen_blocks = set()

for line_index, line_text in enumerate(source_lines):
    lower_line = line_text.lower()

    if not any(
        term in lower_line
        for term in SEARCH_TERMS
    ):
        continue

    start_index = max(
        0,
        line_index - CONTEXT_BEFORE,
    )

    end_index = min(
        len(source_lines),
        line_index + CONTEXT_AFTER + 1,
    )

    block_key = (
        start_index,
        end_index,
    )

    if block_key in seen_blocks:
        continue

    seen_blocks.add(block_key)

    source_rule_records.append(
        {
            "file_path": str(function_source_path),
            "matched_line_number": line_index + 1,
            "matched_line": line_text.strip(),
            "context_start_line": start_index + 1,
            "context_end_line": end_index,
            "context": "\n".join(
                f"{position + 1:>5}: "
                f"{source_lines[position]}"
                for position in range(
                    start_index,
                    end_index,
                )
            ),
        }
    )

    if len(source_rule_records) >= MAX_CONTEXT_BLOCKS:
        break

source_rules_df = pd.DataFrame(
    source_rule_records
)

print("\n=== Real hydrology source-rule contexts ===")

if source_rules_df.empty:
    print("No matching source-rule contexts found.")

else:
    for record in source_rules_df.to_dict(
        orient="records"
    ):
        print("\n" + "=" * 100)
        print(
            f"{record['file_path']}"
            f":{record['matched_line_number']}"
        )
        print(record["context"])


# -----------------------------------------------------------------------------
# 7. Save code evidence
# -----------------------------------------------------------------------------
function_copy_path = (
    OUT_DIR
    / f"cell17_real_hydrology_function_{CELL17_RUN_ID}.py"
)

globals_path = (
    OUT_DIR
    / f"cell17_real_hydrology_referenced_globals_{CELL17_RUN_ID}.csv"
)

dependencies_path = (
    OUT_DIR
    / f"cell17_real_hydrology_dependencies_{CELL17_RUN_ID}.csv"
)

dependency_sources_path = (
    OUT_DIR
    / f"cell17_real_hydrology_dependency_sources_{CELL17_RUN_ID}.json"
)

source_rules_path = (
    OUT_DIR
    / f"cell17_real_hydrology_source_rules_{CELL17_RUN_ID}.csv"
)

summary_path = (
    OUT_DIR
    / f"cell17_real_hydrology_inspection_summary_{CELL17_RUN_ID}.json"
)

function_copy_path.write_text(
    function_source_text,
    encoding="utf-8",
)

referenced_globals_df.to_csv(
    globals_path,
    index=False,
)

dependency_df.to_csv(
    dependencies_path,
    index=False,
)

dependency_sources_path.write_text(
    json.dumps(
        dependency_source_blocks,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

source_rules_df.to_csv(
    source_rules_path,
    index=False,
)

summary_path.write_text(
    json.dumps(
        {
            "run_id": CELL17_RUN_ID,
            "purpose": (
                "Inspect the real hydrology implementation behind "
                "predict_hydrology_proba_df without running inference."
            ),
            "master_inputs_read": False,
            "predictor_source_read": False,
            "model_inference_run": False,
            "function_name": TARGET_FUNCTION_NAME,
            "function_signature": str(
                inspect.signature(
                    hydrology_function
                )
            ),
            "function_source": str(
                function_source_path
            ),
            "direct_callable_names": sorted(
                called_names
            ),
            "outputs": {
                "function_source_py": str(
                    function_copy_path
                ),
                "referenced_globals_csv": str(
                    globals_path
                ),
                "dependencies_csv": str(
                    dependencies_path
                ),
                "dependency_sources_json": str(
                    dependency_sources_path
                ),
                "source_rules_csv": str(
                    source_rules_path
                ),
                "summary_json": str(
                    summary_path
                ),
            },
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved Cell 17 outputs ===")
print("Function source      ->", function_copy_path)
print("Referenced globals   ->", globals_path)
print("Dependencies         ->", dependencies_path)
print("Dependency sources   ->", dependency_sources_path)
print("Source rules         ->", source_rules_path)
print("Summary              ->", summary_path)

print("\nCell 17 complete.")

=== Cell 17 purpose ===
Inspect the actual legacy hydrology implementation and the direct dependencies it uses.

Project root       : C:\Projects\Infer RozviDrought\RozviDrought
master_inputs read : False
predictor read     : False
model inference    : False
Run ID             : 20260620T142848Z

=== Real hydrology function ===
Function : predict_hydrology_proba_from_long_df
Signature: (df: 'pd.DataFrame', pixel_col: 'str' = 'pixel_key', time_col: 'str' = 'yyyymm') -> 'pd.DataFrame'
Source   : c:\Projects\Infer RozviDrought\.venv\Lib\site-packages\rozvidrought_subsystems\hydrology_pipeline.py

=== Exact function source ===
   46: def predict_hydrology_proba_from_long_df(
   47:     df: pd.DataFrame,
   48:     pixel_col: str = "pixel_key",
   49:     time_col: str = "yyyymm",
   50: ) -> pd.DataFrame:
   51:     """
   52:     Build valid hydrology windows from a long dataframe and return probabilities
   53:     for rows whose end-timestep has a valid sequence.
   54: 
   55:     Requ

In [39]:
# validate_events_ng.ipynb — Cell 18 (REWRITE 2)
# Purpose:
# - Run a controlled 32-pixel batch parity test for 201510.
# - Compare batched component execution with repeated one-pixel legacy execution.
# - Preserve and compare FusionService outputs structurally:
#       * nested dictionaries;
#       * lists;
#       * numeric values with tolerance;
#       * non-numeric values exactly.
# - Use only the newest successful Cell 13 context.
#
# No national inference.
# No direct master_inputs read.
# No direct temporal_predictors_full read.

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import gc
import json
import re
import sys
import traceback

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds


# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL18_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

PROBE_TARGET_YYYYMM = 201510
TARGET_MONTH_STRING = str(PROBE_TARGET_YYYYMM)

REFERENCE_PIXEL_ID = "0_101"

CANDIDATE_POOL_SIZE = 512
BATCH_SIZE = 32

ATOL = 1e-6
RTOL = 1e-6

CONTEXT_COLUMNS = [
    "pixel_id",
    "row",
    "col",
    "lon",
    "lat",
    "scenario",
    "yyyymm",
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
    "tws_source",
]

RAW_FEATURE_COLUMNS = [
    "t2m",
    "d2m",
    "pet",
    "sm",
    "ndvi",
    "tws",
]

PROBA_COLUMNS = [
    "p0",
    "p1",
    "p2",
    "p3",
]

METADATA_COLUMNS = {
    "pixel_key",
    "yyyymm",
    "time_type",
    "run_yyyymm",
}

CANDIDATE_AUDIT_PATH = (
    OUT_DIR
    / f"cell18_batch_candidate_audit_{CELL18_RUN_ID}.csv"
)

SELECTED_BATCH_PATH = (
    OUT_DIR
    / f"cell18_selected_batch_{CELL18_RUN_ID}.csv"
)

BATCH_COMPONENT_PATH = (
    OUT_DIR
    / f"cell18_batch_component_probabilities_{CELL18_RUN_ID}.csv"
)

SINGLE_COMPONENT_PATH = (
    OUT_DIR
    / f"cell18_single_component_probabilities_{CELL18_RUN_ID}.csv"
)

BATCH_FUSION_PATH = (
    OUT_DIR
    / f"cell18_batch_fusion_payloads_{CELL18_RUN_ID}.csv"
)

SINGLE_FUSION_PATH = (
    OUT_DIR
    / f"cell18_single_fusion_payloads_{CELL18_RUN_ID}.csv"
)

FUSION_FIELDS_PATH = (
    OUT_DIR
    / f"cell18_fusion_flattened_fields_{CELL18_RUN_ID}.csv"
)

PARITY_PATH = (
    OUT_DIR
    / f"cell18_batch_parity_results_{CELL18_RUN_ID}.csv"
)

SUMMARY_PATH = (
    OUT_DIR
    / f"cell18_batch_parity_summary_{CELL18_RUN_ID}.json"
)

print("=== Cell 18 purpose ===")
print(
    "Test batched legacy execution against repeated one-pixel "
    "legacy execution for one event month."
)

print("\n=== Configuration ===")
print("Target month       :", PROBE_TARGET_YYYYMM)
print("Batch size         :", BATCH_SIZE)
print("Candidate pool     :", CANDIDATE_POOL_SIZE)
print("Reference pixel    :", REFERENCE_PIXEL_ID)
print("Absolute tolerance :", ATOL)
print("Relative tolerance :", RTOL)
print("master_inputs read :", False)
print("predictor read     :", False)
print("national run       :", False)
print("Run ID             :", CELL18_RUN_ID)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root is missing:\n{PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# -----------------------------------------------------------------------------
# 2. Imports and hydrology contract
# -----------------------------------------------------------------------------
from app.services.feature_service import FeatureService
from app.services.fusion_service import FusionService
from app.services.subsystem_service import SubsystemService

from rozvidrought_subsystems import api as subs
from rozvidrought_subsystems.hydrology_windows import (
    build_hydrology_sequences,
)
from rozvidrought_subsystems.loaders import (
    load_subsystem_spec,
)


def to_period(
    yyyymm: int,
) -> pd.Period:
    return pd.Period(
        str(int(yyyymm)),
        freq="M",
    )


def to_yyyymm(
    period: pd.Period,
) -> int:
    return int(
        period.strftime("%Y%m")
    )


def month_sequence(
    start_yyyymm: int,
    end_yyyymm: int,
) -> list[int]:
    start_period = to_period(start_yyyymm)
    end_period = to_period(end_yyyymm)

    if end_period < start_period:
        raise ValueError(
            f"Invalid month range: "
            f"{start_yyyymm} -> {end_yyyymm}"
        )

    return [
        to_yyyymm(period)
        for period in pd.period_range(
            start=start_period,
            end=end_period,
            freq="M",
        )
    ]


def derive_warmup_months(
    feature_list: list[str],
) -> int:
    warmup = 0

    for feature_name in feature_list:
        lag_match = re.search(
            r"_lag(\d+)$",
            feature_name,
        )

        if lag_match:
            warmup = max(
                warmup,
                int(lag_match.group(1)),
            )

        roll_match = re.search(
            r"_roll(?:mean|std)(\d+)$",
            feature_name,
        )

        if roll_match:
            warmup = max(
                warmup,
                int(roll_match.group(1)) - 1,
            )

    return warmup


hydrology_spec = load_subsystem_spec(
    "hydrology"
)

HYDROLOGY_FEATURES = list(
    hydrology_spec.feature_list
)

HYDROLOGY_SEQ_LEN = int(
    hydrology_spec.sequence_config["seq_len"]
)

HYDROLOGY_WARMUP_MONTHS = derive_warmup_months(
    HYDROLOGY_FEATURES
)

TOTAL_REQUIRED_RAW_MONTHS = (
    HYDROLOGY_WARMUP_MONTHS
    + HYDROLOGY_SEQ_LEN
)

CONTEXT_START_YYYYMM = to_yyyymm(
    to_period(PROBE_TARGET_YYYYMM)
    - (TOTAL_REQUIRED_RAW_MONTHS - 1)
)

EXPECTED_MONTHS = month_sequence(
    CONTEXT_START_YYYYMM,
    PROBE_TARGET_YYYYMM,
)

print("\n=== Hydrology contract ===")
print("Features           :", HYDROLOGY_FEATURES)
print("Warm-up months     :", HYDROLOGY_WARMUP_MONTHS)
print("Sequence length    :", HYDROLOGY_SEQ_LEN)
print("Required raw months:", TOTAL_REQUIRED_RAW_MONTHS)
print(
    "Context range      :",
    CONTEXT_START_YYYYMM,
    "->",
    PROBE_TARGET_YYYYMM,
)


# -----------------------------------------------------------------------------
# 3. Resolve newest successful Cell 13 context
# -----------------------------------------------------------------------------
context_paths = sorted(
    OUT_DIR.glob(
        "cell13_legacy_context_tws_backfilled_*.parquet"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not context_paths:
    raise FileNotFoundError(
        "No Cell 13 context parquet was found."
    )

CELL13_CONTEXT_PATH = context_paths[0]

run_id_match = re.search(
    r"cell13_legacy_context_tws_backfilled_"
    r"(\d{8}T\d{6}Z)\.parquet$",
    CELL13_CONTEXT_PATH.name,
)

if not run_id_match:
    raise AssertionError(
        "Could not derive Cell 13 run ID from:\n"
        f"{CELL13_CONTEXT_PATH.name}"
    )

CELL13_RUN_ID = run_id_match.group(1)

CELL13_MANIFEST_PATH = (
    OUT_DIR
    / f"cell13_legacy_context_manifest_{CELL13_RUN_ID}.json"
)

if not CELL13_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Matching Cell 13 manifest is missing:\n"
        f"{CELL13_MANIFEST_PATH}"
    )

cell13_manifest = json.loads(
    CELL13_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if not bool(
    cell13_manifest.get(
        "hydrology_preflight_passed",
        False,
    )
):
    raise AssertionError(
        "The newest Cell 13 context did not pass hydrology preflight."
    )

print("\n=== Cell 13 context ===")
print("Context :", CELL13_CONTEXT_PATH)
print("Manifest:", CELL13_MANIFEST_PATH)
print(
    "Hydrology preflight:",
    cell13_manifest[
        "hydrology_preflight_passed"
    ],
)


# -----------------------------------------------------------------------------
# 4. Load exact 35-month context from Cell 13 only
# -----------------------------------------------------------------------------
def typed_month_value(
    arrow_type: pa.DataType,
    yyyymm: int,
):
    if (
        pa.types.is_string(arrow_type)
        or pa.types.is_large_string(arrow_type)
    ):
        return str(int(yyyymm))

    if pa.types.is_integer(arrow_type):
        return int(yyyymm)

    raise TypeError(
        f"Unsupported yyyymm type: {arrow_type}"
    )


context_dataset = ds.dataset(
    CELL13_CONTEXT_PATH,
    format="parquet",
)

missing_columns = [
    column_name
    for column_name in CONTEXT_COLUMNS
    if column_name not in context_dataset.schema.names
]

if missing_columns:
    raise AssertionError(
        "Cell 13 context lacks required fields:\n"
        f"{missing_columns}"
    )

yyyymm_type = context_dataset.schema.field(
    "yyyymm"
).type

context_table = context_dataset.to_table(
    columns=CONTEXT_COLUMNS,
    filter=(
        ds.field("yyyymm")
        >= typed_month_value(
            yyyymm_type,
            CONTEXT_START_YYYYMM,
        )
    ) & (
        ds.field("yyyymm")
        <= typed_month_value(
            yyyymm_type,
            PROBE_TARGET_YYYYMM,
        )
    ),
)

context_df = context_table.to_pandas()

if context_df.empty:
    raise AssertionError(
        "No Cell 13 context rows were loaded."
    )

for column_name in [
    "row",
    "col",
    "yyyymm",
]:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="raise",
    ).astype("int64")

for column_name in RAW_FEATURE_COLUMNS:
    context_df[column_name] = pd.to_numeric(
        context_df[column_name],
        errors="coerce",
    )

observed_months = sorted(
    context_df["yyyymm"]
    .unique()
    .tolist()
)

if observed_months != EXPECTED_MONTHS:
    raise AssertionError(
        "Cell 13 context does not cover the required "
        "continuous 35-month history.\n"
        f"Expected: {EXPECTED_MONTHS}\n"
        f"Observed: {observed_months}"
    )

if context_df.duplicated(
    subset=[
        "pixel_id",
        "yyyymm",
    ]
).any():
    raise AssertionError(
        "Cell 13 context contains duplicate pixel/month rows."
    )

rows_per_month = (
    context_df.groupby("yyyymm")
    .size()
    .sort_index()
)

if rows_per_month.nunique() != 1:
    raise AssertionError(
        "Cell 13 grid row count changes by month.\n"
        f"{rows_per_month.to_string()}"
    )

print("\n=== Loaded context ===")
print("Rows         :", len(context_df))
print("Months       :", len(observed_months))
print(
    "Pixels/month:",
    int(rows_per_month.iloc[0]),
)


# -----------------------------------------------------------------------------
# 5. Select a deterministic candidate pool
# -----------------------------------------------------------------------------
history_audit_df = (
    context_df.groupby(
        "pixel_id",
        as_index=False,
    )
    .agg(
        history_month_count=(
            "yyyymm",
            "nunique",
        ),
        tws_non_null_month_count=(
            "tws",
            "count",
        ),
    )
)

target_df = (
    context_df.loc[
        context_df["yyyymm"].eq(
            PROBE_TARGET_YYYYMM
        )
    ]
    .copy()
)

target_df["target_raw_complete"] = (
    target_df[
        RAW_FEATURE_COLUMNS
    ]
    .notna()
    .all(axis=1)
)

candidate_audit_df = (
    target_df[
        [
            "pixel_id",
            "row",
            "col",
            "target_raw_complete",
        ]
    ]
    .merge(
        history_audit_df,
        on="pixel_id",
        how="left",
        validate="one_to_one",
    )
)

candidate_audit_df["eligible_raw_history"] = (
    candidate_audit_df[
        "history_month_count"
    ].eq(TOTAL_REQUIRED_RAW_MONTHS)
    & candidate_audit_df[
        "tws_non_null_month_count"
    ].eq(TOTAL_REQUIRED_RAW_MONTHS)
    & candidate_audit_df[
        "target_raw_complete"
    ]
)

raw_eligible_df = (
    candidate_audit_df.loc[
        candidate_audit_df[
            "eligible_raw_history"
        ]
    ]
    .sort_values(
        [
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

if REFERENCE_PIXEL_ID not in set(
    raw_eligible_df["pixel_id"]
):
    raise AssertionError(
        f"Reference pixel {REFERENCE_PIXEL_ID} "
        "is not raw-history eligible."
    )

candidate_pool_ids = [
    REFERENCE_PIXEL_ID,
    *[
        pixel_id
        for pixel_id in raw_eligible_df[
            "pixel_id"
        ].tolist()
        if pixel_id != REFERENCE_PIXEL_ID
    ],
]

candidate_pool_ids = list(
    dict.fromkeys(candidate_pool_ids)
)[:CANDIDATE_POOL_SIZE]

pool_context_df = (
    context_df.loc[
        context_df["pixel_id"].isin(
            candidate_pool_ids
        )
    ]
    .sort_values(
        [
            "pixel_id",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

expected_pool_rows = (
    len(candidate_pool_ids)
    * TOTAL_REQUIRED_RAW_MONTHS
)

if len(pool_context_df) != expected_pool_rows:
    raise AssertionError(
        "Candidate pool does not contain complete histories.\n"
        f"Expected: {expected_pool_rows}\n"
        f"Observed: {len(pool_context_df)}"
    )

pool_context_df["yyyymm"] = (
    pool_context_df["yyyymm"]
    .astype("int64")
    .astype(str)
)

print("\n=== Candidate pool ===")
print(
    "Raw-history eligible:",
    len(raw_eligible_df),
)
print(
    "Pool size           :",
    len(candidate_pool_ids),
)


# -----------------------------------------------------------------------------
# 6. Prepare candidate pool and determine actual model eligibility
# -----------------------------------------------------------------------------
feature_service = FeatureService()

prepared_pool = feature_service.prepare_subsystem_inputs(
    timeseries=pool_context_df.copy(),
    run_yyyymm=PROBE_TARGET_YYYYMM,
)

required_subsystems = [
    "atmospheric",
    "soil",
    "vegetation",
    "hydrology",
]

missing_subsystems = [
    subsystem_name
    for subsystem_name in required_subsystems
    if subsystem_name not in prepared_pool
]

if missing_subsystems:
    raise AssertionError(
        "FeatureService omitted subsystem frames:\n"
        f"{missing_subsystems}"
    )


def branch_feature_columns(
    frame: pd.DataFrame,
) -> list[str]:
    output = [
        column_name
        for column_name in frame.columns
        if column_name not in METADATA_COLUMNS
    ]

    if not output:
        raise AssertionError(
            "Prepared branch has no model-feature columns."
        )

    return output


def finite_feature_rows(
    frame: pd.DataFrame,
    feature_columns: list[str],
) -> pd.Series:
    numeric_frame = frame[
        feature_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    return (
        numeric_frame.notna().all(axis=1)
        & np.isfinite(
            numeric_frame.to_numpy(
                dtype=np.float64
            )
        ).all(axis=1)
    )


branch_valid_keys: dict[str, set[str]] = {}

for subsystem_name in [
    "atmospheric",
    "soil",
    "vegetation",
]:
    branch_df = prepared_pool[
        subsystem_name
    ].copy()

    branch_df["yyyymm"] = (
        branch_df["yyyymm"]
        .astype(str)
    )

    target_branch_df = (
        branch_df.loc[
            branch_df["yyyymm"].eq(
                TARGET_MONTH_STRING
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    features = branch_feature_columns(
        target_branch_df
    )

    valid_mask = finite_feature_rows(
        target_branch_df,
        features,
    )

    branch_valid_keys[subsystem_name] = set(
        target_branch_df.loc[
            valid_mask,
            "pixel_key",
        ]
        .astype(str)
        .tolist()
    )

hydrology_pool_df = (
    prepared_pool["hydrology"]
    .copy()
)

hydrology_pool_df["yyyymm"] = (
    hydrology_pool_df["yyyymm"]
    .astype(str)
)

hydrology_pool_df = (
    hydrology_pool_df.loc[
        hydrology_pool_df["yyyymm"].le(
            TARGET_MONTH_STRING
        )
    ]
    .sort_values(
        [
            "pixel_key",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
)

_, hydrology_valid_index = build_hydrology_sequences(
    df=hydrology_pool_df,
    pixel_col="pixel_key",
    time_col="yyyymm",
)

hydrology_valid_target_df = (
    hydrology_pool_df.loc[
        hydrology_valid_index,
        [
            "pixel_key",
            "yyyymm",
        ],
    ]
    .copy()
)

branch_valid_keys["hydrology"] = set(
    hydrology_valid_target_df.loc[
        hydrology_valid_target_df["yyyymm"].eq(
            TARGET_MONTH_STRING
        ),
        "pixel_key",
    ]
    .astype(str)
    .tolist()
)

full_eligible_keys = set.intersection(
    *[
        branch_valid_keys[subsystem_name]
        for subsystem_name in required_subsystems
    ]
)

for subsystem_name in required_subsystems:
    candidate_audit_df[
        f"{subsystem_name}_valid"
    ] = candidate_audit_df[
        "pixel_id"
    ].astype(str).isin(
        branch_valid_keys[subsystem_name]
    )

candidate_audit_df["eligible_full_hybrid"] = (
    candidate_audit_df[
        "pixel_id"
    ].astype(str).isin(
        full_eligible_keys
    )
)

candidate_audit_df.to_csv(
    CANDIDATE_AUDIT_PATH,
    index=False,
)

full_eligible_df = (
    candidate_audit_df.loc[
        candidate_audit_df[
            "eligible_full_hybrid"
        ]
    ]
    .sort_values(
        [
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

if REFERENCE_PIXEL_ID not in set(
    full_eligible_df["pixel_id"]
):
    raise AssertionError(
        f"Reference pixel {REFERENCE_PIXEL_ID} is not "
        "eligible after feature preparation."
    )

if len(full_eligible_df) < BATCH_SIZE:
    raise AssertionError(
        "Not enough fully eligible pixels for the batch test.\n"
        f"Required: {BATCH_SIZE}\n"
        f"Eligible: {len(full_eligible_df)}"
    )

selected_pixel_ids = [
    REFERENCE_PIXEL_ID,
    *[
        pixel_id
        for pixel_id in full_eligible_df[
            "pixel_id"
        ].tolist()
        if pixel_id != REFERENCE_PIXEL_ID
    ],
][:BATCH_SIZE]

selected_batch_df = (
    full_eligible_df.loc[
        full_eligible_df["pixel_id"].isin(
            selected_pixel_ids
        )
    ]
    .sort_values(
        [
            "row",
            "col",
        ]
    )
    .reset_index(drop=True)
)

selected_batch_df["is_cell14_reference"] = (
    selected_batch_df["pixel_id"].eq(
        REFERENCE_PIXEL_ID
    )
)

selected_batch_df.to_csv(
    SELECTED_BATCH_PATH,
    index=False,
)

if len(selected_batch_df) != BATCH_SIZE:
    raise AssertionError(
        "Selected batch does not equal requested size."
    )

selected_pixel_key_set = set(
    selected_batch_df[
        "pixel_id"
    ].astype(str).tolist()
)

print("\n=== Branch eligibility ===")
for subsystem_name in required_subsystems:
    print(
        f"{subsystem_name:12}:",
        len(branch_valid_keys[subsystem_name]),
    )

print("Full hybrid     :", len(full_eligible_df))
print("Selected batch  :", len(selected_batch_df))
print(
    "Reference present:",
    REFERENCE_PIXEL_ID in selected_pixel_key_set,
)


# -----------------------------------------------------------------------------
# 7. Build batch-ready prepared inputs
# -----------------------------------------------------------------------------
batch_inputs: dict[str, pd.DataFrame] = {}

for subsystem_name in [
    "atmospheric",
    "soil",
    "vegetation",
]:
    branch_df = prepared_pool[
        subsystem_name
    ].copy()

    branch_df["yyyymm"] = (
        branch_df["yyyymm"]
        .astype(str)
    )

    branch_df = (
        branch_df.loc[
            branch_df["pixel_key"]
            .astype(str)
            .isin(selected_pixel_key_set)
            & branch_df["yyyymm"].eq(
                TARGET_MONTH_STRING
            )
        ]
        .sort_values("pixel_key")
        .reset_index(drop=True)
    )

    if len(branch_df) != BATCH_SIZE:
        raise AssertionError(
            f"{subsystem_name}: expected {BATCH_SIZE} "
            f"target rows, got {len(branch_df)}."
        )

    if not finite_feature_rows(
        branch_df,
        branch_feature_columns(branch_df),
    ).all():
        raise AssertionError(
            f"{subsystem_name}: selected batch contains "
            "non-finite target model features."
        )

    batch_inputs[subsystem_name] = branch_df

hydrology_batch_df = (
    hydrology_pool_df.loc[
        hydrology_pool_df["pixel_key"]
        .astype(str)
        .isin(selected_pixel_key_set)
    ]
    .sort_values(
        [
            "pixel_key",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
)

expected_hydrology_rows = (
    BATCH_SIZE
    * TOTAL_REQUIRED_RAW_MONTHS
)

if len(hydrology_batch_df) != expected_hydrology_rows:
    raise AssertionError(
        "Hydrology batch input contains incomplete histories.\n"
        f"Expected: {expected_hydrology_rows}\n"
        f"Observed: {len(hydrology_batch_df)}"
    )

batch_inputs["hydrology"] = hydrology_batch_df


# -----------------------------------------------------------------------------
# 8. Component validation and probability helpers
# -----------------------------------------------------------------------------
def validate_probabilities(
    frame: pd.DataFrame,
    label: str,
):
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            f"{label}: expected DataFrame, got "
            f"{type(frame).__name__}"
        )

    missing_probability_columns = [
        column_name
        for column_name in PROBA_COLUMNS
        if column_name not in frame.columns
    ]

    if missing_probability_columns:
        raise AssertionError(
            f"{label}: missing probability columns:\n"
            f"{missing_probability_columns}"
        )

    numeric_probabilities = frame[
        PROBA_COLUMNS
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    if numeric_probabilities.isna().any().any():
        raise AssertionError(
            f"{label}: contains null probabilities."
        )

    probability_array = numeric_probabilities.to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(probability_array).all():
        raise AssertionError(
            f"{label}: contains non-finite probabilities."
        )

    if not np.allclose(
        probability_array.sum(axis=1),
        np.ones(len(numeric_probabilities)),
        atol=ATOL,
        rtol=RTOL,
    ):
        raise AssertionError(
            f"{label}: probability rows do not sum to one."
        )


def attach_nonsequence_output(
    input_df: pd.DataFrame,
    output_df: pd.DataFrame,
    subsystem_name: str,
) -> pd.DataFrame:
    validate_probabilities(
        output_df,
        subsystem_name,
    )

    input_keys_df = (
        input_df[
            ["pixel_key"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    output_probabilities_df = (
        output_df[
            PROBA_COLUMNS
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(input_keys_df) != len(output_probabilities_df):
        raise AssertionError(
            f"{subsystem_name}: input/output row count mismatch.\n"
            f"Input: {len(input_keys_df)}\n"
            f"Output: {len(output_probabilities_df)}"
        )

    return pd.concat(
        [
            input_keys_df,
            output_probabilities_df,
        ],
        axis=1,
    )


# -----------------------------------------------------------------------------
# 9. Batched package component inference
# -----------------------------------------------------------------------------
batch_exception = None
batch_traceback = None

batch_component_frames: dict[str, pd.DataFrame] = {}

try:
    atmospheric_output_df = subs.predict_atmospheric_proba(
        batch_inputs["atmospheric"]
    )

    soil_output_df = subs.predict_soil_proba(
        batch_inputs["soil"]
    )

    vegetation_output_df = subs.predict_vegetation_proba(
        batch_inputs["vegetation"]
    )

    hydrology_output_df = subs.predict_hydrology_proba_df(
        batch_inputs["hydrology"]
    )

    batch_component_frames["atmospheric"] = (
        attach_nonsequence_output(
            input_df=batch_inputs["atmospheric"],
            output_df=atmospheric_output_df,
            subsystem_name="atmospheric",
        )
    )

    batch_component_frames["soil"] = (
        attach_nonsequence_output(
            input_df=batch_inputs["soil"],
            output_df=soil_output_df,
            subsystem_name="soil",
        )
    )

    batch_component_frames["vegetation"] = (
        attach_nonsequence_output(
            input_df=batch_inputs["vegetation"],
            output_df=vegetation_output_df,
            subsystem_name="vegetation",
        )
    )

    validate_probabilities(
        hydrology_output_df,
        "hydrology",
    )

    hydrology_target_indices = (
        batch_inputs["hydrology"].index[
            batch_inputs["hydrology"]["yyyymm"].eq(
                TARGET_MONTH_STRING
            )
        ]
        .tolist()
    )

    if set(hydrology_output_df.index.tolist()) != set(
        hydrology_target_indices
    ):
        raise AssertionError(
            "Hydrology output indices do not correspond exactly "
            "to selected target-month rows."
        )

    hydrology_keys_df = (
        batch_inputs["hydrology"]
        .loc[
            hydrology_output_df.index,
            [
                "pixel_key",
                "yyyymm",
            ],
        ]
        .copy()
        .reset_index(drop=True)
    )

    if not hydrology_keys_df["yyyymm"].eq(
        TARGET_MONTH_STRING
    ).all():
        raise AssertionError(
            "Hydrology output includes non-target months."
        )

    batch_component_frames["hydrology"] = pd.concat(
        [
            hydrology_keys_df[
                ["pixel_key"]
            ],
            hydrology_output_df[
                PROBA_COLUMNS
            ].reset_index(drop=True),
        ],
        axis=1,
    )

    for subsystem_name in required_subsystems:
        if len(batch_component_frames[subsystem_name]) != BATCH_SIZE:
            raise AssertionError(
                f"{subsystem_name}: expected {BATCH_SIZE} "
                f"batch outputs, got "
                f"{len(batch_component_frames[subsystem_name])}."
            )

except Exception as error:
    batch_exception = repr(error)
    batch_traceback = traceback.format_exc()


# -----------------------------------------------------------------------------
# 10. Robust FusionService-output serialisation and comparison helpers
# -----------------------------------------------------------------------------
def to_plain_python(
    value: Any,
) -> Any:
    """
    Converts arbitrary FusionService output to JSON-compatible Python values
    without assuming a particular output schema.
    """
    if value is None:
        return None

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, pd.DataFrame):
        return {
            "__type__": "DataFrame",
            "columns": [
                str(column_name)
                for column_name in value.columns
            ],
            "rows": value.to_dict(
                orient="records"
            ),
        }

    if isinstance(value, pd.Series):
        return {
            "__type__": "Series",
            "values": value.to_dict(),
        }

    if isinstance(value, np.ndarray):
        return value.tolist()

    if is_dataclass(value):
        return to_plain_python(asdict(value))

    if hasattr(value, "model_dump"):
        return to_plain_python(
            value.model_dump()
        )

    if hasattr(value, "dict"):
        try:
            return to_plain_python(
                value.dict()
            )
        except TypeError:
            pass

    if isinstance(value, Mapping):
        return {
            str(key): to_plain_python(child)
            for key, child in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_plain_python(child)
            for child in value
        ]

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if hasattr(value, "__dict__"):
        return to_plain_python(
            vars(value)
        )

    return {
        "__type__": (
            f"{type(value).__module__}."
            f"{type(value).__name__}"
        ),
        "repr": str(value),
    }


def flatten_payload(
    value: Any,
    path: str = "root",
    numeric_values: dict[str, float] | None = None,
    text_values: dict[str, str] | None = None,
) -> tuple[dict[str, float], dict[str, str]]:
    """
    Extract every numerical leaf and every non-numerical leaf from nested
    fusion output. Lists are indexed explicitly, e.g. root.probabilities[0].
    """
    if numeric_values is None:
        numeric_values = {}

    if text_values is None:
        text_values = {}

    if isinstance(value, bool):
        text_values[path] = json.dumps(value)
        return numeric_values, text_values

    if value is None:
        text_values[path] = "null"
        return numeric_values, text_values

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        numeric_values[path] = float(value)
        return numeric_values, text_values

    if isinstance(value, Mapping):
        if not value:
            text_values[path] = "{}"

        for key, child in value.items():
            flatten_payload(
                value=child,
                path=f"{path}.{key}",
                numeric_values=numeric_values,
                text_values=text_values,
            )

        return numeric_values, text_values

    if isinstance(value, list):
        if not value:
            text_values[path] = "[]"

        for index, child in enumerate(value):
            flatten_payload(
                value=child,
                path=f"{path}[{index}]",
                numeric_values=numeric_values,
                text_values=text_values,
            )

        return numeric_values, text_values

    text_values[path] = str(value)
    return numeric_values, text_values


def fusion_record(
    pixel_key: str,
    fusion_output: Any,
    route: str,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    plain_output = to_plain_python(fusion_output)

    numeric_values, text_values = flatten_payload(
        plain_output
    )

    payload_json = json.dumps(
        plain_output,
        sort_keys=True,
        default=str,
    )

    payload_record = {
        "pixel_key": str(pixel_key),
        "route": route,
        "fusion_payload_json": payload_json,
        "numeric_leaf_count": int(
            len(numeric_values)
        ),
        "text_leaf_count": int(
            len(text_values)
        ),
    }

    field_records: list[dict[str, Any]] = []

    for field_path, value in numeric_values.items():
        field_records.append(
            {
                "pixel_key": str(pixel_key),
                "route": route,
                "field_path": field_path,
                "field_kind": "numeric",
                "numeric_value": float(value),
                "text_value": None,
            }
        )

    for field_path, value in text_values.items():
        field_records.append(
            {
                "pixel_key": str(pixel_key),
                "route": route,
                "field_path": field_path,
                "field_kind": "text",
                "numeric_value": None,
                "text_value": str(value),
            }
        )

    return payload_record, field_records


# -----------------------------------------------------------------------------
# 11. Row-wise fusion for batched component predictions
# -----------------------------------------------------------------------------
batch_fusion_payload_records: list[dict[str, Any]] = []
batch_fusion_field_records: list[dict[str, Any]] = []

if batch_exception is None:
    fusion_service = FusionService()

    component_lookup = {
        subsystem_name: (
            batch_component_frames[
                subsystem_name
            ]
            .set_index("pixel_key")
            .sort_index()
        )
        for subsystem_name in required_subsystems
    }

    for pixel_key in sorted(selected_pixel_key_set):
        component_outputs = {}

        for subsystem_name in required_subsystems:
            probability_row = component_lookup[
                subsystem_name
            ].loc[
                pixel_key,
                PROBA_COLUMNS,
            ]

            component_outputs[subsystem_name] = pd.DataFrame(
                [
                    probability_row.to_dict()
                ]
            )

        fused_output = fusion_service.run(
            subsystem_outputs=component_outputs,
            model="hybrid",
        )

        payload_record, field_records = fusion_record(
            pixel_key=pixel_key,
            fusion_output=fused_output,
            route="batch_components",
        )

        batch_fusion_payload_records.append(
            payload_record
        )

        batch_fusion_field_records.extend(
            field_records
        )

batch_fusion_payload_df = (
    pd.DataFrame(batch_fusion_payload_records)
    .sort_values("pixel_key")
    .reset_index(drop=True)
)

if (
    batch_exception is None
    and len(batch_fusion_payload_df) != BATCH_SIZE
):
    raise AssertionError(
        "Batched fusion output count does not equal batch size."
    )


# -----------------------------------------------------------------------------
# 12. Repeat full original one-pixel route
# -----------------------------------------------------------------------------
single_exception = None
single_traceback = None

single_component_records: list[dict[str, Any]] = []
single_fusion_payload_records: list[dict[str, Any]] = []
single_fusion_field_records: list[dict[str, Any]] = []

selected_context_df = (
    context_df.loc[
        context_df["pixel_id"].isin(
            selected_pixel_key_set
        )
    ]
    .sort_values(
        [
            "pixel_id",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

try:
    subsystem_service = SubsystemService()
    fusion_service = FusionService()

    for pixel_key in sorted(selected_pixel_key_set):
        one_pixel_context_df = (
            selected_context_df.loc[
                selected_context_df["pixel_id"].eq(
                    pixel_key
                )
            ]
            .sort_values("yyyymm")
            .reset_index(drop=True)
            .copy()
        )

        if len(one_pixel_context_df) != TOTAL_REQUIRED_RAW_MONTHS:
            raise AssertionError(
                f"{pixel_key}: incomplete raw-history context."
            )

        one_pixel_context_df["yyyymm"] = (
            one_pixel_context_df["yyyymm"]
            .astype("int64")
            .astype(str)
        )

        one_prepared = (
            feature_service.prepare_subsystem_inputs(
                timeseries=one_pixel_context_df,
                run_yyyymm=PROBE_TARGET_YYYYMM,
            )
        )

        one_target_prepared: dict[str, pd.DataFrame] = {}

        for subsystem_name, subsystem_df in one_prepared.items():
            subsystem_df = subsystem_df.copy()

            subsystem_df["yyyymm"] = (
                subsystem_df["yyyymm"]
                .astype(str)
            )

            if subsystem_name == "hydrology":
                one_target_prepared[subsystem_name] = (
                    subsystem_df.loc[
                        subsystem_df["yyyymm"].le(
                            TARGET_MONTH_STRING
                        )
                    ]
                    .sort_values("yyyymm")
                    .reset_index(drop=True)
                )
            else:
                one_target_df = (
                    subsystem_df.loc[
                        subsystem_df["yyyymm"].eq(
                            TARGET_MONTH_STRING
                        )
                    ]
                    .copy()
                    .reset_index(drop=True)
                )

                if len(one_target_df) != 1:
                    raise AssertionError(
                        f"{pixel_key}: {subsystem_name} expected "
                        "one target row."
                    )

                one_target_prepared[subsystem_name] = (
                    one_target_df
                )

        one_outputs = subsystem_service.run_subsystems(
            one_target_prepared
        )

        for subsystem_name in required_subsystems:
            one_output_df = one_outputs[subsystem_name]

            validate_probabilities(
                one_output_df,
                f"{pixel_key}:{subsystem_name}",
            )

            if len(one_output_df) != 1:
                raise AssertionError(
                    f"{pixel_key}: {subsystem_name} returned "
                    f"{len(one_output_df)} rows instead of one."
                )

            single_component_records.append(
                {
                    "pixel_key": pixel_key,
                    "subsystem": subsystem_name,
                    **{
                        probability_column: float(
                            one_output_df[
                                probability_column
                            ].iloc[0]
                        )
                        for probability_column in PROBA_COLUMNS
                    },
                }
            )

        one_fusion_output = fusion_service.run(
            subsystem_outputs=one_outputs,
            model="hybrid",
        )

        payload_record, field_records = fusion_record(
            pixel_key=pixel_key,
            fusion_output=one_fusion_output,
            route="single_pixel",
        )

        single_fusion_payload_records.append(
            payload_record
        )

        single_fusion_field_records.extend(
            field_records
        )

except Exception as error:
    single_exception = repr(error)
    single_traceback = traceback.format_exc()

single_component_df = (
    pd.DataFrame(single_component_records)
    .sort_values(
        [
            "pixel_key",
            "subsystem",
        ]
    )
    .reset_index(drop=True)
)

single_fusion_payload_df = (
    pd.DataFrame(single_fusion_payload_records)
    .sort_values("pixel_key")
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# 13. Compare batch and one-pixel component outputs
# -----------------------------------------------------------------------------
parity_records: list[dict[str, Any]] = []

batch_component_long_df = pd.DataFrame()

if (
    batch_exception is None
    and single_exception is None
):
    batch_component_long_df = (
        pd.concat(
            [
                frame.assign(
                    subsystem=subsystem_name
                )
                for subsystem_name, frame in (
                    batch_component_frames.items()
                )
            ],
            ignore_index=True,
        )
        .sort_values(
            [
                "pixel_key",
                "subsystem",
            ]
        )
        .reset_index(drop=True)
    )

    for subsystem_name in required_subsystems:
        batch_frame = (
            batch_component_long_df.loc[
                batch_component_long_df[
                    "subsystem"
                ].eq(subsystem_name)
            ]
            .sort_values("pixel_key")
            .reset_index(drop=True)
        )

        single_frame = (
            single_component_df.loc[
                single_component_df[
                    "subsystem"
                ].eq(subsystem_name)
            ]
            .sort_values("pixel_key")
            .reset_index(drop=True)
        )

        if not batch_frame["pixel_key"].equals(
            single_frame["pixel_key"]
        ):
            raise AssertionError(
                f"{subsystem_name}: pixel order differs between "
                "batch and one-pixel results."
            )

        for probability_column in PROBA_COLUMNS:
            batch_values = batch_frame[
                probability_column
            ].to_numpy(
                dtype=np.float64
            )

            single_values = single_frame[
                probability_column
            ].to_numpy(
                dtype=np.float64
            )

            absolute_delta = np.abs(
                batch_values
                - single_values
            )

            parity_records.append(
                {
                    "comparison_type": "component",
                    "subsystem": subsystem_name,
                    "field_path": probability_column,
                    "rows_compared": int(
                        len(batch_values)
                    ),
                    "max_abs_delta": float(
                        absolute_delta.max()
                    ),
                    "mean_abs_delta": float(
                        absolute_delta.mean()
                    ),
                    "passed": bool(
                        np.allclose(
                            batch_values,
                            single_values,
                            atol=ATOL,
                            rtol=RTOL,
                        )
                    ),
                }
            )


# -----------------------------------------------------------------------------
# 14. Compare batch and one-pixel fusion payloads structurally
# -----------------------------------------------------------------------------
batch_fusion_fields_df = pd.DataFrame(
    batch_fusion_field_records
)

single_fusion_fields_df = pd.DataFrame(
    single_fusion_field_records
)

fusion_field_records_for_export = pd.concat(
    [
        batch_fusion_fields_df,
        single_fusion_fields_df,
    ],
    ignore_index=True,
)

if (
    batch_exception is None
    and single_exception is None
):
    if not batch_fusion_payload_df["pixel_key"].equals(
        single_fusion_payload_df["pixel_key"]
    ):
        raise AssertionError(
            "Fusion payload pixel order differs between routes."
        )

    for pixel_key in sorted(selected_pixel_key_set):
        batch_fields = (
            batch_fusion_fields_df.loc[
                batch_fusion_fields_df["pixel_key"].eq(
                    pixel_key
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

        single_fields = (
            single_fusion_fields_df.loc[
                single_fusion_fields_df["pixel_key"].eq(
                    pixel_key
                )
            ]
            .copy()
            .reset_index(drop=True)
        )

        batch_numeric = {
            row["field_path"]: float(
                row["numeric_value"]
            )
            for _, row in batch_fields.loc[
                batch_fields["field_kind"].eq(
                    "numeric"
                )
            ].iterrows()
        }

        single_numeric = {
            row["field_path"]: float(
                row["numeric_value"]
            )
            for _, row in single_fields.loc[
                single_fields["field_kind"].eq(
                    "numeric"
                )
            ].iterrows()
        }

        batch_text = {
            row["field_path"]: str(
                row["text_value"]
            )
            for _, row in batch_fields.loc[
                batch_fields["field_kind"].eq(
                    "text"
                )
            ].iterrows()
        }

        single_text = {
            row["field_path"]: str(
                row["text_value"]
            )
            for _, row in single_fields.loc[
                single_fields["field_kind"].eq(
                    "text"
                )
            ].iterrows()
        }

        numeric_key_match = (
            set(batch_numeric)
            == set(single_numeric)
        )

        text_key_match = (
            set(batch_text)
            == set(single_text)
        )

        if numeric_key_match:
            numeric_keys = sorted(
                batch_numeric.keys()
            )

            batch_numeric_values = np.array(
                [
                    batch_numeric[key]
                    for key in numeric_keys
                ],
                dtype=np.float64,
            )

            single_numeric_values = np.array(
                [
                    single_numeric[key]
                    for key in numeric_keys
                ],
                dtype=np.float64,
            )

            numeric_passed = bool(
                np.allclose(
                    batch_numeric_values,
                    single_numeric_values,
                    atol=ATOL,
                    rtol=RTOL,
                )
            )

            numeric_delta = np.abs(
                batch_numeric_values
                - single_numeric_values
            )

            max_numeric_delta = float(
                numeric_delta.max()
            ) if len(numeric_delta) else 0.0

            mean_numeric_delta = float(
                numeric_delta.mean()
            ) if len(numeric_delta) else 0.0

        else:
            numeric_passed = False
            max_numeric_delta = np.nan
            mean_numeric_delta = np.nan

        text_passed = bool(
            text_key_match
            and batch_text == single_text
        )

        payload_batch_json = batch_fusion_payload_df.loc[
            batch_fusion_payload_df["pixel_key"].eq(
                pixel_key
            ),
            "fusion_payload_json",
        ].iloc[0]

        payload_single_json = single_fusion_payload_df.loc[
            single_fusion_payload_df["pixel_key"].eq(
                pixel_key
            ),
            "fusion_payload_json",
        ].iloc[0]

        parity_records.append(
            {
                "comparison_type": "fusion",
                "subsystem": "hybrid",
                "field_path": "all_payload_fields",
                "pixel_key": pixel_key,
                "rows_compared": 1,
                "numeric_field_count": int(
                    len(batch_numeric)
                ),
                "text_field_count": int(
                    len(batch_text)
                ),
                "max_abs_delta": max_numeric_delta,
                "mean_abs_delta": mean_numeric_delta,
                "payload_json_exact_match": bool(
                    payload_batch_json
                    == payload_single_json
                ),
                "passed": bool(
                    numeric_key_match
                    and text_passed
                    and numeric_passed
                ),
            }
        )

parity_df = pd.DataFrame(
    parity_records
)

if batch_exception is not None:
    CELL18_STATUS = (
        "batch_parity_failed_batched_route_error"
    )

elif single_exception is not None:
    CELL18_STATUS = (
        "batch_parity_failed_single_pixel_route_error"
    )

elif parity_df.empty:
    CELL18_STATUS = (
        "batch_parity_failed_no_comparisons"
    )

elif not parity_df["passed"].all():
    CELL18_STATUS = (
        "batch_parity_failed_output_mismatch"
    )

else:
    CELL18_STATUS = (
        "batch_parity_passed_ready_for_national_runner"
    )


# -----------------------------------------------------------------------------
# 15. Save evidence before enforcing the parity gate
# -----------------------------------------------------------------------------
batch_component_long_df.to_csv(
    BATCH_COMPONENT_PATH,
    index=False,
)

single_component_df.to_csv(
    SINGLE_COMPONENT_PATH,
    index=False,
)

batch_fusion_payload_df.to_csv(
    BATCH_FUSION_PATH,
    index=False,
)

single_fusion_payload_df.to_csv(
    SINGLE_FUSION_PATH,
    index=False,
)

fusion_field_records_for_export.to_csv(
    FUSION_FIELDS_PATH,
    index=False,
)

parity_df.to_csv(
    PARITY_PATH,
    index=False,
)

summary = {
    "run_id": CELL18_RUN_ID,
    "status": CELL18_STATUS,
    "target_yyyymm": PROBE_TARGET_YYYYMM,
    "batch_size": BATCH_SIZE,
    "candidate_pool_size": CANDIDATE_POOL_SIZE,
    "reference_pixel_id": REFERENCE_PIXEL_ID,
    "atol": ATOL,
    "rtol": RTOL,
    "cell13_context": str(
        CELL13_CONTEXT_PATH
    ),
    "master_inputs_read": False,
    "predictor_source_read": False,
    "national_inference_run": False,
    "hydrology_contract": {
        "feature_list": HYDROLOGY_FEATURES,
        "warmup_months": HYDROLOGY_WARMUP_MONTHS,
        "seq_len": HYDROLOGY_SEQ_LEN,
        "required_raw_months": TOTAL_REQUIRED_RAW_MONTHS,
    },
    "candidate_counts": {
        "raw_history_eligible": int(
            len(raw_eligible_df)
        ),
        "atmospheric_valid": int(
            len(branch_valid_keys["atmospheric"])
        ),
        "soil_valid": int(
            len(branch_valid_keys["soil"])
        ),
        "vegetation_valid": int(
            len(branch_valid_keys["vegetation"])
        ),
        "hydrology_valid": int(
            len(branch_valid_keys["hydrology"])
        ),
        "full_hybrid_eligible": int(
            len(full_eligible_df)
        ),
    },
    "batch_exception": batch_exception,
    "batch_traceback": batch_traceback,
    "single_exception": single_exception,
    "single_traceback": single_traceback,
    "parity_passed": bool(
        not parity_df.empty
        and parity_df["passed"].all()
    ),
    "outputs": {
        "candidate_audit_csv": str(
            CANDIDATE_AUDIT_PATH
        ),
        "selected_batch_csv": str(
            SELECTED_BATCH_PATH
        ),
        "batch_components_csv": str(
            BATCH_COMPONENT_PATH
        ),
        "single_components_csv": str(
            SINGLE_COMPONENT_PATH
        ),
        "batch_fusion_payloads_csv": str(
            BATCH_FUSION_PATH
        ),
        "single_fusion_payloads_csv": str(
            SINGLE_FUSION_PATH
        ),
        "fusion_flattened_fields_csv": str(
            FUSION_FIELDS_PATH
        ),
        "parity_csv": str(PARITY_PATH),
        "summary_json": str(
            SUMMARY_PATH
        ),
    },
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# 16. Report and parity gate
# -----------------------------------------------------------------------------
print("\n=== Cell 18 result ===")
print("Status:", CELL18_STATUS)

print("\n=== Exceptions ===")
print("Batch route       :", batch_exception)
print("One-pixel route   :", single_exception)

print("\n=== Parity results ===")
if parity_df.empty:
    print("No parity comparisons were produced.")
else:
    print(
        parity_df.to_string(
            index=False
        )
    )

print("\n=== Saved Cell 18 outputs ===")
print("Candidate audit      ->", CANDIDATE_AUDIT_PATH)
print("Selected batch       ->", SELECTED_BATCH_PATH)
print("Batch components     ->", BATCH_COMPONENT_PATH)
print("Single components    ->", SINGLE_COMPONENT_PATH)
print("Batch fusion payload ->", BATCH_FUSION_PATH)
print("Single fusion payload->", SINGLE_FUSION_PATH)
print("Fusion fields        ->", FUSION_FIELDS_PATH)
print("Parity results       ->", PARITY_PATH)
print("Summary              ->", SUMMARY_PATH)

del context_df
del pool_context_df
del selected_context_df
gc.collect()

if CELL18_STATUS != (
    "batch_parity_passed_ready_for_national_runner"
):
    raise AssertionError(
        "National legacy challenger execution is not justified.\n"
        f"Status: {CELL18_STATUS}\n"
        f"Batch exception: {batch_exception}\n"
        f"Single exception: {single_exception}"
    )

print(
    "\nCell 18 complete: batch and one-pixel outputs match "
    "within the locked tolerance."
)

=== Cell 18 purpose ===
Test batched legacy execution against repeated one-pixel legacy execution for one event month.

=== Configuration ===
Target month       : 201510
Batch size         : 32
Candidate pool     : 512
Reference pixel    : 0_101
Absolute tolerance : 1e-06
Relative tolerance : 1e-06
master_inputs read : False
predictor read     : False
national run       : False
Run ID             : 20260620T153611Z

=== Hydrology contract ===
Features           : ['tws', 'tws_lag1', 'tws_lag2', 'tws_lag3', 'tws_lag6', 'tws_rollmean3', 'tws_rollmean6', 'tws_rollmean12']
Warm-up months     : 11
Sequence length    : 24
Required raw months: 35
Context range      : 201212 -> 201510

=== Cell 13 context ===
Context : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell13_legacy_context_tws_backfilled_20260620T145557Z.parquet
Manifest: C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_

In [41]:
# validate_events_ng.ipynb — Cell 19 (SCHEMA PROBE)
# Purpose:
# - Inspect the actual legacy FusionService output saved by successful Cell 18.
# - Do not assume final fusion exposes p0..p3.
# - Identify scalar, categorical, nested mapping, and probability-like fields.
# - Perform no model inference and no national run.
#
# Inputs:
# - Newest successful Cell 18 batch fusion payload CSV only.

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from collections.abc import Mapping
from typing import Any
import json
import math
import re

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. Paths
# -----------------------------------------------------------------------------
if "OUTPUT_DIR" not in globals():
    raise RuntimeError("OUTPUT_DIR is missing. Rerun Cell 0 first.")

CELL19_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_PATH = (
    OUT_DIR
    / f"cell19_legacy_fusion_schema_probe_{CELL19_RUN_ID}.json"
)

LEAVES_PATH = (
    OUT_DIR
    / f"cell19_legacy_fusion_schema_leaves_{CELL19_RUN_ID}.csv"
)

GROUPS_PATH = (
    OUT_DIR
    / f"cell19_legacy_fusion_probability_candidates_{CELL19_RUN_ID}.csv"
)

print("=== Cell 19 schema probe ===")
print("Model inference      :", False)
print("National run         :", False)
print("master_inputs read   :", False)
print("predictor source read:", False)
print("Run ID               :", CELL19_RUN_ID)


# -----------------------------------------------------------------------------
# 2. Find latest successful Cell 18 fusion payload
# -----------------------------------------------------------------------------
summary_paths = sorted(
    OUT_DIR.glob(
        "cell18_batch_parity_summary_*.json"
    ),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not summary_paths:
    raise FileNotFoundError(
        "No Cell 18 parity summary was found."
    )

CELL18_SUMMARY_PATH = summary_paths[0]

cell18_summary = json.loads(
    CELL18_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    cell18_summary.get("status")
    != "batch_parity_passed_ready_for_national_runner"
):
    raise AssertionError(
        "Newest Cell 18 output did not pass parity.\n"
        f"Status: {cell18_summary.get('status')}"
    )

fusion_payload_path_text = (
    cell18_summary
    .get("outputs", {})
    .get("batch_fusion_payloads_csv")
)

if not fusion_payload_path_text:
    raise KeyError(
        "Cell 18 summary lacks batch_fusion_payloads_csv."
    )

CELL18_FUSION_PAYLOAD_PATH = Path(
    fusion_payload_path_text
)

if not CELL18_FUSION_PAYLOAD_PATH.exists():
    raise FileNotFoundError(
        "Cell 18 fusion payload CSV is missing:\n"
        f"{CELL18_FUSION_PAYLOAD_PATH}"
    )

fusion_payload_df = pd.read_csv(
    CELL18_FUSION_PAYLOAD_PATH
)

required_columns = {
    "pixel_key",
    "fusion_payload_json",
}

if not required_columns.issubset(
    fusion_payload_df.columns
):
    raise AssertionError(
        "Cell 18 fusion payload CSV lacks required columns.\n"
        f"Observed: {fusion_payload_df.columns.tolist()}"
    )

reference_row = (
    fusion_payload_df.loc[
        fusion_payload_df["pixel_key"]
        .astype(str)
        .eq("0_101")
    ]
    .copy()
)

if reference_row.empty:
    reference_row = fusion_payload_df.head(1).copy()

REFERENCE_PIXEL_ID = str(
    reference_row["pixel_key"].iloc[0]
)

FUSION_PAYLOAD = json.loads(
    reference_row[
        "fusion_payload_json"
    ].iloc[0]
)

print("\n=== Cell 18 source ===")
print("Summary       :", CELL18_SUMMARY_PATH)
print("Fusion payload:", CELL18_FUSION_PAYLOAD_PATH)
print("Reference pixel:", REFERENCE_PIXEL_ID)
print(
    "Payload root type:",
    type(FUSION_PAYLOAD).__name__,
)


# -----------------------------------------------------------------------------
# 3. Recursive schema inspection
# -----------------------------------------------------------------------------
def is_finite_number(
    value: Any,
) -> bool:
    if isinstance(value, bool):
        return False

    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def value_type_label(
    value: Any,
) -> str:
    if value is None:
        return "null"

    if isinstance(value, bool):
        return "bool"

    if isinstance(value, (int, np.integer)):
        return "int"

    if isinstance(value, (float, np.floating)):
        return "float"

    if isinstance(value, str):
        return "str"

    if isinstance(value, Mapping):
        return "mapping"

    if isinstance(value, list):
        return "list"

    return type(value).__name__


def flatten_leaves(
    value: Any,
    path: str = "root",
    records: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    if records is None:
        records = []

    if isinstance(value, Mapping):
        if not value:
            records.append(
                {
                    "path": path,
                    "value_type": "empty_mapping",
                    "value": "{}",
                }
            )

        for key, child in value.items():
            flatten_leaves(
                value=child,
                path=f"{path}.{key}",
                records=records,
            )

        return records

    if isinstance(value, list):
        if not value:
            records.append(
                {
                    "path": path,
                    "value_type": "empty_list",
                    "value": "[]",
                }
            )

        for index, child in enumerate(value):
            flatten_leaves(
                value=child,
                path=f"{path}[{index}]",
                records=records,
            )

        return records

    records.append(
        {
            "path": path,
            "value_type": value_type_label(value),
            "value": json.dumps(
                value,
                default=str,
            ),
        }
    )

    return records


def find_probability_candidates(
    value: Any,
    path: str = "root",
    records: list[dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    """
    Find several possible probability formats:
    - mappings with p0..p3;
    - mappings with class_0..class_3;
    - numeric lists of length 4 summing to 1;
    - mappings containing a nested key like probabilities/proba.
    """
    if records is None:
        records = []

    if isinstance(value, Mapping):
        keys = set(value.keys())

        candidate_key_sets = [
            ["p0", "p1", "p2", "p3"],
            ["class_0", "class_1", "class_2", "class_3"],
            ["prob_0", "prob_1", "prob_2", "prob_3"],
            ["probability_0", "probability_1", "probability_2", "probability_3"],
        ]

        for candidate_keys in candidate_key_sets:
            if set(candidate_keys).issubset(keys):
                values = [
                    value[key]
                    for key in candidate_keys
                ]

                if all(
                    is_finite_number(item)
                    for item in values
                ):
                    numeric_values = [
                        float(item)
                        for item in values
                    ]

                    records.append(
                        {
                            "path": path,
                            "candidate_type": "mapping_4class",
                            "keys": "|".join(candidate_keys),
                            "values": json.dumps(
                                numeric_values
                            ),
                            "sum": float(
                                sum(numeric_values)
                            ),
                            "sums_to_one": bool(
                                np.isclose(
                                    sum(numeric_values),
                                    1.0,
                                    atol=1e-6,
                                    rtol=1e-6,
                                )
                            ),
                        }
                    )

        for key, child in value.items():
            find_probability_candidates(
                value=child,
                path=f"{path}.{key}",
                records=records,
            )

        return records

    if isinstance(value, list):
        if (
            len(value) == 4
            and all(
                is_finite_number(item)
                for item in value
            )
        ):
            numeric_values = [
                float(item)
                for item in value
            ]

            records.append(
                {
                    "path": path,
                    "candidate_type": "numeric_list_4",
                    "keys": None,
                    "values": json.dumps(
                        numeric_values
                    ),
                    "sum": float(
                        sum(numeric_values)
                    ),
                    "sums_to_one": bool(
                        np.isclose(
                            sum(numeric_values),
                            1.0,
                            atol=1e-6,
                            rtol=1e-6,
                        )
                    ),
                }
            )

        for index, child in enumerate(value):
            find_probability_candidates(
                value=child,
                path=f"{path}[{index}]",
                records=records,
            )

    return records


leaves_df = pd.DataFrame(
    flatten_leaves(FUSION_PAYLOAD)
)

candidates_df = pd.DataFrame(
    find_probability_candidates(FUSION_PAYLOAD)
)

if candidates_df.empty:
    candidates_df = pd.DataFrame(
        columns=[
            "path",
            "candidate_type",
            "keys",
            "values",
            "sum",
            "sums_to_one",
        ]
    )

leaves_df.to_csv(
    LEAVES_PATH,
    index=False,
)

candidates_df.to_csv(
    GROUPS_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# 4. Print compact, exact evidence
# -----------------------------------------------------------------------------
payload_pretty = json.dumps(
    FUSION_PAYLOAD,
    indent=2,
    default=str,
)

MAX_PRINT_CHARS = 12000

print("\n=== Exact fusion payload ===")

if len(payload_pretty) <= MAX_PRINT_CHARS:
    print(payload_pretty)

else:
    print(
        payload_pretty[:MAX_PRINT_CHARS]
    )
    print(
        "\n[Payload print truncated; full payload is in the "
        "saved schema JSON.]"
    )

print("\n=== Scalar and categorical leaves ===")
print(
    leaves_df.to_string(
        index=False,
        max_colwidth=180,
    )
)

print("\n=== Probability-like candidates ===")
if candidates_df.empty:
    print(
        "None found. The fusion result may expose a class, "
        "severity score, confidence, or another non-probability structure."
    )
else:
    print(
        candidates_df.to_string(
            index=False,
            max_colwidth=180,
        )
    )


# -----------------------------------------------------------------------------
# 5. Save full schema artifact
# -----------------------------------------------------------------------------
schema_payload = {
    "run_id": CELL19_RUN_ID,
    "purpose": (
        "Exact schema inspection of the saved Cell 18 "
        "legacy FusionService output."
    ),
    "cell18_summary_path": str(
        CELL18_SUMMARY_PATH
    ),
    "cell18_fusion_payload_path": str(
        CELL18_FUSION_PAYLOAD_PATH
    ),
    "reference_pixel_id": REFERENCE_PIXEL_ID,
    "fusion_payload": FUSION_PAYLOAD,
    "leaf_count": int(len(leaves_df)),
    "probability_candidate_count": int(
        len(candidates_df)
    ),
    "outputs": {
        "schema_json": str(SCHEMA_PATH),
        "leaves_csv": str(LEAVES_PATH),
        "probability_candidates_csv": str(
            GROUPS_PATH
        ),
    },
}

SCHEMA_PATH.write_text(
    json.dumps(
        schema_payload,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("\n=== Saved schema-probe outputs ===")
print("Schema JSON            ->", SCHEMA_PATH)
print("Leaf inventory         ->", LEAVES_PATH)
print("Probability candidates ->", GROUPS_PATH)

print(
    "\nCell 19 schema probe complete. "
    "Do not run national inference until the actual fusion "
    "fields are selected from this output."
)

=== Cell 19 schema probe ===
Model inference      : False
National run         : False
master_inputs read   : False
predictor source read: False
Run ID               : 20260620T154648Z

=== Cell 18 source ===
Summary       : C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell18_batch_parity_summary_20260620T153611Z.json
Fusion payload: C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell18_batch_fusion_payloads_20260620T153611Z.csv
Reference pixel: 0_101
Payload root type: dict

=== Exact fusion payload ===
{
  "fusion_input": {
    "__type__": "DataFrame",
    "columns": [
      "atm_p0",
      "atm_p1",
      "atm_p2",
      "atm_p3",
      "soil_p0",
      "soil_p1",
      "soil_p2",
      "soil_p3",
      "veg_p0",
      "veg_p1",
      "veg_p2",
      "veg_p3",
      "hyd_p0",
      "hyd_p1",
      "hyd_p2",
      "hyd_p3"
    ],
    "rows": [
      {


In [12]:
# =============================================================================
# CELL 20 (FULLY VECTORIZED NATIONAL LEGACY CHALLENGER)
#
# Proven by debugging:
# - Atmospheric batch inference supported
# - Soil batch inference supported
# - Vegetation batch inference supported
# - Hydrology batch inference supported
# - predict_spi3 batch inference supported
#
# This version:
# - Eliminates per-pixel subsystem inference
# - Eliminates per-pixel fusion inference
# - Processes pixels in batches
# - Uses vectorized subsystem execution
# - Uses vectorized fusion execution
# - Writes monthly checkpoints
# - Memory safe
# - Restart friendly
# =============================================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# =============================================================================
# CONFIG
# =============================================================================

CELL20_RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

PROJECT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\RozviDrought"
)

OUTPUT_ROOT_CANDIDATES = [
    Path(
        r"C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng"
    ),
    PROJECT_ROOT
    / "data"
    / "backtests"
    / "validate_events_ng",
]

OUTPUT_DIR = None

for candidate in OUTPUT_ROOT_CANDIDATES:
    if candidate.exists():
        OUTPUT_DIR = candidate
        break

if OUTPUT_DIR is None:
    raise FileNotFoundError(
        "Could not locate validate_events_ng."
    )

OUT_DIR = (
    OUTPUT_DIR
    / "real_event_validation"
    / "legacy_package_challenger"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVENT_MONTHS = {
    "zimbabwe_2015_2016": [
        201510,
        201511,
        201512,
        201601,
        201602,
    ],
    "zimbabwe_2018_2019": [
        201810,
        201811,
        201812,
        201901,
        201902,
        201903,
        201904,
        201905,
    ],
    "zimbabwe_2023_2024": [
        202310,
        202311,
        202312,
        202401,
        202402,
        202403,
    ],
}

TARGET_MONTHS = sorted(
    {
        m
        for months in EVENT_MONTHS.values()
        for m in months
    }
)

BATCH_PIXELS = 512

CHECKPOINT_DIR = (
    OUT_DIR
    / f"cell20_vectorized_checkpoints_{CELL20_RUN_ID}"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PIXEL_OUTPUT_PATH = (
    OUT_DIR
    / f"cell20_vectorized_pixel_predictions_{CELL20_RUN_ID}.parquet"
)

MONTHLY_OUTPUT_PATH = (
    OUT_DIR
    / f"cell20_vectorized_monthly_summary_{CELL20_RUN_ID}.csv"
)

EVENT_OUTPUT_PATH = (
    OUT_DIR
    / f"cell20_vectorized_event_summary_{CELL20_RUN_ID}.csv"
)

MANIFEST_PATH = (
    OUT_DIR
    / f"cell20_vectorized_manifest_{CELL20_RUN_ID}.json"
)

# =============================================================================
# IMPORTS
# =============================================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from app.services.feature_service import FeatureService
import rozvidrought_subsystems as subs
from rozvidrought.pipeline.predict import predict_spi3

feature_service = FeatureService()

# =============================================================================
# CELL13
# =============================================================================

context_candidates = sorted(
    OUT_DIR.glob(
        "cell13_legacy_context_tws_backfilled_*.parquet"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not context_candidates:
    raise FileNotFoundError(
        "No Cell13 context found."
    )

CELL13_CONTEXT_PATH = context_candidates[0]

print("\n=== CELL 20 VECTORIZED ===")
print("Run ID:", CELL20_RUN_ID)
print("Context:", CELL13_CONTEXT_PATH)

# =============================================================================
# LOAD
# =============================================================================

context_df = (
    ds.dataset(
        CELL13_CONTEXT_PATH,
        format="parquet",
    )
    .to_table()
    .to_pandas()
)

context_df["yyyymm"] = (
    pd.to_numeric(
        context_df["yyyymm"]
    )
    .astype(np.int64)
)

context_df = (
    context_df
    .sort_values(
        [
            "pixel_id",
            "yyyymm",
        ]
    )
    .reset_index(drop=True)
)

pixel_ids = (
    context_df["pixel_id"]
    .drop_duplicates()
    .tolist()
)

TOTAL_PIXELS = len(
    pixel_ids
)

print(
    f"Rows: {len(context_df):,}"
)

print(
    f"Pixels: {TOTAL_PIXELS:,}"
)

# =============================================================================
# HELPERS
# =============================================================================

def chunked(
    values,
    size,
):
    for i in range(
        0,
        len(values),
        size,
    ):
        yield values[
            i:i+size
        ]

def prediction_result_to_arrays(
    result,
):
    probs = np.asarray(
        result.probs
    )

    pred = np.asarray(
        result.drought_class_spi3_pred
    )

    conf = np.asarray(
        result.confidence
    )

    return probs, pred, conf

monthly_summary_rows = []

# =============================================================================
# MONTH LOOP
# =============================================================================

for month_position, target_month in enumerate(
    TARGET_MONTHS,
    start=1,
):

    print(
        "\n" + "=" * 80
    )

    print(
        f"[{month_position}/{len(TARGET_MONTHS)}] "
        f"{target_month}"
    )

    month_start = time.time()

    month_records = []

    processed_pixels = 0

    for batch_no, pixel_batch in enumerate(
        chunked(
            pixel_ids,
            BATCH_PIXELS,
        ),
        start=1,
    ):

        atm_frames = []
        soil_frames = []
        veg_frames = []
        hyd_frames = []

        target_rows = []

        for pixel_id in pixel_batch:

            history = context_df.loc[
                (
                    context_df["pixel_id"]
                    == pixel_id
                )
                &
                (
                    context_df["yyyymm"]
                    <= target_month
                )
            ]

            if history.empty:
                continue

            target_row = history.loc[
                history["yyyymm"]
                == target_month
            ]

            if target_row.empty:
                continue

            try:

                prepared = (
                    feature_service
                    .prepare_subsystem_inputs(
                        timeseries=history,
                        run_yyyymm=target_month,
                    )
                )

                atm = prepared[
                    "atmospheric"
                ].copy()

                soil = prepared[
                    "soil"
                ].copy()

                veg = prepared[
                    "vegetation"
                ].copy()

                hyd = prepared[
                    "hydrology"
                ].copy()

                for df in (
                    atm,
                    soil,
                    veg,
                    hyd,
                ):
                    df["yyyymm"] = (
                        pd.to_numeric(
                            df["yyyymm"],
                            errors="coerce",
                        )
                    )

                atm = atm.loc[
                    atm["yyyymm"]
                    == target_month
                ]

                soil = soil.loc[
                    soil["yyyymm"]
                    == target_month
                ]

                veg = veg.loc[
                    veg["yyyymm"]
                    == target_month
                ]

                hyd["pixel_key"] = (
                    pixel_id
                )

                if (
                    atm.empty
                    or soil.empty
                    or veg.empty
                    or hyd.empty
                ):
                    continue

                atm["pixel_id"] = pixel_id
                soil["pixel_id"] = pixel_id
                veg["pixel_id"] = pixel_id

                atm_frames.append(
                    atm
                )

                soil_frames.append(
                    soil
                )

                veg_frames.append(
                    veg
                )

                hyd_frames.append(
                    hyd
                )

                target_rows.append(
                    target_row.iloc[0]
                )

            except Exception:
                continue

        if not target_rows:
            continue

        atm_batch = pd.concat(
            atm_frames,
            ignore_index=True,
        )

        soil_batch = pd.concat(
            soil_frames,
            ignore_index=True,
        )

        veg_batch = pd.concat(
            veg_frames,
            ignore_index=True,
        )

        hyd_batch = pd.concat(
            hyd_frames,
            ignore_index=True,
        )

        atm_probs = (
            subs.predict_atmospheric_proba(
                atm_batch
            )
            .reset_index(
                drop=True
            )
        )

        soil_probs = (
            subs.predict_soil_proba(
                soil_batch
            )
            .reset_index(
                drop=True
            )
        )

        veg_probs = (
            subs.predict_vegetation_proba(
                veg_batch
            )
            .reset_index(
                drop=True
            )
        )

        hyd_probs = (
            subs.predict_hydrology_proba_df(
                hyd_batch,
                pixel_col="pixel_key",
                time_col="yyyymm",
            )
            .reset_index(
                drop=True
            )
        )

        n = min(
            len(atm_probs),
            len(soil_probs),
            len(veg_probs),
            len(hyd_probs),
            len(target_rows),
        )

        fusion_rows = []

        for i in range(n):

            fusion_rows.append(
                {
                    "atm_p0": float(atm_probs.iloc[i]["p0"]),
                    "atm_p1": float(atm_probs.iloc[i]["p1"]),
                    "atm_p2": float(atm_probs.iloc[i]["p2"]),
                    "atm_p3": float(atm_probs.iloc[i]["p3"]),
                    "soil_p0": float(soil_probs.iloc[i]["p0"]),
                    "soil_p1": float(soil_probs.iloc[i]["p1"]),
                    "soil_p2": float(soil_probs.iloc[i]["p2"]),
                    "soil_p3": float(soil_probs.iloc[i]["p3"]),
                    "veg_p0": float(veg_probs.iloc[i]["p0"]),
                    "veg_p1": float(veg_probs.iloc[i]["p1"]),
                    "veg_p2": float(veg_probs.iloc[i]["p2"]),
                    "veg_p3": float(veg_probs.iloc[i]["p3"]),
                    "hyd_p0": float(hyd_probs.iloc[i]["p0"]),
                    "hyd_p1": float(hyd_probs.iloc[i]["p1"]),
                    "hyd_p2": float(hyd_probs.iloc[i]["p2"]),
                    "hyd_p3": float(hyd_probs.iloc[i]["p3"]),
                }
            )

        fusion_result = predict_spi3(
            rows=fusion_rows,
            model="hybrid",
        )

        probs, preds, confs = (
            prediction_result_to_arrays(
                fusion_result
            )
        )

        for i in range(
            len(preds)
        ):

            row = target_rows[i]

            month_records.append(
                {
                    "yyyymm": target_month,
                    "pixel_id": row["pixel_id"],
                    "row": row["row"],
                    "col": row["col"],
                    "lon": row["lon"],
                    "lat": row["lat"],
                    "predicted_class": int(
                        preds[i]
                    ),
                    "confidence": float(
                        confs[i]
                    ),
                    "p0": float(
                        probs[i][0]
                    ),
                    "p1": float(
                        probs[i][1]
                    ),
                    "p2": float(
                        probs[i][2]
                    ),
                    "p3": float(
                        probs[i][3]
                    ),
                }
            )

        processed_pixels += len(
            pixel_batch
        )

        elapsed = (
            time.time()
            - month_start
        )

        rate = (
            processed_pixels
            / max(
                elapsed,
                1e-9,
            )
        )

        eta = (
            TOTAL_PIXELS
            - processed_pixels
        ) / max(
            rate,
            1e-9,
        )

        print(
            f"{target_month} | "
            f"batch={batch_no} | "
            f"{processed_pixels:,}/{TOTAL_PIXELS:,} | "
            f"{rate:.1f} px/sec | "
            f"ETA={eta/60:.1f} min"
        )

        del atm_batch
        del soil_batch
        del veg_batch
        del hyd_batch
        gc.collect()

    month_df = pd.DataFrame(
        month_records
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{target_month}.parquet"
    )

    month_df.to_parquet(
        checkpoint_path,
        index=False,
    )

    monthly_summary_rows.append(
        {
            "yyyymm": target_month,
            "pixel_count": len(
                month_df
            ),
            "mean_class": month_df[
                "predicted_class"
            ].mean(),
            "mean_confidence": month_df[
                "confidence"
            ].mean(),
            "mean_p3": month_df[
                "p3"
            ].mean(),
        }
    )

    print(
        f"Month complete -> "
        f"{len(month_df):,} rows"
    )

    del month_df
    del month_records
    gc.collect()

# =============================================================================
# COMBINE
# =============================================================================

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "*.parquet"
    )
)

final_df = pd.concat(
    [
        pd.read_parquet(
            p
        )
        for p in checkpoint_files
    ],
    ignore_index=True,
)

final_df.to_parquet(
    PIXEL_OUTPUT_PATH,
    index=False,
)

monthly_df = pd.DataFrame(
    monthly_summary_rows
)

monthly_df.to_csv(
    MONTHLY_OUTPUT_PATH,
    index=False,
)

event_rows = []

for event_id, months in EVENT_MONTHS.items():

    subset = monthly_df.loc[
        monthly_df["yyyymm"]
        .isin(months)
    ]

    event_rows.append(
        {
            "event_id": event_id,
            "mean_class": subset[
                "mean_class"
            ].mean(),
            "mean_confidence": subset[
                "mean_confidence"
            ].mean(),
            "mean_p3": subset[
                "mean_p3"
            ].mean(),
        }
    )

event_df = pd.DataFrame(
    event_rows
)

event_df.to_csv(
    EVENT_OUTPUT_PATH,
    index=False,
)

manifest = {
    "run_id": CELL20_RUN_ID,
    "cell13_context": str(
        CELL13_CONTEXT_PATH
    ),
    "pixel_predictions": int(
        len(final_df)
    ),
    "outputs": {
        "pixel_predictions": str(
            PIXEL_OUTPUT_PATH
        ),
        "monthly_summary": str(
            MONTHLY_OUTPUT_PATH
        ),
        "event_summary": str(
            EVENT_OUTPUT_PATH
        ),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 80)
print("CELL20 COMPLETE")
print("=" * 80)
print(PIXEL_OUTPUT_PATH)
print(MONTHLY_OUTPUT_PATH)
print(EVENT_OUTPUT_PATH)
print(MANIFEST_PATH)


=== CELL 20 VECTORIZED ===
Run ID: 20260622T055931Z
Context: C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell13_legacy_context_tws_backfilled_20260620T145557Z.parquet
Rows: 3,159,450
Pixels: 26,775

[1/19] 201510
201510 | batch=1 | 512/26,775 | 9.0 px/sec | ETA=48.7 min
201510 | batch=2 | 1,024/26,775 | 9.2 px/sec | ETA=46.8 min
201510 | batch=3 | 1,536/26,775 | 8.3 px/sec | ETA=50.4 min
201510 | batch=4 | 2,048/26,775 | 8.1 px/sec | ETA=50.6 min
201510 | batch=5 | 2,560/26,775 | 7.8 px/sec | ETA=52.0 min
201510 | batch=6 | 3,072/26,775 | 7.7 px/sec | ETA=51.5 min
201510 | batch=7 | 3,584/26,775 | 7.6 px/sec | ETA=50.6 min
201510 | batch=8 | 4,096/26,775 | 7.9 px/sec | ETA=47.7 min
201510 | batch=9 | 4,608/26,775 | 8.0 px/sec | ETA=46.3 min
201510 | batch=10 | 5,120/26,775 | 7.9 px/sec | ETA=45.6 min
201510 | batch=11 | 5,632/26,775 | 8.1 px/sec | ETA=43.4 min
201510 | batch=12 | 6,144/26,775 | 7.9 px/sec | ETA=43.7 

In [2]:
# =============================================================================
# CELL 21
# Analyze national drought signal from pixel-level outputs.
#
# Produces:
# 1. Class distribution by month
# 2. P3 percentiles by month
# 3. Top 1%, 5%, 10% drought coverage
# 4. Event summaries
# 5. CSV outputs
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# =============================================================================
# PATHS
# =============================================================================

RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

BASE_DIR = Path(
    r"C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng"
    r"\real_event_validation\legacy_package_challenger"
)

pixel_files = sorted(
    BASE_DIR.glob(
        "cell20_vectorized_pixel_predictions_*.parquet"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not pixel_files:
    raise FileNotFoundError(
        "No Cell20 pixel prediction parquet found."
    )

PIXEL_PATH = pixel_files[0]

print("=" * 80)
print("CELL 21")
print("=" * 80)

print("Input:")
print(PIXEL_PATH)

# =============================================================================
# LOAD
# =============================================================================

df = pd.read_parquet(
    PIXEL_PATH
)

print("\nRows:", f"{len(df):,}")
print("Months:", df["yyyymm"].nunique())
print("Pixels:", df["pixel_id"].nunique())

# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

class_distribution_rows = []

for month, g in df.groupby("yyyymm"):

    total = len(g)

    record = {
        "yyyymm": month,
        "pixel_count": total,
    }

    for drought_class in [0, 1, 2, 3]:

        count = int(
            (g["predicted_class"] == drought_class)
            .sum()
        )

        record[f"class_{drought_class}_count"] = count

        record[f"class_{drought_class}_pct"] = (
            100.0 * count / total
        )

    class_distribution_rows.append(
        record
    )

class_distribution_df = pd.DataFrame(
    class_distribution_rows
).sort_values(
    "yyyymm"
)

# =============================================================================
# P3 PERCENTILES
# =============================================================================

percentile_rows = []

for month, g in df.groupby("yyyymm"):

    p3 = g["p3"].to_numpy()

    percentile_rows.append(
        {
            "yyyymm": month,
            "p3_mean": np.mean(p3),
            "p3_max": np.max(p3),
            "p3_p90": np.percentile(p3, 90),
            "p3_p95": np.percentile(p3, 95),
            "p3_p99": np.percentile(p3, 99),
            "p3_p995": np.percentile(p3, 99.5),
            "p3_p999": np.percentile(p3, 99.9),
        }
    )

percentiles_df = pd.DataFrame(
    percentile_rows
).sort_values(
    "yyyymm"
)

# =============================================================================
# TOP DROUGHT PIXELS
# =============================================================================

coverage_rows = []

for month, g in df.groupby("yyyymm"):

    total = len(g)

    p3_sorted = (
        g["p3"]
        .sort_values(
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    top_1_n = max(
        1,
        int(total * 0.01),
    )

    top_5_n = max(
        1,
        int(total * 0.05),
    )

    top_10_n = max(
        1,
        int(total * 0.10),
    )

    coverage_rows.append(
        {
            "yyyymm": month,

            "top1_mean_p3":
                p3_sorted.iloc[:top_1_n].mean(),

            "top5_mean_p3":
                p3_sorted.iloc[:top_5_n].mean(),

            "top10_mean_p3":
                p3_sorted.iloc[:top_10_n].mean(),

            "top1_min_p3":
                p3_sorted.iloc[top_1_n - 1],

            "top5_min_p3":
                p3_sorted.iloc[top_5_n - 1],

            "top10_min_p3":
                p3_sorted.iloc[top_10_n - 1],
        }
    )

coverage_df = pd.DataFrame(
    coverage_rows
).sort_values(
    "yyyymm"
)

# =============================================================================
# EVENT WINDOWS
# =============================================================================

EVENTS = {
    "zimbabwe_2015_2016": [
        201510,
        201511,
        201512,
        201601,
        201602,
    ],
    "zimbabwe_2018_2019": [
        201810,
        201811,
        201812,
        201901,
        201902,
        201903,
        201904,
        201905,
    ],
    "zimbabwe_2023_2024": [
        202310,
        202311,
        202312,
        202401,
        202402,
        202403,
    ],
}

event_rows = []

for event_id, months in EVENTS.items():

    subset = df.loc[
        df["yyyymm"].isin(months)
    ]

    event_rows.append(
        {
            "event_id": event_id,

            "rows":
                len(subset),

            "mean_class":
                subset["predicted_class"].mean(),

            "pct_class1plus":
                (
                    subset["predicted_class"] >= 1
                ).mean() * 100,

            "pct_class2plus":
                (
                    subset["predicted_class"] >= 2
                ).mean() * 100,

            "pct_class3":
                (
                    subset["predicted_class"] == 3
                ).mean() * 100,

            "mean_p3":
                subset["p3"].mean(),

            "p3_p95":
                subset["p3"].quantile(0.95),

            "p3_p99":
                subset["p3"].quantile(0.99),

            "max_p3":
                subset["p3"].max(),
        }
    )

event_df = pd.DataFrame(
    event_rows
)

# =============================================================================
# SAVE
# =============================================================================

class_path = (
    BASE_DIR
    / f"cell21_class_distribution_{RUN_ID}.csv"
)

percentile_path = (
    BASE_DIR
    / f"cell21_p3_percentiles_{RUN_ID}.csv"
)

coverage_path = (
    BASE_DIR
    / f"cell21_top_drought_pixels_{RUN_ID}.csv"
)

event_path = (
    BASE_DIR
    / f"cell21_event_analysis_{RUN_ID}.csv"
)

class_distribution_df.to_csv(
    class_path,
    index=False,
)

percentiles_df.to_csv(
    percentile_path,
    index=False,
)

coverage_df.to_csv(
    coverage_path,
    index=False,
)

event_df.to_csv(
    event_path,
    index=False,
)

# =============================================================================
# PRINT
# =============================================================================

print("\n" + "=" * 80)
print("EVENT SUMMARY")
print("=" * 80)

print(
    event_df.to_string(
        index=False
    )
)

print("\nSaved:")
print(class_path)
print(percentile_path)
print(coverage_path)
print(event_path)

print("\nCell 21 complete.")

CELL 21
Input:
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell20_vectorized_pixel_predictions_20260622T055931Z.parquet

Rows: 508,725
Months: 19
Pixels: 26775

EVENT SUMMARY
          event_id   rows  mean_class  pct_class1plus  pct_class2plus  pct_class3  mean_p3   p3_p95   p3_p99   max_p3
zimbabwe_2015_2016 133875    0.155757       12.729785        2.745098    0.100840 0.003147 0.000818 0.086994 0.786701
zimbabwe_2018_2019 214200    0.118058        8.118114        3.677404    0.010271 0.000937 0.000778 0.009798 0.836939
zimbabwe_2023_2024 160650    0.083517        8.248988        0.102085    0.000622 0.000280 0.000778 0.000791 0.488006

Saved:
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell21_class_distribution_20260623T100306Z.csv
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cel

In [3]:
# =============================================================================
# CELL 22
# FINAL AUDIT REPORT
#
# Consolidates:
# - Architecture investigations
# - Cell 18 parity validation
# - Cell 19 schema validation
# - Cell 20 national challenger run
# - Cell 21 event analysis
# - Excel severity comparison
#
# Outputs:
#   cell22_final_audit_report_<timestamp>.md
#   cell22_final_audit_summary_<timestamp>.json
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

RUN_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

BASE_DIR = Path(
    r"C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng"
    r"\real_event_validation\legacy_package_challenger"
)

REPORT_PATH = (
    BASE_DIR
    / f"cell22_final_audit_report_{RUN_ID}.md"
)

SUMMARY_PATH = (
    BASE_DIR
    / f"cell22_final_audit_summary_{RUN_ID}.json"
)

# =============================================================================
# AUDIT CONTENT
# =============================================================================

report_lines = []

report_lines.append(
"# RozviDrought Legacy Challenger Final Audit"
)

report_lines.append("")
report_lines.append(
f"Generated: {RUN_ID}"
)

report_lines.append("")
report_lines.append(
"---"
)

# =============================================================================
# EXECUTIVE SUMMARY
# =============================================================================

report_lines.append("")
report_lines.append("## Executive Summary")
report_lines.append("")

report_lines.append(
"Objective: determine whether the legacy challenger pipeline "
"should replace the validated fusion package."
)

report_lines.append("")

report_lines.append(
"Result: challenger does not outperform the validated fusion model."
)

report_lines.append("")

report_lines.append(
"Recommendation: retain the validated fusion package as the "
"operational candidate."
)

# =============================================================================
# ARCHITECTURE FINDINGS
# =============================================================================

report_lines.append("")
report_lines.append("## Architecture Investigation")
report_lines.append("")

report_lines.append(
"- Atmospheric subsystem supports vectorized inference."
)

report_lines.append(
"- Soil subsystem supports vectorized inference."
)

report_lines.append(
"- Vegetation subsystem supports vectorized inference."
)

report_lines.append(
"- Hydrology subsystem supports vectorized inference."
)

report_lines.append(
"- Fusion model supports vectorized inference."
)

report_lines.append(
"- SubsystemService tail(1) operation destroys hydrology vectorization."
)

report_lines.append(
"- FusionService run() performs single-row fusion only."
)

report_lines.append(
"- Original challenger implementation was heavily bottlenecked by per-pixel loops."
)

# =============================================================================
# CELL 18
# =============================================================================

report_lines.append("")
report_lines.append("## Cell 18 Batch Parity Validation")
report_lines.append("")

report_lines.append(
"- Batch parity passed."
)

report_lines.append(
"- No numerical differences detected between batch and single execution."
)

report_lines.append(
"- National inference approved."
)

report_lines.append(
"- Hybrid eligibility: 512 / 512."
)

# =============================================================================
# CELL 19
# =============================================================================

report_lines.append("")
report_lines.append("## Cell 19 Fusion Schema Validation")
report_lines.append("")

report_lines.append(
"- Final fusion probabilities located successfully."
)

report_lines.append(
"- Valid probability path:"
)

report_lines.append(
"  fusion_result.probs[0][0:3]"
)

report_lines.append(
"- Fusion output sums to 1.0."
)

report_lines.append(
"- Fusion extraction contract confirmed."
)

# =============================================================================
# CELL 20
# =============================================================================

report_lines.append("")
report_lines.append("## Cell 20 National Legacy Challenger")
report_lines.append("")

report_lines.append(
"- National inference completed successfully."
)

report_lines.append(
"- Pixels evaluated: 26,775."
)

report_lines.append(
"- Monthly outputs generated."
)

report_lines.append(
"- Event outputs generated."
)

report_lines.append(
"- No inference failures."
)

report_lines.append(
"- Runtime instability observed across months."
)

report_lines.append(
"- Significant throughput variation despite identical workloads."
)

# =============================================================================
# EVENT RESULTS
# =============================================================================

report_lines.append("")
report_lines.append("## Legacy Challenger Event Results")
report_lines.append("")

event_table = [
    (
        "2015-2016",
        0.156,
        12.73,
        2.75,
        0.10,
    ),
    (
        "2018-2019",
        0.118,
        8.12,
        3.68,
        0.01,
    ),
    (
        "2023-2024",
        0.084,
        8.25,
        0.10,
        0.00,
    ),
]

for (
    event_name,
    mean_class,
    class1plus,
    class2plus,
    class3,
) in event_table:

    report_lines.append(
        f"- {event_name}: "
        f"mean_class={mean_class:.3f}, "
        f"class1+={class1plus:.2f}%, "
        f"class2+={class2plus:.2f}%, "
        f"class3={class3:.2f}%"
    )

report_lines.append("")

report_lines.append(
"Finding: challenger identifies event onset but loses persistence."
)

report_lines.append(
"Finding: severe drought probabilities collapse after initial months."
)

report_lines.append(
"Finding: national signal remains weak."
)

# =============================================================================
# EXCEL COMPARISON
# =============================================================================

report_lines.append("")
report_lines.append("## Excel Severity Comparison")
report_lines.append("")

report_lines.append(
"| Event | Excel Severity | Fusion Mean Severity |"
)

report_lines.append(
"|------|------|------|"
)

report_lines.append(
"| 2015/16 | 4 Extreme | 1.704 |"
)

report_lines.append(
"| 2018/19 | 2 Moderate | 1.079 |"
)

report_lines.append(
"| 2023/24 | 5 Catastrophic | 1.100 |"
)

report_lines.append("")

report_lines.append(
"Fusion ranking:"
)

report_lines.append(
"2015/16 > 2023/24 > 2018/19"
)

report_lines.append("")

report_lines.append(
"Excel ranking:"
)

report_lines.append(
"2023/24 > 2015/16 > 2018/19"
)

report_lines.append("")

report_lines.append(
"Only one of three event rankings matches exactly."
)

report_lines.append(
"Top two events are reversed."
)

# =============================================================================
# INTERPRETATION
# =============================================================================

report_lines.append("")
report_lines.append("## Interpretation")
report_lines.append("")

report_lines.append(
"The fusion model detects all three major drought periods."
)

report_lines.append(
"The challenger detects drought onset but struggles with persistence."
)

report_lines.append(
"The fusion model better reproduces historical event behaviour."
)

report_lines.append(
"Severity calibration remains compressed."
)

report_lines.append(
"Physical drought intensity and humanitarian impact severity are not equivalent targets."
)

report_lines.append(
"Excel severity scores incorporate impacts not directly represented in the model feature space."
)

# =============================================================================
# FINAL DECISION
# =============================================================================

report_lines.append("")
report_lines.append("## Final Decision")
report_lines.append("")

report_lines.append(
"PASS: validated fusion package"
)

report_lines.append(
"PASS: batch parity"
)

report_lines.append(
"PASS: national-scale execution"
)

report_lines.append(
"PASS: event detection"
)

report_lines.append(
"WARN: severity ranking mismatch"
)

report_lines.append(
"WARN: compressed severity range"
)

report_lines.append(
"FAIL: legacy challenger does not outperform validated fusion package"
)

report_lines.append("")

report_lines.append(
"Operational Recommendation:"
)

report_lines.append(
"Retain the validated fusion package as the primary drought model."
)

report_lines.append(
"Archive the legacy challenger as an investigated alternative."
)

# =============================================================================
# WRITE REPORT
# =============================================================================

report_text = "\n".join(
    report_lines
)

REPORT_PATH.write_text(
    report_text,
    encoding="utf-8",
)

summary = {
    "run_id": RUN_ID,
    "decision": "retain_validated_fusion_package",
    "legacy_challenger_status": "not_selected",
    "event_detection": "passed",
    "batch_parity": "passed",
    "national_inference": "passed",
    "severity_calibration": "warning",
    "ranking_match": "partial",
    "recommended_model": "validated_fusion_package",
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

# =============================================================================
# DISPLAY
# =============================================================================

print("=" * 80)
print("FINAL AUDIT COMPLETE")
print("=" * 80)

print("\nReport:")
print(REPORT_PATH)

print("\nSummary:")
print(SUMMARY_PATH)

FINAL AUDIT COMPLETE

Report:
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell22_final_audit_report_20260623T102010Z.md

Summary:
C:\Projects\Infer RozviDrought\data\backtests\validate_events_ng\real_event_validation\legacy_package_challenger\cell22_final_audit_summary_20260623T102010Z.json
